# PaperDos_SI

In [ ]:
%matplotlib inline
import re
import sys
from pathlib import Path

import hyperspy.api as hs
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib import gridspec, patches, ticker
from matplotlib.colorbar import Colorbar
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, to_hex, to_rgba
from matplotlib.lines import Line2D
from matplotlib.transforms import Bbox
from mpl_toolkits.mplot3d import Axes3D
from scipy import ndimage


In [ ]:
# Ensure custom module Path is set before import
sys.path.append(r"E:\Desktop\PaperDos\Plottings")
from colors import tol_cmap, tol_cset  # type: ignore

# 画图的初始设置
plt.style.use(r"E:\Desktop\PaperDos\Plottings\liuchzzyy.mplstyle")
# print(plt.style.available)

# xarray setting
xr.set_options(
    cmap_sequential="viridis",
    cmap_divergent="viridis",
    display_width=150,
)  # viridis, gray

# 颜色设定
colors = tol_cset("vibrant")
if colors is not None:
    colors = list(colors)
    colors_opt = ["#b0a3d1", "#8bd0d5", "#a8e0ee", "#c5e1a3", "#ffe48b", "#f5a37d", "#e88db1"]
    colors_opt2 = list(tol_cset("bright"))
else:
    # Fallback colors in case tol_cset returns None
    colors = ["#0077BB", "#33BBEE", "#009988", "#EE7733", "#CC3311", "#EE3377", "#BBBBBB"]
if r"sunset" not in plt.colormaps():
    cmap = tol_cmap("sunset")
    if isinstance(cmap, LinearSegmentedColormap):
        plt.colormaps.register(cmap)
if r"rainbow_PuRd" not in plt.colormaps():
    cmap = tol_cmap("rainbow_PuRd")
    if isinstance(cmap, LinearSegmentedColormap):
        plt.colormaps.register(cmap)  # 备用 plasma

# Set math font
mpl.rcParams["mathtext.fontset"] = "custom"
mpl.rcParams["mathtext.rm"] = "Arial"
mpl.rcParams["mathtext.it"] = "Arial:italic"
mpl.rcParams["mathtext.bf"] = "Arial:bold"
mpl.rcParams["mathtext.sf"] = "Arial"
mpl.rcParams["mathtext.tt"] = "Arial"
mpl.rcParams["mathtext.cal"] = "Arial"
mpl.rcParams["mathtext.default"] = "regular"

# 数据/输出的文件夹
path_data_root = Path(r"D:\PaperDos.20260304\Data")
path_out = Path(r"E:\Desktop\Figures")
path_out.mkdir(parents=True, exist_ok=True)


## Figure MCR-ALS-PHASE3-ZN

In [ ]:
path_cell2 = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")
path_file = Path.joinpath(path_cell2, r"FC1st-FD2nd\Oct2022_cell2_P2b\XANES\20260515\画图需要的")

zn_mcr_st = pd.read_csv(Path.joinpath(path_file, r"spectraZn.csv"))
zn_mcr_st = zn_mcr_st.iloc[:, [0, 2, 4, 6]]

echem_c = pd.read_csv(Path.joinpath(path_file, r"cell2_P2b_echem_mn_zn_aligned.csv"))
echem_c = echem_c[[
    "zn_frac_P1b_phase1",
    "zn_frac_P1b_phase2",
    "zn_frac_ZHS_seed",
    "dt",
    "ewe_v",
]].copy()

echem_c = echem_c.iloc[9:, :].reset_index(drop=True)
echem_c["dt"] = pd.to_datetime(echem_c["dt"], format="mixed", errors="coerce")

zn_phase_cols = [
    r"zn_frac_P1b_phase1",
    r"zn_frac_P1b_phase2",
    r"zn_frac_ZHS_seed",
]
zn_phase_labels = [
    r"Phase 1",
    r"Phase 2",
    r"ZHS",
]

path_filelist_cell2 = list(Path.joinpath(path_cell2, r"Echem").glob(r"**\*.txt"))
if not path_filelist_cell2:
    raise FileNotFoundError(f"No electrochemistry files found in {Path.joinpath(path_cell2, 'Echem')}")

echem_all_cell2 = []
for path_file_echem in path_filelist_cell2:
    with open(path_file_echem, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break

    df = pd.read_csv(path_file_echem, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = pd.to_datetime(df["time/s"], format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype("Int64")
    echem_all_cell2.append(df)

echem_all_cell2 = pd.concat(echem_all_cell2, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)
echem_all_cell2 = echem_all_cell2.dropna(subset=["time/s", "Ewe/V", "<I>/mA"]).reset_index(drop=True)
echem_index_low = echem_all_cell2[echem_all_cell2["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high = echem_all_cell2[echem_all_cell2["cycle number"] == 1]["Ewe/V"].idxmax()
echem_all_cell2 = echem_all_cell2.loc[echem_index_low : echem_index_high].reset_index(drop=True)



echem_all_cell2['Time'] = (echem_all_cell2['time/s'] - echem_all_cell2['time/s'].iloc[0]).dt.total_seconds() / 3600
echem_c['Time'] = (echem_c['dt'] - echem_all_cell2['time/s'].iloc[0]).dt.total_seconds() / 3600
special_idx_cell2 = np.array([0, 4, 9, 12, 14, 17, 24, 30, 34, 38], dtype=int)
special_idx_cell2 = special_idx_cell2[(special_idx_cell2 >= 0) & (special_idx_cell2 < len(echem_c))]
special_echem_c = echem_c.iloc[special_idx_cell2].copy() 

In [ ]:
# 画图
plt.close("all")
fig = plt.figure(figsize=(7.0, 2.5), dpi=300)
gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.0], width_ratios=[1.0, 1.0], hspace=0.0, wspace=0.0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.3)

ax.plot(echem_all_cell2["Time"], echem_all_cell2["Ewe/V"], lw=1.0, color=colors[0], label=r"Voltage", zorder=0)
ax.scatter(echem_c['Time'], echem_c['ewe_v'], c="lightgrey", s=18, edgecolors="none", label=None, zorder=0)
ax.scatter(special_echem_c['Time'], special_echem_c['ewe_v'], marker="*", c=colors[0], edgecolors="k", linewidths=0.3, s=72, zorder=1)


ax.set_xlim(-0.1, 20)
ax.set_xlabel(r"Time (h)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=0))

ax.set_ylim(0.8, 2.0)
ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2+}}$)", fontsize=9, color=colors[0])
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=-0.1))
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter("{x:.1f}"))
ax.tick_params(axis="both", which="both", labelsize=8)

ax2 = ax.twinx()
line_i, = ax2.plot(echem_all_cell2["Time"], echem_all_cell2["<I>/mA"] * 1000.0, lw=1.0, ls="--", color=colors[1], label=r"Current", zorder=1)
ax2.set_ylim(-100, 100)
ax2.set_ylabel(r"Current (\mu A)", fontsize=9, color=colors[1])
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=50, offset=0.0))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=25, offset=0.0))
ax2.yaxis.set_major_formatter(ticker.StrMethodFormatter("{x:.0f}"))
ax2.tick_params(axis="y", which="both", labelsize=8, colors=colors[1])

ax.legend(
    loc="upper right",
    bbox_to_anchor=(0.25, 1.02),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)

ax2.legend(
    loc="upper right",
    bbox_to_anchor=(0.5, 1.02),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)

ax.text(-0.15, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
axb = subfig.add_subplot(sharex=ax)
axb.set_position((0.0, -0.25, 1.0, 1.0))
axb.set_box_aspect(0.3)

for idx, (column, label) in enumerate(zip(zn_phase_cols, zn_phase_labels), start=1):
    color = colors[idx]
    axb.plot(echem_c["Time"], echem_c[column], lw=1.0, marker="o", markersize=3.2, color=color, label=label, zorder=3)
    axb.scatter(special_echem_c["Time"], special_echem_c[column], marker="*", c=color, edgecolors="k", linewidths=0.25, s=54, zorder=5)

axb.set_xlim(-0.1, 20)
axb.set_ylim(-0.05, 1.2)
axb.set_xlabel("Time (h)", fontsize=9)
axb.set_ylabel("Fraction (arb. u.)", fontsize=9)
axb.xaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=0))
axb.xaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=0))
axb.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=0.0))
axb.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=0.0))
axb.tick_params(axis="both", which="both", labelsize=8)

axb.legend(
    loc="upper right",
    bbox_to_anchor=(1.0, 1.02),
    ncols=2,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)

axb.text(-0.15, 1.0, r"B", transform=axb.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[1, :], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.1, 0.0, 2.0, 2.0))
ax.set_box_aspect(0.8)

name_labels = [r" ", r"Phase 1", r"Phase 2", r"ZHS"]
for i in range(1, zn_mcr_st.shape[1]):
    ax.plot(zn_mcr_st.iloc[:, 0], zn_mcr_st.iloc[:, i], ls="-", lw=1.0, c=colors[i], label=name_labels[i])

ax.set_xlim(9640, 9710)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))

ax.set_ylim(-0.1, 2.5)
ax.set_ylabel("Relative intensity (arb. u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.tick_params(axis="both", labelsize=8, bottom=True, left=True, labelbottom=True, labelleft=True)
ax.legend(
    loc="upper right",
    bbox_to_anchor=(1.0, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)

ax.text(-0.15, 1.0, r"C", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_55_XAS_10_V2_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_55_XAS_10_V2_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_55_XAS_10_V2_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_55_XAS_10_V2_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## Figure Spectrum of STXM, Mn and Zn

In [ ]:
# 读取数据 Mn
file_path = list(Path(r"D:\PaperDos.20260304\Data\STXM\ExSitu\Andrea").glob(r"**/*Mn.txt"))
file_path = [file_path[i] for i in [8, 6, 1, 3, 2, 0, 4, 5]]
# file_path = [file_path[i] for i in [8, 6, 3, 2, 0, 4, 5]]
data_dict_mn = {}
for f in file_path:
    df = pd.read_csv(f, sep="\t", header=None, skiprows=0, index_col=None, comment="#")
    df.columns = ["Energy", "Absorption"]
    name = f.stem
    data_dict_mn[name] = df


In [ ]:
from scipy.stats import linregress


def remove_linear_background(spectrum, energy_threshold):
    x = spectrum["Energy"].values
    y = spectrum["Absorption"].values

    # 背景拟合区域
    mask = x < energy_threshold
    if np.sum(mask) < 2:
        raise ValueError("背景区间点数不足，无法拟合。")

    x_bg = x[mask]
    y_bg = y[mask]

    # 线性拟合
    slope, intercept, *_ = linregress(x_bg, y_bg)
    background = slope * x + intercept

    return y - background

In [ ]:
for _, df in data_dict_mn.items():
    data = remove_linear_background(df, energy_threshold=635.0)
    # df["Absorption"] = (data - data.min()) / (data.max() - data.min())

In [ ]:
# TXM 的数据路径, Zn
path_data = Path(r"D:\PaperDos.20260304\Data\STXM\ExSitu\Andrea")


def read_data(file_path):
    df = pd.read_csv(file_path, sep=r"\s+", header=None, comment="#")
    df.columns = ["Energy", "Absorption"]
    df["Absorption"] = (df["Absorption"] - df["Absorption"].min()) / (df["Absorption"].max() - df["Absorption"].min())
    return df


# 定义所有数据集的信息
datasets = [
    (r"ZSH", r"Pristine/E5 ZHS/ref_E5_ZHS_Zn.txt"),
    (r"1stDischarge", r"Charge/B6 1st0.9V/B6_1stDisch_Zn.txt"),
    (r"1stHalfCharge#2", r"Charge/E3 1st1.63V/E3_1stHCh_1p63V_Zn.txt"),
    (r"1stFullCharge", r"Charge/B2 1st1.80V/B2_1stCh_Zn.txt"),
    (r"2ndHalfDischarge", r"Charge/F4 2nd1.30V/F4_2ndDisch_1p3V_Zn.txt"),
    (r"2ndFullDischarge", r"Charge/G1 2nd0.9V/G1_2ndDisch_Zn.txt"),
]

# 加载和处理数据
data_dict_zn = {name: read_data(path_data.joinpath(path)) for name, path in datasets}

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1], height_ratios=None, wspace=0.0, hspace=0.0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.2)

names = [r'Ref. MnOOH', r'Pristine', r'FD#A', r"HC#B", r"HC#C", r"FC#D", r"HD#E", r"FD#F"]
# names = [r"Ref. MnOOH", r"Pristine", r"HC#B", r"HC#C", r"FC#D", r"HD#E", r"FD#F"]

for i, (_, data) in enumerate(data_dict_mn.items()):
    ax.plot(data["Energy"], data["Absorption"] + 0.5 * i, label=names[i], color=colors_opt2[i], lw=1.0)

line_paras = [(644.2, 0.25, 0.40, "grey"), (644.0, 0.45, 0.6, "grey"), (644.0, 0.6, 0.7, "grey"), (644.0, 0.7, 0.8, "grey"), (644.0, 0.8, 0.9, "grey"), (644.0, 0.9, 0.98, "grey")]
# line_paras = [(644.2, 0.23, 0.33, 'grey'), (644.2, 0.35, 0.45, 'grey'), (644.0, 0.5, 0.6, 'grey'), (644.0, 0.6, 0.7, 'grey'), (644.0, 0.7, 0.8, 'grey'), (644.0, 0.8, 0.9, 'grey'), (644.0, 0.9, 0.98, 'grey')]
for x, ymin, ymax, color in line_paras:
    ax.axvline(x=x, ymin=ymin, ymax=ymax, ls="--", color=color, lw=1.0)
ax.plot([644.2, 644.0], [1.5, 1.75], ls="dotted", color="grey", lw=1.0)

# 设置刻度线等格式
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.set_xlim(635, 665.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=5))

ax.set_ylabel(r"Absorption (arb.u.)", fontsize=11)
ax.set_yticks([])
ax.tick_params(axis="x", which="both", direction="out", labelsize=9)
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.04), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(-0.13, 1.0, r"A", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.1, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.2)

names = ["Ref. ZSH", r"FD#A", r"HC#C", r"FC#D", r"HD#E", r"FD#F"]
# names = ["Ref. ZSH", r"HC#B", r"FC#C", r"HD#D", r"FD#E"]

for i, (_, data) in enumerate(data_dict_zn.items()):
    if i == 0:
        ax.plot(data["Energy"], data["Absorption"] + 0.5 * i, label=names[i], color=colors_opt2[i], lw=1.0)
    if i > 0:
        ax.plot(data["Energy"], data["Absorption"] + 0.5 * i, label=names[i], color=colors_opt2[i + 2], lw=1.0)


# 设置刻度线等格式
ax.set_xlabel(r"Energy (eV)", fontsize=11)
ax.set_xlim(1010.0, 1050.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))

ax.set_ylabel(r"Absorption (arb.u.)", fontsize=11)
ax.set_yticks([])
ax.tick_params(axis="x", which="both", direction="out", labelsize=9)
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.04), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(-0.13, 1.0, r"B", transform=ax.transAxes, fontsize=13, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_02_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_02_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_02_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"Chapter_IV_Figure_STXM_02_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

## Figure MCR-ALS-PHASE3-FC-DINO

In [ ]:
path_cell2 = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")
path_file = Path.joinpath(path_cell2, r"FC1st-FD2nd\Oct2022_cell2_P2b\XANES\20260515\画图需要的")

mn_mcr_st = pd.read_csv(Path.joinpath(path_file, r"spectraMn.csv"))
mn_mcr_st = mn_mcr_st.iloc[:, [0, 5, 6, 7, 8]]

echem_c = pd.read_csv(Path.joinpath(path_file, r"cell2_P2b_echem_mn_zn_aligned.csv"))
echem_c = echem_c[[
    "mn_frac_0.2M_MnSO4",
    "mn_frac_alpha_MnO2",
    "mn_frac_FC1st_candidate_3",
    "mn_frac_Phase4_free",
    "dt",
    "ewe_v",
]].copy()

echem_c = echem_c.iloc[9:, :].reset_index(drop=True)
echem_c["dt"] = pd.to_datetime(echem_c["dt"], format="mixed", errors="coerce")

mn_phase_cols = [
    r"mn_frac_0.2M_MnSO4",
    r"mn_frac_alpha_MnO2",
    r"mn_frac_FC1st_candidate_3",
    r"mn_frac_Phase4_free",
]
mn_phase_labels = [
    r"$\mathrm{0.2 \ M \ Mn^{2+}}$",
    r"$\mathrm{\alpha-MnO_2}$",
    r"Phase 3",
    r"Phase 4",
]

path_filelist_cell2 = list(Path.joinpath(path_cell2, r"Echem").glob(r"**\*.txt"))
if not path_filelist_cell2:
    raise FileNotFoundError(f"No electrochemistry files found in {Path.joinpath(path_cell2, 'Echem')}")

echem_all_cell2 = []
for path_file_echem in path_filelist_cell2:
    with open(path_file_echem, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break

    df = pd.read_csv(path_file_echem, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = pd.to_datetime(df["time/s"], format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype("Int64")
    echem_all_cell2.append(df)

echem_all_cell2 = pd.concat(echem_all_cell2, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)
echem_all_cell2 = echem_all_cell2.dropna(subset=["time/s", "Ewe/V", "<I>/mA"]).reset_index(drop=True)
echem_index_low = echem_all_cell2[echem_all_cell2["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high = echem_all_cell2[echem_all_cell2["cycle number"] == 1]["Ewe/V"].idxmax()
echem_all_cell2 = echem_all_cell2.loc[echem_index_low : echem_index_high].reset_index(drop=True)



echem_all_cell2['Time'] = (echem_all_cell2['time/s'] - echem_all_cell2['time/s'].iloc[0]).dt.total_seconds() / 3600
echem_c['Time'] = (echem_c['dt'] - echem_all_cell2['time/s'].iloc[0]).dt.total_seconds() / 3600
special_idx_cell2 = np.array([0, 4, 9, 12, 14, 17, 24, 30, 34, 38], dtype=int)
special_idx_cell2 = special_idx_cell2[(special_idx_cell2 >= 0) & (special_idx_cell2 < len(echem_c))]
special_echem_c = echem_c.iloc[special_idx_cell2].copy() 

In [ ]:
# 画图
plt.close("all")
fig = plt.figure(figsize=(7.0, 2.5), dpi=300)
gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.0], width_ratios=[1.0, 1.0], hspace=0.0, wspace=0.0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.3)

ax.plot(echem_all_cell2["Time"], echem_all_cell2["Ewe/V"], lw=1.0, color=colors[0], label=r"Voltage", zorder=0)
ax.scatter(echem_c['Time'], echem_c['ewe_v'], c="lightgrey", s=18, edgecolors="none", label=None, zorder=0)
ax.scatter(special_echem_c['Time'], special_echem_c['ewe_v'], marker="*", c=colors[0], edgecolors="k", linewidths=0.3, s=72, zorder=1)


ax.set_xlim(-0.1, 20)
ax.set_xlabel(r"Time (h)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=0))

ax.set_ylim(0.8, 2.0)
ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2+}}$)", fontsize=9, color=colors[0])
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=-0.1))
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter("{x:.1f}"))
ax.tick_params(axis="both", which="both", labelsize=8)

ax2 = ax.twinx()
line_i, = ax2.plot(echem_all_cell2["Time"], echem_all_cell2["<I>/mA"] * 1000.0, lw=1.0, ls="--", color=colors[1], label=r"Current", zorder=1)
ax2.set_ylim(-100, 100)
ax2.set_ylabel(r"Current (\mu A)", fontsize=9, color=colors[1])
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=50, offset=0.0))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=25, offset=0.0))
ax2.yaxis.set_major_formatter(ticker.StrMethodFormatter("{x:.0f}"))
ax2.tick_params(axis="y", which="both", labelsize=8, colors=colors[1])

ax.legend(
    loc="upper right",
    bbox_to_anchor=(0.25, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)

ax2.legend(
    loc="upper right",
    bbox_to_anchor=(0.5, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)

ax.text(-0.15, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
axb = subfig.add_subplot(sharex=ax)
axb.set_position((0.0, -0.25, 1.0, 1.0))
axb.set_box_aspect(0.3)

for idx, (column, label) in enumerate(zip(mn_phase_cols, mn_phase_labels), start=1):
    color = colors[idx]
    axb.plot(echem_c["Time"], echem_c[column], lw=1.0, marker="o", markersize=3.2, color=color, label=label, zorder=3)
    axb.scatter(special_echem_c["Time"], special_echem_c[column], marker="*", c=color, edgecolors="k", linewidths=0.25, s=54, zorder=5)

axb.set_xlim(-0.1, 20)
axb.set_ylim(-0.05, 1.2)
axb.set_xlabel("Time (h)", fontsize=9)
axb.set_ylabel("Fraction (arb. u.)", fontsize=9)
axb.xaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=0))
axb.xaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=0))
axb.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=0.0))
axb.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=0.0))
axb.tick_params(axis="both", which="both", labelsize=8)

axb.legend(
    loc="upper right",
    bbox_to_anchor=(1.0, 1.02),
    ncols=2,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)

axb.text(-0.15, 1.0, r"B", transform=axb.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[1, :], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.1, 0.0, 2.0, 2.0))
ax.set_box_aspect(0.8)

name_labels = [r' ', r"$\mathrm{0.2 \ M \ Mn^{2+}}$", r"$\mathrm{\alpha-MnO_2}$", r"FC#D", r"Phase 4"]
for i in range(1, mn_mcr_st.shape[1]):
    ax.plot(mn_mcr_st.iloc[:, 0], mn_mcr_st.iloc[:, i], ls="-", lw=1.0, c=colors[i], label=name_labels[i])

ax.set_xlim(6530, 6600)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))

ax.set_ylim(-0.1, 2.5)
ax.set_ylabel("Relative intensity (arb. u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.tick_params(axis="both", labelsize=8, bottom=True, left=True, labelbottom=True, labelleft=True)
ax.legend(
    loc="upper right",
    bbox_to_anchor=(1.0, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)

ax.text(-0.15, 1.0, r"C", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_55_XAS_09_V2_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_55_XAS_09_V2_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_55_XAS_09_V2_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_55_XAS_09_V2_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## Figure MCR-ALS trial

In [ ]:
path_file = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\FC1st-FD2nd\Oct2022_cell2_P2b\XANES\20260515\P2b")

mn_mcr_st = pd.read_csv(Path.joinpath(path_file, r"Mn\mcrals_results_cell2_P2b_Mn_4phase_no_P2b_10\spectra.csv"))
mn_mcr_st = mn_mcr_st.iloc[:, [0, 5,6,7,8]]

mn_mcr_c = pd.read_csv(Path.joinpath(path_file, r"Mn\mcrals_results_cell2_P2b_Mn_4phase_no_P2b_10\concentrations.csv"))
mn_mcr_c = mn_mcr_c.iloc[:, [0, 6,7,8,9]]

zn_mcr_st = pd.read_csv(Path.joinpath(path_file, r"Zn\mcrals_results_cell2_P2b\spectra.csv"))
zn_mcr_st = zn_mcr_st.iloc[:, [0, 4,5,6]]
zn_mcr_c = pd.read_csv(Path.joinpath(path_file, r"Zn\mcrals_results_cell2_P2b\concentrations.csv"))
zn_mcr_c = zn_mcr_c.iloc[:, [0, 5,6,7]]

In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 5.0), dpi=300)
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0, hspace=0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

name_labels = [r' ', r"$\mathrm{0.2 \ M \ Mn^{2+}}$", r"$\mathrm{\alpha-MnO_2}$", r"Phase 3", r"Phase 4"]
for i in range(1, mn_mcr_st.shape[1]):
    ax.plot(mn_mcr_st.iloc[:, 0], mn_mcr_st.iloc[:, i], ls="-", lw=1.0, c=colors[i], label=name_labels[i])

ax.set_xlim(6530, 6600)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))

ax.set_ylim(-0.1, 6)
ax.set_ylabel("Relative intensity (arb. u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=1, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.tick_params(axis="both", labelsize=8, bottom=True, left=True, labelbottom=True, labelleft=True)
ax.legend(
    loc="upper right",
    bbox_to_anchor=(1.0, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)

ax.text(-0.12, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.1, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

name_labels = [r' ', r"$\mathrm{0.2 \ M \ Mn^{2+}}$", r"$\mathrm{\alpha-MnO_2}$", r"Phase 3", r"Phase 4"]
for i in range(1, mn_mcr_c.shape[1]):
    ax.plot(mn_mcr_c.iloc[:, 0], mn_mcr_c.iloc[:, i], ls="-", lw=1.0, c=colors[i], label=name_labels[i])

ax.set_xlim(-0.5, 40)
ax.set_xlabel(r"Charge and Discharge", fontsize=9)
ax.set_xticks([])

ax.set_ylim(-0.05, 1.0)
ax.set_ylabel("Fraction (arb. u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))

ax.tick_params(axis="both", labelsize=8, bottom=True, left=True, labelbottom=True, labelleft=True)
ax.legend(
    loc="upper right",
    bbox_to_anchor=(0.7, 1),
    ncols=2,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)

ax.text(-0.12, 1.0, r"B", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, -0.3, 1.0, 1.0))
ax.set_box_aspect(0.8)

name_labels = [r' ', r"$\mathrm{0.5 \ M \ Zn^{2+}}$", r"ZSH", r"Phase 3", r"Phase 4"]
for i in range(1, zn_mcr_st.shape[1]):
    ax.plot(zn_mcr_st.iloc[:, 0], zn_mcr_st.iloc[:, i], ls="-", lw=1.0, c=colors[i], label=name_labels[i])

ax.set_xlim(9640, 9710)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))

ax.set_ylim(-0.1, 3)
ax.set_ylabel("Relative intensity (arb. u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=1, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.tick_params(axis="both", labelsize=8, bottom=True, left=True, labelbottom=True, labelleft=True)
ax.legend(
    loc="upper right",
    bbox_to_anchor=(1.0, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)

ax.text(-0.12, 1.0, r"C", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.1, -0.3, 1.0, 1.0))
ax.set_box_aspect(0.8)

name_labels = [r' ', r"$\mathrm{0.5 \ M \ Zn^{2+}}$", r"ZSH", r"Phase 3", r"Phase 4"]
for i in range(1, zn_mcr_c.shape[1]):
    ax.plot(zn_mcr_c.iloc[:, 0], zn_mcr_c.iloc[:, i], ls="-", lw=1.0, c=colors[i], label=name_labels[i])

ax.set_xlim(-0.5, 40)
ax.set_xlabel(r"Charge and Discharge", fontsize=9)
ax.set_xticks([])

ax.set_ylim(-0.05, 1.0)
ax.set_ylabel("Fraction (arb. u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))

ax.tick_params(axis="both", labelsize=8, bottom=True, left=True, labelbottom=True, labelleft=True)
ax.legend(
    loc="upper right",
    bbox_to_anchor=(0.7, 1),
    ncols=2,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)

ax.text(-0.12, 1.0, r"D", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_55_XAS_08_V2_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_55_XAS_08_V2_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_55_XAS_08_V2_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_55_XAS_08_V2_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 01 -- XRD of pristine αMnO2 powder + EMD - V2

In [ ]:
# 读取数据
path_xrd = Path.joinpath(path_data_root, r"XRD\ExSitu\αMnO2\Pristine\Results\2024-ICMAB")
xrd = pd.read_csv(Path.joinpath(path_xrd, r"αMnO2_Pristine_2024.UXD"), sep=r"\s+", index_col=None, header=0, comment="#")
pdf = pd.read_csv(Path.joinpath(path_xrd, r"PDF-00-044-0141.csv"), sep=r",", index_col=None, header=0, comment="#")
path_img = Path.joinpath(path_xrd, r"PDF Card - 00-044-0141_withK.tif")

xrd["Spacing"] = 1.5406 / (2 * np.sin(np.radians(xrd["2THETA"] / 2)))


# 读取数据
path_xrd = Path.joinpath(path_data_root, r"XRD\ExSitu\EMD\Pristine\Results\EMD-1.60V-2mAh_cm-2-1MZnAC2+02M-pH5.0_1M+0.2M")
xrd_emd = pd.read_csv(Path.joinpath(path_xrd, r"#1EMD_CP.uxd"), sep=r"\s+", index_col=None, header=0, comment="#")
xrd_cp = pd.read_csv(Path.joinpath(path_xrd, r"#2CP.uxd"), sep=r"\s+", index_col=None, header=0, comment="#")
pdf2 = pd.read_csv(Path.joinpath(path_xrd, r"PDF Card - 00-030-0820.csv"), sep=r",", index_col=None, header=0, comment="#")
xrd_emd["Spacing"] = 1.5406 / (2 * np.sin(np.radians(xrd_emd["2THETA"] / 2)))
xrd_cp["Spacing"] = 1.5406 / (2 * np.sin(np.radians(xrd_cp["2THETA"] / 2)))


In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = fig.add_gridspec(1, 2, width_ratios=[1, 1], height_ratios=None, wspace=0, hspace=0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

line2d = ax.plot(xrd.iloc[:, 2], xrd.iloc[:, 1] + 50, ls="-", lw=1.0, c=colors[0], label=r"$\alpha$-MnO${_2}$")
stem = ax.stem(
    pdf.iloc[:, 1],
    pdf.iloc[:, 2] * 0.8,
    linefmt=colors[3],
    markerfmt="none",
    bottom=0,
    orientation="vertical",
    label=r"PDF #00-044-0141",
)

ax.set_xlim(1, 9)
ax.set_xlabel(r"$\mathrm{d \ Spacing \ (\AA)}$", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1, offset=-1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=-1))

ax.set_ylim(0, 700)
ax.set_ylabel("Relative intensity (arb. u.)", fontsize=9)
ax.set(yticks=[])
ax.tick_params(axis="both", labelsize=8, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.legend(
    handles=line2d,
    loc="upper right",
    bbox_to_anchor=(1.0, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)
ax.text(
    0.97,
    0.88,
    r"PDF#00-044-0141",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],  # type: ignore
    va="top",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    zorder=5,
)

axins = ax.inset_axes((0.28, 0.5, 0.45, 0.45))
AA = plt.imread(path_img, format="tif")
axins.imshow(AA, origin="upper", zorder=2, interpolation="bilinear")
axins.set_axis_off()
ax.text(-0.07, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.05, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

line2d_emd = ax.plot(xrd_emd.iloc[:, 2], xrd_emd.iloc[:, 1] + 50, ls="-", lw=1.0, c=colors[0], label=r"$\mathrm{EMD, \epsilon -MnO_2}$")
line2d_cp = ax.plot(xrd_cp.iloc[:, 2], xrd_cp.iloc[:, 1] * 0.1 + 150, ls="-", lw=1.0, c=colors[1], label=r"Carbon Paper")
stem = ax.stem(
    pdf2.iloc[:, 1],
    pdf2.iloc[:, 2] * 2.0,
    linefmt=colors[3],
    markerfmt="none",
    bottom=0,
    orientation="vertical",
    label=r"PDF #00-030-0820",
)

ax.set_xlim(1, 9)
ax.set_xlabel(r"$\mathrm{d \ Spacing \ (\AA)}$", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1, offset=-1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=-1))

ax.set_ylim(0, 1800)
ax.set_ylabel("Relative intensity (arb. u.)", fontsize=9)
ax.set(yticks=[])
ax.tick_params(axis="both", labelsize=8, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.legend(
    handles=line2d_emd + line2d_cp,
    loc="upper right",
    bbox_to_anchor=(1.0, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)

ax.text(
    0.96,
    0.82,
    r"PDF #00-030-0820",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],  # type: ignore
    va="top",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    zorder=5,
)
ax.text(-0.07, 1.0, r"B", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_01_XRD_00_V2_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_01_XRD_00_V2_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_01_XRD_00_V2_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_01_XRD_00_V2_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 02 -- Echem - Limited electrolyte governs multi-stage features

In [ ]:
# αMnO2 in Bulk Cells, 1M ZnSO4 + 0.2M MnSO4, no buffering
# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r'Echem\αMnO2\SpecialTest\BulkCell\NopHBufffer\1M ZnSO4 + 0.2M MnSO4\Results\02C-aMnO2-1M+02M-1').glob(r'*.txt'))

# 读取电化学数据
echemb = []
for path_file in path_filelist:
    line_skip = None
    with open(path_file, 'r', encoding='latin_1') as file:
        for line in file:
            if line.startswith('Nb header lines'):
                line_skip = int(line.split(':')[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep='\t', comment='#', skiprows=line_skip-1, encoding='latin_1', index_col=None, decimal=',').dropna(axis=1, how='all')
     # # 转换数据格式
    df[['Ewe/V', 'Ece/V', '<I>/mA','Capacity/mA.h']] = df[['Ewe/V', 'Ece/V', '<I>/mA','Capacity/mA.h']].apply(pd.to_numeric, errors='coerce')
    df['cycle number'] = df['cycle number'].astype(float).astype(np.int16)
    echemb.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemb)):
    echemb[i] = echemb[i][echemb[i].iloc[:, 0].isin([0,1,2,3])]
    echemb[i] = echemb[i].copy()
    echemb[i]['Voltage'] = echemb[i]['Ewe/V'] - echemb[i]['Ece/V']
    echemb[i] = echemb[i][echemb[i]["Voltage"] > 0.95]


# αMnO2 in Bulk Cells, 1M ZnSO4 + 0.2M MnSO4, no buffering => extended windows
# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r'Echem\αMnO2\SpecialTest\BulkCell\NopHBufffer\1M ZnSO4 + 0.2M MnSO4\Results\02C-aMnO2-1M+02M-2-extendedwindows').glob(r'*.txt'))

# 读取电化学数据
echemb2 = []
for path_file in path_filelist:
    line_skip = None
    with open(path_file, 'r', encoding='latin_1') as file:
        for line in file:
            if line.startswith('Nb header lines'):
                line_skip = int(line.split(':')[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep='\t', comment='#', skiprows=line_skip-1, encoding='latin_1', index_col=None, decimal=',').dropna(axis=1, how='all')
     # # 转换数据格式
    df[['Ewe/V', 'Ece/V', '<I>/mA','Capacity/mA.h']] = df[['Ewe/V', 'Ece/V', '<I>/mA','Capacity/mA.h']].apply(pd.to_numeric, errors='coerce')
    df['cycle number'] = df['cycle number'].astype(float).astype(np.int16)
    echemb2.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemb2)):
    echemb2[i] = echemb2[i][echemb2[i].iloc[:, 0].isin([0, 1])]
    echemb2[i] = echemb2[i].copy()
    echemb2[i].loc[:, 'Voltage'] = echemb2[i].loc[:, 'Ewe/V'] - echemb2[i].loc[:, 'Ece/V']
    df[['Ewe/V', 'Ece/V', '<I>/mA','Capacity/mA.h']] = df[['Ewe/V', 'Ece/V', '<I>/mA','Capacity/mA.h']].apply(pd.to_numeric, errors='coerce')
    df['cycle number'] = df['cycle number'].astype(float).astype(np.int16)
    echemb2.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemb2)):
    echemb2[i] = echemb2[i][echemb2[i].iloc[:, 0].isin([0,1,2])]
    echemb2[i] = echemb2[i].copy()
    echemb2[i]['Voltage'] = echemb2[i]['Ewe/V'] - echemb2[i]['Ece/V']


# αMnO2 in Bulk Cells, 1M ZnSO4 + 0.2M MnSO4 + Buffering
# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r'Echem\αMnO2\SpecialTest\BulkCell\pHBuffer').glob(r'Bulk-Buffer-Siavash*/**/*.txt'))
# 读取电化学数据
echemc = []
for path_file in path_filelist:
    line_skip = None
    with open(path_file, 'r', encoding='latin_1') as file:
        for line in file:
            if line.startswith('Nb header lines'):
                line_skip = int(line.split(':')[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep='\t', comment='#', skiprows=line_skip-1, encoding='latin_1', index_col=None, decimal=',').dropna(axis=1, how='all')
     # # 转换数据格式
    df[['Ewe/V', 'Ece/V', '<I>/mA','Capacity/mA.h']] = df[['Ewe/V', 'Ece/V', '<I>/mA','Capacity/mA.h']].apply(pd.to_numeric, errors='coerce')
    df['cycle number'] = df['cycle number'].astype(float).astype(np.int16)
    echemc.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemc)):
    echemc[i] = echemc[i][echemc[i].iloc[:, 0].isin([0, 1])]
    echemc[i] = echemc[i].copy()
    echemc[i].loc[:, 'Voltage'] = echemc[i].loc[:, 'Ewe/V'] - echemc[i].loc[:, 'Ece/V']


In [ ]:
# EMD in Bulk Cells, 1M ZnSO4 + 0.2M MnSO4, no buffering, 估计也是 C/5
# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r'Echem\EMD\GCD\1M ZnSO4 + 0.2M MnSO4\Results\BulkCell-2V-2mAh-1M+02M\01-02').glob(r'*.txt'))

# 读取电化学数据
echeme = []
for path_file in path_filelist:
    line_skip = None
    with open(path_file, 'r', encoding='latin_1') as file:
        for line in file:
            if line.startswith('Nb header lines'):
                line_skip = int(line.split(':')[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep='\t', comment='#', skiprows=line_skip-1, encoding='latin_1', index_col=None, decimal=',').dropna(axis=1, how='all')
     # # 转换数据格式
    df[['Ewe/V', 'Ece/V', '<I>/mA','Capacity/mA.h']] = df[['Ewe/V', 'Ece/V', '<I>/mA','Capacity/mA.h']].apply(pd.to_numeric, errors='coerce')
    df['cycle number'] = df['cycle number'].astype(float).astype(np.int16)
    echeme.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echeme)):
    echeme[i] = echeme[i][echeme[i].iloc[:, 0].isin([0,1,2])]
    echeme[i] = echeme[i].copy()
    echeme[i]['Voltage'] = echeme[i]['Ewe/V'] - echeme[i]['Ece/V']


# EMD in Bulk Cells, 1M ZnSO4 + 0.2M MnSO4 + Buffering
# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r'Echem\αMnO2\SpecialTest\BulkCell\pHBuffer\BulkCell_pHbuffer_pH4.2_0.5mA-2h-Zn-CP-0.5+0.2+pH4.2').glob(r'*.txt'))
# 读取电化学数据
echemf = []
for path_file in path_filelist:
    line_skip = None
    with open(path_file, 'r', encoding='latin_1') as file:
        for line in file:
            if line.startswith('Nb header lines'):
                line_skip = int(line.split(':')[1].strip())
                break  # 发现后立即退出循环，提高效率
    df = pd.read_csv(path_file, sep='\t', comment='#', skiprows=line_skip-1, encoding='latin_1', index_col=None, decimal=',').dropna(axis=1, how='all')
     # # 转换数据格式
    df[['Ewe/V', 'Ece/V', '<I>/mA','Capacity/mA.h']] = df[['Ewe/V', 'Ece/V', '<I>/mA','Capacity/mA.h']].apply(pd.to_numeric, errors='coerce')
    df['cycle number'] = df['cycle number'].astype(float).astype(np.int16)
    echemf.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemf)):
    echemf[i] = echemf[i][echemf[i].iloc[:, 0].isin([0, 1, 2])]
    echemf[i] = echemf[i].copy()
    echemf[i].loc[:, 'Voltage'] = echemf[i].loc[:, 'Ewe/V'] - echemf[i].loc[:, 'Ece/V']


In [ ]:
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 5.0), dpi=300)
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0.0, hspace=0.0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

labels = [[r'#1', None, None, None], [r"#2", None, None, None]]

for _, data in enumerate(echemb2):
    data_capacity_max = data['Capacity/mA.h'].max()
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)

        # 找到容量最大的时候的索引
        idx_max = temp['Capacity/mA.h'].idxmax()
        ax.plot(temp.loc[:idx_max, 'Capacity/mA.h']/data_capacity_max*100, temp.loc[:idx_max, 'Voltage'], ls='-', lw=1.0, c=colors[i+j], label=None, zorder=0)
        ax.plot(temp.loc[idx_max+1:, 'Capacity/mA.h']/data_capacity_max*100, temp.loc[idx_max+1:, 'Voltage'], ls='-', lw=1.0, c=colors[i+j], label=None, zorder=0)

for i, data in enumerate(echemb[0:1]):
    # data_capacity_max = data['Capacity/mA.h'].max()
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)

        # 找到电压的最大值的索引
        idx_max = temp['Voltage'].idxmax()
        if j == 0 :
            ax.plot(temp.loc[idx_max+2:, 'Capacity/mA.h']/data_capacity_max*100, temp.loc[idx_max+2:, 'Voltage'], ls='-', lw=1.0, c=colors[i+j], label=None, zorder=0)

# 设置刻度线等格式
ax.set_xlabel(r"Full Charge Percentage (%)", fontsize=9)
ax.set_xlim(0, 105)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1,  offset=0.1))

ax.tick_params(axis='both', direction='out', labelsize=8)
# ax.legend(loc='upper right', bbox_to_anchor=(1.0, 1.03), ncols=1, frameon=False, labelcolor='linecolor', fontsize=6)
ax.text(0.03, 0.15, r'$\mathrm{Bulk \ Cell}$' + '\n' + r'$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$', ha='left', va='top', transform=ax.transAxes, fontsize=8, c='k')
ax.text(-0.15, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.1, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

labels = [[r'#1', None, None, None], [r"#2", None, None, None]]
for i, data in enumerate(echemc[0:1]):
    data_capacity_max = data['Capacity/mA.h'].max()
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)

        # 找到容量最大的时候的索引
        idx_max = temp['Capacity/mA.h'].idxmax()
        ax.plot(temp.loc[:idx_max, 'Capacity/mA.h']/data_capacity_max*100, temp.loc[:idx_max, 'Voltage'], ls='-', lw=1.0, c=colors[i+j], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_max+1:, 'Capacity/mA.h']/data_capacity_max*100, temp.loc[idx_max+1:, 'Voltage'], ls='-', lw=1.0, c=colors[i+j], label=None, zorder=0)

# 设置刻度线等格式
ax.set_xlabel(r"Full Charge Percentage (%)", fontsize=9)
ax.set_xlim(0, 105)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1,  offset=0.1))

ax.tick_params(axis='both', direction='out', labelsize=8)
# ax.legend(loc='upper right', bbox_to_anchor=(1.0, 1.0), ncols=1, frameon=False, labelcolor='linecolor', fontsize=6)
ax.text(0.03, 0.15, r'$\mathrm{Bulk \ Cell, \ Buffered}$' + '\n' + r'$\mathrm{0.5 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$', ha='left', va='top', transform=ax.transAxes, fontsize=8, c='k')
ax.text(-0.15, 1.0, r"B", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, -0.3, 1.0, 1.0))
ax.set_box_aspect(0.8)

labels = [[r'#1', None, None, None], [r"#2", None, None, None]]

for _, data in enumerate(echeme[0:1]):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
        capacity_max = temp['Capacity/mA.h'].max()

        # 找到最小电压的索引
        idx_max = temp['Voltage'].idxmin()
        ax.plot(temp.loc[:idx_max, 'Capacity/mA.h']/capacity_max*100, temp.loc[:idx_max, 'Voltage'], ls='-', lw=1.0, c=colors[j], label=None, zorder=0)
        ax.plot(temp.loc[idx_max+1:, 'Capacity/mA.h']/capacity_max*100, temp.loc[idx_max+1:, 'Voltage'], ls='-', lw=1.0, c=colors[j], label=None, zorder=0)


# 设置刻度线等格式
ax.set_xlabel(r"Full Charge Percentage (%)", fontsize=9)
ax.set_xlim(0, 105)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1,  offset=-0.1))

ax.tick_params(axis='both', direction='out', labelsize=8)
# ax.legend(loc='upper right', bbox_to_anchor=(1.0, 0.7), ncols=1, frameon=False, labelcolor='linecolor', fontsize=6)
ax.text(0.03, 0.15, r'$\mathrm{Bulk \ Cell}$' + '\n' + r'$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$', ha='left', va='top', transform=ax.transAxes, fontsize=8, c='k')
ax.text(-0.15, 1.0, r"C", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.1, -0.3, 1.0, 1.0))
ax.set_box_aspect(0.8)

labels = [[r'#1', None, None, None], [r"#2", None, None, None]]

for i, data in enumerate(echemf):
    data_capacity_max = data['Capacity/mA.h'].max()
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)

        # 找到容量最大的时候的索引
        idx_max = temp['Capacity/mA.h'].idxmax()
        ax.plot(temp.loc[:idx_max, 'Capacity/mA.h']/data_capacity_max*100, temp.loc[:idx_max, 'Voltage'], ls='-', lw=1.0, c=colors[i+j], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_max+1:, 'Capacity/mA.h']/data_capacity_max*100, temp.loc[idx_max+1:, 'Voltage'], ls='-', lw=1.0, c=colors[i+j], label=None, zorder=0)

# 设置刻度线等格式
ax.set_xlabel(r"Full Charge Percentage (%)", fontsize=9)
ax.set_xlim(0, 105)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1,  offset=0.1))

ax.tick_params(axis='both', direction='out', labelsize=8)
# ax.legend(loc='upper right', bbox_to_anchor=(1.0, 0.9), ncols=1, frameon=False, labelcolor='linecolor', fontsize=6)
ax.text(0.03, 0.15, r'$\mathrm{Bulk \ Cell, \ Buffered}$' + '\n' + r'$\mathrm{0.5 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$', ha='left', va='top', transform=ax.transAxes, fontsize=8, c='k')
ax.text(-0.15, 1.0, r"D", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_02_Echem_00_V2_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_02_Echem_00_V2_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_02_Echem_00_V2_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_02_Echem_00_V2_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 05 -- Echem - αMnO2 in 1 M ZnSO4 和 Mn2+ 不同溶度下的沉积

In [ ]:
# α-MnO2 in 1 M ZnSO4

# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r"Echem\αMnO2\GCD\1M ZnSO4").glob(r"LC*.xlsx"))
echema = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echema.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echema)):
    echema[i] = echema[i][echema[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echema[i] = echema[i][echema[i].iloc[:, 2] >= 0]


In [ ]:
# Carbon Paper in Swagelok Cells, 1M ZnSO4 + 0.2M MnSO4

# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r"Echem\Substrates\CarbonPaper_H23\GCD\DifferentConcentration\Results").glob(r"**\*009*.xlsx"))

# 选取需要的数据
path_filelist = [path_filelist[i] for i in [0, 1, 3, 4]]

# 读取电化学数据
echemb = []
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[["Voltage/V", "Current/uA", "Capacity/uAh"]] = df[["Voltage/V", "Current/uA", "Capacity/uAh"]].apply(pd.to_numeric, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemb.append(df)

# 选取每个数据的第一到第三圈
for i in range(len(echemb)):
    echemb[i] = echemb[i][echemb[i]["Cycle"].isin([1, 2, 3])]


In [ ]:
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 2.5), dpi=300)
gs = fig.add_gridspec(1, 2, width_ratios=[1, 1], height_ratios=None, wspace=0, hspace=0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{Cell\#1-1.96mg/cm^2}$", None, None, None], [r"$\mathrm{Cell\#2-1.96mg/cm^2}$", None, None, None]]
for i, data in enumerate(echema):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)

        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"]*0.984, temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"]*0.984, temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)


# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (\mu Ah \ cm^{-2})}$", fontsize=9)
ax.set_xlim(-2, 350)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=70, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=35, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9, labelpad=1.0)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.1))

ax.tick_params(axis="both", direction="out", labelsize=8)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.8), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.text(0.03, 0.15, r"$\mathrm{C/10}$" + "\n" + r"$\mathrm{1 \ M \ ZnSO_4}$", ha="left", va="top", transform=ax.transAxes, fontsize=8, c="k")
ax.text(-0.15, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.1, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

labels = [
    [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", None, None],
    [r"$\mathrm{0 \ M \ ZnSO_4 + 1 \ M \ MnSO_4}$", None, None],
    [r"$\mathrm{1 \ M \ ZnSO_4 + 0 \ M \ MnSO_4}$", None, None],
    [r"$\mathrm{1 \ M \ ZnSO_4 + 1 \ M \ MnSO_4}$", None, None],
]
for i, data in enumerate(echemb):
    if i == 0:
        for j, idx in enumerate(data.iloc[:, 0].unique()):
            temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
            # 找到电压最小值的索引
            idx_min = temp["Voltage/V"].idxmin()
            ax.plot(temp.loc[: idx_min - 1, "Capacity/uAh"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
            ax.plot(temp.loc[idx_min + 1 :, "Capacity/uAh"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
    if i != 0:
        for j, idx in enumerate(data.iloc[:, 0].unique()):
            temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
            # 找到电压最大值的索引
            idx_max = temp["Voltage/V"].idxmax()
            ax.plot(temp.loc[: idx_max - 1, "Capacity/uAh"] / 0.503, temp.loc[: idx_max - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
            ax.plot(temp.loc[idx_max + 1 :, "Capacity/uAh"] / 0.503, temp.loc[idx_max + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (\mu Ah \ cm^{-2})}$", fontsize=9)
ax.set_xlim(-0.2, 20)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2.5, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax.set_ylim(0.8, 2.0)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=0.2))

ax.tick_params(axis="both", direction="out", labelsize=8)
ax.legend(loc="upper right", bbox_to_anchor=(1.02, 0.8), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.text(0.03, 0.08, r"C/10", ha="left", va="top", transform=ax.transAxes, fontsize=8, c="k")
ax.text(-0.15, 1.0, r"B", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_05_Echem_04_V2_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_05_Echem_04_V2_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_05_Echem_04_V2_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_05_Echem_04_V2_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 06 -- Echem - Organic Echem - 1 Zn(Otf)2 in DMSO

In [ ]:
# 1st1.80V - ZHS - 1M ZnSO4 + 0.2M MnSO4

# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r"Echem\αMnO2\SpecialTest\OrganicEchem\1st1.80V\ZHS\1M ZnSO4 + 0.2M MnSO4\Results\15#-03-2023").glob(r"LC*.xlsx"))
echemc = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemc.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemc)):
    echemc[i] = echemc[i][echemc[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echemc[i] = echemc[i][echemc[i].iloc[:, 2] >= 0]

# 调整电化学数据的顺序，满足，首先制备对应的样品，然后再有机电解液，然后再含水的有机电解液中的反应
indices = [0, 1, 2]  # 指定新顺序
echemc = [echemc[i] for i in indices]


# 1st1.80V - ZHS - 1M ZnSO4

# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r"Echem\αMnO2\SpecialTest\OrganicEchem\1st1.80V\ZHS\1M ZnSO4\Results\15#-03-2023").glob(r"LC*.xlsx"))
echemd = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemd.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemd)):
    echemd[i] = echemd[i][echemd[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echemd[i] = echemd[i][echemd[i].iloc[:, 2] >= 0]

# 调整电化学数据的顺序，满足，首先制备对应的样品，然后再有机电解液，然后再含水的有机电解液中的反应
indices = [0, 1, 2]  # 指定新顺序
echemd = [echemd[i] for i in indices]

# 1st1.80V - No ZHS - 1M ZnSO4

# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r"Echem\αMnO2\SpecialTest\OrganicEchem\1st1.80V\No ZHS\1M ZnSO4 + 0.2M MnSO4\Results\15#-03-2023").glob(r"LC*.xlsx"))
echeme = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echeme.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echeme)):
    echeme[i] = echeme[i][echeme[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echeme[i] = echeme[i][echeme[i].iloc[:, 2] >= 0]

# 调整电化学数据的顺序，满足，首先制备对应的样品，然后再有机电解液，然后再含水的有机电解液中的反应
indices = [0, 1, 2]  # 指定新顺序
echeme = [echeme[i] for i in indices]

# 1st1.80V - No ZHS - 1M ZnSO4

# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r"Echem\αMnO2\SpecialTest\OrganicEchem\1st1.80V\No ZHS\1M ZnSO4\Results\15#-03-2023").glob(r"LC*.xlsx"))
echemf = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemf.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemf)):
    echemf[i] = echemf[i][echemf[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echemf[i] = echemf[i][echemf[i].iloc[:, 2] >= 0]

# 调整电化学数据的顺序，满足，首先制备对应的样品，然后再有机电解液，然后再含水的有机电解液中的反应
indices = [0, 1, 2]  # 指定新顺序
echemf = [echemf[i] for i in indices]


In [ ]:
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 5.0), dpi=300)
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0, hspace=0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_axes((0, 0.0, 1, 1))
ax.set_box_aspect(0.8)

labels = [
    [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", None, None, None],
    [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO}$", None, None, None],
    [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO + H_2O}$", None, None, None],
]
for i, data in enumerate(echemc):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(433, 1.80, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=9)
ax.set_xlim(-1, 450)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=90, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=45, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=8)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.65), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.text(0.6, 0.07, r"No Acid-treated, C/10", ha="left", va="top", transform=ax.transAxes, fontsize=8, c="k")
ax.text(-0.18, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.1, 0.0, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{1 \ M \ ZnSO_4}$", None, None, None], [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO}$", None, None, None], [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO + H_2O}$", None, None, None]]
for i, data in enumerate(echemd):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(155, 1.80, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=9)
ax.set_xlim(-0.1, 300)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=60, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=30, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=8)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.65), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.text(0.6, 0.07, r"No Acid-treated, C/10", ha="left", va="top", transform=ax.transAxes, fontsize=8, c="k")
ax.text(-0.18, 1.0, r"B", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[1, 0])
ax = subfig.add_axes((0.0, -0.3, 1, 1))
ax.set_box_aspect(0.8)

labels = [
    [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2M \ MnSO_4}$", None, None, None],
    [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO}$", None, None, None],
    [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO + H_2O}$", None, None, None],
]
for i, data in enumerate(echeme):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(330, 1.80, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=9)
ax.set_xlim(-0.1, 350)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=70, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=35, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=8)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.65), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.text(0.65, 0.07, r"Acid-treated, C/10", ha="left", va="top", transform=ax.transAxes, fontsize=8, c="k")
ax.text(-0.18, 1.0, r"C", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
subfig = fig.add_subfigure(gs[1, 1])
ax = subfig.add_axes((0.1, -0.3, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{1 \ M \ ZnSO_4}$", None, None, None], [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO}$", None, None, None], [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO + H_2O}$", None, None, None]]
for i, data in enumerate(echemf):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(263, 1.80, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=9)
ax.set_xlim(-0.1, 450)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=90, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=45, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=8)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.67), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.text(0.65, 0.07, r"Acid-treated, C/10", ha="left", va="top", transform=ax.transAxes, fontsize=8, c="k")
ax.text(-0.18, 1.0, r"D", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_06_Echem_08_V2_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_06_Echem_08_V2_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_06_Echem_08_V2_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_06_Echem_08_V2_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 07 -- XRD - Operando XRD, αMnO2 -V3

In [ ]:
path_xrd = Path.joinpath(path_data_root, r"XRD\Operando\αMnO2\2022-ICMAB\Results")

# 读取 PDF card 的数据
pdfmo = pd.read_csv(
    Path.joinpath(path_xrd, r"PDFCards", "PDF_1stDischarge-d-spacing.csv"),
    sep=",",
    index_col=None,
    header=0,
    skiprows=1,
    comment="#",
)

# 读取 spectrum 的数据
spectrum_all = pd.read_csv(
    Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", "Spectum_all_d spacing.csv"),
    sep=r",",
    index_col=None,
    header=0,
    comment="#",
)
spectrum_all.set_index(keys=["d_spacing"], inplace=True)
spectrum_all.index = pd.to_numeric(spectrum_all.index, errors="coerce")
spectrum_all.columns = pd.to_numeric(spectrum_all.columns, errors="coerce")

# 电化学上的时间
with open(Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", r"EchemOperando1_c10_1544mg_C02.txt"), "r") as file:
    lines = file.readlines()
for line in lines:
    if line.startswith("Nb header lines"):
        line_skip = int(line.split(":")[1].strip())

echem_all = pd.read_csv(
    Path.joinpath(path_xrd, r"Coin1", r"1stCharge+2ndDischarge", r"EchemOperando1_c10_1544mg_C02.txt"),
    sep="	",
    index_col=None,
    header=0,
    comment="#",
    skiprows=line_skip - 1,
    encoding="latin_1",
    date_format="%m/%d/%y %H:%M:%S.%f",
    parse_dates=[1, 2],
).dropna(axis=1, how="all", inplace=False)
echem_all["time/s"] = pd.to_datetime(echem_all["time/s"], errors="coerce")
echem_all["Ewe/V"] = pd.to_numeric(echem_all["Ewe/V"], errors="coerce")
echem_all["<I>/mA"] = pd.to_numeric(echem_all["<I>/mA"], errors="coerce")
echem_all = echem_all.dropna(subset=["time/s", "Ewe/V", "<I>/mA"]).sort_values("time/s").reset_index(drop=True)

# 谱线上的时间（使用你更新的 xlsx）
path_spec_time = Path(r"D:\PaperDos.20260304\Data\XRD\Operando\αMnO2\2022-ICMAB\Results\Coin1\Overivew\Time_index_spectrum.xlsx")
spectrum_time_all = pd.read_excel(path_spec_time)
spectrum_time_all["scan_idx"] = pd.to_numeric(spectrum_time_all["scan_idx"], errors="coerce").astype("Int64")
spectrum_time_all["spectrum_col"] = pd.to_numeric(spectrum_time_all["spectrum_col"], errors="coerce").astype("Int64")
spectrum_time_all["Time"] = pd.to_datetime(spectrum_time_all["Time"], errors="coerce")
spectrum_time_all = spectrum_time_all.dropna(subset=["Time", "scan_idx", "spectrum_col"]).sort_values("Time").reset_index(drop=True)

if spectrum_time_all.empty:
    raise ValueError("FigureSI 35: Time_index_spectrum.xlsx has no valid rows")

# 匹配谱线和电化学上的时间（最近邻）
map_df = pd.merge_asof(
    spectrum_time_all[["scan_idx", "spectrum_col", "Time"]].sort_values("Time"),
    echem_all.reset_index(names="echem_idx").sort_values("time/s")[["echem_idx", "time/s", "Ewe/V", "<I>/mA"]],
    left_on="Time",
    right_on="time/s",
    direction="nearest",
)
map_df = map_df.dropna(subset=["echem_idx"]).copy()
map_df["echem_idx"] = map_df["echem_idx"].astype(int)
map_df["dt_s"] = (map_df["time/s"] - map_df["Time"]).dt.total_seconds().abs()

# 使用最新字段：scan_idx / echem_idx
spectrum_time = map_df[["scan_idx", "echem_idx", "Ewe/V", "<I>/mA", "spectrum_col", "Time", "dt_s"]].copy()


In [ ]:
# 选择需要的电化学数据以及对应的谱线
index_labels = [17, 26, 31, 37, 40, 45, 50, 54]
selected_spectrum_time = (
    spectrum_time[(spectrum_time["scan_idx"] <= index_labels[-1] + 1) & (spectrum_time["scan_idx"] >= index_labels[0])]
    .sort_values(by="scan_idx", ascending=True)
    .reset_index(drop=True)
)

if selected_spectrum_time.empty:
    raise ValueError("FigureSI 35: selected_spectrum_time is empty after time mapping")

# 统一时间零点：使用首条映射谱线时间，确保谱线映射与电化学时间同一基准
time_zero = selected_spectrum_time["Time"].iloc[0]

selected_echem = echem_all[(echem_all.index <= selected_spectrum_time["echem_idx"].iloc[-1]) & (echem_all.index >= selected_spectrum_time["echem_idx"].iloc[0])]
selected_echem = selected_echem.copy()
selected_echem["FD1st_time"] = (selected_echem["time/s"] - time_zero).dt.total_seconds() / 3600

# 同步给谱线时间（便于后续检查/标注）
selected_spectrum_time = selected_spectrum_time.copy()
selected_spectrum_time["spec_time_h"] = (selected_spectrum_time["Time"] - time_zero).dt.total_seconds() / 3600

# 修复：使用正确的列选择方式
# spectrum_all 的列是数字1-88，对应不同的谱线编号
column_range_start = selected_spectrum_time["scan_idx"].iloc[0] + 1
column_range_end = selected_spectrum_time["scan_idx"].iloc[-1] + 1

# 选择对应范围的列
selected_columns = [col for col in spectrum_all.columns if col >= column_range_start and col <= column_range_end]
selected_spectrum = spectrum_all[selected_columns]


In [ ]:
plt.close("all")

# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 4, 4], height_ratios=None, wspace=0, hspace=0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Ewe/V"], selected_echem["FD1st_time"], ls="-", lw=1.0, c=colors[0], label=r"opCoin_1", zorder=5)

for i, idx in enumerate(selected_spectrum_time["scan_idx"]):
    if idx in index_labels:
        index_value = selected_spectrum_time.loc[selected_spectrum_time["scan_idx"] == idx, "echem_idx"].iloc[0]
        ax.scatter(
            selected_echem.loc[index_value, "Ewe/V"],
            selected_echem.loc[index_value, "FD1st_time"],
            c=colors[0],
            edgecolors="face",
            marker="*",
            s=50,
        )
    else:
        index_value = selected_spectrum_time.loc[i, "echem_idx"]
        ax.scatter(
            selected_echem.loc[index_value, "Ewe/V"],
            selected_echem.loc[index_value, "FD1st_time"],
            c="lightgrey",
            edgecolors="face",
            s=15,
        )

ax.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax.set_xlim(0.85, 1.85)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=-0.15))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=-0.15))

ax.set_ylabel(r"Duration time (h)", fontsize=9)
ax.set_ylim(-0.5, 30.5)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))

# ax.legend(loc='upper right', bbox_to_anchor=(0.9, 1.0), ncols=1, frameon=False, labelcolor='linecolor', fontsize=7)
# ax.text(-0.1, -0.06, r'$\mathrm{3.288 mg\ cm^{-2}}$', transform=ax.transAxes, fontsize=6, color=colors[3], va='top', ha='right', fontfamily='Arial',)
# ax.text(-0.1, 0, r'C/10, 1C=250 mA/g', transform=ax.transAxes, fontsize=6, color=colors[3], va='top', ha='right', fontfamily='Arial',)

ax2 = ax.twiny()
ax2.set_position((0, 0, 1, 1))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"], selected_echem["FD1st_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(r"Current (mA)", fontsize=9)
ax2.set_xlim(-0.2, 0.2)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax2.tick_params(
    axis="both",
    which="both",
    labelsize=8,
    bottom=False,
    top=True,
    labeltop=True,
    labelright=False,
)
# ax2.legend(loc='upper left', bbox_to_anchor=(0.15, 1.05), ncols=1, frameon=False, labelcolor='linecolor', fontsize=7)
ax.text(-0.5, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 按时间与谱线映射构建谱图网格
scan_cols = (selected_spectrum_time["scan_idx"].astype(int) + 1).tolist()
missing_scan_cols = [c for c in scan_cols if c not in selected_spectrum.columns]
if missing_scan_cols:
    raise ValueError(f"FigureSI 35: missing mapped spectrum columns: {missing_scan_cols[:6]}")
selected_spectrum_plot = selected_spectrum.reindex(columns=scan_cols)
spec_time_arr = selected_spectrum_time["spec_time_h"].to_numpy(dtype=float)
d_axis = selected_spectrum_plot.index.to_numpy(dtype=float)
d_grid, t_grid = np.meshgrid(d_axis, spec_time_arr)
special_y_time = (
    selected_spectrum_time[selected_spectrum_time["scan_idx"].isin(index_labels)]["spec_time_h"]
    .dropna()
    .astype(float)
    .tolist()
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.05, 0.035, 0.94, 0.94))
ax.set_box_aspect(1.0)

ax.pcolormesh(
    d_grid,
    t_grid,
    selected_spectrum_plot.T.to_numpy(dtype=float),
    cmap="coolwarm",
    shading="gouraud",
)
ax.set_yticks([])
ax.set_ylim(float(spec_time_arr.min()), float(spec_time_arr.max()))
ax.set_xlim(2.0, 8.0)
ax.set_xlabel(
    r"d Spacing ($\AA$)",
    fontsize=9,
)
# # 修改刻度设置以适配d_spacing
# ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=0))
# ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.03, 0.1, 0.08, 0.8)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.text(
    1.08,
    0.97,
    "High",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="center",
    fontfamily="Arial",
)
ax.text(
    1.08,
    0.08,
    "Low",
    transform=ax.transAxes,
    fontsize=9,
    va="top",
    ha="center",
    fontfamily="Arial",
)

pdf_scale = [0.001, 0.006, 0.001, 0.03, 0.002, 0.001, 0.0001, 0.0005]
XRD_y = [0, 0.15, 0.15, 0.15, 0.15, 0.07, 0.0, 0.09]
XRDcolors = ["blue", "k", "k", "k", "k", "orange", "purple", "green"]

for i in range(pdfmo.shape[0]):
    temp = pdfmo.iloc[:, 2 * i : 2 * i + 2].dropna(axis="index", how="all")
    for j in range((temp.shape[0])):
        ax.axvline(x=temp.iloc[j, 0], ymin=XRD_y[i], ymax=(XRD_y[i] + temp.iloc[j, 1] * pdf_scale[i]), lw=1, c=XRDcolors[i])

ax.text(
    0,
    1.08,
    r"$\alpha$-MnO$\mathrm{_2}$ #00-044-0141",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="blue",
)
ax.text(
    0.55,
    1.08,
    r"Zn(HSO$_{4}$)$_{2}$ #00-052-0258",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="orange",
)
ax.text(0, 1.02, r"ZSH", horizontalalignment="left", verticalalignment="bottom", transform=ax.transAxes, fontsize=8, c="k")
ax.text(
    0.12,
    1.01,
    r"K$_{0.66}$Mn$_4$O$_{8}$ #04-023-7544",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="purple",
)
ax.text(
    0.77,
    1.02,
    r"Ti #00-044-1294",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=9,
    c="green",
)

ax.text(-0.05, 1.0, r"B", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)

opxrd_ref_d_spacing = [2.150451, 2.390493, 3.099262, 4.895775, 6.93]
peak_widths = [(0.02, 0.02), (0.02, 0.02), (0.05, 0.05), (0.05, 0.05), (0.1, 0.1)]
vmin_vmax = [(None, None), (None, None), (None, None), (None, None), (None, None)]

for i in range(len(opxrd_ref_d_spacing)):
    ax = subfig.add_axes((0.1 + i * 0.14, 0.035, 0.12, 0.94), zorder=0)
    temp = selected_spectrum_plot.loc[(selected_spectrum_plot.index >= opxrd_ref_d_spacing[i] - peak_widths[i][0]) & (selected_spectrum_plot.index <= opxrd_ref_d_spacing[i] + peak_widths[i][1])]
    vmin = vmin_vmax[i][0] if vmin_vmax[i][0] is not None else temp.min().min()
    vmax = vmin_vmax[i][1] if vmin_vmax[i][1] is not None else temp.max().max()
    temp_d_grid, temp_t_grid = np.meshgrid(temp.index.to_numpy(dtype=float), spec_time_arr)
    ax.pcolormesh(temp_d_grid, temp_t_grid, temp.T.to_numpy(dtype=float), cmap="coolwarm", shading="gouraud", vmin=vmin, vmax=vmax)
    ax.axvline(opxrd_ref_d_spacing[i], ls="--", lw=1.0, color="grey", alpha=0.5)
    ax.yaxis.set_ticks([])
    ax.set_ylim(float(spec_time_arr.min()), float(spec_time_arr.max()))
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_d_spacing[i]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor="k",
        bottom=True,
        left=True,
        labelbottom=True,
        labelleft=True,
    )
    # 图C按时间映射后的特殊点位置标注
    for y_sp in special_y_time:
        ax.axhline(y=y_sp, lw=0.6, ls="--", color=colors[1], alpha=0.5)

ax.text(-4.9, 1.0, r"C", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_07_XRD_01_V4_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_07_XRD_01_V4_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_07_XRD_01_V4_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_07_XRD_01_V4_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


In [ ]:
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 5.0))
gs = fig.add_gridspec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 0.3, 0.3))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Ewe/V"], selected_echem["FD1st_time"], ls="-", lw=1.0, c=colors[0], label=r"opCoin_1", zorder=5)


for i, idx in enumerate(selected_spectrum_time["scan_idx"]):
    if idx in index_labels:
        index_value = selected_spectrum_time.loc[selected_spectrum_time["scan_idx"] == idx, "echem_idx"].iloc[0]
        ax.scatter(
            selected_echem.loc[index_value, "Ewe/V"],
            selected_echem.loc[index_value, "FD1st_time"],
            c=colors[0],
            edgecolors="face",
            marker="*",
            s=50,
            zorder=10,
        )
    else:
        index_value = selected_spectrum_time.loc[i, "echem_idx"]
        ax.scatter(
            selected_echem.loc[index_value, "Ewe/V"],
            selected_echem.loc[index_value, "FD1st_time"],
            c="lightgrey",
            edgecolors="face",
            s=5,
            zorder=1,
        )

ax.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=8)
ax.set_xlim(0.85, 1.85)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=-0.15))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=-0.15))

ax.set_ylabel(r"Duration time (h)", fontsize=8)
ax.set_ylim(-0.5, 30.5)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))

# ax.legend(loc='upper right', bbox_to_anchor=(0.9, 1.0), ncols=1, frameon=False, labelcolor='linecolor', fontsize=7)
# ax.text(-0.1, -0.06, r'$\mathrm{3.288 mg\ cm^{-2}}$', transform=ax.transAxes, fontsize=6, color=colors[3], va='top', ha='right', fontfamily='Arial',)
# ax.text(-0.1, 0, r'C/10, 1C=250 mA/g', transform=ax.transAxes, fontsize=6, color=colors[3], va='top', ha='right', fontfamily='Arial',)

ax2 = ax.twiny()
ax2.set_position((0, 0, 0.3, 0.3))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"], selected_echem["FD1st_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(r"Current (mA)", fontsize=8)
ax2.set_xlim(-0.2, 0.2)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax2.tick_params(
    axis="both",
    which="both",
    labelsize=8,
    bottom=False,
    top=True,
    labeltop=True,
    labelright=False,
)
# ax2.legend(loc='upper left', bbox_to_anchor=(0.15, 1.05), ncols=1, frameon=False, labelcolor='linecolor', fontsize=7)
ax.text(-0.8, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 按时间与谱线映射构建谱图网格
scan_cols = (selected_spectrum_time["scan_idx"].astype(int) + 1).tolist()
missing_scan_cols = [c for c in scan_cols if c not in selected_spectrum.columns]
if missing_scan_cols:
    raise ValueError(f"FigureSI 35 V3: missing mapped spectrum columns: {missing_scan_cols[:6]}")
selected_spectrum_plot = selected_spectrum.reindex(columns=scan_cols)
spec_time_arr = selected_spectrum_time["spec_time_h"].to_numpy(dtype=float)
special_y_time = (
    selected_spectrum_time[selected_spectrum_time["scan_idx"].isin(index_labels)]["spec_time_h"]
    .dropna()
    .astype(float)
    .tolist()
)

# 图 B
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.06, -0.85, 0.55, 0.08))

K1 = "#295488"
K2 = "k"
K3 = "purple"
K4 = "orange"

ax.text(
    0,
    0.55,
    r"$\mathrm{Zn_4SO_4(OH)_6 \cdot 5H_2O}$ #00-060-0655, #04-012-8189",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=8,
    c=K1,
)
ax.text(
    0,
    0.05,
    r"$\mathrm{Zn_4SO_4(OH)_6 \cdot 3H_2O}$ #01-082-3605, #00-039-0689",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=8,
    c=K2,
)
ax.text(
    0.65,
    0.55,
    r"$\mathrm{K_{0.66}Mn_4O_8}$ #04-023-7544",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=8,
    c=K3,
)
ax.text(
    0.66,
    0.05,
    r"$\mathrm{Zn(HSO_4)_2}$ #00-052-0258",
    horizontalalignment="left",
    verticalalignment="bottom",
    transform=ax.transAxes,
    fontsize=8,
    c=K4,
)
ax.spines[:].set_visible(False)
ax.set_xticks([])
ax.set_yticks([])
ax.set_facecolor("gainsboro")

# opxrd_ref_peak = [
#     [7.451],[9.775],[11.194],[11.591],[11.837],[11.945],[12.611],[12.755],
#     [13.060],[14.918],[15.055],[15.276],[15.452],[15.802],[15.914],
#     [16.103],[16.541],[17.585],[18.457],
# ]

opxrd_ref_peak = [
    [7.46],
    [9.78],
    [11.20],
    [11.60],
    [11.84],
    [11.95],
    [12.60],
    [12.76],
    [13.06],
    [14.92],
    [15.055],
    [15.28],
    [15.45],
    [15.80],
    [15.92],
    [16.103],
    [16.54],
    [17.585],
    [18.457],
]

opxrd_ref_d_spacing = [[5.45], [4.16], [3.63], [3.51], [3.44], [3.40], 
[3.24], [3.19], [3.12], [2.73], [2.71], [2.67], [2.64], [2.58], 
[2.56], [2.53], [2.47], [2.32], [2.21]]

# KMnO
opxrd_colors = [K1, K1, K2, K1, K1, K1, K1, K1, K3, K2, K4, K1, K2, K2, K1, K1, K3, K4, K1]
vmin_vmax = [
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
]

for i in range(0, len(opxrd_ref_peak) // 3 - 1):
    # print(i)
    ax = subfig.add_axes((0.22 + i * 0.08, 0.0, 0.07, 0.3), zorder=0)
    temp = selected_spectrum_plot.loc[(selected_spectrum_plot.index >= opxrd_ref_d_spacing[i][0] - 0.1) & (selected_spectrum_plot.index <= opxrd_ref_d_spacing[i][0] + 0.1)]
    if temp.empty:
        continue
    temp_d_grid, temp_t_grid = np.meshgrid(temp.index.to_numpy(dtype=float), spec_time_arr)
    ax.pcolormesh(temp_d_grid, temp_t_grid, temp.T.to_numpy(dtype=float), cmap="coolwarm", shading="gouraud", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1])
    ax.axvline(opxrd_ref_d_spacing[i][0], ls="--", lw=1.0, color="grey", alpha=0.3)
    # ax.text(0.75 + 0.001 * i, -0.30, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=7, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(float(spec_time_arr.min()), float(spec_time_arr.max()))
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_d_spacing[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    # ax.tick_params(axis='both', which='both', labelcolor='k', bottom=True, labelbottom=True)
    ax.tick_params(axis="both", which="both", labelcolor=opxrd_colors[i], bottom=True, labelbottom=True)
    for y_sp in special_y_time:
        ax.axhline(y=y_sp, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(len(opxrd_ref_peak) // 3 - 1, 2 * len(opxrd_ref_peak) // 3):
    # print(i)
    ax = subfig.add_axes((0.06 + (i - len(opxrd_ref_peak) // 3 + 1) * 0.08, -0.38, 0.07, 0.3), zorder=0)
    temp = selected_spectrum_plot.loc[(selected_spectrum_plot.index >= opxrd_ref_d_spacing[i][0] - 0.1) & (selected_spectrum_plot.index <= opxrd_ref_d_spacing[i][0] + 0.1)]
    if temp.empty:
        continue
    temp_d_grid, temp_t_grid = np.meshgrid(temp.index.to_numpy(dtype=float), spec_time_arr)
    ax.pcolormesh(temp_d_grid, temp_t_grid, temp.T.to_numpy(dtype=float), cmap="coolwarm", shading="gouraud", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1])
    ax.axvline(opxrd_ref_d_spacing[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    # ax.text(0.75 + 0.001 * i, -0.23, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=7, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(float(spec_time_arr.min()), float(spec_time_arr.max()))
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_d_spacing[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    # ax.tick_params(axis='both', which='both', labelcolor='k', bottom=True, labelbottom=True)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor=opxrd_colors[i],
        bottom=True,
        labelbottom=True,
    )
    for y_sp in special_y_time:
        ax.axhline(y=y_sp, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(2 * len(opxrd_ref_peak) // 3, len(opxrd_ref_peak)):
    # print(i)
    ax = subfig.add_axes((0.06 + (i - 2 * len(opxrd_ref_peak) // 3) * 0.08, -0.73, 0.07, 0.3), zorder=0)
    temp = selected_spectrum_plot.loc[(selected_spectrum_plot.index >= opxrd_ref_d_spacing[i][0] - 0.1) & (selected_spectrum_plot.index <= opxrd_ref_d_spacing[i][0] + 0.1)]
    if temp.empty:
        continue
    temp_d_grid, temp_t_grid = np.meshgrid(temp.index.to_numpy(dtype=float), spec_time_arr)
    ax.pcolormesh(temp_d_grid, temp_t_grid, temp.T.to_numpy(dtype=float), cmap="coolwarm", shading="gouraud", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1])
    ax.axvline(opxrd_ref_d_spacing[i][0], ls="--", lw=1.0, color="grey", alpha=0.5)
    # ax.text(0.75 + 0.001 * i, -0.23, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=7, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(float(spec_time_arr.min()), float(spec_time_arr.max()))
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_d_spacing[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    # ax.tick_params(axis='both', which='both', labelcolor='k', bottom=True, labelbottom=True)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor=opxrd_colors[i],
        bottom=True,
        labelbottom=True,
    )
    for y_sp in special_y_time:
        ax.axhline(y=y_sp, lw=0.6, ls="--", color=colors[0], alpha=1.0)
ax.text(-1.0, 2.23, r"$\mathrm{d \ Spacing\ (\AA)}$", transform=ax.transAxes, fontsize=9, va="center", ha="right", fontfamily="Arial", fontweight="bold", c="blue")
ax.text(-4.65, 3.45, r"B", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_07_XRD_02_V3_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_07_XRD_02_V3_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_07_XRD_02_V3_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_07_XRD_02_V3_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 08 -- XRD - Operando, ex situ XRD 比较

In [ ]:
# 读取数据
operando_csv_emd = path_data_root.joinpath(r"XRD\Operando\EMD\2025-MSPD\Results\IS17_D\Data\spectrum_all_d_spacing.csv")
operando_map_emd = {"OCV": "17_Da_f00001", "FD#A": "17_Da_f00016", "FC#D": "17_Da_f00047"}
operando_csv_mno2 = path_data_root.joinpath(r"XRD\Operando\αMnO2\2022-ICMAB\Results\Coin1\1stCharge+2ndDischarge\Spectum_all_d spacing.csv")
operando_map_mno2 = {"OCV": "1", "FD#A": "17", "FC#D": "37"}
path_si01_mno2 = path_data_root.joinpath(r"XRD\ExSitu\αMnO2\Pristine\Results\2024-ICMAB\αMnO2_Pristine_2024.UXD")
path_si01_emd = path_data_root.joinpath(r"XRD\ExSitu\EMD\Pristine\Results\EMD-1.60V-2mAh_cm-2-1MZnAC2+02M-pH5.0_1M+0.2M\#1EMD_CP.uxd")

# Operando
df_op_mno2 = pd.read_csv(operando_csv_mno2)
d_op_mno2 = pd.to_numeric(df_op_mno2["d_spacing"], errors="coerce").to_numpy(float)

spectra_mno2 = {
    "OCV": {
        "d": d_op_mno2,
        "y": pd.to_numeric(df_op_mno2[operando_map_mno2["OCV"]], errors="coerce").to_numpy(float),
    },
    "FD#A": {
        "d": d_op_mno2,
        "y": pd.to_numeric(df_op_mno2[operando_map_mno2["FD#A"]], errors="coerce").to_numpy(float),
    },
    "FC#D": {
        "d": d_op_mno2,
        "y": pd.to_numeric(df_op_mno2[operando_map_mno2["FC#D"]], errors="coerce").to_numpy(float),
    },
}

df_op_emd = pd.read_csv(operando_csv_emd)
d_op_emd = pd.to_numeric(df_op_emd["d_spacing"], errors="coerce").to_numpy(float)

spectra_emd = {
    "OCV": {
        "d": d_op_emd,
        "y": pd.to_numeric(df_op_emd[operando_map_emd["OCV"]], errors="coerce").to_numpy(float),
    },
    "FD#A": {
        "d": d_op_emd,
        "y": pd.to_numeric(df_op_emd[operando_map_emd["FD#A"]], errors="coerce").to_numpy(float),
    },
    "FC#D": {
        "d": d_op_emd,
        "y": pd.to_numeric(df_op_emd[operando_map_emd["FC#D"]], errors="coerce").to_numpy(float),
    },
}

# Pristine
def _load_uxd(path_uxd: Path):
    raw = pd.read_csv(path_uxd, sep=r"\s+", index_col=None, header=0, comment="#")
    two_theta = pd.to_numeric(raw["2THETA"], errors="coerce").to_numpy(float)
    intensity = pd.to_numeric(raw.iloc[:, 1], errors="coerce").to_numpy(float)
    d_spacing = 1.5406 / (2.0 * np.sin(np.radians(two_theta / 2.0)))

    m = np.isfinite(d_spacing) & np.isfinite(intensity)
    d_spacing = d_spacing[m]
    intensity = intensity[m]

    # 统一为 d-spacing 从大到小
    order = np.argsort(d_spacing)[::-1]
    return d_spacing[order], intensity[order]

alpha_d, alpha_y = _load_uxd(path_si01_mno2)
spectra_mno2["alpha_MnO2"] = {"d": alpha_d, "y": alpha_y}  
emd_d, emd_y = _load_uxd(path_si01_emd)
spectra_emd["EMD"] = {"d": emd_d, "y": emd_y} 

In [ ]:
# 画图
plt.close("all")

fig = plt.figure(figsize=(7.2, 2.5), dpi=300)
gs = fig.add_gridspec(1, 2, height_ratios=[1], width_ratios=[1, 1], wspace=0, hspace=0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

detail_d_values_mno2 = [2.15, 2.39, 3.10, 4.90, 6.93]

# -------------------------
# 1) OCV
# -------------------------
d_ocv = spectra_mno2["OCV"]["d"]
y_ocv = spectra_mno2["OCV"]["y"]
y_ocv_norm = (y_ocv - np.nanmin(y_ocv)) / (np.nanmax(y_ocv) - np.nanmin(y_ocv) + 1e-12)
y_ocv_plot = y_ocv_norm * 3.0 + 0.8
ax.plot(d_ocv, y_ocv_plot, ls="-", lw=1.0, c=colors[0], label="OCV", zorder=1)

# -------------------------
# 2) FD#A
# -------------------------
d_fda = spectra_mno2["FD#A"]["d"]
y_fda = spectra_mno2["FD#A"]["y"]
y_fda_norm = (y_fda - np.nanmin(y_fda)) / (np.nanmax(y_fda) - np.nanmin(y_fda) + 1e-12)
y_fda_plot = y_fda_norm * 3 + 2.6
ax.plot(d_fda, y_fda_plot, ls="-", lw=1.0, c=colors[1], label="FD#A", zorder=1)

# -------------------------
# 3) FC#D
# -------------------------
d_fcd = spectra_mno2["FC#D"]["d"]
y_fcd = spectra_mno2["FC#D"]["y"]
y_fcd_norm = (y_fcd - np.nanmin(y_fcd)) / (np.nanmax(y_fcd) - np.nanmin(y_fcd) + 1e-12)
y_fcd_plot = y_fcd_norm * 3 + 4.5
ax.plot(d_fcd, y_fcd_plot, ls="-", lw=1.0, c=colors[2], label="FC#D", zorder=1)

# -------------------------
# 4) alpha-MnO2
# -------------------------
d_alpha = spectra_mno2["alpha_MnO2"]["d"]
y_alpha = spectra_mno2["alpha_MnO2"]["y"]
y_alpha_norm = (y_alpha - np.nanmin(y_alpha)) / (np.nanmax(y_alpha) - np.nanmin(y_alpha) + 1e-12)
y_alpha_plot = y_alpha_norm * 0.8
ax.plot(d_alpha, y_alpha_plot, ls="-", lw=1.0, c=colors[3], label=r"$\alpha$-MnO$_2$", zorder=1)

# 标注给定 d 点
for x in detail_d_values_mno2:
    ax.axvline(x=x, ymin=0.0, ymax=0.8, ls="--", lw=0.6, color="0.55", alpha=0.7, zorder=1)

# for k, x in enumerate(sorted(detail_d_values_mno2, reverse=True)):
#     y_txt = 5.5 + 0.12 * (k % 2)
#     ax.text(x, y_txt, f"{x:.3f}", fontsize=7, rotation=90, ha="center", va="top", color="0.25", clip_on=False)

ax.set_xlim(2, 8)
ax.set_xlabel(r"$\mathrm{d \ Spacing \ (\AA)}$", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.set_ylabel("Relative intensity (arb. u.)", fontsize=9)
ax.set_ylim(-0.1, 8.0)
ax.set(yticks=[])
ax.tick_params(axis="both", labelsize=8, bottom=True, left=True, labelbottom=True, labelleft=True)

ax.legend(loc="upper right", frameon=False, labelcolor="linecolor", fontsize=10, ncol=2)
ax.text(-0.06, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

detail_d_values_emd = [10.9626, 7.9882, 2.7172, 2.4451, 2.0694, 1.5770, 1.41497238067471]

# -------------------------
# 1) OCV
# -------------------------
d_ocv = spectra_emd["OCV"]["d"]
y_ocv = spectra_emd["OCV"]["y"]
y_ocv_norm = (y_ocv - np.nanmin(y_ocv)) / (np.nanmax(y_ocv) - np.nanmin(y_ocv) + 1e-12)
y_ocv_plot = y_ocv_norm * 4 - 0.6
ax.plot(d_ocv, y_ocv_plot, ls="-", lw=1.0, c=colors[0], label="OCV", zorder=1)

# -------------------------
# 2) FD#A
# -------------------------
d_fda = spectra_emd["FD#A"]["d"]
y_fda = spectra_emd["FD#A"]["y"]
y_fda_norm = (y_fda - np.nanmin(y_fda)) / (np.nanmax(y_fda) - np.nanmin(y_fda) + 1e-12)
y_fda_plot = y_fda_norm * 4 + 0.9
ax.plot(d_fda, y_fda_plot, ls="-", lw=1.0, c=colors[1], label="FD#A", zorder=1)

# -------------------------
# 3) FC#D
# -------------------------
d_fcd = spectra_emd["FC#D"]["d"]
y_fcd = spectra_emd["FC#D"]["y"]
y_fcd_norm = (y_fcd - np.nanmin(y_fcd)) / (np.nanmax(y_fcd) - np.nanmin(y_fcd) + 1e-12)
y_fcd_plot = y_fcd_norm * 4 + 0.7
ax.plot(d_fcd, y_fcd_plot, ls="-", lw=1.0, c=colors[2], label="FC#D", zorder=1)

# -------------------------
# 4) EMD
# -------------------------
d_alpha = spectra_emd["EMD"]["d"]
y_alpha = spectra_emd["EMD"]["y"]
y_alpha_norm = (y_alpha - np.nanmin(y_alpha)) / (np.nanmax(y_alpha) - np.nanmin(y_alpha) + 1e-12)
y_alpha_plot = y_alpha_norm * 0.8 - 0.1
ax.plot(d_alpha, y_alpha_plot, ls="-", lw=1.0, c=colors[3], label=r"EMD", zorder=1)

# 标注给定 d 点
for x in detail_d_values_emd:
    ax.axvline(x=x, ymin=0.0, ymax=0.8, ls="--", lw=0.6, color="0.55", alpha=0.7, zorder=1)

# for k, x in enumerate(sorted(detail_d_values_emd, reverse=True)):
#     y_txt = 5.5 + 0.12 * (k % 2)
#     ax.text(x, y_txt, f"{x:.3f}", fontsize=7, rotation=90, ha="center", va="top", color="0.25", clip_on=False)

ax.set_xlim(2, 5)
ax.set_xlabel(r"$\mathrm{d \ Spacing \ (\AA)}$", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.set_ylabel("Relative intensity (arb. u.)", fontsize=9)
ax.set_ylim(-0.1, 3.2)
ax.set(yticks=[])
ax.tick_params(axis="both", labelsize=8, bottom=True, left=True, labelbottom=True, labelleft=True)
ax.legend(loc="upper right", frameon=False, labelcolor="linecolor", fontsize=10, ncol=2)
ax.text(-0.06, 1.0, r"B", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_08_XRD_02_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_08_XRD_02_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_08_XRD_02_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_08_XRD_02_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 09 -- XAS - Operando XANES and PCA on electrolyte + Cathode, Zn

In [ ]:
# 读取 reference + operando 数据

path_file = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")

# Zn
data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P1_Zn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P1_Zn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
xas_Zn = pd.concat([data1, data2.iloc[:, 3:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, 2, *list(range(12, xas_Zn.shape[1], 1))]
xas_Zn = xas_Zn.iloc[:, charge_index]  # noqa: N816

pdf_Zn = xas_Zn.iloc[:, 0:3]  # noqa: N816
pdf_Zn.columns = [r"Energy", r"0.5M_ZnSO4(aq.)", r"ZSH"]
opData_Zn = xas_Zn.iloc[:, [0, *list(range(3, xas_Zn.shape[1]))]]  # noqa: N816
opData_Zn.columns = list(range(0, opData_Zn.shape[1], 1))

# 读取 PCA 数据

path_file = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\FC1st-FD2nd\Oct2022_cell2_P1\XANES")
# Zn
pca_Zn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"PCA", r"cell2_1stFC_opXAS_P1_Zn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Zn2 = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Zn", r"PCA", r"Cumulative_variance.csv"), sep=",", index_col=None, header=0, comment="#"
)


In [ ]:
# 剔除不好的谱线， Zn
opData_Zn = opData_Zn.drop(columns=opData_Zn.columns[14])  # noqa: N816


In [ ]:
# 读取 reference + operando 数据

path_file_ca = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")

# Zn
data1_ca = pd.read_csv(
    Path.joinpath(path_file_ca, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
data2_ca = pd.read_csv(
    Path.joinpath(path_file_ca, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
xas_Zn_ca = pd.concat([data1_ca, data2_ca.iloc[:, 3:]], axis=1, ignore_index=True)  # noqa: N816
charge_index_ca = [0, 1, 2, *list(range(12, xas_Zn_ca.shape[1], 1))]
xas_Zn_ca = xas_Zn_ca.iloc[:, charge_index_ca]  # noqa: N816

pdf_Zn_ca = xas_Zn_ca.iloc[:, 0:3]  # noqa: N816
pdf_Zn_ca.columns = [r"Energy", r"0.5M_ZnSO4(aq.)", r"ZHS"]
opData_Zn_ca = xas_Zn_ca.iloc[:, [0, *list(range(3, xas_Zn_ca.shape[1]))]]  # noqa: N816
opData_Zn_ca.columns = list(range(0, opData_Zn_ca.shape[1], 1))

# 读取 PCA 数据
path_file_ca = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\FC1st-FD2nd\Oct2022_cell3_P2b\XANES")
# Zn
pca_Zn_ca = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file_ca, r"Zn", r"PCA", r"cell2_1stFC_opXAS_P2b_Zn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Zn2_ca = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file_ca, r"Zn", r"PCA", r"Cumulative_variance.csv"), sep=",", index_col=None, header=0, comment="#"
)


In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 5.0))
gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1], wspace=0, hspace=0)

xascolors = [[colors[0], colors[4]], [colors[2], colors[1]]]
xas_colors = ListedColormap(
    mpl.colormaps["sunset"](np.linspace(1.0, 0.0, opData_Zn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

xascolors = [[colors[0], colors[4]], [colors[2], colors[1]]]
xas_colors = ListedColormap(
    mpl.colormaps["coolwarm"](np.linspace(0.0, 1.0, opData_Zn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]

for i in range(pdf_Zn.shape[1] - 1):
    ax.plot(pdf_Zn.iloc[:, 0], pdf_Zn.iloc[:, i + 1], c=xascolors[1][i], ls="--", lw=1.0, label=labels[1][i], zorder=5)
ax.tick_params(
    axis="both",
    which="both",
    labelsize=8,
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
for i in range(opData_Zn.shape[1] - 1):
    ax.plot(opData_Zn.iloc[:, 0], opData_Zn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0)  # type: ignore

ax.set_xlim(9650, 9750)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax.set_ylim(0, 2.4)
ax.set_ylabel(r"Absorption (arb. u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((0.5, 0.05, 0.42, 0.05)),
    location="bottom",
    orientation="horizontal",
    cmap=xas_colors,
    ticklocation="top",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.45, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    handlelength=3,
    handletextpad=0.2,
)
ax.text(
    0.49,
    0.18,
    r"Charge",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.84,
    0.26,
    r"Discharge",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="right",
    fontfamily="Arial",
)
ax.text(
    0.92,
    0.18,
    r"Charge",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="right",
    fontfamily="Arial",
)
colorbar.outline.set_visible(False)  # type: ignore

ax.text(-0.18, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.3, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

line = []
for i in range(pca_Zn.shape[1] - 1):
    (lineA,) = ax.plot(pca_Zn.iloc[:, 0], pca_Zn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0, zorder=i)  # type: ignore # noqa: N816
    line.append(lineA)
ax.set_xlim(9650, 9750)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-15.0, 1)
ax.set_ylabel(r"PCA intensity (arb. u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1]],
    labels=[r"Component#1", r"Component#2"],
    loc="upper left",
    bbox_to_anchor=(0.45, 0.25),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)
ax.text(-0.18, 1.0, r"B", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.6, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Zn2.shape[0], 1), np.log10(pca_Zn2.iloc[:, 1]), s=50, c=colors[1], edgecolors="face")

ax.set_xlim(-0.5, pca_Zn2.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-8, 0.5)
ax.set_ylabel(r"Log variance", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.text(-0.18, 1.0, r"C", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

xascolors = [[colors[0], colors[4]], [colors[2], colors[1]]]
xas_colors = ListedColormap(
    mpl.colormaps["coolwarm"](np.linspace(0.0, 1.0, opData_Zn_ca.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]

for i in range(pdf_Zn_ca.shape[1] - 1):
    ax.plot(pdf_Zn_ca.iloc[:, 0], pdf_Zn_ca.iloc[:, i + 1], c=xascolors[1][i], ls="--", lw=1.0, label=labels[1][i], zorder=5)
ax.tick_params(
    axis="both",
    which="both",
    labelsize=8,
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
for i in range(opData_Zn_ca.shape[1] - 1):
    ax.plot(opData_Zn_ca.iloc[:, 0], opData_Zn_ca.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0)  # type: ignore

ax.set_xlim(9650, 9750)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax.set_ylim(0, 2.4)
ax.set_ylabel(r"Absorption (arb. u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((0.5, 0.05, 0.42, 0.05)),
    location="bottom",
    orientation="horizontal",
    cmap=xas_colors,
    ticklocation="top",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.42, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    handlelength=3,
    handletextpad=0.2,
)
ax.text(
    0.49,
    0.18,
    r"Charge",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.84,
    0.26,
    r"Discharge",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="right",
    fontfamily="Arial",
)
ax.text(
    0.92,
    0.18,
    r"Charge",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="right",
    fontfamily="Arial",
)
colorbar.outline.set_visible(False)  # type: ignore
ax.text(-0.18, 1.0, r"D", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 图 E
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.3, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

line = []
for i in range(pca_Zn_ca.shape[1] - 1):
    (lineA,) = ax.plot(pca_Zn_ca.iloc[:, 0], pca_Zn_ca.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0, zorder=i)  # type: ignore # noqa: N816
    line.append(lineA)
ax.set_xlim(9650, 9750)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-15, 1)
ax.set_ylabel(r"PCA intensity (arb. u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1], line[2]],
    labels=[r"Component#1", r"Component#2", r"Component#3"],
    loc="upper left",
    bbox_to_anchor=(0.45, 0.35),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)
ax.text(-0.18, 1.0, r"E", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 图 F
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.6, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Zn2_ca.shape[0], 1), np.log10(pca_Zn2_ca.iloc[:, 1]), s=50, c=colors[1], edgecolors="face")

ax.set_xlim(-0.5, pca_Zn2_ca.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-8, 0.5)
ax.set_ylabel(r"Log variance", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.text(-0.18, 1.0, r"F", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_09_XAS_XANES_00_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_09_XAS_XANES_00_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_09_XAS_XANES_00_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_09_XAS_XANES_00_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigrueSI 10 -- XAS - Operando XANES and PCA on electrolyte + Cathode, Mn

In [ ]:
# 读取 reference + operando 数据 # 放电数据
path_file = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")
# Mn
data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P1_Mn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P1_Mn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
xas_Mn = pd.concat([data1, data2.iloc[:, 5:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, 2, 3, 4, *list(range(14, xas_Mn.shape[1], 1))]
xas_Mn = xas_Mn.iloc[:, charge_index]  # noqa: N816
pdf_Mn = xas_Mn.iloc[:, 0:5]  # noqa: N816
pdf_Mn.columns = [
    r"Energy",
    r"0.2M_MnSO4(aq.)",
    r"alpha_MnO2_Electrode",
    r"alpha_MnO2_Powder",
    r"FC1st",
]
opData_Mn = xas_Mn.iloc[:, [0, *list(range(5, xas_Mn.shape[1]))]]  # noqa: N816
opData_Mn.columns = list(range(0, opData_Mn.shape[1], 1))


# 读取 PCA 数据
path_file = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\FC1st-FD2nd\Oct2022_cell2_P1\XANES")
# Mn
pca_Mn = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"PCA", r"cell2_1stFC_opXAS_P1_Mn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Mn2 = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file, r"Mn", r"PCA", r"Cumulative_variance.csv"), sep=",", index_col=None, header=0, comment="#"
)


In [ ]:
# 读取 reference + operando 数据
path_file_ca = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")

# Mn
data1_ca = pd.read_csv(
    Path.joinpath(path_file_ca, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
data2_ca = pd.read_csv(
    Path.joinpath(path_file_ca, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
xas_Mn_ca = pd.concat([data1_ca, data2_ca.iloc[:, 5:]], axis=1, ignore_index=True)  # noqa: N816
charge_index_ca = [0, 1, 2, 3, 4, *list(range(14, xas_Mn_ca.shape[1], 1))]
xas_Mn_ca = xas_Mn_ca.iloc[:, charge_index_ca]  # noqa: N816
pdf_Mn_ca = xas_Mn_ca.iloc[:, 0:5]  # noqa: N816
pdf_Mn_ca.columns = [
    r"Energy",
    r"0.2M_MnSO4(aq.)",
    r"alpha_MnO2_Electrode",
    r"alpha_MnO2_Powder",
    r"FC1st",
]
opData_Mn_ca = xas_Mn_ca.iloc[:, [0, *list(range(5, xas_Mn_ca.shape[1]))]]  # noqa: N816
opData_Mn_ca.columns = list(range(0, opData_Mn_ca.shape[1], 1))

# 读取 PCA 数据
path_file_ca = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\FC1st-FD2nd\Oct2022_cell3_P2b\XANES")
# Mn
pca_Mn_ca = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file_ca, r"Mn", r"PCA", r"cell2_1stFC_opXAS_P2b_Mn_components.pca"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
pca_Mn2_ca = pd.read_csv(  # noqa: N816
    Path.joinpath(path_file_ca, r"Mn", r"PCA", r"Cumulative_variance.csv"), sep=",", index_col=None, header=0, comment="#"
)


In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 5.0))
gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 1], height_ratios=[1, 1], wspace=0, hspace=0)

xascolors = [[colors[0], colors[4]], [colors[2], colors[1]]]
xas_colors = ListedColormap(
    mpl.colormaps["sunset"](np.linspace(1.0, 0.0, opData_Mn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

for i in range(pdf_Mn.shape[1] - 3):
    ax.plot(pdf_Mn.iloc[:, 0], pdf_Mn.iloc[:, i + 1], c=xascolors[0][i], ls="--", lw=1.0, label=labels[0][i], zorder=5)
ax.tick_params(
    axis="both",
    which="both",
    labelsize=8,
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
for i in range(opData_Mn.shape[1] - 1):
    ax.plot(opData_Mn.iloc[:, 0], opData_Mn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0)  # type: ignore

ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax.set_ylim(0, 2.4)
ax.set_ylabel(r"Absorption (arb. u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((0.5, 0.05, 0.42, 0.05)),
    location="bottom",
    orientation="horizontal",
    cmap=xas_colors,
    ticklocation="top",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.45, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    handlelength=3,
    handletextpad=0.2,
)
ax.text(
    0.49,
    0.18,
    r"Charge",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.84,
    0.26,
    r"Discharge",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="right",
    fontfamily="Arial",
)
ax.text(
    0.92,
    0.18,
    r"Charge",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="right",
    fontfamily="Arial",
)
colorbar.outline.set_visible(False)  # type: ignore
ax.text(-0.18, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.3, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

line = []
for i in range(pca_Mn.shape[1] - 1):
    (lineA,) = ax.plot(pca_Mn.iloc[:, 0], pca_Mn.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0, zorder=i)  # type: ignore # noqa: N816
    line.append(lineA)
ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-15, 1)
ax.set_ylabel(r"PCA intensity (arb. u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1]],
    labels=[r"Component#1", r"Component#2"],
    loc="upper left",
    bbox_to_anchor=(0.4, 0.3),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)
ax.text(-0.18, 1.0, r"B", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.6, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Mn2.shape[0], 1), np.log10(pca_Mn2.iloc[:, 1]), s=50, c=colors[0], edgecolors="face")

ax.set_xlim(-0.5, pca_Mn2.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-6.0, 0.5)
ax.set_ylabel(r"Log variance", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.text(-0.18, 1.0, r"C", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 图 D
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

xascolors = [[colors[0], colors[4]], [colors[2], colors[1]]]
xas_colors = ListedColormap(
    mpl.colormaps["sunset"](np.linspace(1.0, 0.0, opData_Mn.shape[1])),
    name="xas_colors",
)
labels = [[r"0.2 M Mn$\mathrm{^{2\!+}}$", r"${\alpha}$-MnO$\mathrm{_2}$"], [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"]]

for i in range(pdf_Mn_ca.shape[1] - 3):
    ax.plot(pdf_Mn_ca.iloc[:, 0], pdf_Mn_ca.iloc[:, i + 1], c=xascolors[0][i], ls="--", lw=1.0, label=labels[0][i], zorder=5)
ax.tick_params(
    axis="both",
    which="both",
    labelsize=8,
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labeltop=False,
    labelleft=True,
    labelright=False,
)
for i in range(opData_Mn_ca.shape[1] - 1):
    ax.plot(opData_Mn_ca.iloc[:, 0], opData_Mn_ca.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0)  # type: ignore

ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax.set_ylim(0, 2.4)
ax.set_ylabel(r"Absorption (arb. u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((0.5, 0.05, 0.42, 0.05)),
    location="bottom",
    orientation="horizontal",
    cmap=xas_colors,
    ticklocation="top",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.42, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=9,
    handlelength=3,
    handletextpad=0.2,
)
ax.text(
    0.49,
    0.18,
    r"Charge",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.84,
    0.26,
    r"Discharge",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="right",
    fontfamily="Arial",
)
ax.text(
    0.92,
    0.18,
    r"Charge",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="right",
    fontfamily="Arial",
)
colorbar.outline.set_visible(False)  # type: ignore
ax.text(-0.18, 1.0, r"D", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 E
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.3, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

line = []
for i in range(pca_Mn_ca.shape[1] - 1):
    (lineA,) = ax.plot(pca_Mn_ca.iloc[:, 0], pca_Mn_ca.iloc[:, i + 1], c=xas_colors.colors[i], ls="-", lw=1.0, zorder=i)  # type: ignore # noqa: N816
    line.append(lineA)
ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax.set_ylim(-10, 2.0)
ax.set_ylabel(r"PCA intensity (arb. u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=3, offset=-1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1.5, offset=-1))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(
    handles=[line[0], line[1], line[2]],
    labels=[r"Component#1", r"Component#2", r"Component#3"],
    loc="upper left",
    bbox_to_anchor=(0.4, 0.33),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)
ax.text(-0.18, 1.0, r"E", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 F
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.6, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(np.arange(0, pca_Mn2_ca.shape[0], 1), np.log10(pca_Mn2_ca.iloc[:, 1]), s=50, c=colors[0], edgecolors="face")

ax.set_xlim(-0.5, pca_Mn2_ca.shape[0] + 1)
ax.set_xlabel(r"Components", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-8.0, 0.5)
ax.set_ylabel(r"Log variance", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.text(-0.18, 1.0, r"F", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_10_XAS_XANES_00_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_10_XAS_XANES_00_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_10_XAS_XANES_00_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_10_XAS_XANES_00_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 11 -- XAS - Cell2 中第三相的存在 -V1

In [ ]:
# 读取数据

path_cell2 = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")
path_file = Path.joinpath(path_cell2, r"FC1st-FD2nd\Oct2022_cell2_P2b\XANES\Mn\PCA_Larch_Mn")
xas_Mn_path_1 = Path.joinpath(path_file, r"lcf_alpha_MnO2_electrode_pristine_residual_1.nor")
xas_Mn_path_2 = Path.joinpath(path_file, r"lcf_alpha_MnO2_electrode_pristine_residual_2.nor")

# 电化学时间序列
path_filelist = list(Path.joinpath(path_cell2, r"Echem").glob(r"**\*.txt"))
echem_all = []
for path_file_echem in path_filelist:
    with open(path_file_echem, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break

    df = pd.read_csv(path_file_echem, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem_all.append(df)

echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 当前对齐后的 Mn operando 数据
xas_Mn_1 = pd.read_csv(
    xas_Mn_path_1,
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
xas_Mn_2 = pd.read_csv(
    xas_Mn_path_2,
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)

# 两段 Mn residual .nor 的 energy 网格略有差异
energy_Mn = xas_Mn_1.iloc[:, 0].to_numpy(dtype=float)
energy_Mn_2 = xas_Mn_2.iloc[:, 0].to_numpy(dtype=float)
energy_mask_Mn = (energy_Mn >= float(np.nanmin(energy_Mn_2))) & (energy_Mn <= float(np.nanmax(energy_Mn_2)))
xas_Mn = xas_Mn_1.loc[energy_mask_Mn].reset_index(drop=True)
energy_Mn_common = xas_Mn.iloc[:, 0].to_numpy(dtype=float)
for col in range(1, xas_Mn_2.shape[1]):
    xas_Mn[xas_Mn.shape[1]] = np.interp(
        energy_Mn_common,
        energy_Mn_2,
        xas_Mn_2.iloc[:, col].to_numpy(dtype=float),
    )

pdf_Mn = xas_Mn.iloc[:, [0, 1, 2]]  # noqa: N816
pdf_Mn.columns = [r"Energy", r"0.2M_MnSO4(aq.)", r"alpha-MnO2"]
opData_Mn = xas_Mn.iloc[:, [0, *list(range(3, xas_Mn.shape[1]))]]  # noqa: N816
opData_Mn.columns = list(range(0, opData_Mn.shape[1], 1))


In [ ]:
# 聚焦当前对齐后 Mn 数据
opData_Mn = opData_Mn[opData_Mn.iloc[:, 0].between(6530.0, 6610.0)].copy()

# # 保留原来标记为异常的谱线
# bad_spec_cols = [11,]
# bad_spec_cols = [idx for idx in bad_spec_cols if idx < opData_Mn.shape[1]]
# if bad_spec_cols:
#     opData_Mn.iloc[:, bad_spec_cols] = np.nan



In [ ]:
n_spec = opData_Mn.shape[1] - 1
special_idx = np.array([0, 4, 9, 12, 14, 17, 24, 30, 34, 38], dtype=int)
special_idx = special_idx[(special_idx >= 0) & (special_idx < n_spec)]

energy = opData_Mn.iloc[:, 0].to_numpy(dtype=float)
mu_mat = opData_Mn.iloc[:, 1:].to_numpy(dtype=float).T  # (n_spec, n_energy)

# 电化学窗口：首圈放电谷值到次圈充电峰值
echem_index_low = echem_all[echem_all["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high = echem_all[echem_all["cycle number"] == 1]["Ewe/V"].idxmax()
echem_seg = echem_all.iloc[int(echem_index_low) : int(echem_index_high) + 1].copy()
echem_seg = echem_seg.dropna(subset=["time/s", "Ewe/V", "<I>/mA"]).sort_values("time/s").reset_index(drop=True)

# 直接从当前 LCF .nor 表头读取列名，用于谱线到采谱时间的映射
def _read_nor_cols(path_nor):
    cols = {}
    with open(path_nor, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.startswith("# Column."):
                if line.startswith("#------------------------"):
                    break
                continue
            left, right = line.split(":", 1)
            cols[int(left.split(".")[1])] = right.strip().lstrip("-").strip()
    return cols

cols_Mn_1 = _read_nor_cols(xas_Mn_path_1)
cols_Mn_2 = _read_nor_cols(xas_Mn_path_2)
op_labels = [cols_Mn_1[i] for i in range(4, xas_Mn_1.shape[1] + 1) if i in cols_Mn_1 and re.search(r"P2b_(\d+)", cols_Mn_1[i])]
op_labels.extend([cols_Mn_2[i] for i in range(2, xas_Mn_2.shape[1] + 1) if i in cols_Mn_2 and re.search(r"P2b_(\d+)", cols_Mn_2[i])])

if len(op_labels) != n_spec:
    raise ValueError(f"FigureSI 12A: op_labels={len(op_labels)} != n_spec={n_spec}")

map_df = pd.read_csv(Path.joinpath(path_cell2, r"Overview", r"Time_Index_Spectrum.csv"))
map_df = map_df[
    (map_df["Folder"].astype(str).str.contains("cell2_P2b", case=False, na=False))
    & (map_df["Element"].astype(str).str.upper() == "MN")
    & map_df["NorName"].notna()
].copy()
map_df["Time"] = pd.to_datetime(map_df["Time"], errors="coerce")
map_df["NorName"] = map_df["NorName"].astype(str).str.strip().str.lstrip("-")
map_df = map_df.dropna(subset=["Time"]).sort_values("Time")
name_to_time = map_df.drop_duplicates(subset=["NorName"], keep="first").set_index("NorName")["Time"]

label_s = pd.Series(op_labels, dtype="string").str.strip().str.lstrip("-")
spec_time_dt = label_s.map(name_to_time)
missing_labels = label_s[spec_time_dt.isna()].tolist()
if missing_labels:
    raise ValueError(f"FigureSI 12A: missing NorName->Time mapping for labels: {missing_labels[:6]}")

# 只保留落在电化学窗口内的谱线，避免 np.interp 对窗口外谱线使用边界值
echem_start = echem_seg["time/s"].iloc[0]
echem_end = echem_seg["time/s"].iloc[-1]
in_echem_window = spec_time_dt.between(echem_start, echem_end).to_numpy(dtype=bool)
if not np.all(in_echem_window):
    op_labels = [label for label, keep in zip(op_labels, in_echem_window) if keep]
    spec_time_dt = spec_time_dt[in_echem_window].reset_index(drop=True)
    mu_mat = mu_mat[in_echem_window, :]
    n_spec = mu_mat.shape[0]
special_idx = special_idx[(special_idx >= 0) & (special_idx < n_spec)]

# 统一时间零点，并把采谱点映射到电化学轨迹上
time_zero = min(echem_seg["time/s"].iloc[0], spec_time_dt.min())
echem_time = (echem_seg["time/s"] - time_zero).dt.total_seconds().to_numpy(dtype=float) / 3600.0
echem_voltage = echem_seg["Ewe/V"].to_numpy(dtype=float)
echem_current_ua = echem_seg["<I>/mA"].to_numpy(dtype=float) * 1000.0

spec_time_raw = (spec_time_dt - time_zero).dt.total_seconds().to_numpy(dtype=float) / 3600.0
spec_time = spec_time_raw.copy()
spec_volt = np.interp(spec_time, echem_time, echem_voltage)
spec_curr_ua = np.interp(spec_time, echem_time, echem_current_ua)

# White-line left half maximum energy, aligned to the filtered operando spectra.
white_line_path = Path.joinpath(path_file, r"white_line_left_half_max_energies.csv")
white_line_df = pd.read_csv(white_line_path)
white_line_df = white_line_df[white_line_df["spectrum"].isin(op_labels)].copy()
white_line_df["spectrum"] = pd.Categorical(white_line_df["spectrum"], categories=op_labels, ordered=True)
white_line_df = white_line_df.sort_values("spectrum").reset_index(drop=True)
if white_line_df.shape[0] != len(op_labels):
    missing_wl = [label for label in op_labels if label not in set(white_line_df["spectrum"].astype(str))]
    raise ValueError(f"FigureSI 12B: missing white-line values for labels: {missing_wl[:6]}")
white_line_energy = white_line_df["left_half_max_energy_eV"].to_numpy(dtype=float)


In [ ]:
# 读取当前 Mn residual PCA 数据
path_pca_cell2 = Path.joinpath(path_file, r"lcf_alpha_MnO2_electrode_pristine_residual.pca")
if not path_pca_cell2.exists():
    raise FileNotFoundError(f"Missing required input: {path_pca_cell2}")

# PCA 主成分矩阵：第1列能量，其余为 component 1..N
pca_raw_cell2 = pd.read_csv(path_pca_cell2, sep=r"\s+", index_col=None, header=None, comment="#")

# 从注释行提取 eignevalue/variance/cumulative variance
variance_rows_cell2 = []
with open(path_pca_cell2, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        m = re.match(r"^#\s*(\d+):\s*([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)\s+([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)\s+([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)", line)
        if m:
            variance_rows_cell2.append(
                {
                    "component_id": int(m.group(1)),
                    "eigenvalue": float(m.group(2)),
                    "variance": float(m.group(3)),
                    "cumulative_variance": float(m.group(4)),
                }
            )

pca_variance_cell2 = pd.DataFrame(variance_rows_cell2)



In [ ]:
# 清洗数据
if pca_raw_cell2.shape[1] < 2:
    raise ValueError(f"Unexpected PCA matrix shape: {pca_raw_cell2.shape}")

component_cols_cell2 = [f"component_{i}" for i in range(1, pca_raw_cell2.shape[1])]
pca_components_cell2 = pca_raw_cell2.copy()
pca_components_cell2.columns = ["energy_eV", *component_cols_cell2]
pca_components_cell2 = pca_components_cell2.apply(pd.to_numeric, errors="coerce")
pca_components_cell2 = pca_components_cell2.dropna(subset=["energy_eV"]).reset_index(drop=True)

if pca_variance_cell2.empty:
    raise ValueError(f"No variance metadata parsed from {path_pca_cell2.name}")

pca_variance_cell2 = pca_variance_cell2.sort_values("component_id").reset_index(drop=True)
pca_variance_cell2["log10_variance"] = np.log10(np.clip(pca_variance_cell2["variance"].to_numpy(dtype=float), 1e-30, None))


In [ ]:
# 画图
plt.close("all")
%matplotlib inline

fig = plt.figure(figsize=(7.0, 5.0), dpi=300)
gs = fig.add_gridspec(2, 4, width_ratios=[1.0, 1.0, 1.0, 1.0], height_ratios=[1.0, 1.0], wspace=0.0, hspace=0.0)

# 图 A
ax_echem = fig.add_subplot(gs[:, 0])
pos = ax_echem.get_position()
ax_echem.set_position((pos.x0, pos.y0, pos.width, pos.height))
ax_echem.set_box_aspect(3.0)

y_min = min(float(np.nanmin(echem_time)), float(np.nanmin(spec_time)))
y_max = max(float(np.nanmax(echem_time)), float(np.nanmax(spec_time)))

ax_echem.plot(echem_voltage, echem_time, c=colors[0], ls="-", lw=1.0, zorder=1)

all_idx = np.arange(len(spec_time), dtype=int)
special_mask = np.zeros(len(spec_time), dtype=bool)
if special_idx.size > 0:
    special_mask[special_idx[special_idx < len(spec_time)]] = True
normal_idx = all_idx[(~special_mask) & (all_idx != len(spec_time) - 1)]

if normal_idx.size > 0:
    ax_echem.scatter(spec_volt[normal_idx], spec_time[normal_idx], c="lightgrey", edgecolors="face", s=18, zorder=2)
if special_idx.size > 0:
    sp = special_idx[special_idx < len(spec_time)]
    if sp.size > 0:
        ax_echem.scatter(spec_volt[sp], spec_time[sp], c=colors[0], edgecolors="w", marker="*", s=42, linewidths=0.35, zorder=4)

ax_echem.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax_echem.set_xlim(0.8, 2.0)
ax_echem.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=0.0))
ax_echem.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=0.0))
ax_echem.set_ylabel("Duration time (h)", fontsize=9)
ax_echem.set_ylim(y_min - 0.25, y_max + 5)
ax_echem.yaxis.set_major_locator(ticker.MultipleLocator(base=3.0, offset=0.0))
ax_echem.yaxis.set_minor_locator(ticker.MultipleLocator(base=1.5, offset=0.0))
ax_echem.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)

ax_echem.text(-0.25, 1.0, r"A", transform=ax_echem.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

ax_echem_top = ax_echem.twiny()
pos = ax_echem_top.get_position()
ax_echem_top.set_position((pos.x0, pos.y0, pos.width, pos.height))
ax_echem_top.set_box_aspect(3.0)

ax_echem_top.plot(echem_current_ua, echem_time, c=colors[3], ls="--", lw=1.0, alpha=0.9, zorder=1)
ax_echem_top.set_xlim(-60, 60)
ax_echem_top.xaxis.set_major_locator(ticker.MultipleLocator(base=60.0, offset=0.0))
ax_echem_top.xaxis.set_minor_locator(ticker.MultipleLocator(base=30.0, offset=0.0))
ax_echem_top.set_xlabel(r"Current ($\mu$A)", fontsize=9)
ax_echem_top.set_ylim(y_min - 0.25, y_max + 5)
ax_echem_top.tick_params(axis="x", which="both", labelsize=8)

# 图 B
ax_wl = fig.add_subplot(gs[:, 1], sharey=ax_echem)
pos = ax_wl.get_position()
ax_wl.set_position((pos.x0-0.02, pos.y0, pos.width, pos.height))
ax_wl.set_box_aspect(6.0)

ax_wl.plot(white_line_energy, spec_time, c=colors[2], ls="-", lw=1.0, marker="o", ms=2.8, mfc=colors[2], mec="w", mew=0.25, zorder=3)
if special_idx.size > 0:
    sp = special_idx[special_idx < len(spec_time)]
    if sp.size > 0:
        ax_wl.scatter(white_line_energy[sp], spec_time[sp], c=colors[0], edgecolors="w", marker="*", s=42, linewidths=0.35, zorder=4)

ax_wl.set_xlabel(r"$\mathrm{E_0 \ (eV)}$", fontsize=9)
ax_wl.set_xlim(6553.8, 6555.6)
ax_wl.xaxis.set_major_locator(ticker.MultipleLocator(base=0.8, offset=0.2))
ax_wl.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.4, offset=0.2))
ax_wl.set_ylim(y_min - 0.25, y_max + 5)
ax_wl.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)
ax_wl.tick_params(axis="y", which="both", left=False, labelleft=False)
ax_wl.set_ylabel("")
ax_wl.text(-0.06, 1.0, r"B", transform=ax_wl.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
ax_spec = fig.add_subplot(gs[:, 2], sharey=ax_echem)
pos = ax_spec.get_position()
ax_spec.set_position((pos.x0-0.05, pos.y0, pos.width, pos.height))
# ax_spec.set_box_aspect(3.0)

xas_colors = ListedColormap(
    mpl.colormaps["coolwarm"](np.linspace(0.0, 1.0, n_spec)),
    name="mn_colors",
)

time_sorted = np.sort(np.unique(np.asarray(spec_time, dtype=float)))
time_diff = np.diff(time_sorted)
time_diff = time_diff[time_diff > 0]
time_step = float(np.nanmedian(time_diff)) if time_diff.size > 0 else 0.45
trace_half_height = max(0.12, 0.34 * time_step)

for m in range(n_spec):
    spectrum = mu_mat[m, :]
    if np.all(np.isnan(spectrum)):
        continue

    spec_min = float(np.nanmin(spectrum))
    spec_max = float(np.nanmax(spectrum))
    y_plot = spec_time[m] + (spectrum - spec_min) * 3.0
    ax_spec.plot(energy, y_plot, c=xas_colors.colors[m], ls="-", lw=1.0, zorder=2)

# 特殊索引谱线再覆盖画一遍黑线
for m in special_idx:
    spectrum = mu_mat[m, :]
    if np.all(np.isnan(spectrum)):
        continue

    spec_min = float(np.nanmin(spectrum))
    spec_max = float(np.nanmax(spectrum))
    y_plot = spec_time[m] + (spectrum - spec_min) * 3.0
    ax_spec.plot(energy, y_plot, c="k", ls="-", lw=1.2, zorder=4)

ax_spec.set_xlim(6530.0, 6610.0)
ax_spec.set_xlabel(r"Energy (eV)", fontsize=9)
ax_spec.xaxis.set_major_locator(ticker.MultipleLocator(base=20.0, offset=-10.0))
ax_spec.xaxis.set_minor_locator(ticker.MultipleLocator(base=10.0, offset=-10.0))
ax_spec.set_ylim(y_min - 0.25, y_max + 5)
ax_spec.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)
ax_spec.tick_params(axis="y", which="both", left=False, labelleft=False)
ax_spec.set_ylabel("")
ax_spec.text(-0.04, 1.0, r"C", transform=ax_spec.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
xas_colors_cell2 = ListedColormap(
    mpl.colormaps["coolwarm"](np.linspace(1.0, 0.5, pca_components_cell2.shape[1] - 1)),
    name="xas_colors_cell2",
)
ax_e_pca = fig.add_subplot(gs[0, 3])
pos = ax_e_pca.get_position()
ax_e_pca.set_position((pos.x0+0.05, pos.y0+0.01, pos.width*1.1, pos.height*1.1))
ax_e_pca.set_box_aspect(1.1)

line_handles_cell2 = []
for i, comp_col in enumerate(component_cols_cell2):
    (line_i,) = ax_e_pca.plot(
        pca_components_cell2["energy_eV"],
        pca_components_cell2[comp_col],
        c=xas_colors_cell2.colors[i],
        ls="-",
        lw=1.0,
        zorder=i,
    )
    line_handles_cell2.append(line_i)

ax_e_pca.set_xlim(6530, 6610)
ax_e_pca.set_xlabel(r"Energy (eV)", fontsize=9)
ax_e_pca.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax_e_pca.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax_e_pca.set_ylim(-10, 1)
ax_e_pca.set_ylabel(r"PCA intensity (arb. u.)", fontsize=9)
ax_e_pca.yaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax_e_pca.yaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))
ax_e_pca.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)
ax_e_pca.legend(
    handles=[line_handles_cell2[0], line_handles_cell2[1], line_handles_cell2[2]],
    labels=[r"Component#1", r"Component#2", r"Component#3"],
    loc="upper left",
    bbox_to_anchor=(0.3, 0.8),
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)
ax_e_pca.text(-0.23, 1.0, r"D", transform=ax_e_pca.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 E
ax_f_pca = fig.add_subplot(gs[1, 3])
pos = ax_f_pca.get_position()
ax_f_pca.set_position((pos.x0+0.05, pos.y0-0.06, pos.width*1.1, pos.height*1.1))
ax_f_pca.set_box_aspect(1.1)

ax_f_pca.scatter(
    pca_variance_cell2["component_id"],
    pca_variance_cell2["log10_variance"],
    s=40,
    c=colors[0],
    edgecolors="face",
)
ax_f_pca.set_xlim(-0.5, float(pca_variance_cell2["component_id"].max()) + 1.0)
ax_f_pca.set_xlabel(r"Components", fontsize=9)
ax_f_pca.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax_f_pca.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax_f_pca.set_ylim(-8.0, 0.5)
ax_f_pca.set_ylabel(r"Log variance", fontsize=9)
ax_f_pca.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax_f_pca.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax_f_pca.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)
ax_f_pca.text(-0.23, 1.0, r"E", transform=ax_f_pca.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_11_XAS_XANES_Mn_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_11_XAS_XANES_Mn_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_11_XAS_XANES_Mn_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_11_XAS_XANES_Mn_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 11 -- XAS - Cell2 中第三相的存在

In [ ]:
# 读取数据
path_thirdphase = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\FC1st-FD2nd\Oct2022_cell3_P2b\XANES\Mn\ThirdPhase_LCF")
path_cell2 = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")
path_thirdphase_xlsx = Path.joinpath(path_thirdphase, r"RemoveLiquid3.xlsx")

required_paths = [
    path_thirdphase_xlsx,
    Path.joinpath(path_cell2, r"Overview", r"Time_Index_Spectrum.csv"),
    Path.joinpath(path_cell2, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_1.chir2_mag"),
    Path.joinpath(path_cell2, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_2.chir2_mag"),
    Path.joinpath(path_cell2, r"Echem"),
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required input(s):\n" + "\n".join(missing))

xmu_data_cell2 = pd.read_excel(path_thirdphase_xlsx, sheet_name="RemoveLiquid3_xmu")
xmu_data_cell2 = xmu_data_cell2.rename(columns={"energy_eV": "Energy (eV)"}).set_index("Energy (eV)", drop=True)

energy_cut_raw_cell2 = pd.read_excel(path_thirdphase_xlsx, sheet_name="Energy Cuts")
half_peak_raw_cell2 = pd.read_excel(path_thirdphase_xlsx, sheet_name="HalfPeakEnergy")
time_index_cell2 = pd.read_csv(Path.joinpath(path_cell2, r"Overview", r"Time_Index_Spectrum.csv"))

if "column_id" not in half_peak_raw_cell2.columns:
    op_mask = half_peak_raw_cell2["series_type"].astype(str).eq("operando")
    op_col = half_peak_raw_cell2["column_name"].astype(str)
    op_id = op_col.str.extract(r"Mn(\d+)\.recon", expand=False)
    op_id = pd.to_numeric(op_id, errors="coerce") + 1

    ref_mask = half_peak_raw_cell2["series_type"].astype(str).eq("reference")
    ref_seq = np.arange(1, int(ref_mask.sum()) + 1, dtype=int)

    half_peak_raw_cell2["column_id"] = np.nan
    half_peak_raw_cell2.loc[op_mask, "column_id"] = op_id.loc[op_mask]
    half_peak_raw_cell2.loc[ref_mask, "column_id"] = ref_seq
    half_peak_raw_cell2["column_id"] = pd.to_numeric(half_peak_raw_cell2["column_id"], errors="coerce")

def _read_chir_cols(path_chir):
    cols = {}
    with open(path_chir, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.startswith("# Column."):
                if line.startswith("#------------------------"):
                    break
                continue
            left, right = line.split(":", 1)
            cols[int(left.split(".")[1])] = right.strip().lstrip("-").strip()
    return cols

chir1_cols = _read_chir_cols(Path.joinpath(path_cell2, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_1.chir2_mag"))
chir2_cols = _read_chir_cols(Path.joinpath(path_cell2, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_2.chir2_mag"))


In [ ]:
# 清洗数据
combined_data_cell2 = energy_cut_raw_cell2.rename(columns={"Energy Cuts": "Spectrum Number"}).copy()
combined_data_cell2["Spectrum Number"] = pd.to_numeric(combined_data_cell2["Spectrum Number"], errors="raise").astype(int)
combined_data_cell2 = combined_data_cell2.set_index("Spectrum Number")
combined_data_cell2.columns = [f"{float(col):.2f}" for col in combined_data_cell2.columns]

label_all = [chir1_cols[i] for i in sorted(chir1_cols) if i >= 2] + [chir2_cols[i] for i in sorted(chir2_cols) if i >= 6]
keep_idx = [0, 1, 2, 3, *range(13, len(label_all))]
op_labels_cell2 = [label_all[i] for i in keep_idx][4:]
if len(op_labels_cell2) != combined_data_cell2.shape[0]:
    raise ValueError(f"Label count {len(op_labels_cell2)} does not match spectrum count {combined_data_cell2.shape[0]}")

time_index_cell2 = time_index_cell2[(time_index_cell2["Folder"] == "cell2_P2b_ca") & (time_index_cell2["Element"] == "Mn") & time_index_cell2["NorName"].notna()].copy()
time_index_cell2["Time"] = pd.to_datetime(time_index_cell2["Time"], errors="coerce")
time_index_cell2["NorName"] = time_index_cell2["NorName"].astype(str).str.strip().str.lstrip("-")
time_index_cell2 = time_index_cell2.dropna(subset=["Time"]).sort_values("Time")
name_to_time_cell2 = time_index_cell2.drop_duplicates(subset=["NorName"], keep="first").set_index("NorName")["Time"]
spec_time_dt_cell2 = pd.Series(op_labels_cell2, dtype="string").str.strip().str.lstrip("-").map(name_to_time_cell2)
missing_labels = pd.Series(op_labels_cell2, dtype="string")[spec_time_dt_cell2.isna()].tolist()
if missing_labels:
    raise ValueError(f"Missing NorName->Time mapping for labels: {missing_labels[:6]}")

path_filelist_cell2 = list(Path.joinpath(path_cell2, r"Echem").glob(r"**\*.txt"))
echem_all_cell2 = []
for path_file in path_filelist_cell2:
    with open(path_file, "r", encoding="latin_1") as f:
        for line in f:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break
    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = pd.to_datetime(df["time/s"], format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype("Int64")
    echem_all_cell2.append(df)

echem_all_cell2 = pd.concat(echem_all_cell2, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)
echem_all_cell2 = echem_all_cell2.dropna(subset=["time/s", "Ewe/V", "<I>/mA"]).reset_index(drop=True)
echem_index_low_cell2 = echem_all_cell2[echem_all_cell2["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high_cell2 = echem_all_cell2[echem_all_cell2["cycle number"] == 1]["Ewe/V"].idxmax()
selected_echem_cell2 = echem_all_cell2.iloc[echem_index_low_cell2 : echem_index_high_cell2 + 1].copy().reset_index(drop=True)
selected_echem_cell2["charge_time"] = (selected_echem_cell2["time/s"] - selected_echem_cell2["time/s"].iloc[0]).dt.total_seconds() / 3600.0

spec_time_h_cell2 = ((spec_time_dt_cell2 - echem_all_cell2["time/s"].iloc[echem_index_low_cell2]).dt.total_seconds() / 3600.0).to_numpy(dtype=float)
spec_voltage_cell2 = np.interp(spec_time_h_cell2, selected_echem_cell2["charge_time"].to_numpy(dtype=float), selected_echem_cell2["Ewe/V"].to_numpy(dtype=float))
spec_current_ua_cell2 = np.interp(spec_time_h_cell2, selected_echem_cell2["charge_time"].to_numpy(dtype=float), (selected_echem_cell2["<I>/mA"] * 1000.0).to_numpy(dtype=float))

half_peak_ref_cell2 = half_peak_raw_cell2[half_peak_raw_cell2["series_type"] == "reference"].copy()
half_peak_op_cell2 = half_peak_raw_cell2[(half_peak_raw_cell2["series_type"] == "operando") & (half_peak_raw_cell2["source_file"] == "RemoveLiquid3.xmu")].copy()
half_peak_op_cell2 = half_peak_op_cell2.sort_values("column_id").reset_index(drop=True)
if len(half_peak_op_cell2) != len(spec_time_h_cell2):
    raise ValueError(f"Half-peak table length {len(half_peak_op_cell2)} does not match spectrum count {len(spec_time_h_cell2)}")

half_peak_op_cell2["time_h"] = spec_time_h_cell2
half_peak_op_cell2["voltage_V"] = spec_voltage_cell2
ref_half_peak_cell2 = half_peak_ref_cell2["half_peak_energy_interp_eV"].to_numpy(dtype=float)

plot_df_cell2 = combined_data_cell2.copy()
plot_df_cell2.index = pd.Index(spec_time_h_cell2, name="Time (h)")
plot_df_cell2["Voltage (V)"] = spec_voltage_cell2
plot_df_cell2["Current (uA)"] = spec_current_ua_cell2
plot_df_cell2["Spectrum Number"] = np.arange(1, len(plot_df_cell2) + 1)


In [ ]:
# 读取数据
path_pca_cell2 = Path.joinpath(path_thirdphase, r"RemoveLiquid2.pca")
if not path_pca_cell2.exists():
    raise FileNotFoundError(f"Missing required input: {path_pca_cell2}")

# PCA 主成分矩阵：第1列能量，其余为 component 1..N
pca_raw_cell2 = pd.read_csv(path_pca_cell2, sep=r"\s+", index_col=None, header=None, comment="#")

# 从注释行提取 eignevalue/variance/cumulative variance
variance_rows_cell2 = []
with open(path_pca_cell2, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        m = re.match(r"^#\s*(\d+):\s*([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)\s+([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)\s+([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)", line)
        if m:
            variance_rows_cell2.append(
                {
                    "component_id": int(m.group(1)),
                    "eigenvalue": float(m.group(2)),
                    "variance": float(m.group(3)),
                    "cumulative_variance": float(m.group(4)),
                }
            )

pca_variance_cell2 = pd.DataFrame(variance_rows_cell2)


In [ ]:
# 清洗数据
if pca_raw_cell2.shape[1] < 2:
    raise ValueError(f"Unexpected PCA matrix shape: {pca_raw_cell2.shape}")

component_cols_cell2 = [f"component_{i}" for i in range(1, pca_raw_cell2.shape[1])]
pca_components_cell2 = pca_raw_cell2.copy()
pca_components_cell2.columns = ["energy_eV", *component_cols_cell2]
pca_components_cell2 = pca_components_cell2.apply(pd.to_numeric, errors="coerce")
pca_components_cell2 = pca_components_cell2.dropna(subset=["energy_eV"]).reset_index(drop=True)

if pca_variance_cell2.empty:
    raise ValueError("No variance metadata parsed from RemoveLiquid2.pca")

pca_variance_cell2 = pca_variance_cell2.sort_values("component_id").reset_index(drop=True)
pca_variance_cell2["log10_variance"] = np.log10(np.clip(pca_variance_cell2["variance"].to_numpy(dtype=float), 1e-30, None))


In [ ]:
# 画图
plt.close("all")
%matplotlib inline

fig = plt.figure(figsize=(7.0, 5.0), dpi=300)
gs = fig.add_gridspec(4, 2, width_ratios=[1.0, 1.0], height_ratios=[1.0, 1.0, 1.0, 1.0], wspace=0.0, hspace=0.0)

# 图 A
ax_a = fig.add_subplot(gs[:3, 0])
pos = ax_a.get_position()
ax_a.set_position((pos.x0, pos.y0, pos.width*1.3, pos.height*1.3))
ax_a.set_box_aspect(1.5)
xas_colors = mpl.colormaps["coolwarm"](np.linspace(1.0, 0.0, xmu_data_cell2.shape[1]))
for m in range(xmu_data_cell2.shape[1]):
    ax_a.plot(xmu_data_cell2.index, xmu_data_cell2.iloc[:, m], color=xas_colors[m], linewidth=1.0)
for m, column in enumerate(combined_data_cell2.columns):
    energy_value = float(column)
    ax_a.axvline(energy_value, color=colors[m % len(colors)], lw=1.0, ls="--", alpha=0.5, zorder=6)
    ax_a.text(energy_value, 0.07 + 0.10 * (m % 2), f"{energy_value:.2f} eV", color=colors[m % len(colors)], fontsize=8, rotation=90, ha="center", va="bottom", bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.8), zorder=7)
ax_a.set_xlim(6530, 6630)
ax_a.set_ylim(-0.01, 1.6)
ax_a.set_xlabel("Energy (eV)", fontsize=9)
ax_a.set_ylabel("Absorbance (arb. u.)", fontsize=9)
ax_a.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax_a.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax_a.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=0.0))
ax_a.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=0.0))
ax_a.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)
ax_a.text(-0.15, 1.0, r"A", transform=ax_a.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
stage_break_idx_cell2 = [13, 24]
time_edges_cell2 = np.empty(spec_time_h_cell2.size + 1, dtype=float)
time_edges_cell2[1:-1] = 0.5 * (spec_time_h_cell2[:-1] + spec_time_h_cell2[1:])
time_edges_cell2[0] = spec_time_h_cell2[0] - 0.5 * (spec_time_h_cell2[1] - spec_time_h_cell2[0])
time_edges_cell2[-1] = spec_time_h_cell2[-1] + 0.5 * (spec_time_h_cell2[-1] - spec_time_h_cell2[-2])
stage_spans_cell2 = [
    (time_edges_cell2[0], time_edges_cell2[stage_break_idx_cell2[0]], "Charge", "#FDE6A8"),
    (time_edges_cell2[stage_break_idx_cell2[0]], time_edges_cell2[stage_break_idx_cell2[1]], "Discharge", "#DCEBFF"),
    (time_edges_cell2[stage_break_idx_cell2[1]], time_edges_cell2[-1], "Charge", "#FDE6A8"),
]

ax_b = fig.add_subplot(gs[0, 1])
pos = ax_b.get_position()
ax_b.set_position((pos.x0+0.1, pos.y0+0.23, pos.width, pos.height))
ax_b.set_box_aspect(0.5)

for x0, x1, _, bgc in stage_spans_cell2:
    ax_b.axvspan(x0, x1, facecolor=bgc, alpha=0.35, lw=0, zorder=0)
ax_b.plot(selected_echem_cell2["charge_time"], selected_echem_cell2["Ewe/V"], lw=1.1, color=colors[0], zorder=2)
ax_b.scatter(spec_time_h_cell2, spec_voltage_cell2, c="lightgrey", s=20, edgecolors="none", zorder=3)
ax_b.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9, color=colors[0])
ax_b.tick_params(axis="both", which="both", labelsize=8)
ax_b.set_xlim(time_edges_cell2[0], time_edges_cell2[-1])
ax_b.set_xlabel("Time (h)", fontsize=9)
ax_b.set_xticks([])
ax_b.set_ylim(0.8, 2.0)
ax_b.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=-0.1))
ax_b.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15, offset=-0.1))
sig2_fmt = ticker.FuncFormatter(lambda x, pos: f"{x:.1f}")
ax_b.yaxis.set_major_formatter(sig2_fmt)
ax_b.text(-0.21, 1.0, r"B", transform=ax_b.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

ax_b2 = ax_b.twinx()
pos = ax_b2.get_position()
ax_b2.set_position((pos.x0+0.1, pos.y0+0.22, pos.width, pos.height))
ax_b2.set_box_aspect(0.5)

ax_b2.plot(selected_echem_cell2["charge_time"], selected_echem_cell2["<I>/mA"] * 1000.0, lw=1.0, ls="--", color=colors[1], zorder=1)
ax_b2.set_ylabel("Current (uA)", fontsize=9, color=colors[1])
ax_b2.set_ylim(-100, 100)
ax_b2.tick_params(axis="y", which="both", labelsize=8, colors=colors[1])
ax_b2.yaxis.set_major_locator(ticker.MultipleLocator(base=50, offset=0.0))
ax_b2.yaxis.set_minor_locator(ticker.MultipleLocator(base=25, offset=0.0))
ax_b2.yaxis.set_major_formatter(ticker.StrMethodFormatter("{x:.0f}"))
for x0, x1, stage_name, _ in stage_spans_cell2:
    ax_b.text(0.5 * (x0 + x1), ax_b.get_ylim()[1] - 0.1 * (ax_b.get_ylim()[1] - ax_b.get_ylim()[0]), stage_name, fontsize=9, color="0.25", ha="center", va="bottom")

# 图 C
ax_c = fig.add_subplot(gs[1, 1], sharex=ax_b)
pos = ax_c.get_position()
ax_c.set_position((pos.x0+0.1, pos.y0+0.1, pos.width, pos.height))
ax_c.set_box_aspect(0.5)

for x0, x1, _, bgc in stage_spans_cell2:
    ax_c.axvspan(x0, x1, facecolor=bgc, alpha=0.35, lw=0, zorder=0)
for m, column in enumerate(combined_data_cell2.columns):
    trace = plot_df_cell2[column].to_numpy(dtype=float)
    trace_norm = trace / np.nanmax(trace)
    ax_c.plot(spec_time_h_cell2, trace_norm, color=colors[m % len(colors)], linewidth=2.0, marker="o", markersize=4, label=f"{float(column):.2f} eV", zorder=3)
ax_c.set_xlabel("Time (h)", fontsize=9)
ax_c.set_ylabel("Normalized intensity (arb. u.)", fontsize=9)
ax_c.set_xlim(time_edges_cell2[0], time_edges_cell2[-1])
ax_c.set_ylim(0.6, 1.05)
ax_c.xaxis.set_major_locator(ticker.MultipleLocator(base=3.0, offset=0.0))
ax_c.xaxis.set_minor_locator(ticker.MultipleLocator(base=1.5, offset=0.0))
ax_c.yaxis.set_major_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax_c.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.05, offset=0))
ax_c.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)
ax_c.legend(fontsize=8, frameon=False, loc="lower right", ncol=2, labelcolor="linecolor")
ax_c.text(-0.21, 1.0, r"C", transform=ax_c.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
ax_d = fig.add_subplot(gs[2, 1], sharex=ax_b)
pos = ax_d.get_position()
ax_d.set_position((pos.x0+0.1, pos.y0, pos.width, pos.height))
ax_d.set_box_aspect(0.5)

for x0, x1, _, bgc in stage_spans_cell2:
    ax_d.axvspan(x0, x1, facecolor=bgc, alpha=0.35, lw=0, zorder=0)
for idx, ref_value in enumerate(ref_half_peak_cell2[2:]):
    if np.isfinite(ref_value):
        ax_d.axhline(ref_value, color="0.35", lw=0.9, ls="--", alpha=0.6, zorder=1, label="Reference" if idx == 0 else None)
half_peak_y_cell2 = half_peak_op_cell2["half_peak_energy_interp_eV"].to_numpy(dtype=float)
ax_d.plot(spec_time_h_cell2, half_peak_y_cell2, color=colors[4], linewidth=1.8, marker="o", markersize=4, label=r"$\mathrm{E}_{0}$", zorder=3)
ax_d.set_xlabel("Time (h)", fontsize=9)
ax_d.set_ylabel("Energy (eV)", fontsize=9)
ax_d.set_xlim(time_edges_cell2[0], time_edges_cell2[-1])
ax_d.xaxis.set_major_locator(ticker.MultipleLocator(base=3.0, offset=0.0))
ax_d.xaxis.set_minor_locator(ticker.MultipleLocator(base=1.5, offset=0.0))
ax_d.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)
ymin = float(np.nanmin(half_peak_y_cell2))
ymax = float(np.nanmax(half_peak_y_cell2))
ypad = max((ymax - ymin) * 0.15, 0.03)
ax_d.set_ylim(6553.5, 6555.5)
ax_d.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=0))
ax_d.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=0))
formatter = ticker.ScalarFormatter(useOffset=6550)
ax_d.yaxis.set_major_formatter(formatter)
ax_d.legend(fontsize=8, frameon=False, loc="best")
ax_d.text(-0.21, 1.0, r"D", transform=ax_d.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 E
xas_colors_cell2 = ListedColormap(
    mpl.colormaps["sunset"](np.linspace(1.0, 0.0, pca_components_cell2.shape[1] - 1)),
    name="xas_colors_cell2",
)
ax_e_pca = fig.add_subplot(gs[3, 0])
pos = ax_e_pca.get_position()
ax_e_pca.set_position([pos.x0+0.04, pos.y0-0.1, pos.width, pos.height])
ax_e_pca.set_box_aspect(0.45)

line_handles_cell2 = []
for i, comp_col in enumerate(component_cols_cell2):
    (line_i,) = ax_e_pca.plot(
        pca_components_cell2["energy_eV"],
        pca_components_cell2[comp_col],
        c=xas_colors_cell2.colors[i],
        ls="-",
        lw=1.0,
        zorder=i,
    )
    line_handles_cell2.append(line_i)

ax_e_pca.set_xlim(6530, 6610)
ax_e_pca.set_xlabel(r"Energy (eV)", fontsize=9)
ax_e_pca.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax_e_pca.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax_e_pca.set_ylim(-12, 1)
ax_e_pca.set_ylabel(r"PCA intensity (arb. u.)", fontsize=9)
ax_e_pca.yaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=1))
ax_e_pca.yaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=1))
ax_e_pca.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)
ax_e_pca.legend(
    handles=[line_handles_cell2[0], line_handles_cell2[1]],
    labels=[r"Component#1", r"Component#2"],
    loc="upper left",
    bbox_to_anchor=(0.45, 0.4),
    frameon=False,
    labelcolor="linecolor",
    fontsize=10,
)
ax_e_pca.text(-0.18, 1.0, r"E", transform=ax_e_pca.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 F
ax_f_pca = fig.add_subplot(gs[3, 1])
pos = ax_f_pca.get_position()
ax_f_pca.set_position([pos.x0+0.08, pos.y0-0.1, pos.width, pos.height])
ax_f_pca.set_box_aspect(0.45)

ax_f_pca.scatter(
    pca_variance_cell2["component_id"],
    pca_variance_cell2["log10_variance"],
    s=40,
    c=colors[0],
    edgecolors="face",
)
ax_f_pca.set_xlim(-0.5, float(pca_variance_cell2["component_id"].max()) + 1.0)
ax_f_pca.set_xlabel(r"Components", fontsize=9)
ax_f_pca.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax_f_pca.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax_f_pca.set_ylim(-8.0, 0.5)
ax_f_pca.set_ylabel(r"Log variance", fontsize=9)
ax_f_pca.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax_f_pca.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax_f_pca.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)
ax_f_pca.text(-0.18, 1.0, r"F", transform=ax_f_pca.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_11_XAS_XANES_00_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_11_XAS_XANES_00_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_11_XAS_XANES_00_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_11_XAS_XANES_00_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 12 -- XAS - Operando XAS, Zn, WhiteLine Intensity - V2

In [ ]:
# 读取数据

path_cell2 = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")
path_file = Path.joinpath(path_cell2, r"FC1st-FD2nd\Oct2022_cell2_P2b\XANES\Zn\PCA_Larch_Zn")
xas_Zn_path = Path.joinpath(path_file, r"lcf_ZHS_residual.nor")

# 电化学时间序列
path_filelist = list(Path.joinpath(path_cell2, r"Echem").glob(r"**\*.txt"))
echem_all = []
for path_file_echem in path_filelist:
    with open(path_file_echem, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break

    df = pd.read_csv(path_file_echem, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem_all.append(df)

echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 当前对齐后的 Zn operando 数据
xas_Zn = pd.read_csv(
    xas_Zn_path,
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)

charge_index = [0, 1, 2, *list(range(12, xas_Zn.shape[1], 1))]
xas_Zn = xas_Zn.iloc[:, charge_index]  # noqa: N816

pdf_Zn = xas_Zn.iloc[:, [0, 1, 2]]  # noqa: N816
pdf_Zn.columns = [r"Energy", r"0.5M_ZnSO4(aq.)", r"ZHS"]
opData_Zn = xas_Zn.iloc[:, [0, *list(range(3, xas_Zn.shape[1]))]]  # noqa: N816
opData_Zn.columns = list(range(0, opData_Zn.shape[1], 1))


In [ ]:
# 聚焦当前对齐后 Zn 数据
opData_Zn = opData_Zn[opData_Zn.iloc[:, 0].between(9640.0, 9740.0)].copy()

# 保留原来标记为异常的两条谱线
bad_spec_cols = [11, 36, 37, 38]
bad_spec_cols = [idx for idx in bad_spec_cols if idx < opData_Zn.shape[1]]
if bad_spec_cols:
    opData_Zn.iloc[:, bad_spec_cols] = np.nan


In [ ]:
n_spec = opData_Zn.shape[1] - 1
special_idx = np.array([0, 4, 9, 12, 14, 17, 24, 30, 34, 38], dtype=int)
special_idx = special_idx[(special_idx >= 0) & (special_idx < n_spec)]

energy = opData_Zn.iloc[:, 0].to_numpy(dtype=float)
mu_mat = opData_Zn.iloc[:, 1:].to_numpy(dtype=float).T  # (n_spec, n_energy)

# 电化学窗口：首圈放电谷值到次圈充电峰值
echem_index_low = echem_all[echem_all["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high = echem_all[echem_all["cycle number"] == 1]["Ewe/V"].idxmax()
echem_seg = echem_all.iloc[int(echem_index_low) : int(echem_index_high) + 1].copy()
echem_seg = echem_seg.dropna(subset=["time/s", "Ewe/V", "<I>/mA"]).sort_values("time/s").reset_index(drop=True)

# 直接从当前 LCF .nor 表头读取列名，用于谱线到采谱时间的映射
def _read_nor_cols(path_nor):
    cols = {}
    with open(path_nor, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.startswith("# Column."):
                if line.startswith("#------------------------"):
                    break
                continue
            left, right = line.split(":", 1)
            cols[int(left.split(".")[1])] = right.strip().lstrip("-").strip()
    return cols

cols = _read_nor_cols(xas_Zn_path)
op_source_cols = charge_index[3:]
op_labels = [cols[i + 1] for i in op_source_cols if i + 1 in cols and re.search(r"P2b_(\d+)", cols[i + 1])]

if len(op_labels) != n_spec:
    raise ValueError(f"FigureSI 12A: op_labels={len(op_labels)} != n_spec={n_spec}")

map_df = pd.read_csv(Path.joinpath(path_cell2, r"Overview", r"Time_Index_Spectrum.csv"))
map_df = map_df[
    (map_df["Folder"].astype(str).str.contains("cell2_P2b", case=False, na=False))
    & (map_df["Element"].astype(str).str.upper() == "ZN")
    & map_df["NorName"].notna()
].copy()
map_df["Time"] = pd.to_datetime(map_df["Time"], errors="coerce")
map_df["NorName"] = map_df["NorName"].astype(str).str.strip().str.lstrip("-")
map_df = map_df.dropna(subset=["Time"]).sort_values("Time")
name_to_time = map_df.drop_duplicates(subset=["NorName"], keep="first").set_index("NorName")["Time"]

label_s = pd.Series(op_labels, dtype="string").str.strip().str.lstrip("-")
spec_time_dt = label_s.map(name_to_time)
missing_labels = label_s[spec_time_dt.isna()].tolist()
if missing_labels:
    raise ValueError(f"FigureSI 12A: missing NorName->Time mapping for labels: {missing_labels[:6]}")

# 只保留落在电化学窗口内的谱线，避免 np.interp 对窗口外谱线使用边界值
echem_start = echem_seg["time/s"].iloc[0]
echem_end = echem_seg["time/s"].iloc[-1]
in_echem_window = spec_time_dt.between(echem_start, echem_end).to_numpy(dtype=bool)
if not np.all(in_echem_window):
    op_labels = [label for label, keep in zip(op_labels, in_echem_window) if keep]
    spec_time_dt = spec_time_dt[in_echem_window].reset_index(drop=True)
    mu_mat = mu_mat[in_echem_window, :]
    old_special_idx = special_idx.copy()
    kept_old_idx = np.flatnonzero(in_echem_window)
    special_idx = np.array([np.where(kept_old_idx == idx)[0][0] for idx in old_special_idx if idx in kept_old_idx], dtype=int)
    n_spec = mu_mat.shape[0]
special_idx = np.unique(np.r_[special_idx, n_spec - 1]).astype(int)

# 统一时间零点，并把采谱点映射到电化学轨迹上
time_zero = min(echem_seg["time/s"].iloc[0], spec_time_dt.min())
echem_time = (echem_seg["time/s"] - time_zero).dt.total_seconds().to_numpy(dtype=float) / 3600.0
echem_voltage = echem_seg["Ewe/V"].to_numpy(dtype=float)
echem_current_ua = echem_seg["<I>/mA"].to_numpy(dtype=float) * 1000.0

spec_time_raw = (spec_time_dt - time_zero).dt.total_seconds().to_numpy(dtype=float) / 3600.0
spec_time = spec_time_raw.copy()
spec_volt = np.interp(spec_time, echem_time, echem_voltage)
spec_curr_ua = np.interp(spec_time, echem_time, echem_current_ua)


In [ ]:
# 读取当前 Zn residual PCA 数据
path_pca_cell2 = Path.joinpath(path_file, r"lcf_ZHS_residual.pca")
if not path_pca_cell2.exists():
    raise FileNotFoundError(f"Missing required input: {path_pca_cell2}")

# PCA 主成分矩阵：第1列能量，其余为 component 1..N
pca_raw_cell2 = pd.read_csv(path_pca_cell2, sep=r"\s+", index_col=None, header=None, comment="#")

# 从注释行提取 eignevalue/variance/cumulative variance
variance_rows_cell2 = []
with open(path_pca_cell2, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        m = re.match(r"^#\s*(\d+):\s*([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)\s+([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)\s+([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)", line)
        if m:
            variance_rows_cell2.append(
                {
                    "component_id": int(m.group(1)),
                    "eigenvalue": float(m.group(2)),
                    "variance": float(m.group(3)),
                    "cumulative_variance": float(m.group(4)),
                }
            )

pca_variance_cell2 = pd.DataFrame(variance_rows_cell2)


In [ ]:
# 清洗数据
if pca_raw_cell2.shape[1] < 2:
    raise ValueError(f"Unexpected PCA matrix shape: {pca_raw_cell2.shape}")

component_cols_cell2 = [f"component_{i}" for i in range(1, pca_raw_cell2.shape[1])]
pca_components_cell2 = pca_raw_cell2.copy()
pca_components_cell2.columns = ["energy_eV", *component_cols_cell2]
pca_components_cell2 = pca_components_cell2.apply(pd.to_numeric, errors="coerce")
pca_components_cell2 = pca_components_cell2.dropna(subset=["energy_eV"]).reset_index(drop=True)

if pca_variance_cell2.empty:
    raise ValueError(f"No variance metadata parsed from {path_pca_cell2.name}")

pca_variance_cell2 = pca_variance_cell2.sort_values("component_id").reset_index(drop=True)
pca_variance_cell2["log10_variance"] = np.log10(np.clip(pca_variance_cell2["variance"].to_numpy(dtype=float), 1e-30, None))


In [ ]:
# 画图
plt.close("all")
%matplotlib inline

fig = plt.figure(figsize=(7.0, 5.0), dpi=300)
gs = fig.add_gridspec(2, 3, width_ratios=[1.0, 1.0, 1.0], height_ratios=[1.0, 1.0], wspace=0.0, hspace=0.0)

# 图 A
ax_echem = fig.add_subplot(gs[:, 0])
pos = ax_echem.get_position()
ax_echem.set_position((pos.x0, pos.y0, pos.width, pos.height))
ax_echem.set_box_aspect(3.0)

y_min = min(float(np.nanmin(echem_time)), float(np.nanmin(spec_time)))
y_max = max(float(np.nanmax(echem_time)), float(np.nanmax(spec_time)))

ax_echem.plot(echem_voltage, echem_time, c=colors[0], ls="-", lw=1.0, zorder=1)

all_idx = np.arange(len(spec_time), dtype=int)
special_mask = np.zeros(len(spec_time), dtype=bool)
if special_idx.size > 0:
    special_mask[special_idx[special_idx < len(spec_time)]] = True
normal_idx = all_idx[(~special_mask) & (all_idx != len(spec_time) - 1)]

if normal_idx.size > 0:
    ax_echem.scatter(spec_volt[normal_idx], spec_time[normal_idx], c="lightgrey", edgecolors="face", s=18, zorder=2)
if special_idx.size > 0:
    sp = special_idx[special_idx < len(spec_time)]
    if sp.size > 0:
        ax_echem.scatter(spec_volt[sp], spec_time[sp], c=colors[0], edgecolors="w", marker="*", s=42, linewidths=0.35, zorder=4)

ax_echem.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax_echem.set_xlim(0.8, 2.0)
ax_echem.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=0.0))
ax_echem.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=0.0))
ax_echem.set_ylabel("Duration time (h)", fontsize=9)
ax_echem.set_ylim(y_min - 0.25, y_max + 1)
ax_echem.yaxis.set_major_locator(ticker.MultipleLocator(base=3.0, offset=0.0))
ax_echem.yaxis.set_minor_locator(ticker.MultipleLocator(base=1.5, offset=0.0))
ax_echem.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)

ax_echem.text(-0.25, 1.0, r"A", transform=ax_echem.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

ax_echem_top = ax_echem.twiny()
pos = ax_echem_top.get_position()
ax_echem_top.set_position((pos.x0, pos.y0, pos.width, pos.height))
ax_echem_top.set_box_aspect(3.0)

ax_echem_top.plot(echem_current_ua, echem_time, c=colors[3], ls="--", lw=1.0, alpha=0.9, zorder=1)
ax_echem_top.set_xlim(-60, 60)
ax_echem_top.xaxis.set_major_locator(ticker.MultipleLocator(base=60.0, offset=0.0))
ax_echem_top.xaxis.set_minor_locator(ticker.MultipleLocator(base=30.0, offset=0.0))
ax_echem_top.set_xlabel(r"Current ($\mu$A)", fontsize=9)
ax_echem_top.set_ylim(y_min - 0.25, y_max + 1)
ax_echem_top.tick_params(axis="x", which="both", labelsize=8)

# 图 B
ax_spec = fig.add_subplot(gs[:, 1], sharey=ax_echem)
pos = ax_spec.get_position()
ax_spec.set_position((pos.x0, pos.y0, pos.width, pos.height))
# ax_spec.set_box_aspect(3.0)

xas_colors = ListedColormap(
    mpl.colormaps["coolwarm"](np.linspace(0.0, 1.0, n_spec)),
    name="zn_colors",
)

time_sorted = np.sort(np.unique(np.asarray(spec_time, dtype=float)))
time_diff = np.diff(time_sorted)
time_diff = time_diff[time_diff > 0]
time_step = float(np.nanmedian(time_diff)) if time_diff.size > 0 else 0.45
trace_half_height = max(0.12, 0.34 * time_step)

for m in range(n_spec):
    spectrum = mu_mat[m, :]
    if np.all(np.isnan(spectrum)):
        continue

    spec_min = float(np.nanmin(spectrum))
    spec_max = float(np.nanmax(spectrum))
    y_plot = spec_time[m] - spec_min + spectrum
    ax_spec.plot(energy, y_plot, c=xas_colors.colors[m], ls="-", lw=1.0, zorder=2)

# 特殊索引谱线再覆盖画一遍黑线
for m in special_idx:
    spectrum = mu_mat[m, :]
    if np.all(np.isnan(spectrum)):
        continue

    spec_min = float(np.nanmin(spectrum))
    spec_max = float(np.nanmax(spectrum))
    y_plot = spec_time[m] - spec_min + spectrum
    ax_spec.plot(energy, y_plot, c="k", ls="-", lw=1.2, zorder=4)

ax_spec.set_xlim(9640.0, 9740.0)
ax_spec.set_xlabel(r"Energy (eV)", fontsize=9)
ax_spec.xaxis.set_major_locator(ticker.MultipleLocator(base=20.0, offset=0.0))
ax_spec.xaxis.set_minor_locator(ticker.MultipleLocator(base=10.0, offset=0.0))
ax_spec.set_ylim(y_min - 0.25, y_max + 1)
ax_spec.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)
ax_spec.tick_params(axis="y", which="both", left=False, labelleft=False)
ax_spec.set_ylabel("")
ax_spec.text(-0.04, 1.0, r"B", transform=ax_spec.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
xas_colors_cell2 = ListedColormap(
    mpl.colormaps["coolwarm"](np.linspace(1.0, 0.5, pca_components_cell2.shape[1] - 1)),
    name="xas_colors_cell2",
)
ax_e_pca = fig.add_subplot(gs[0, 2])
pos = ax_e_pca.get_position()
ax_e_pca.set_position((pos.x0+0.1, pos.y0+0.02, pos.width*1.1, pos.height*1.1))
ax_e_pca.set_box_aspect(0.8)

line_handles_cell2 = []
for i, comp_col in enumerate(component_cols_cell2):
    (line_i,) = ax_e_pca.plot(
        pca_components_cell2["energy_eV"],
        pca_components_cell2[comp_col],
        c=xas_colors_cell2.colors[i],
        ls="-",
        lw=1.0,
        zorder=i,
    )
    line_handles_cell2.append(line_i)

ax_e_pca.set_xlim(9640, 9710)
ax_e_pca.set_xlabel(r"Energy (eV)", fontsize=9)
ax_e_pca.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=-10))
ax_e_pca.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=-10))
ax_e_pca.set_ylim(-13, 7)
ax_e_pca.set_ylabel(r"PCA intensity (arb. u.)", fontsize=9)
ax_e_pca.yaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=2))
ax_e_pca.yaxis.set_minor_locator(ticker.MultipleLocator(base=2.5, offset=2))
ax_e_pca.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)
ax_e_pca.legend(
    handles=[line_handles_cell2[0], line_handles_cell2[1], line_handles_cell2[2]],
    labels=[r"Component#1", r"Component#2", r"Component#3"],
    loc="upper left",
    bbox_to_anchor=(0.5, 0.3),
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
)
ax_e_pca.text(-0.18, 1.0, r"C", transform=ax_e_pca.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
ax_f_pca = fig.add_subplot(gs[1, 2])
pos = ax_f_pca.get_position()
ax_f_pca.set_position((pos.x0+0.1, pos.y0-0.07, pos.width*1.1, pos.height*1.1))
ax_f_pca.set_box_aspect(0.8)

ax_f_pca.scatter(
    pca_variance_cell2["component_id"],
    pca_variance_cell2["log10_variance"],
    s=40,
    c=colors[0],
    edgecolors="face",
)
ax_f_pca.set_xlim(-0.5, float(pca_variance_cell2["component_id"].max()) + 1.0)
ax_f_pca.set_xlabel(r"Components", fontsize=9)
ax_f_pca.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax_f_pca.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax_f_pca.set_ylim(-8.0, 0.5)
ax_f_pca.set_ylabel(r"Log variance", fontsize=9)
ax_f_pca.yaxis.set_major_locator(ticker.MultipleLocator(base=2))
ax_f_pca.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax_f_pca.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)
ax_f_pca.text(-0.18, 1.0, r"D", transform=ax_f_pca.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_12_XAS_XANES_10_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_12_XAS_XANES_10_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_12_XAS_XANES_10_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_12_XAS_XANES_10_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 12 -- XAS - Operando XAS, Zn, WhiteLine Intensity

In [ ]:
# 读取数据

path_cell2 = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")
path_file = Path.joinpath(path_cell2, r"FC1st-FD2nd\Oct2022_cell3_P2b\XANES\Zn\PCA_Larch")

# 电化学时间序列
path_filelist = list(Path.joinpath(path_cell2, r"Echem").glob(r"**\*.txt"))
echem_all = []
for path_file_echem in path_filelist:
    with open(path_file_echem, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break

    df = pd.read_csv(path_file_echem, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem_all.append(df)

echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 当前对齐后的 Zn operando 数据
xas_Zn = pd.read_csv(
    Path.joinpath(path_file, r"Normalized_zhs_component_with_e0_zhs_aligned.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)

charge_index = [0, 2, 3, *list(range(13, xas_Zn.shape[1], 1))]
xas_Zn = xas_Zn.iloc[:, charge_index]  # noqa: N816

pdf_Zn = xas_Zn.iloc[:, [0, 2, 3]]  # noqa: N816
pdf_Zn.columns = [r"Energy", r"0.5M_ZnSO4(aq.)", r"ZHS"]
opData_Zn = xas_Zn.iloc[:, [0, *list(range(3, xas_Zn.shape[1]))]]  # noqa: N816
opData_Zn.columns = list(range(0, opData_Zn.shape[1], 1))


In [ ]:
# 聚焦当前对齐后 Zn 数据
opData_Zn = opData_Zn[opData_Zn.iloc[:, 0].between(9640.0, 9740.0)].copy()

# 保留原来标记为异常的两条谱线
bad_spec_cols = [11, 36, 39, 40]
bad_spec_cols = [idx for idx in bad_spec_cols if idx < opData_Zn.shape[1]]
if bad_spec_cols:
    opData_Zn.iloc[:, bad_spec_cols] = np.nan


In [ ]:
n_spec = opData_Zn.shape[1] - 1
special_idx = np.array([0, 4, 9, 12, 14, 17, 24, 30, 34], dtype=int)
special_idx = special_idx[(special_idx >= 0) & (special_idx < n_spec)]

energy = opData_Zn.iloc[:, 0].to_numpy(dtype=float)
mu_mat = opData_Zn.iloc[:, 1:].to_numpy(dtype=float).T  # (n_spec, n_energy)

# 电化学窗口：首圈放电谷值到次圈充电峰值
echem_index_low = echem_all[echem_all["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high = echem_all[echem_all["cycle number"] == 1]["Ewe/V"].idxmax()
echem_seg = echem_all.iloc[int(echem_index_low) : int(echem_index_high) + 1].copy()
echem_seg = echem_seg.dropna(subset=["time/s", "Ewe/V", "<I>/mA"]).sort_values("time/s").reset_index(drop=True)

# operando Zn 列名恢复，用于谱线到采谱时间的映射
def _read_nor_cols(path_nor):
    cols = {}
    with open(path_nor, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.startswith("# Column."):
                if line.startswith("#------------------------"):
                    break
                continue
            left, right = line.split(":", 1)
            cols[int(left.split(".")[1])] = right.strip().lstrip("-").strip()
    return cols

nor1 = Path.joinpath(path_cell2, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_1.nor")
nor2 = Path.joinpath(path_cell2, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_2.nor")
cols_1 = _read_nor_cols(nor1)
cols_2 = _read_nor_cols(nor2)
cols1_vals = [cols_1[i] for i in sorted(cols_1)]
cols2_vals = [cols_2[i] for i in sorted(cols_2)]
all_cols = [*cols1_vals, *cols2_vals]
charge_index_overview = [0, 1, 2, *list(range(12, len(all_cols), 1))]
selected_cols = [all_cols[i] for i in charge_index_overview if i < len(all_cols)]
op_labels = [str(x).strip().lstrip("-") for x in selected_cols if re.search(r"P2b_(\d+)", str(x))]

if len(op_labels) != n_spec:
    raise ValueError(f"FigureSI 12A: op_labels={len(op_labels)} != n_spec={n_spec}")

map_df = pd.read_csv(Path.joinpath(path_cell2, r"Overview", r"Time_Index_Spectrum.csv"))
map_df = map_df[
    (map_df["Folder"].astype(str).str.contains("cell2_P2b", case=False, na=False))
    & (map_df["Element"].astype(str).str.upper() == "ZN")
    & map_df["NorName"].notna()
].copy()
map_df["Time"] = pd.to_datetime(map_df["Time"], errors="coerce")
map_df["NorName"] = map_df["NorName"].astype(str).str.strip().str.lstrip("-")
map_df = map_df.dropna(subset=["Time"]).sort_values("Time")
name_to_time = map_df.drop_duplicates(subset=["NorName"], keep="first").set_index("NorName")["Time"]

label_s = pd.Series(op_labels, dtype="string").str.strip().str.lstrip("-")
spec_time_dt = label_s.map(name_to_time)
missing_labels = label_s[spec_time_dt.isna()].tolist()
if missing_labels:
    raise ValueError(f"FigureSI 12A: missing NorName->Time mapping for labels: {missing_labels[:6]}")

# 统一时间零点，并把采谱点映射到电化学轨迹上
time_zero = min(echem_seg["time/s"].iloc[0], spec_time_dt.min())
echem_time = (echem_seg["time/s"] - time_zero).dt.total_seconds().to_numpy(dtype=float) / 3600.0
echem_voltage = echem_seg["Ewe/V"].to_numpy(dtype=float)
echem_current_ua = echem_seg["<I>/mA"].to_numpy(dtype=float) * 1000.0

spec_time_raw = (spec_time_dt - time_zero).dt.total_seconds().to_numpy(dtype=float) / 3600.0
spec_time = spec_time_raw.copy()
spec_volt = np.interp(spec_time, echem_time, echem_voltage)
spec_curr_ua = np.interp(spec_time, echem_time, echem_current_ua)


In [ ]:
# 画图
plt.close("all")
%matplotlib inline

fig = plt.figure(figsize=(7.0, 5.0), dpi=300)
gs = fig.add_gridspec(4, 3, width_ratios=[1.0, 1.0, 1.0], height_ratios=[1.0, 1.0, 1.0, 1.0], wspace=0.0, hspace=0.0)

# 图 A
ax_echem = fig.add_subplot(gs[:, 0])
pos = ax_echem.get_position()
ax_echem.set_position((pos.x0, pos.y0, pos.width, pos.height))
ax_echem.set_box_aspect(3.0)

y_min = min(float(np.nanmin(echem_time)), float(np.nanmin(spec_time)))
y_max = max(float(np.nanmax(echem_time)), float(np.nanmax(spec_time)))

ax_echem.plot(echem_voltage, echem_time, c=colors[0], ls="-", lw=1.0, zorder=1)

all_idx = np.arange(len(spec_time), dtype=int)
special_mask = np.zeros(len(spec_time), dtype=bool)
if special_idx.size > 0:
    special_mask[special_idx[special_idx < len(spec_time)]] = True
normal_idx = all_idx[(~special_mask) & (all_idx != len(spec_time) - 1)]

if normal_idx.size > 0:
    ax_echem.scatter(spec_volt[normal_idx], spec_time[normal_idx], c="lightgrey", edgecolors="face", s=18, zorder=2)
if special_idx.size > 0:
    sp = special_idx[special_idx < len(spec_time)]
    if sp.size > 0:
        ax_echem.scatter(spec_volt[sp], spec_time[sp], c=colors[0], edgecolors="w", marker="*", s=42, linewidths=0.35, zorder=4)

ax_echem.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax_echem.set_xlim(0.8, 2.0)
ax_echem.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=0.0))
ax_echem.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=0.0))
ax_echem.set_ylabel("Duration time (h)", fontsize=9)
ax_echem.set_ylim(y_min - 0.25, y_max + 2.5)
ax_echem.yaxis.set_major_locator(ticker.MultipleLocator(base=3.0, offset=0.0))
ax_echem.yaxis.set_minor_locator(ticker.MultipleLocator(base=1.5, offset=0.0))
ax_echem.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)

ax_echem.text(-0.25, 1.0, r"A", transform=ax_echem.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

ax_echem_top = ax_echem.twiny()
pos = ax_echem_top.get_position()
ax_echem_top.set_position((pos.x0, pos.y0, pos.width, pos.height))
ax_echem_top.set_box_aspect(3.0)

ax_echem_top.plot(echem_current_ua, echem_time, c=colors[3], ls="--", lw=1.0, alpha=0.9, zorder=1)
ax_echem_top.set_xlim(-60, 60)
ax_echem_top.xaxis.set_major_locator(ticker.MultipleLocator(base=60.0, offset=0.0))
ax_echem_top.xaxis.set_minor_locator(ticker.MultipleLocator(base=30.0, offset=0.0))
ax_echem_top.set_xlabel(r"Current ($\mu$A)", fontsize=9)
ax_echem_top.set_ylim(y_min - 0.25, y_max + 2.5)
ax_echem_top.tick_params(axis="x", which="both", labelsize=8)

# 图 B
ax_spec = fig.add_subplot(gs[:, 1], sharey=ax_echem)
pos = ax_spec.get_position()
ax_spec.set_position((pos.x0, pos.y0, pos.width, pos.height))
# ax_spec.set_box_aspect(3.0)

xas_colors = ListedColormap(
    mpl.colormaps["coolwarm"](np.linspace(0.0, 1.0, n_spec)),
    name="zn_colors",
)

time_sorted = np.sort(np.unique(np.asarray(spec_time, dtype=float)))
time_diff = np.diff(time_sorted)
time_diff = time_diff[time_diff > 0]
time_step = float(np.nanmedian(time_diff)) if time_diff.size > 0 else 0.45
trace_half_height = max(0.12, 0.34 * time_step)

for m in range(n_spec):
    spectrum = mu_mat[m, :]
    if np.all(np.isnan(spectrum)):
        continue

    spec_min = float(np.nanmin(spectrum))
    spec_max = float(np.nanmax(spectrum))
    y_plot = spec_time[m] - spec_min + spectrum
    ax_spec.plot(energy, y_plot, c=xas_colors.colors[m], ls="-", lw=1.0, zorder=2)

# 特殊索引谱线再覆盖画一遍黑线
for m in special_idx:
    spectrum = mu_mat[m, :]
    if np.all(np.isnan(spectrum)):
        continue

    spec_min = float(np.nanmin(spectrum))
    spec_max = float(np.nanmax(spectrum))
    y_plot = spec_time[m] - spec_min + spectrum
    ax_spec.plot(energy, y_plot, c="k", ls="-", lw=1.2, zorder=4)

ax_spec.set_xlim(9640.0, 9740.0)
ax_spec.set_xlabel(r"Energy (eV)", fontsize=9)
ax_spec.xaxis.set_major_locator(ticker.MultipleLocator(base=20.0, offset=0.0))
ax_spec.xaxis.set_minor_locator(ticker.MultipleLocator(base=10.0, offset=0.0))
ax_spec.set_ylim(y_min - 0.25, y_max + 2.5)
ax_spec.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)
ax_spec.tick_params(axis="y", which="both", left=False, labelleft=False)
ax_spec.set_ylabel("")
ax_spec.text(-0.04, 1.0, r"B", transform=ax_spec.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_12_XAS_XANES_10_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_12_XAS_XANES_10_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_12_XAS_XANES_10_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_12_XAS_XANES_10_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 12 -- XAS - Operando XAS, Zn, WhiteLine Intensity - V0

In [ ]:
# 读取数据

# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\Echem").glob(r"**\*.txt"))
# 读取电化学数据
echem_all = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem_all.append(df)

# 合并所有电化学数据为一个二维表格
echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 读取 reference + operando 数据
path_file = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")

# Zn
data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
xas_Zn = pd.concat([data1, data2.iloc[:, 3:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, 2, *list(range(12, xas_Zn.shape[1], 1))]
xas_Zn = xas_Zn.iloc[:, charge_index]  # noqa: N816

pdf_Zn = xas_Zn.iloc[:, 0:3]  # noqa: N816
pdf_Zn.columns = [r"Energy", r"0.5M_ZnSO4(aq.)", r"ZHS"]
opData_Zn = xas_Zn.iloc[:, [0, *list(range(3, xas_Zn.shape[1]))]]  # noqa: N816
opData_Zn.columns = list(range(0, opData_Zn.shape[1], 1))


In [ ]:
n_spec = opData_Zn.shape[1] - 1
special_idx = np.array([0, 4, 9, 12, 14, 17, 24, 30, 34, 38], dtype=int)
special_idx = special_idx[(special_idx >= 0) & (special_idx < n_spec)]

energy = opData_Zn.iloc[:, 0].to_numpy(dtype=float)
mu_mat = opData_Zn.iloc[:, 1:].to_numpy(dtype=float).T  # (n_spec, n_energy)


In [ ]:
# 画图
plt.close("all")
%matplotlib inline

fig = plt.figure(figsize=(3.0, 2.5), dpi=300)
gs = fig.add_gridspec(1, 1, width_ratios=None, wspace=None, hspace=0.0)

# 图 
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

# 在 white-line 能区内提取每条谱线的峰强度与峰位
wl_mask = (energy >= 9662.0) & (energy <= 9676.0)
if not np.any(wl_mask):
    raise ValueError("Panel C: white-line energy window is empty")

energy_wl = energy[wl_mask]
mu_wl = mu_mat[:, wl_mask]
peak_pos = np.argmax(mu_wl, axis=1)
wl_energy = energy_wl[peak_pos]
wl_intensity = mu_wl[np.arange(mu_wl.shape[0]), peak_pos]

state_idx = np.arange(1, n_spec + 1)
wl_intensity_norm = wl_intensity / (wl_intensity[0] + 1e-12)

# 基于同批数据的谱线序号分段：1-13 / 14-24 / 25-end
stage_break_idx = [13, 24]
b1 = min(stage_break_idx[0], n_spec)
b2 = min(stage_break_idx[1], n_spec)
stage_spans = []
if b1 >= 1:
    stage_spans.append((1, b1, "Charge", "#FDE6A8"))
if b2 >= b1 + 1:
    stage_spans.append((b1 + 1, b2, "Discharge", "#DCEBFF"))
if n_spec >= b2 + 1:
    stage_spans.append((b2 + 1, n_spec, "Charge", "#FDE6A8"))

for s_idx, e_idx, _, bgc in stage_spans:
    ax.axvspan(s_idx - 0.5, e_idx + 0.5, facecolor=bgc, alpha=0.35, lw=0, zorder=0)

ax.plot(state_idx, wl_intensity_norm, '-o', lw=1.1, ms=3.2, color='#1f77b4', label='WL intensity', zorder=3)

if special_idx.size > 0:
    valid_special = special_idx[(special_idx >= 0) & (special_idx < n_spec)]
    ax.scatter(
        state_idx[valid_special],
        wl_intensity_norm[valid_special],
        marker='*',
        c=colors[0],
        edgecolors='k',
        linewidths=0.3,
        s=74,
        zorder=6,
    )

ax.set_ylabel('Normalized WL intensity (arb. u.)', fontsize=9, color='#1f77b4')
ax.tick_params(axis='y', which='both', labelsize=10, colors='#1f77b4')
ax.set_xlim(1, n_spec)
ax.set_xlabel('Charge and Discharge', fontsize=9)
ax.set_xticks([])

ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.4)

if stage_spans:
    y_top = float(np.nanmax(wl_intensity_norm))
    y_bot = float(np.nanmin(wl_intensity_norm))
    y_text = y_top - 0.04 * max(y_top - y_bot, 1e-6)
    for s_idx, e_idx, stage_name, _ in stage_spans:
        x_mid = 0.5 * (s_idx + e_idx)
        ax.text(x_mid, y_text, stage_name, fontsize=9, ha='center', va='bottom', color='0.25')

# ax.text(-0.15, 1.0, 'A', transform=ax.transAxes, fontsize=8, fontweight='bold', va="center", ha="right")


plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_12_XAS_XANES_10_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches='tight',
    dpi=300,
    transparent=False,
    pil_kwargs={'compression': 'tiff_lzw'},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_12_XAS_XANES_10_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_12_XAS_XANES_10_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_12_XAS_XANES_10_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 13 -- XAS - XANES MCR-ALS: Third Phase

In [ ]:
path_ref = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")
path_dino = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\Dino")

ref = pd.read_csv(
    path_ref.joinpath(r"Overview", r"cell2_opXAS_P1_Mn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
).iloc[:, [0, 2, 4]]

ref = ref[(ref.iloc[:, 0] >= 6530.0) & (ref.iloc[:, 0] <= 6630.0)].copy().reset_index(drop=True)


dino = pd.read_csv(Path.joinpath(path_dino, r"FC1st_3rd_phase.txt"), sep=r"\s+", header=None, index_col=None)
dino = dino[(dino.iloc[:, 0] >= 6530.0) & (dino.iloc[:, 0] <= 6630.0)].copy()


In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 2.5))
gs = fig.add_gridspec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(ref.iloc[:, 0], ref.iloc[:, 1], c=colors[0], ls="-", lw=1.0, label=r"$\mathrm{\alpha-MnO_2}$")
ax.plot(ref.iloc[:, 0], ref.iloc[:, 2], c=colors[1], ls="-", lw=1.0, label=r"FC#C")
ax.plot(dino.iloc[:, 0], dino.iloc[:, 3], c=colors[3], ls="--", lw=1.0, label=r"$\mathrm{3^{rd} \ Phase}$")

# 设置刻度线等格式
ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax.set_ylim(0, 1.6)
ax.set_ylabel(r"Absorption (arb. u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))

ax.tick_params(axis="both", which="both", bottom=True, left=True, labelbottom=True, labelleft=True, labelsize=8)
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.55, 1),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    handlelength=3,
    handletextpad=0.2,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_13_XAS_XANES_03_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_13_XAS_XANES_03_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_13_XAS_XANES_03_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_13_XAS_XANES_03_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 14 -- XAS - Mn, MCR-ALS, 挣扎, 第三个物相

In [ ]:
# 读取数据
mcr_path = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\FC1st-FD2nd\Oct2022_cell3_P2b\XANES\Mn\MCR")

Mn_MCR_2 = xr.open_dataset(mcr_path.joinpath(r"P2b_Mn_MCR_2_stFixed.NETCDF4"), engine="h5netcdf")
Mn_MCR_3 = xr.open_dataset(mcr_path.joinpath(r"P2b_Mn_MCR_3_stFixed.NETCDF4"), engine="h5netcdf")


In [ ]:
# 画图
# gridspec inside gridspec
plt.close("all")
%matplotlib inline
fig = plt.figure(figsize=(7.0, 5.0))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

xascolors = [[colors[0], colors[1], colors[3]], [colors[4], colors[1]]]
xas_colors = ListedColormap(
    mpl.colormaps["coolwarm"](np.linspace(1.0, 0.0, Mn_MCR_2["MCR_residual"].shape[0])),
    name="xas_colors",
)
labels = [
    [r"${\alpha}$-MnO$\mathrm{_2}$", r"0.2 M Mn$\mathrm{^{2\!+}}$", r"FC#D"],
    [r"0.5 M Zn$\mathrm{^{2\!+}}$", r"ZSH"],
]

# 分段标注充放电状态, 1-13 / 14-24 / 25-end
n_spec = int(Mn_MCR_3[r"MCR_C"].shape[0])
stage_break_idx = [13, 24]
b1 = min(stage_break_idx[0], n_spec)
b2 = min(stage_break_idx[1], n_spec)
stage_spans = []
if b1 >= 1:
    stage_spans.append((1, b1, "Charge", "#FDE6A8"))
if b2 >= b1 + 1:
    stage_spans.append((b1 + 1, b2, "Discharge", "#DCEBFF"))
if n_spec >= b2 + 1:
    stage_spans.append((b2 + 1, n_spec, "Charge", "#FDE6A8"))

special_idx = np.array([0, 4, 9, 12, 14, 17, 24, 30, 34, 38], dtype=int)
valid_special = special_idx[(special_idx >= 0) & (special_idx < n_spec)]

for i in range(Mn_MCR_3[r"MCR_C"].shape[1]):
    x_vals = np.asarray(Mn_MCR_3[r"spectrum_number"].values, dtype=float)
    y_vals = np.asarray(Mn_MCR_3[r"MCR_C"][:, i].values, dtype=float)

    ax.plot(
        x_vals,
        y_vals,
        marker="o",
        markerfacecolor=xascolors[0][i],
        markeredgecolor=xascolors[0][i],
        markersize=5,
        c=xascolors[0][i],
        label=labels[0][i],
        ls="-",
    )

    if valid_special.size > 0:
        ax.scatter(
            x_vals[valid_special],
            y_vals[valid_special],
            marker="*",
            c=xascolors[0][i],
            edgecolors="k",
            linewidths=0.3,
            s=62,
            zorder=6,
        )

# 背景标注
for s_idx, e_idx, _, bgc in stage_spans:
    ax.axvspan(s_idx-0.5, e_idx+0.5, facecolor=bgc, alpha=0.5, lw=0, zorder=0)

if stage_spans:
    y_top = 0.9
    y_bot = 0.7
    y_text = y_top - 0.04 * max(y_top - y_bot, 1e-6)
    for s_idx, e_idx, stage_name, _ in stage_spans:
        x_mid = 0.5 * (s_idx + e_idx)
        ax.text(x_mid, y_text, stage_name, fontsize=8, ha='center', va='bottom', color='0.25')


#  刻度线
ax.set_xlim(-0.5, Mn_MCR_3[r"MCR_C"].shape[0])
ax.set_xlabel(r"Spectrum Number", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylim(-0.05, 1.0)
ax.set_ylabel(r"Mn molar fraction (arb. u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax.tick_params(
    axis="both",
    which="both",
    direction="out",
    bottom=True,
    left=True,
    labelbottom=True,
    labelleft=True,
    labelsize=8,
)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.0, 0.9),
    ncols=3,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.1,
)

ax.text(
    -0.23,
    1.0,
    r"A",
    transform=ax.transAxes,
    fontsize=10,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.35, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)
Ref_XANES = Mn_MCR_3["Ref_XANES"].isel(mcr_number=[1, 0, 2])
for i in range(Ref_XANES.shape[0]):
    ax.plot(
        Ref_XANES["pca_energy"],
        Ref_XANES[i],
        c="b",
        ls="--",
        lw=1.0,
        label=None,
        zorder=5,
        alpha=0.5,
    )
ax.tick_params(
    axis="both",
    which="both",
    labelsize=8,
    direction="out",
    bottom=True,
    left=True,
    labelbottom=True,
    labelleft=True,
)
for i in range(Mn_MCR_3["MCR_St"].shape[0]):
    ax.plot(Mn_MCR_3["MCR_St"]["pca_energy"], Mn_MCR_3["MCR_St"][i, :], c=xascolors[0][i], label=labels[0][i], ls="-", lw=1.0)

ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax.set_ylim(0, 2.0)
ax.set_ylabel(r"Absorption (arb. u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))

ax.legend(loc="upper left", bbox_to_anchor=(0.5, 1.0), ncols=1, handlelength=1, labelcolor="linecolor", fontsize=8)

ax.text(
    -0.23,
    1.0,
    r"B",
    transform=ax.transAxes,
    fontsize=10,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.7, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

for i in range(Mn_MCR_3["MCR_residual"].shape[0]):
    ax.plot(Mn_MCR_3["MCR_recon_Data"]["pca_energy"], Mn_MCR_3["MCR_recon_Data"][i, :], c=xas_colors.colors[i], label=None, ls="-", lw=1.0)
    ax.plot(Mn_MCR_3["MCR_residual"]["pca_energy"], Mn_MCR_3["MCR_residual"][i, :] - 0.2, c=xas_colors.colors[i], label=None, ls="-", lw=1.0)

# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((0.5, 0.15, 0.42, 0.05)),
    location="bottom",
    orientation="horizontal",
    cmap=xas_colors,
    ticklocation="top",
    spacing="proportional",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.outline.set_visible(False)  # type: ignore

ax.text(
    0.49,
    0.28,
    r"Charge",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
)
ax.text(
    0.84,
    0.36,
    r"Discharge",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="right",
    fontfamily="Arial",
)
ax.text(
    0.92,
    0.28,
    r"Charge",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="right",
    fontfamily="Arial",
)

ax.set_xlim(6530, 6630)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=10))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=10))
ax.set_ylim(-0.4, 1.6)
ax.set_ylabel(r"Absorption (arb. u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2))

ax.tick_params(
    axis="both",
    which="both",
    labelsize=8,
    direction="out",
    bottom=True,
    left=True,
    labelbottom=True,
    labelleft=True,
)

ax.text(
    -0.23,
    1.0,
    r"C",
    transform=ax.transAxes,
    fontsize=10,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_14_XAS_XANES_04_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_14_XAS_XANES_04_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_14_XAS_XANES_04_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_14_XAS_XANES_04_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 15 -- MCR - ALS - STXM 数据

In [ ]:
# 读取 MCR 的数据
path_mcr = Path.joinpath(path_data_root, r"STXM\ExSitu\Andrea\SpectrumAll").joinpath(r"sTXM_MCR-ALS-4comp.xlsx")
data = pd.read_excel(path_mcr, sheet_name=0, index_col=0, header=0, comment="#", engine="openpyxl")

# 交换顺序，使得符合放电状态
order_index = [*list(range(0, 5, 1)), *list(range(20, 25, 1)), *list(range(40, 50, 1)), *list(range(30, 35, 1)), *list(range(10, 15, 1)), *list(range(50, 60, 1)), *list(range(60, 65, 1))]
data = data.iloc[:, order_index]


In [ ]:
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 2.5))
gs = fig.add_gridspec(1, 1, width_ratios=None, height_ratios=None, wspace=0.0, hspace=0.0)

# 图
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.0)

names = [r"Pristine", r"FD#A", r"HC#B", r"HC#C", r"FC#D", r"HD#E", r"FD#F"]

ax.imshow(data.T.values, aspect="auto", cmap="coolwarm", origin="lower", extent=(data.index[0], data.index[-1], 0, data.shape[1]), vmin=0, vmax=2.0, interpolation="bilinear")  #'jet'

# 添加 colorbar
# Add colorbar and adjust ticks afterwards
colorbar = Colorbar(
    ax=ax.inset_axes((1.04, 0.1, 0.08, 0.8), transform=ax.transAxes),
    location="right",
    orientation="vertical",
    cmap="coolwarm",  #'jet'
    ticklocation="right",
    spacing="proportional",
    label=r"Absorption (arb. u.)",
)

colorbar.set_ticks([])
# colorbar.outline.set_visible(False)  # type: ignore
colorbar.ax.text(0.5, -0.08, 0.0, ha="center", va="bottom", fontsize=8, transform=colorbar.ax.transAxes)
colorbar.ax.text(0.5, 1.06, 2.0, ha="center", va="top", fontsize=8, transform=colorbar.ax.transAxes)

# 设置刻度线等格式
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.set_xlim(630, 670)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))

groups = [range(0, 5 + 1), range(5, 10 + 1), range(10, 20 + 1), range(20, 25 + 1), range(25, 30 + 1), range(30, 40 + 1), range(40, 45 + 1)]

# === 计算上下边界和中点 ===
boundaries = []
centers = []
for g in groups:
    boundaries.extend([min(g), max(g)])
    centers.append((min(g) + max(g)) / 2)

# 设置 y 轴刻度与标签
ax.set_yticks(boundaries)
ax.set_yticklabels([])
for y, label in zip(centers, names, strict=False):
    ax.text(-0.1, y, label, ha="right", va="center", transform=ax.get_yaxis_transform(), fontsize=8)

ax.tick_params(axis="x", which="both", direction="out", labelsize=8)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_15_STXM_03_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_15_STXM_03_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_15_STXM_03_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_15_STXM_03_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 16 -- MCR-ALS of STXM

In [ ]:
# 读取 MCR 的数据
path_mcr = Path.joinpath(path_data_root, r"STXM\ExSitu\Andrea\SpectrumAll").joinpath(r"sTXM_MCR-ALS-4comp.xlsx")
mcr_st = pd.read_excel(path_mcr, sheet_name=1, index_col=None, header=0, comment="#", engine="openpyxl")
mcr_c = pd.read_excel(path_mcr, sheet_name=2, index_col=None, header=0, comment="#", engine="openpyxl")


In [ ]:
# 画图
%matplotlib inline
fig = plt.figure(figsize=(7.0, 2.5))
gs = fig.add_gridspec(1, 2, width_ratios=[1, 1], height_ratios=None, wspace=0.0, hspace=0.0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(1.1)

ax.plot(mcr_st.iloc[:, 0], mcr_st.iloc[:, 1], ls="-", lw=1.0, c=colors[0], label=r"#1")
ax.plot(mcr_st.iloc[:, 0], mcr_st.iloc[:, 2], ls="-", lw=1.0, c=colors[1], label=r"#2")
ax.plot(mcr_st.iloc[:, 0], mcr_st.iloc[:, 3], ls="-", lw=1.0, c=colors[3], label=r"#3")
ax.plot(mcr_st.iloc[:, 0], mcr_st.iloc[:, 4], ls="-", lw=1.0, c=colors[5], label=r"#4")
ax.legend(loc="upper right", bbox_to_anchor=(0.98, 1.0), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)

# 刻度线
ax.set_xlim(635, 665)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=5))
ax.set_ylim(-0.1, 3.5)
ax.set_ylabel(r"Absorption (arb. u.)", fontsize=9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.7, offset=0.0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.35, offset=0.0))
ax.tick_params(
    axis="both",
    which="both",
    labelsize=8,
    bottom=True,
    left=True,
    labelbottom=True,
    labelleft=True,
)
ax.text(-0.23, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.8)

colors_mcr = [colors[0], colors[0], colors[1], colors[3], colors[5]]
# Pristine
for i in range(1, 5):
    ax.plot(np.linspace(0, 1, num=5), mcr_c.iloc[0:5, i], label=f"#{i}", c=colors_mcr[i])
# 1st Discharge
for i in range(1, 5):
    ax.plot(np.linspace(1, 2, num=5), mcr_c.iloc[10:15, i], label=None, c=colors_mcr[i])
# HC#1
for i in range(1, 5):
    ax.plot(np.linspace(2, 3, num=10), mcr_c.iloc[20:30, i], label=None, c=colors_mcr[i])
# HC#2
for i in range(1, 5):
    ax.plot(np.linspace(3, 4, num=5), mcr_c.iloc[15:20, i], label=None, c=colors_mcr[i])
# FC
for i in range(1, 5):
    ax.plot(np.linspace(4, 5, num=5), mcr_c.iloc[5:10, i], label=None, c=colors_mcr[i])
# HD
for i in range(1, 5):
    ax.plot(np.linspace(6, 5, num=10), mcr_c.iloc[30:40, i], label=None, c=colors_mcr[i])
# FD
for i in range(1, 5):
    ax.plot(np.linspace(6, 7, num=5), mcr_c.iloc[40:45, i], label=None, c=colors_mcr[i])

ax.set_xlabel("", fontsize=9)
ax.set_xlim(0, 7)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.set_ylabel(r"Relative concentrations (arb. u.)", fontsize=9)
ax.set_ylim(-0.01, 1.0)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax.legend(loc="upper left", bbox_to_anchor=(0.52, 1.0), ncols=2, frameon=False, labelcolor="linecolor", fontsize=8)

ax.vlines(
    [1, 2, 3, 4, 5, 6],
    [0, 0, 0, 0, 0, 0],
    [1, 1, 1, 1, 1, 1],
    colors=["grey", "magenta", "grey", "magenta", "grey", "magenta"],
    linestyles="--",
    lw=0.5,
    alpha=0.8,
    zorder=0,
)

for i in [0, 0.3, 0.6]:
    ax.text(
        0.06 + i,
        1.05,
        r"Border",
        transform=ax.transAxes,
        fontsize=9,
        va="top",
        ha="left",
        fontfamily="Arial",
        color="grey",
    )
    ax.text(
        0.22 + i,
        1.05,
        r"Bulk",
        transform=ax.transAxes,
        fontsize=9,
        va="top",
        ha="left",
        fontfamily="Arial",
        color="magenta",
    )

names = [r"Pristine", r"FD#A", r"HC#B", r"HC#C", r"FC#D", r"HD#E", r"FD#F"]

# x 轴
ax.set_xticks(range(len(names)))  # 设置 tick 的位置
ax.set_xticklabels([])
ax.tick_params(axis="x", which="minor", bottom=False, top=False, left=False, right=False)
centers = np.linspace(0.5, len(names) - 0.5, len(names))
for x, label in zip(centers, names, strict=False):
    ax.text(
        x,
        -0.05,
        label,
        ha="center",
        va="top",
        transform=ax.get_xaxis_transform(),
        fontsize=8,
        rotation=60,
    )

ax.text(-0.17, 1.0, r"B", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_16_STXM_04_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_16_STXM_04_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_16_STXM_04_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_16_STXM_04_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 17 -- TXM -Zn 谱线

In [ ]:
# TXM 的数据路径, Zn
path_data = Path.joinpath(path_data_root, r"STXM\ExSitu\Andrea")


def read_data(file_path):
    df = pd.read_csv(file_path, sep=r"\s+", header=None, comment="#")
    df.columns = ["Energy", "Absorption"]
    df["Absorption"] = (df["Absorption"] - df["Absorption"].min()) / (df["Absorption"].max() - df["Absorption"].min())
    return df


# 定义所有数据集的信息
datasets = [
    (r"ZSH", r"Pristine/E5 ZHS/ref_E5_ZHS_Zn.txt"),
    (r"1stDischarge", r"Charge/B6 1st0.9V/B6_1stDisch_Zn.txt"),
    (r"1stHalfCharge#2", r"Charge/E3 1st1.63V/E3_1stHCh_1p63V_Zn.txt"),
    (r"1stFullCharge", r"Charge/B2 1st1.80V/B2_1stCh_Zn.txt"),
    (r"2ndHalfDischarge", r"Charge/F4 2nd1.30V/F4_2ndDisch_1p3V_Zn.txt"),
    (r"2ndFullDischarge", r"Charge/G1 2nd0.9V/G1_2ndDisch_Zn.txt"),
]

# 加载和处理数据
data_dict_zn = {name: read_data(path_data.joinpath(path)) for name, path in datasets}


In [ ]:
# 画图
%matplotlib inline
plt.close("all")

fig = plt.figure(figsize=(7.0, 2.5))
gs = fig.add_gridspec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(1.2)

names = ["Ref.ZSH", r"FD#A", r"HC#C", r"FC#D", r"HD#E", r"FD#F"]

for i, (sample_name, data) in enumerate(data_dict_zn.items()):
    label = names[i] if i < len(names) else sample_name
    if i == 0:
        ax.plot(data["Energy"], data["Absorption"] + 0.5 * i, label=label, color=colors_opt2[i % len(colors_opt2)], lw=1.0)
    if i > 0:
        ax.plot(data["Energy"], data["Absorption"] + 0.5 * i, label=label, color=colors_opt2[(i + 2) % len(colors_opt2)], lw=1.0)


# 设置刻度线等格式
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.set_xlim(1010.0, 1050.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))

ax.set_ylabel(r"Absorption (arb. u.)", fontsize=9)
ax.set_ylim(-0.1, 4.5)
ax.set_yticks([])
ax.tick_params(axis="x", which="both", direction="out", labelsize=8)
ax.legend(loc="upper left", bbox_to_anchor=(0.0, 1.0), ncols=2, frameon=False, labelcolor="linecolor", fontsize=8)

# ax.text(-0.13, 1.0, r"A", transform=ax.transAxes, fontsize=8, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_17_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_17_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_17_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_17_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 19 -- TXM Zn Clusters 的图 Mantins

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        if len(bardata.axes_manager.navigation_shape) == 2:
            barsize = bardata.axes_manager.navigation_axes[0].scale
            unit = barunits if barunits is not None else bardata.axes_manager.navigation_axes[0].units
        elif len(bardata.axes_manager.signal_shape) == 2:
            barsize = bardata.axes_manager.signal_axes[0].scale
            unit = barunits if barunits is not None else bardata.axes_manager.signal_axes[0].units
        asb = AnchoredSizeBar(
            ax.transData,
            size / barsize,  # type: ignore
            "{} {}".format(size, unit),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])


from scipy.stats import linregress  # noqa: E402


def remove_linear_background(spectrum, energy_threshold):
    x = spectrum["Energy"].values
    y = spectrum["Absorption"].values

    # 背景拟合区域
    mask = x < energy_threshold
    if np.sum(mask) < 2:
        raise ValueError("背景区间点数不足，无法拟合。")

    x_bg = x[mask]
    y_bg = y[mask]

    # 线性拟合
    slope, intercept, *_ = linregress(x_bg, y_bg)
    background = slope * x + intercept

    return y - background


In [ ]:
# 读取数据
path_zsh = Path.joinpath(path_data_root, r"STXM\ExSitu\ZHS\Pristine\E5 ZHS\Results\20230630_E6Zn_798.3x826.9y_specnorm\MANTIS\k=3\ca")
path_fd1st = Path.joinpath(path_data_root, r"STXM\ExSitu\αMnO2\Charge\Electrode\1st0.9V\B6 1st0.9V\Results\20230630_B6Zn_632.7x1193.9y_specnorm\MANTIS\k=3\ca")
path_hca = Path.joinpath(path_data_root, r"STXM\ExSitu\αMnO2\Charge\Electrode\1st1.53V\F2 1st1.53V\Results\20230630_F2ZnB_881.9x731.1y_specnorm\k=3\ca")
path_hcb = Path.joinpath(path_data_root, r"STXM\ExSitu\αMnO2\Charge\Electrode\1st1.63V\E3 1st1.63V\Results\20230701_E3Zn_515.7x521.1y_specnorm\k=3\ca")
path_fc1st = Path.joinpath(path_data_root, r"STXM\ExSitu\αMnO2\Charge\Electrode\1st1.80V\B2 1st1.80V\Results\20230630_B2Zn_270.2x1228.4y_specnorm\k=3\ca")
path_hd = Path.joinpath(path_data_root, r"STXM\ExSitu\αMnO2\Charge\Electrode\2nd1.30V\F4 2nd1.30V\Results\20230701_F4Zn_868.2x982.9y_specnorm\k=3\ca")
path_fd2nd = Path.joinpath(path_data_root, r"STXM\ExSitu\αMnO2\Charge\Electrode\2nd0.9V\G1 2nd0.9V\Results\20230701_G1Zn_949.2x834.6y_specnorm\MANTIS\k=3\ca")

paths = [path_fd1st, path_hca, path_hcb, path_fc1st, path_hd, path_fd2nd]
# paths = [path_zsh, path_fd1st, path_hca, path_hcb, path_fc1st, path_hd, path_fd2nd]

data_dict = {}
for path in paths:
    name = path.parts[11]
    image_path = list(path.glob(r"*_CAcimg.png"))
    image = plt.imread(image_path[0])
    spectrum_path = sorted(path.glob(r"ca_CAspectrum_*.csv"), key=lambda p: int(re.search(r"_(\d+)\.csv$", p.name).group(1)))
    spectrum = []
    for file_path in spectrum_path:
        df = pd.read_csv(
            file_path,
            sep=r"\s+|\t+|,",
            engine="python",
            comment="*",
            header=None,
            names=["Energy", "Absorption"],
            skip_blank_lines=True,
        )
        df["Absorption"] = remove_linear_background(df, energy_threshold=1020.0)
        # df["Absorption"] = df["Absorption"] / df["Absorption"].max()
        spectrum.append(df)
    spectrum = pd.concat(spectrum, axis=1, ignore_index=True)
    spectrum = spectrum.iloc[:, [0, *list(range(1, spectrum.shape[1], 2))]]
    spectrum.columns = ["Energy", *[f"Spec{i}" for i in range(1, spectrum.shape[1])]]
    data_dict[name] = {"image": image, "spectrum": spectrum}


In [ ]:
# 画图
from matplotlib.colors import to_rgb

%matplotlib inline
plt.close("all")

fig = plt.figure(figsize=(7.0, 5.0))
gs = fig.add_gridspec(3, 4, width_ratios=[1, 1, 1, 1], height_ratios=[1, 1, 1], wspace=0.0, hspace=0.0)
Letters = ["A", "B", "C", "D", "E", "F"]

def _remap_stxm_cluster_colors(img, palette):
    arr = np.asarray(img, dtype=float)
    if arr.ndim != 3 or arr.shape[-1] not in (3, 4):
        return img

    if arr.max() > 1.0:
        arr = arr / 255.0

    alpha = arr[..., 3:4] if arr.shape[-1] == 4 else None
    rgb = np.clip(arr[..., :3], 0.0, 1.0)

    hsv = mpl.colors.rgb_to_hsv(rgb)
    h = hsv[..., 0]
    s = hsv[..., 1]

    # 依据色相将原始聚类图的主色（红/绿黄/蓝）映射到 colors[0/1/2]
    labels = np.full(h.shape, 2, dtype=np.int8)
    red_mask = (h < 0.07) | (h > 0.93)
    green_mask = (h >= 0.10) & (h <= 0.40)
    blue_mask = (h >= 0.45) & (h <= 0.80)

    labels[red_mask] = 0
    labels[green_mask] = 1
    labels[blue_mask] = 2
    labels[s < 0.10] = 2

    palette_rgb = np.array([to_rgb(palette[0]), to_rgb(palette[1]), to_rgb(palette[2])], dtype=float)
    mapped = palette_rgb[labels]

    # 略微向白色混合，降低“荧光感”
    mapped = 0.85 * mapped + 0.15

    if alpha is not None:
        mapped = np.concatenate([mapped, alpha], axis=-1)

    return np.clip(mapped, 0.0, 1.0)

# 图
for i, (_, data) in enumerate(data_dict.items()):
    subfig = fig.add_subfigure(gs[i // 2, 0]) if i % 2 == 0 else fig.add_subfigure(gs[i // 2, 2])
    ax = subfig.add_axes((0, 0, 0.9, 0.9)) if i % 2 == 0 else subfig.add_axes((0.25, 0, 0.9, 0.9))
    ax.set_box_aspect(1.0)
    ax.imshow(_remap_stxm_cluster_colors(data["image"], colors), interpolation="bilinear")
    ax.set_axis_off()
    # add_sizebar(ax, 2, 0.013000000268220901, r"w", r"$\mu m$").set_bbox_to_anchor(Bbox.from_bounds(0, 0.03, 0, 0), ax.transAxes)
    ax.text(-0.05, 0.95, f"{Letters[i]}", transform=ax.transAxes, fontsize=10, va="center", ha="center", fontfamily="Arial", fontweight="bold")

    subfig = fig.add_subfigure(gs[i // 2, 1]) if i % 2 == 0 else fig.add_subfigure(gs[i // 2, 3])
    ax = subfig.add_axes((0.15, 0.05, 0.9, 0.9)) if i % 2 == 0 else subfig.add_axes((0.4, 0.05, 0.9, 0.9))
    ax.set_box_aspect(0.8)

    # Spec1 为背景簇，不绘制；Spec2/Spec3 对应小图中的两类非背景簇颜色
    spec_color_map = {2: colors[0], 3: colors[1]}
    for spec_id, line_color in spec_color_map.items():
        col_name = f"Spec{spec_id}"
        if col_name in data["spectrum"].columns:
            ax.plot(
                data["spectrum"]["Energy"],
                data["spectrum"][col_name],
                color=line_color,
                linewidth=1.0,
            )
    # 刻度线
    ax.set_xlim(1010, 1050)
    ax.set_xlabel(r"Energy (eV)", fontsize=9)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=5))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=5))

    ax.set_ylabel(r"Absorption (arb. u.)", fontsize=9)
    ax.set_yticks([])
    ax.tick_params(
        axis="both",
        which="both",
        labelsize=8,
        bottom=True,
        left=False,
        labelbottom=True,
        labelleft=False,
    )

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_19_STXM_07_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_19_STXM_07_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_19_STXM_07_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_19_STXM_07_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 20 -- TXM Mn Clusters 的图 Mantins -V1

In [ ]:
from hyperspy.axes import DataAxis, UniformDataAxis
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        if len(bardata.axes_manager.navigation_shape) == 2:
            barsize = bardata.axes_manager.navigation_axes[0].scale
            unit = barunits if barunits is not None else bardata.axes_manager.navigation_axes[0].units
        elif len(bardata.axes_manager.signal_shape) == 2:
            barsize = bardata.axes_manager.signal_axes[0].scale
            unit = barunits if barunits is not None else bardata.axes_manager.signal_axes[0].units
        asb = AnchoredSizeBar(
            ax.transData,
            size / barsize,  # type: ignore
            "{} {}".format(size, unit),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])


def hs_spect(data):
    energy_axis = DataAxis(
        axis=data.iloc[:, 0].values,
        index_in_array=None,
        name="Energy",
        units="eV",
    )
    index = UniformDataAxis(
        offset=0,
        scale=1,
        size=data.shape[1] - 1,
        navigate=True,
        name=r"Index",
        units=None,
    )
    data_spc = hs.signals.Signal1D(data.iloc[:, 1:].values, axes=[energy_axis, index])
    data_spc = data_spc.inav[1:].deepcopy()
    return data_spc


In [ ]:
# 读取数据
path_pristine = Path.joinpath(path_data_root, r"STXM\ExSitu\αMnO2\Pristine\Powder\E1 Pristine\Results\SecondTrail_ScanMn_night\20230629_E1A_749.7x177.5y_specnorm\MANTIS\k=3\ca")
path_fd1st = Path.joinpath(path_data_root, r"STXM\ExSitu\αMnO2\Charge\Electrode\1st0.9V\B6 1st0.9V\Results\20230630_B6Mn_632.7x1193.9y_specnorm\MANTIS\k=4\ca")
path_hca = Path.joinpath(path_data_root, r"STXM\ExSitu\αMnO2\Charge\Electrode\1st1.53V\F2 1st1.53V\Results\20230630_F2MnB_881.9x731.1y_specnorm\MANTIS\k=4\ca")
path_hcb = Path.joinpath(path_data_root, r"STXM\ExSitu\αMnO2\Charge\Electrode\1st1.63V\E3 1st1.63V\Results\20230701_E3Mn_515.7x521.1y_specnorm\MANTIS\k=4\ca")
path_fc1st = Path.joinpath(path_data_root, r"STXM\ExSitu\αMnO2\Charge\Electrode\1st1.80V\B2 1st1.80V\Results\20230630_B2Mn_270.2x1228.4y_specnorm\MANTIS\k=4\ca")
path_hd = Path.joinpath(path_data_root, r"STXM\ExSitu\αMnO2\Charge\Electrode\2nd1.30V\F4 2nd1.30V\Results\20230701_F4Mn_868.2x982.9y_specnorm\MANTIS\k=4\ca")
path_fd2nd = Path.joinpath(path_data_root, r"STXM\ExSitu\αMnO2\Charge\Electrode\2nd0.9V\G1 2nd0.9V\Results\20230701_G1Mn_949.2x834.6y_specnorm\MANTIS\k=5\ca")

paths = [path_pristine, path_hca, path_hcb, path_fc1st, path_hd, path_fd2nd]
# paths = [path_pristine, path_fd1st, path_hca, path_hcb, path_fc1st, path_hd, path_fd2nd]

data_dict = {}
for path in paths:
    name = path.parts[11]
    image_path = list(path.glob(r"*_CAcimg.png"))
    image = plt.imread(image_path[0])
    spectrum_path = sorted(path.glob(r"ca_CAspectrum_*.csv"), key=lambda p: int(re.search(r"_(\d+)\.csv$", p.name).group(1)))
    spectrum = []
    for file_path in spectrum_path:
        df = pd.read_csv(
            file_path,
            sep=r"\s+|\t+|,",
            engine="python",
            comment="*",
            header=None,
            names=["Energy", "Absorption"],
            skip_blank_lines=True,
        )
        spectrum.append(df)
    spectrum = pd.concat(spectrum, axis=1, ignore_index=True)
    spectrum = spectrum.iloc[:, [0, *list(range(1, spectrum.shape[1], 2))]]
    spectrum.columns = ["Energy", *[f"Spec{i}" for i in range(1, spectrum.shape[1])]]
    spectrum = hs_spect(spectrum)
    data_dict[name] = {"image": image, "spectrum": spectrum}


In [ ]:
# 画图
from matplotlib.colors import to_rgb

%matplotlib inline
plt.close("all")

fig = plt.figure(figsize=(7.0, 5.0))
gs = fig.add_gridspec(3, 4, width_ratios=[1, 1, 1, 1], height_ratios=[1, 1, 1], wspace=0.0, hspace=0.0)
Letters = ["A", "B", "C", "D", "E", "F"]


def _mn_cluster_line_colors(n_fg, palette):
    # 使用 Tol 调色板，前景簇按顺序取色，避免高饱和红绿。
    base = [palette[0], palette[1], palette[3], palette[4]]
    n_fg = int(max(0, n_fg))
    return base[:n_fg]


def _remap_mn_cluster_colors(img, palette, n_fg):
    arr = np.asarray(img, dtype=float)
    if arr.ndim != 3 or arr.shape[-1] not in (3, 4):
        return img

    if arr.max() > 1.0:
        arr = arr / 255.0

    alpha = arr[..., 3:4] if arr.shape[-1] == 4 else None
    rgb = np.clip(arr[..., :3], 0.0, 1.0)

    hsv = mpl.colors.rgb_to_hsv(rgb)
    h = hsv[..., 0]
    s = hsv[..., 1]

    # 背景通常为蓝青色（以及低饱和像素）
    bg_mask = ((h >= 0.45) & (h <= 0.78)) | (s < 0.10)

    fg_h_centers = np.array([0.00, 0.25, 0.84, 0.08], dtype=float)  # red, green, purple, orange
    n_fg = int(max(1, min(4, n_fg)))
    fg_h = fg_h_centers[:n_fg]

    # hue 环形距离，将前景像素映射到 n_fg 个簇索引
    h_fg = h.copy()
    d = np.abs(h_fg[..., None] - fg_h[None, None, :])
    d = np.minimum(d, 1.0 - d)
    fg_idx = np.argmin(d, axis=-1)

    fg_colors = np.array([to_rgb(c) for c in _mn_cluster_line_colors(n_fg, palette)], dtype=float)
    bg_color = np.array(to_rgb(palette[2]), dtype=float)

    mapped = fg_colors[fg_idx]
    mapped[bg_mask] = bg_color

    # 轻度去荧光
    mapped = 0.88 * mapped + 0.12

    if alpha is not None:
        mapped = np.concatenate([mapped, alpha], axis=-1)

    return np.clip(mapped, 0.0, 1.0)

# 图
for i, (_, data) in enumerate(data_dict.items()):
    subfig = fig.add_subfigure(gs[i // 2, 0]) if i % 2 == 0 else fig.add_subfigure(gs[i // 2, 2])
    ax = subfig.add_axes((0, 0, 0.9, 0.9)) if i % 2 == 0 else subfig.add_axes((0.25, 0, 0.9, 0.9))
    ax.set_box_aspect(1.0)
    n_fg = int(data["spectrum"].axes_manager.navigation_size)
    ax.imshow(_remap_mn_cluster_colors(data["image"], colors, n_fg), interpolation="bilinear")
    ax.set_axis_off()
    # add_sizebar(ax, 2, 0.013000000268220901, r"w", r"$\mu m$").set_bbox_to_anchor(Bbox.from_bounds(0, 0.03, 0, 0), ax.transAxes)
    ax.text(-0.05, 0.95, f"{Letters[i]}", transform=ax.transAxes, fontsize=10, va="center", ha="center", fontfamily="Arial", fontweight="bold")

    subfig = fig.add_subfigure(gs[i // 2, 1]) if i % 2 == 0 else fig.add_subfigure(gs[i // 2, 3])
    ax = subfig.add_axes((0.15, 0.05, 0.9, 0.9)) if i % 2 == 0 else subfig.add_axes((0.4, 0.05, 0.9, 0.9))
    ax.set_box_aspect(0.8)

    hs.plot.plot_spectra(
        data["spectrum"],
        ax=ax,
        style="overlap",
        color=_mn_cluster_line_colors(data["spectrum"].axes_manager.navigation_size, colors),
        normalise=True,
    )

    # 刻度线
    ax.set_xlim(635, 665)
    ax.set_xlabel(r"Energy (eV)", fontsize=9)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=5))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=5))

    ax.set_ylabel(r"Absorption (arb. u.)", fontsize=9)
    ax.set_ylim(-0.05, 1.05)
    ax.set_yticks([])
    ax.tick_params(
        axis="both",
        which="both",
        labelsize=8,
        bottom=True,
        left=False,
        labelbottom=True,
        labelleft=False,
    )


plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_20_STXM_06_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_20_STXM_06_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_20_STXM_06_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_20_STXM_06_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 21 -- TEM-EELS 的各个状态 + 新相: 谱线

In [ ]:
# 读取 STEM-EELS 谱线的数据
path_eels = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\TEM-EDS-EELS-all")

TEM_EELS_Mn = pd.read_csv(path_eels.joinpath(r"TEM-EELS_Mn_selected.nor"), sep=r"\s+", header=None, index_col=None, comment=r"#")
TEM_EELS_O = pd.read_csv(path_eels.joinpath(r"TEM-EELS_O_selected.nor"), sep=r"\s+", header=None, index_col=None, comment=r"#")

# TEM_EELS_Mn = {
#     "energy": TEM_EELS_Mn.iloc[:, 0].copy(),
#     "bulk": TEM_EELS_Mn.iloc[:, [1, 3, 7, 9, 12, 19, 21]].copy(),
#     "surface": TEM_EELS_Mn.iloc[:, [2, 6, 8, 13, 18, 20]].copy(),
#     "new_phase": TEM_EELS_Mn.iloc[:, [4, 5, 10, 11, 14, 15, 16, 17]].copy(),
# }
TEM_EELS_O = {
    "energy": TEM_EELS_O.iloc[:, 0].copy(),
    "bulk": TEM_EELS_O.iloc[:, [1, 4, 8, 9, 13, 20, 22]].copy(),
    "surface": TEM_EELS_O.iloc[:, [3, 7, 10, 14, 19, 21]].copy(),
    "new_phase": TEM_EELS_O.iloc[:, [5, 6, 11, 12, 15, 16, 17, 18]].copy(),
    "ZSH": TEM_EELS_O.iloc[:, 2].copy(),
}


In [ ]:
# 删除多余的 1stFD
# TEM_EELS_Mn["bulk"].drop(columns=TEM_EELS_Mn["bulk"].columns[1], inplace=True)
# TEM_EELS_Mn["surface"].drop(columns=TEM_EELS_Mn["surface"].columns[1], inplace=True)
TEM_EELS_O["bulk"].drop(columns=TEM_EELS_O["bulk"].columns[1], inplace=True)
TEM_EELS_O["surface"].drop(columns=TEM_EELS_O["surface"].columns[1], inplace=True)


In [ ]:
colors_opt2 = [colors[0], colors[2], colors[3], colors[4], colors[5], colors[6]]

# EELS energy shifts (eV), confirmed by bulk-peak alignment vs FC#D
shift_energy_hd_mn = 3.6  # Mn, HD#E vs FC#D
shift_energy_fd_mn = 2.7  # Mn, FD#F vs FC#D
shift_energy_hd_o = 1.8   # O,  HD#E vs FC#D
shift_energy_fd_o = 0.9   # O,  FD#F vs FC#D


In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(3.0, 2.5))
gs = fig.add_gridspec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0)

# 图
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0, 1.0, 1.0))
ax.set_box_aspect(1.5)

labels = [r"Pristine", r"HC#B", r"HC#C", r"FC#D", r"HD#E", r"FD#F"]
for i in range(TEM_EELS_O["bulk"].shape[1]):
    if i < 4:
        ax.plot(TEM_EELS_O["energy"], TEM_EELS_O["bulk"].iloc[:, i] + 0.4 * i, label=labels[i], lw=1.0, color=colors_opt2[i])
    else:
        shift_i = shift_energy_hd_o if i == 4 else shift_energy_fd_o
        ax.plot(TEM_EELS_O["energy"] + shift_i, TEM_EELS_O["bulk"].iloc[:, i] + 0.4 * i, label=labels[i], lw=1.0, color=colors_opt2[i])

ax.plot(TEM_EELS_O["energy"], TEM_EELS_O["ZSH"], label=r"ZSH", color="blue", linestyle="--")


for i in range(0, TEM_EELS_O["surface"].shape[1]):
    if i < 3:
        ax.plot(TEM_EELS_O["energy"], TEM_EELS_O["surface"].iloc[:, i] + 0.5 * i + 3.5, linestyle="-", lw=1.0, label=None, color=colors_opt2[i + 1])
    else:
        shift_i = shift_energy_hd_o
        ax.plot(TEM_EELS_O["energy"] + shift_i, TEM_EELS_O["surface"].iloc[:, i] + 0.5 * i + 3.5, linestyle="-", lw=1.0, label=None, color=colors_opt2[i + 1])

for i in range(0, TEM_EELS_O["new_phase"].shape[1] // 2):
    if i < 3:
        ax.plot(TEM_EELS_O["energy"], TEM_EELS_O["new_phase"].iloc[:, 2 * i] + 0.7 * i + 7, linestyle="-", lw=1.0, label=None, color=colors_opt2[i + 2])
        ax.plot(TEM_EELS_O["energy"], TEM_EELS_O["new_phase"].iloc[:, 2 * i + 1] + 0.7 * i + 7, linestyle="-", lw=1.0, label=None, color=colors_opt2[i + 2])
    else:
        shift_i = shift_energy_hd_o
        ax.plot(TEM_EELS_O["energy"] + shift_i, TEM_EELS_O["new_phase"].iloc[:, 2 * i] + 0.7 * i + 7, linestyle="-", lw=1.0, label=None, color=colors_opt2[i + 2])
        ax.plot(TEM_EELS_O["energy"] + shift_i, TEM_EELS_O["new_phase"].iloc[:, 2 * i + 1] + 0.7 * i + 7, linestyle="-", lw=1.0, label=None, color=colors_opt2[i + 2])

# 刻度线
ax.set_xlim(520, 550)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2.5, offset=0))
ax.set_ylim(-0.1, 15)
ax.set_ylabel(r"Relative absorption (arb. u.)", fontsize=9)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    which="both",
    labelsize=8,
    bottom=True,
    left=True,
    labelbottom=True,
    labelleft=True,
)
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.0, 1.01),
    ncols=2,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.5,
)
ax.text(0.6, 0.25, r"Bulk", transform=ax.transAxes, fontsize=9, va="center", ha="left", fontfamily="Arial")
ax.text(0.6, 0.5, r"Surface", transform=ax.transAxes, fontsize=9, va="center", ha="left", fontfamily="Arial")
ax.text(0.6, 0.75, r"New Phase", transform=ax.transAxes, fontsize=9, va="center", ha="left", fontfamily="Arial")
ax.text(0.9, 0.95, r"O", transform=ax.transAxes, fontsize=10, va="center", ha="left", fontfamily="Arial", fontweight="bold")
ax.vlines(528.4, -0.1, 10, colors="gray", linestyles="dashed", linewidth=1.0, zorder=1)
ax.vlines(537, -0.1, 10.6, colors="gray", linestyles="dashed", linewidth=1.0, zorder=1)

# ax.text(-0.10, 1.0, r"B", transform=ax.transAxes, fontsize=8, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_21_TEM_EELS_01_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_21_TEM_EELS_01_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_21_TEM_EELS_01_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_21_TEM_EELS_01_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 22 -- EELS Clustering - FD1 1stDischarge

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def cluster(signal_list, key_word=(r"labels", r"ADF"), crop_box=None):
    cluster_data = {}

    for s in signal_list:
        try:
            title = str(s.metadata["General"]["title"])
        except Exception:
            title = ""

        match = re.search(r"\b(" + "|".join(key_word) + r")\b", title, re.IGNORECASE)
        # print(title, match)

        # 判断是否包含关键词
        if match:
            try:
                if crop_box:
                    s_crop = s.isig[crop_box].copy() if s.axes_manager.signal_dimension == 2 else s.inav[crop_box].copy()
            except Exception:
                s_crop = s.copy()
            cluster_data[f"{match.group(0)}"] = s_crop
        else:
            cluster_data["others"] = s

    return cluster_data


In [ ]:
# 读取 Clustering_Mn 的数据
path_1stDischarge_clustering = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\EELS\Results\SI2_80pA_10ms")
FD1_clustering_haadf = hs.load(path_1stDischarge_clustering.joinpath(r"Corrected SI", r"STEM SI processed.dm4"))
FD1_clustering_clusterings_total_mn = hs.load(path_1stDischarge_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_Mn_all_N4_*.hspy"))
FD1_clustering_clusterings_total_o = hs.load(path_1stDischarge_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_O_all_N4_*.hspy"))


In [ ]:
FD1_clustering_haadf = convert_units(FD1_clustering_haadf[1])
FD1_clustering_clusterings_total_mn = convert_units(FD1_clustering_clusterings_total_mn)
FD1_clustering_clusterings_total_o = convert_units(FD1_clustering_clusterings_total_o)

FD1_clustering_haadf = cluster(FD1_clustering_haadf, crop_box=(slice(None), slice(None)))
FD1_clustering_clusterings_total_mn = cluster(FD1_clustering_clusterings_total_mn, crop_box=(slice(None), slice(None)))
FD1_clustering_clusterings_total_o = cluster(FD1_clustering_clusterings_total_o, crop_box=(slice(None), slice(None)))


In [ ]:
# 画图
%matplotlib inline
plt.close("all")
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 5.0))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0, hspace=0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

ax.imshow(FD1_clustering_haadf["ADF"].data, cmap="gray", aspect=1.0, interpolation="bilinear")

scalebar = add_sizebar(
    ax,
    size=20,
    bardata=FD1_clustering_haadf["ADF"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(0.03, 0.90, r"HAADF", transform=ax.transAxes, fontsize=8, va="center", ha="left", fontfamily="Arial", color="w")
# ax.text(-0.03, 1.0, r'A', transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily='Arial', fontweight='bold')

# 图 B
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.25, 0.25, 0.7, 0.7))

hs.plot.plot_images(
    [FD1_clustering_clusterings_total_mn["labels"].inav[i] for i in range(FD1_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[r"k", colors_opt[2], colors_opt[4], colors_opt[6]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * FD1_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=FD1_clustering_clusterings_total_mn['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"Mn", transform=ax.transAxes, color="k", fontsize=10, va="center", ha="left", fontfamily="Arial")

# 插入图 B1
axins = ax.inset_axes((0.0, -2.0, 1.0, 1.85), zorder=1)

hs.plot.plot_spectra(
    [FD1_clustering_clusterings_total_mn["others"].inav[0], FD1_clustering_clusterings_total_mn["others"].inav[2], FD1_clustering_clusterings_total_mn["others"].inav[3]],
    ax=axins,
    style="overlap",
    color=[colors_opt[2], colors_opt[4], colors_opt[6]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=8, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=8, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(630, 660)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=8)

# 图 C
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.04, 0.25, 0.7, 0.7))

hs.plot.plot_images(
    [FD1_clustering_clusterings_total_o["labels"].inav[i] for i in range(FD1_clustering_clusterings_total_o["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[colors_opt[2], colors_opt[4], r"k", colors_opt[6]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * FD1_clustering_clusterings_total_o["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=FD1_clustering_clusterings_total_o['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"O", transform=ax.transAxes, color="k", fontsize=10, va="center", ha="left", fontfamily="Arial")

# 插入图 C1
axins = ax.inset_axes((0.0, -2.0, 1.0, 1.85), zorder=1)

hs.plot.plot_spectra(
    [FD1_clustering_clusterings_total_o["others"].inav[0], FD1_clustering_clusterings_total_o["others"].inav[1], FD1_clustering_clusterings_total_o["others"].inav[3]],
    ax=axins,
    style="overlap",
    color=[colors_opt[2], colors_opt[4], colors_opt[6]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=8, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=8, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(520, 550)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=8)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_22_TEM_EELS_03_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_22_TEM_EELS_03_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_22_TEM_EELS_03_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_22_TEM_EELS_03_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 23 -- EELS Clustering - HC#1 1sthalfcharge -V1

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def cluster(signal_list, key_word=(r"labels", r"ADF"), crop_box=None):
    cluster_data = {}

    for s in signal_list:
        try:
            title = str(s.metadata["General"]["title"])
        except Exception:
            title = ""

        match = re.search(r"\b(" + "|".join(key_word) + r")\b", title, re.IGNORECASE)
        # print(title, match)

        # 判断是否包含关键词
        if match:
            try:
                if crop_box:
                    s_crop = s.isig[crop_box].copy() if s.axes_manager.signal_dimension == 2 else s.inav[crop_box].copy()
            except Exception:
                s_crop = s.copy()
            cluster_data[f"{match.group(0)}"] = s_crop
        else:
            cluster_data["others"] = s

    return cluster_data


In [ ]:
# 读取 Clustering_Mn 的数据
path_1sthalfcharge_clustering = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EELS\Results\SI 2_630mm_5mm_0.9eVCh_0.9nm_15ms_150pA")
HC1_clustering_haadf = hs.load(path_1sthalfcharge_clustering.joinpath(r"Corrected SI", r"STEM SI processed.dm4"))
HC1_clustering_clusterings_total_mn = hs.load(path_1sthalfcharge_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_Mn_all_N4_*.hspy"))
HC1_clustering_clusterings_total_o = hs.load(path_1sthalfcharge_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_O_all_N4_*.hspy"))


In [ ]:
HC1_clustering_haadf = convert_units(HC1_clustering_haadf[1])
HC1_clustering_clusterings_total_mn = convert_units(HC1_clustering_clusterings_total_mn)
HC1_clustering_clusterings_total_o = convert_units(HC1_clustering_clusterings_total_o)

HC1_clustering_haadf = cluster(HC1_clustering_haadf, crop_box=(slice(None), slice(None)))
HC1_clustering_clusterings_total_mn = cluster(HC1_clustering_clusterings_total_mn, crop_box=(slice(None), slice(None)))
HC1_clustering_clusterings_total_o = cluster(HC1_clustering_clusterings_total_o, crop_box=(slice(None), slice(None)))


In [ ]:
# 画图
%matplotlib inline
plt.close("all")
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

ax.imshow(HC1_clustering_haadf["ADF"].data, cmap="gray", aspect=1.0, interpolation="bilinear")

scalebar = add_sizebar(
    ax,
    size=20,
    bardata=HC1_clustering_haadf["ADF"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(0.03, 0.90, r"HAADF", transform=ax.transAxes, fontsize=9, va="center", ha="left", fontfamily="Arial", color="w")
# ax.text(-0.03, 1.0, r'A', transform=ax.transAxes, fontsize=8, va="center", ha="right", fontfamily='Arial', fontweight='bold')

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.05, 0.41, 0.5, 0.5))

hs.plot.plot_images(
    [HC1_clustering_clusterings_total_mn["labels"].inav[i] for i in range(HC1_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[colors_opt[2], colors_opt[4], r"k", colors_opt[6]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * HC1_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=HC1_clustering_clusterings_total_mn['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"Mn", transform=ax.transAxes, color="k", fontsize=9, va="center", ha="left", fontfamily="Arial")

# 插入图 B1
axins = ax.inset_axes((0.0, -2.0, 1.0, 1.85), zorder=1)

hs.plot.plot_spectra(
    [HC1_clustering_clusterings_total_mn["others"].inav[0], HC1_clustering_clusterings_total_mn["others"].inav[1], HC1_clustering_clusterings_total_mn["others"].inav[3]],
    ax=axins,
    style="overlap",
    color=[colors_opt[2], colors_opt[4], colors_opt[6]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=8, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=8, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(630, 660)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=8)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.35, 0.41, 0.5, 0.5))
# ax.set_box_aspect(1.0)

hs.plot.plot_images(
    [HC1_clustering_clusterings_total_o["labels"].inav[i] for i in range(HC1_clustering_clusterings_total_o["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[colors_opt[2], colors_opt[4], r"k", colors_opt[6]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * HC1_clustering_clusterings_total_o["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=HC1_clustering_clusterings_total_o['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"O", transform=ax.transAxes, color="b", fontsize=9, va="center", ha="left", fontfamily="Arial")

# 插入图 C1
axins = ax.inset_axes((0.0, -2.0, 1.0, 1.85), zorder=1)

hs.plot.plot_spectra(
    [HC1_clustering_clusterings_total_o["others"].inav[0], HC1_clustering_clusterings_total_o["others"].inav[1], HC1_clustering_clusterings_total_o["others"].inav[3]],
    ax=axins,
    style="overlap",
    color=[colors_opt[2], colors_opt[4], colors_opt[6]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=8, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=8, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(520, 550)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=8)


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_23_TEM_EELS_04_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_23_TEM_EELS_04_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_23_TEM_EELS_04_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_23_TEM_EELS_04_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 24 -- EELS Clustering - HC#2 1sthalfcharge

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def cluster(signal_list, key_word=(r"labels", r"ADF"), crop_box=None):
    cluster_data = {}

    for s in signal_list:
        try:
            title = str(s.metadata["General"]["title"])
        except Exception:
            title = ""

        match = re.search(r"\b(" + "|".join(key_word) + r")\b", title, re.IGNORECASE)
        # print(title, match)

        # 判断是否包含关键词
        if match:
            try:
                if crop_box:
                    s_crop = s.isig[crop_box].copy() if s.axes_manager.signal_dimension == 2 else s.inav[crop_box].copy()
            except Exception:
                s_crop = s.copy()
            cluster_data[f"{match.group(0)}"] = s_crop
        else:
            cluster_data["others"] = s

    return cluster_data


In [ ]:
# 读取 Clustering_Mn 的数据
path_1sthalfcharge2_clustering = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.63V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EELS\Results\SI 1_630mm_5mm_0.9eVCh_3nm_15ms_150pA")
HC2_clustering_haadf = hs.load(path_1sthalfcharge2_clustering.joinpath(r"Corrected SI", r"STEM SI processed.dm4"))
HC2_clustering_clusterings_total_mn = hs.load(path_1sthalfcharge2_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_Mn_all_N3_*.hspy"))
HC2_clustering_clusterings_total_o = hs.load(path_1sthalfcharge2_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_O_all_N3_*.hspy"))


In [ ]:
HC2_clustering_haadf = convert_units(HC2_clustering_haadf[1])
HC2_clustering_clusterings_total_mn = convert_units(HC2_clustering_clusterings_total_mn)
HC2_clustering_clusterings_total_o = convert_units(HC2_clustering_clusterings_total_o)

HC2_clustering_haadf = cluster(HC2_clustering_haadf, crop_box=(slice(None), slice(None)))
HC2_clustering_clusterings_total_mn = cluster(HC2_clustering_clusterings_total_mn, crop_box=(slice(None), slice(None)))
HC2_clustering_clusterings_total_o = cluster(HC2_clustering_clusterings_total_o, crop_box=(slice(None), slice(None)))


In [ ]:
# 画图
%matplotlib inline
plt.close("all")
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

ax.imshow(HC2_clustering_haadf["ADF"].data, cmap="gray", aspect=1.0, interpolation="bilinear")

scalebar = add_sizebar(
    ax,
    size=50,
    bardata=HC2_clustering_haadf["ADF"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.02, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(0.03, 0.90, r"HAADF", transform=ax.transAxes, fontsize=9, va="center", ha="left", fontfamily="Arial", color="w")
# ax.text(-0.03, 1.0, r'A', transform=ax.transAxes, fontsize=8, va="center", ha="right", fontfamily='Arial', fontweight='bold')

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.45, 0.45, 0.55, 0.55))

hs.plot.plot_images(
    [HC2_clustering_clusterings_total_mn["labels"].inav[i] for i in range(HC2_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[r"k", colors_opt[2], colors_opt[4]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * HC2_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=HC2_clustering_clusterings_total_mn['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.05, r"Mn", transform=ax.transAxes, color="k", fontsize=9, va="center", ha="left", fontfamily="Arial")

# 插入图 B1
axins = ax.inset_axes((0.0, -0.8, 1.0, 0.75), zorder=1)

hs.plot.plot_spectra(
    [HC2_clustering_clusterings_total_mn["others"].inav[1], HC2_clustering_clusterings_total_mn["others"].inav[2]],
    ax=axins,
    style="overlap",
    color=[colors_opt[2], colors_opt[4]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=8, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=8, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(630, 660)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=8)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.45, 0.55, 0.55))
# ax.set_box_aspect(1.0)

hs.plot.plot_images(
    [HC2_clustering_clusterings_total_o["labels"].inav[i] for i in range(HC2_clustering_clusterings_total_o["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[colors_opt[4], r"k", colors_opt[2]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * HC2_clustering_clusterings_total_o["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=HC2_clustering_clusterings_total_o['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.05, r"O", transform=ax.transAxes, color="k", fontsize=9, va="center", ha="left", fontfamily="Arial")

# 插入图 C1
axins = ax.inset_axes((0.0, -0.8, 1.0, 0.75), zorder=1)

hs.plot.plot_spectra(
    [HC2_clustering_clusterings_total_o["others"].inav[0], HC2_clustering_clusterings_total_o["others"].inav[2]],
    ax=axins,
    style="overlap",
    color=[colors_opt[4], colors_opt[2]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=8, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=8, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(520, 550)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=8)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_24_TEM_EELS_05_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_24_TEM_EELS_05_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_24_TEM_EELS_05_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_24_TEM_EELS_05_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 25 -- EELS Clustering - FC1 1stCharge

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def cluster(signal_list, key_word=(r"labels", r"ADF"), crop_box=None):
    cluster_data = {}

    for s in signal_list:
        try:
            title = str(s.metadata["General"]["title"])
        except Exception:
            title = ""

        match = re.search(r"\b(" + "|".join(key_word) + r")\b", title, re.IGNORECASE)
        # print(title, match)

        # 判断是否包含关键词
        if match:
            try:
                if crop_box:
                    s_crop = s.isig[crop_box].copy() if s.axes_manager.signal_dimension == 2 else s.inav[crop_box].copy()
            except Exception:
                s_crop = s.copy()
            cluster_data[f"{match.group(0)}"] = s_crop
        else:
            cluster_data["others"] = s

    return cluster_data


In [ ]:
# 读取 Clustering_Mn 的数据
path_FC1_clustering = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\EELS\Results\SI2_EFTEM400m_5mm_50ms_200pA_5nm")
FC1_clustering_haadf = hs.load(path_FC1_clustering.joinpath(r"Corrected SI", r"STEM SI processed.dm4"))
FC1_clustering_clusterings_total_mn = hs.load(path_FC1_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_Mn_all_N3_*.hspy"))
FC1_clustering_clusterings_total_o = hs.load(path_FC1_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_O_all_N3_*.hspy"))


In [ ]:
FC1_clustering_haadf = convert_units(FC1_clustering_haadf[1])
FC1_clustering_clusterings_total_mn = convert_units(FC1_clustering_clusterings_total_mn)
FC1_clustering_clusterings_total_o = convert_units(FC1_clustering_clusterings_total_o)

FC1_clustering_haadf = cluster(FC1_clustering_haadf, crop_box=(slice(None), slice(0, 70)))
FC1_clustering_clusterings_total_mn = cluster(FC1_clustering_clusterings_total_mn, crop_box=(slice(None), slice(0, 70)))
FC1_clustering_clusterings_total_o = cluster(FC1_clustering_clusterings_total_o, crop_box=(slice(None), slice(0, 70)))


In [ ]:
# 画图
%matplotlib inline
plt.close("all")
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

ax.imshow(FC1_clustering_haadf["ADF"].data.T, cmap="gray", aspect=1.0, interpolation="bilinear")

scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FC1_clustering_haadf["ADF"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(0.03, 0.95, r"HAADF", transform=ax.transAxes, fontsize=9, va="center", ha="left", fontfamily="Arial", color="w")
# ax.text(-0.03, 1.0, r'A', transform=ax.transAxes, fontsize=8, va="center", ha="right", fontfamily='Arial', fontweight='bold')

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.05, 0.42, 0.7, 0.7))
# ax.set_box_aspect(1.0)

hs.plot.plot_images(
    [FC1_clustering_clusterings_total_mn["labels"].inav[i] for i in range(FC1_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[colors_opt[0], r"k", colors_opt[2]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * FC1_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=FC1_clustering_clusterings_total_mn['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"Mn", transform=ax.transAxes, color="k", fontsize=9, va="center", ha="left", fontfamily="Arial")

# 插入图 B1
axins = ax.inset_axes((0.0, -1.18, 1.0, 1.1), zorder=1)

hs.plot.plot_spectra(
    [FC1_clustering_clusterings_total_mn["others"].inav[0], FC1_clustering_clusterings_total_mn["others"].inav[2]],
    ax=axins,
    style="overlap",
    color=[colors_opt[0], colors_opt[2]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=8, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=8, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(630, 660)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=8)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.2, 0.42, 0.7, 0.7))
# ax.set_box_aspect(1.0)

hs.plot.plot_images(
    [FC1_clustering_clusterings_total_o["labels"].inav[i] for i in range(FC1_clustering_clusterings_total_o["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[r"k", colors_opt[2], colors_opt[0]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * FC1_clustering_clusterings_total_o["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=FC1_clustering_clusterings_total_o['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"O", transform=ax.transAxes, color="k", fontsize=9, va="center", ha="left", fontfamily="Arial")

# 插入图 C1
axins = ax.inset_axes((0.0, -1.18, 1.0, 1.1), zorder=1)

hs.plot.plot_spectra(
    [FC1_clustering_clusterings_total_o["others"].inav[1], FC1_clustering_clusterings_total_o["others"].inav[2]],
    ax=axins,
    style="overlap",
    color=[colors_opt[2], colors_opt[0]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=8, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=8, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(520, 550)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=8)


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_25_TEM_EELS_06_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_25_TEM_EELS_06_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_25_TEM_EELS_06_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_25_TEM_EELS_06_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 26 -- EELS Clustering - HD1 2ndHalfDischarge -V1

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def cluster(signal_list, key_word=(r"labels", r"ADF"), crop_box=None):
    cluster_data = {}

    for s in signal_list:
        try:
            title = str(s.metadata["General"]["title"])
        except Exception:
            title = ""

        match = re.search(r"\b(" + "|".join(key_word) + r")\b", title, re.IGNORECASE)
        # print(title, match)

        # 判断是否包含关键词
        if match:
            try:
                if crop_box:
                    s_crop = s.isig[crop_box].copy() if s.axes_manager.signal_dimension == 2 else s.inav[crop_box].copy()
            except Exception:
                s_crop = s.copy()
            cluster_data[f"{match.group(0)}"] = s_crop
        else:
            cluster_data["others"] = s

    return cluster_data


In [ ]:
# 读取 Clustering_Mn 的数据
path_HD1_clustering = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\EELS\Results\SI3_CL320mm_0.9eV_25ms_1.5nm step_120pA")
HD1_clustering_haadf = hs.load(path_HD1_clustering.joinpath(r"Corrected SI", r"STEM SI processed.dm4"))
HD1_clustering_clusterings_total_mn = hs.load(path_HD1_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_Mn_all_N3_*.hspy"))
HD1_clustering_clusterings_total_o = hs.load(path_HD1_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_O_all_N3_*.hspy"))


In [ ]:
HD1_clustering_haadf = convert_units(HD1_clustering_haadf[1])
HD1_clustering_clusterings_total_mn = convert_units(HD1_clustering_clusterings_total_mn)
HD1_clustering_clusterings_total_o = convert_units(HD1_clustering_clusterings_total_o)

HD1_clustering_haadf = cluster(HD1_clustering_haadf, crop_box=(slice(None), slice(None)))
HD1_clustering_clusterings_total_mn = cluster(HD1_clustering_clusterings_total_mn, crop_box=(slice(None), slice(None)))
HD1_clustering_clusterings_total_o = cluster(HD1_clustering_clusterings_total_o, crop_box=(slice(None), slice(None)))


In [ ]:
energy_shift = shift_energy_hd_mn  # eV (Mn, HD#E vs FC#D bulk peak alignment)
HD1_clustering_clusterings_total_mn["others"].axes_manager.signal_axes[0].offset += energy_shift
energy_shift = shift_energy_hd_o  # eV (O, HD#E vs FC#D bulk peak alignment)
HD1_clustering_clusterings_total_o["others"].axes_manager.signal_axes[0].offset += energy_shift


In [ ]:
# 画图
%matplotlib inline
plt.close("all")

fig = plt.figure(figsize=(7.0, 2.5))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

ax.imshow(HD1_clustering_haadf["ADF"].data, cmap="gray", aspect=1.0, interpolation="bilinear")

scalebar = add_sizebar(
    ax,
    size=20,
    bardata=HD1_clustering_haadf["ADF"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(0.03, 0.92, r"HAADF", transform=ax.transAxes, fontsize=9, va="center", ha="left", fontfamily="Arial", color="w")
# ax.text(-0.03, 1.0, r'A', transform=ax.transAxes, fontsize=8, va="center", ha="right", fontfamily='Arial', fontweight='bold')

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.1, 0.38, 0.6, 0.6))

hs.plot.plot_images(
    [HD1_clustering_clusterings_total_mn["labels"].inav[i] for i in range(HD1_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[r"k", colors_opt[4], colors_opt[2]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * HD1_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=HD1_clustering_clusterings_total_mn['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"Mn", transform=ax.transAxes, color="k", fontsize=9, va="center", ha="left", fontfamily="Arial")

# 插入图 B1
axins = ax.inset_axes((0.0, -2.0, 1.0, 1.85), zorder=1)

hs.plot.plot_spectra(
    [HD1_clustering_clusterings_total_mn["others"].inav[1], HD1_clustering_clusterings_total_mn["others"].inav[2]],
    ax=axins,
    style="overlap",
    color=[colors_opt[4], colors_opt[2]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=8, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=8, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(630, 660)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=8)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.20, 0.38, 0.6, 0.6))
# ax.set_box_aspect(1.0)

hs.plot.plot_images(
    [HD1_clustering_clusterings_total_o["labels"].inav[i] for i in range(HD1_clustering_clusterings_total_o["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[r"k", colors_opt[4], colors_opt[2]],
    axes_decor="off",
    label=None,
    alphas=[1.0] * HD1_clustering_clusterings_total_o["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=HD1_clustering_clusterings_total_o['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"O", transform=ax.transAxes, color="k", fontsize=9, va="center", ha="left", fontfamily="Arial")

# 插入图 C1
axins = ax.inset_axes((0.0, -2.0, 1.0, 1.85), zorder=1)

hs.plot.plot_spectra(
    [HD1_clustering_clusterings_total_o["others"].inav[1], HD1_clustering_clusterings_total_o["others"].inav[2]],
    ax=axins,
    style="overlap",
    color=[colors_opt[4], colors_opt[2]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=8, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=8, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(520, 550)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=8)


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_26_TEM_EELS_07_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_26_TEM_EELS_07_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_26_TEM_EELS_07_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_26_TEM_EELS_07_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 27 -- EELS Clustering - FD2 2ndFullDischarge

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def cluster(signal_list, key_word=(r"labels", r"ADF"), crop_box=None):
    cluster_data = {}

    for s in signal_list:
        try:
            title = str(s.metadata["General"]["title"])
        except Exception:
            title = ""

        match = re.search(r"\b(" + "|".join(key_word) + r")\b", title, re.IGNORECASE)
        # print(title, match)

        # 判断是否包含关键词
        if match:
            try:
                if crop_box:
                    s_crop = s.isig[crop_box].copy() if s.axes_manager.signal_dimension == 2 else s.inav[crop_box].copy()
            except Exception:
                s_crop = s.copy()
            cluster_data[f"{match.group(0)}"] = s_crop
        else:
            cluster_data["others"] = s

    return cluster_data


In [ ]:
# 读取 Clustering_Mn 的数据
path_HD2_clustering = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B2\EELS\Results\SI1_100pA_10ms_1.5nm")
HD2_clustering_haadf = hs.load(path_HD2_clustering.joinpath(r"Corrected SI", r"STEM SI processed.dm4"))
HD2_clustering_clusterings_total_mn = hs.load(path_HD2_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_Mn_all_N3_*.hspy"))
HD2_clustering_clusterings_total_o = hs.load(path_HD2_clustering.joinpath(r"Hyperspy").glob(r"5-Clusters_O_all_N3_*.hspy"))


In [ ]:
HD2_clustering_haadf = convert_units(HD2_clustering_haadf[1])
HD2_clustering_clusterings_total_mn = convert_units(HD2_clustering_clusterings_total_mn)
HD2_clustering_clusterings_total_o = convert_units(HD2_clustering_clusterings_total_o)

HD2_clustering_haadf = cluster(HD2_clustering_haadf, crop_box=(slice(None), slice(None)))
HD2_clustering_clusterings_total_mn = cluster(HD2_clustering_clusterings_total_mn, crop_box=(slice(None), slice(None)))
HD2_clustering_clusterings_total_o = cluster(HD2_clustering_clusterings_total_o, crop_box=(slice(None), slice(None)))


In [ ]:
energy_shift = shift_energy_fd_mn  # eV (Mn, FD#F vs FC#D bulk peak alignment)
HD2_clustering_clusterings_total_mn["others"].axes_manager.signal_axes[0].offset += energy_shift
energy_shift = shift_energy_fd_o  # eV (O, FD#F vs FC#D bulk peak alignment)
HD2_clustering_clusterings_total_o["others"].axes_manager.signal_axes[0].offset += energy_shift


In [ ]:
# 画图
%matplotlib inline
plt.close("all")
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0, hspace=0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

ax.imshow(HD2_clustering_haadf["ADF"].data, cmap="gray", aspect=1.0, interpolation="bilinear")

scalebar = add_sizebar(
    ax,
    size=20,
    bardata=HD2_clustering_haadf["ADF"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(0.03, 0.90, r"HAADF", transform=ax.transAxes, fontsize=9, va="center", ha="left", fontfamily="Arial", color="w")
# ax.text(-0.03, 1.0, r'A', transform=ax.transAxes, fontsize=8, va="center", ha="right", fontfamily='Arial', fontweight='bold')

# 图 B
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.54, 0.25, 0.5, 0.5))

hs.plot.plot_images(
    [HD2_clustering_clusterings_total_mn["labels"].inav[i] for i in range(HD2_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[colors_opt[2], colors_opt[4], "k"],
    axes_decor="off",
    label=None,
    alphas=[1.0] * HD2_clustering_clusterings_total_mn["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=HD2_clustering_clusterings_total_mn['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"Mn", transform=ax.transAxes, color="k", fontsize=9, va="center", ha="left", fontfamily="Arial")

# 插入图 B1
axins = ax.inset_axes((0.0, -2.0, 1.0, 1.85), zorder=1)

hs.plot.plot_spectra(
    [HD2_clustering_clusterings_total_mn["others"].inav[1], HD2_clustering_clusterings_total_mn["others"].inav[2]],
    ax=axins,
    style="overlap",
    color=[colors_opt[2], colors_opt[4]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=8, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=8, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(630, 660)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=8)

# 图 C
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.05, 0.25, 0.5, 0.5))
# ax.set_box_aspect(1.0)

hs.plot.plot_images(
    [HD2_clustering_clusterings_total_o["labels"].inav[i] for i in range(HD2_clustering_clusterings_total_o["labels"].axes_manager.navigation_size)],
    ax=ax,
    overlay=True,
    colors=[colors_opt[2], colors_opt[4], r"k"],
    axes_decor="off",
    label=None,
    alphas=[1.0] * HD2_clustering_clusterings_total_o["labels"].axes_manager.navigation_size,
    vmax="100thh",
)
# scalebar =add_sizebar(
#     ax,
#     size=20,
#     bardata=HD2_clustering_clusterings_total_o['labels'],
#     color="w",
# )
# scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)
ax.text(0.0, 1.1, r"O", transform=ax.transAxes, color="k", fontsize=9, va="center", ha="left", fontfamily="Arial")

# 插入图 C1
axins = ax.inset_axes((0.0, -2.0, 1.0, 1.85), zorder=1)

hs.plot.plot_spectra(
    [HD2_clustering_clusterings_total_o["others"].inav[0], HD2_clustering_clusterings_total_o["others"].inav[1]],
    ax=axins,
    style="overlap",
    color=[colors_opt[2], colors_opt[4]],
    normalise=True,
)

axins.set_xlabel(r"Energy Loss (eV)", fontsize=8, fontfamily="Arial")
axins.set_ylabel(r"", fontsize=8, fontfamily="Arial")
axins.set_yticks([])
axins.set_ylim(-0.01, 1.05)
axins.set_xlim(520, 550)
axins.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
axins.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
axins.tick_params(axis="both", which="major", labelsize=8)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_27_TEM_EELS_08_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_27_TEM_EELS_08_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_27_TEM_EELS_08_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_27_TEM_EELS_08_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 28 -- TEM-EDX, Initial Full Charged sample and others -V1

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        if len(bardata.axes_manager.navigation_shape) == 2:
            barsize = bardata.axes_manager.navigation_axes[0].scale
            unit = bardata.axes_manager.navigation_axes[0].units
        elif len(bardata.axes_manager.signal_shape) == 2:
            barsize = bardata.axes_manager.signal_axes[0].scale
            unit = bardata.axes_manager.signal_axes[0].units
        asb = AnchoredSizeBar(
            ax.transData,
            size / barsize,  # type: ignore
            "{} {}".format(size, unit),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    results = []
    for file in data:
        if len(file.axes_manager.shape) >= 2:
            if len(file.axes_manager.navigation_shape) == 2:
                if file.axes_manager.navigation_axes[0].units == r"µm":
                    file.axes_manager.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(file.axes_manager.signal_shape) == 2:  # noqa: SIM102
                if file.axes_manager.signal_axes[0].units == r"µm":
                    file.axes_manager.convert_units(axes="signal", units="nm", same_units=True, factor=1000)

        if len(file.axes_manager.shape) == 3:
            if len(file.axes_manager.navigation_shape) == 2:
                for axis in file.axes_manager.navigation_axes:
                    axis.offset = 0
            elif len(file.axes_manager.signal_shape) == 2:
                for axis in file.axes_manager.signal_axes:
                    axis.offset = 0
        results.append(file)
    return results


def element_maps(signal_list, elements=(r"HAADF", r"K", r"Mn", r"Zn"), crop_box=None):
    element_maps = {}
    for signal in signal_list:
        title = signal.metadata["General"]["title"]
        if title in elements:
            if crop_box:
                if signal.axes_manager.navigation_shape:
                    signal_crop = signal.inav[crop_box].copy()
                elif signal.axes_manager.signal_shape:
                    signal_crop = signal.isig[crop_box].copy()
            else:
                signal_crop = signal.copy()

            element_maps[title] = signal_crop
    return element_maps


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    if color0 is None:
        return LinearSegmentedColormap.from_list("", [to_rgba(color1, 0), to_rgba(color1, 1)])
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])


eds_colors = {
    "C": "#FF0000",
    "K": "#00ff00",
    "Mn": "#FFFF00",
    "Zn": "#FF8000",
    "S": "#1f3cef",
    "O": "#00ffff",
}


In [ ]:
# 读取 EDX 数据
path_file_hc1 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EDS\Results\0005-20241211_HAADF_82000_x_EDS\Sum")
eds_hc1 = hs.load(path_file_hc1.joinpath(r"0005-20241211_HAADF_82000_x_EDS.emd"))
path_file_hc2 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.63V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EDS\Results\0012-20241211_HAADF_58000_x_EDS\Sum")
eds_hc2 = hs.load(path_file_hc2.joinpath(r"0012-20241211_HAADF_58000_x_EDS.emd"))
path_file_fc = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\EDS\Results\0021 - 1500 Pristine MnO2 HAADF 58000 x\Sum")
eds_fc = hs.load(path_file_fc.joinpath(r"0021 - 1500 Pristine MnO2 HAADF 58000 x.emd"))
path_file_hd1 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\EDS\Results\0013-HAADF_33000_x_EDS_SI\Sum")
eds_hd1 = hs.load(path_file_hd1.joinpath(r"0013-HAADF_33000_x_EDS_SI.emd"))
path_file_fd = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B2\EDS\Results\0028-HAADF_67000_x_EDS_SI\Sum")
eds_fd = hs.load(path_file_fd.joinpath(r"0028-HAADF_67000_x_EDS_SI.emd"))


In [ ]:
eds_hc1 = convert_units(eds_hc1)
eds_hc2 = convert_units(eds_hc2)
eds_fc = convert_units(eds_fc)
eds_hd1 = convert_units(eds_hd1)
eds_fd = convert_units(eds_fd)


In [ ]:
eds_hc1 = element_maps(eds_hc1, elements=[r"HAADF", r"K", r"Mn", r"Zn"], crop_box=(slice(None), slice(None)))
eds_hc2 = element_maps(eds_hc2, elements=[r"HAADF", r"K", r"Mn", r"Zn"], crop_box=(slice(40, 185), slice(None)))
eds_fc = element_maps(eds_fc, elements=[r"HAADF", r"K", r"Mn", r"Zn"], crop_box=(slice(27, 160), slice(None)))
eds_hd1 = element_maps(eds_hd1, elements=[r"HAADF", r"K", r"Mn", r"Zn"], crop_box=(slice(None), slice(None)))
eds_fd = element_maps(eds_fd, elements=[r"HAADF", r"K", r"Mn", r"Zn"], crop_box=(slice(10, 60), slice(None)))


In [ ]:
# 画图
%matplotlib inline
plt.close("all")
fig = plt.figure(figsize=(7.0, 5.0))
gs = fig.add_gridspec(5, 4, width_ratios=[1, 1, 1, 1], height_ratios=[1, 1, 1, 1, 1], wspace=0.0, hspace=0.0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hc1["HAADF"].data
ax.imshow(img, cmap="gray", alpha=1.0, interpolation="bilinear")
sizebar = add_sizebar(ax, 100, eds_hc1["HAADF"], "w")
sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"A",
#     transform=ax.transAxes,
#     fontsize=8,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.65, 0.0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hc1["K"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["K"]), alpha=1.0, interpolation="bilinear")
# sizebar = add_sizebar(ax, 100, eds_hc1['K'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"K $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"B",
#     transform=ax.transAxes,
#     fontsize=8,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-1.3, 0.0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hc1["Mn"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["Mn"]), alpha=1.0, interpolation="bilinear")
# sizebar = add_sizebar(ax, 100, eds_hc1['Mn'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"Mn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"C",
#     transform=ax.transAxes,
#     fontsize=8,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 D
subfig = fig.add_subfigure(gs[0, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-1.95, 0.0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hc1["Zn"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["Zn"]), alpha=1.0, interpolation="bilinear")
# sizebar = add_sizebar(ax, 100, eds_hc1['Zn'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"Zn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"D",
#     transform=ax.transAxes,
#     fontsize=8,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 E
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, -0.1, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hc2["HAADF"].data
ax.imshow(img, cmap="gray", alpha=1.0, interpolation="bilinear")
sizebar = add_sizebar(ax, 100, eds_hc2["HAADF"], "w")
sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"E",
#     transform=ax.transAxes,
#     fontsize=8,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 F
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.65, -0.1, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hc2["K"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["K"]), alpha=1.0, interpolation="bilinear")
# sizebar = add_sizebar(ax, 100, eds_hc2['K'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"K $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"F",
#     transform=ax.transAxes,
#     fontsize=8,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 G
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-1.3, -0.1, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hc2["Mn"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["Mn"]), alpha=1.0, interpolation="bilinear")
# sizebar = add_sizebar(ax, 100, eds_hc2['Mn'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"Mn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"G",
#     transform=ax.transAxes,
#     fontsize=8,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 H
subfig = fig.add_subfigure(gs[1, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-1.95, -0.1, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hc2["Zn"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["Zn"]), alpha=1.0, interpolation="bilinear")
# sizebar = add_sizebar(ax, 100, eds_hc2['Zn'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"Zn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"H",
#     transform=ax.transAxes,
#     fontsize=8,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 M
subfig = fig.add_subfigure(gs[2, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, -0.2, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hd1["HAADF"].data
ax.imshow(img, cmap="gray", alpha=1.0, interpolation="bilinear")
sizebar = add_sizebar(ax, 500, eds_hd1["HAADF"], "w")
sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"M",
#     transform=ax.transAxes,
#     fontsize=8,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 N
subfig = fig.add_subfigure(gs[2, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.65, -0.2, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hd1["K"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["K"]), alpha=1.0, interpolation="bilinear")
# sizebar = add_sizebar(ax, 500, eds_hd1['K'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"K $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"N",
#     transform=ax.transAxes,
#     fontsize=8,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 O
subfig = fig.add_subfigure(gs[2, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-1.3, -0.2, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hd1["Mn"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["Mn"]), alpha=1.0, interpolation="bilinear")
# sizebar = add_sizebar(ax, 500, eds_hd1['Mn'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"Mn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"O",
#     transform=ax.transAxes,
#     fontsize=8,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 图 P
subfig = fig.add_subfigure(gs[2, 3], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-1.95, -0.2, 1.0, 1.0))
# ax.set_box_aspect(1.0)

img = eds_hd1["Zn"].data
ax.imshow(img, cmap=transparent_single_color_cmap(r"k", eds_colors["Zn"]), alpha=1.0, interpolation="bilinear")
# sizebar = add_sizebar(ax, 500, eds_hd1['Zn'], "w")
# sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.02,
    0.98,
    r"Zn $\mathit{K \alpha}$",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
    color="w",
)
# ax.text(
#     0.10,
#     1.0,
#     r"P",
#     transform=ax.transAxes,
#     fontsize=8,
#     va="center",
#     ha="right",
#     fontfamily="Arial",
#     fontweight="bold",
# )

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_28_TEM_EDS_00_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_28_TEM_EDS_00_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_28_TEM_EDS_00_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_28_TEM_EDS_00_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 29 -- TEM-EDX - 对应的 Surface and bulk EDS 的结果

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        display_size = size
        display_units = bardata.axes_manager[0].units  # type: ignore
        if display_units == "nm" and size >= 1000 and size % 1000 == 0:
            display_size = size // 1000
            display_units = "µm"
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[0].scale,  # type: ignore
            "{} {}".format(display_size, display_units),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    results = []
    for file in data:
        if len(file.axes_manager.shape) >= 2:
            if len(file.axes_manager.navigation_shape) == 2:
                if file.axes_manager.navigation_axes[0].units == r"µm":
                    file.axes_manager.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(file.axes_manager.signal_shape) == 2:  # noqa: SIM102
                if file.axes_manager.signal_axes[0].units == r"µm":
                    file.axes_manager.convert_units(axes="signal", units="nm", same_units=True, factor=1000)

        if len(file.axes_manager.shape) == 3:
            if len(file.axes_manager.navigation_shape) == 2:
                for axis in file.axes_manager.navigation_axes:
                    axis.offset = 0
            elif len(file.axes_manager.signal_shape) == 2:
                for axis in file.axes_manager.signal_axes:
                    axis.offset = 0
        results.append(file)
    return results


def element_maps(signal_list, elements=(r"K", r"Mn", r"Zn"), crop_box=None):
    element_maps = {}
    for signal in signal_list:
        title = signal.metadata["General"]["title"]
        if title in elements:
            if crop_box:
                if signal.axes_manager.navigation_shape:
                    signal_crop = signal.inav[crop_box].copy()
                elif signal.axes_manager.signal_shape:
                    signal_crop = signal.isig[crop_box].copy()
            else:
                signal_crop = signal.copy()

            element_maps[title] = signal_crop
    return element_maps


eds_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def rois_plot(ax, roi_coords, data, textcolor=None, text_shift=([10, 5], [10, 5]), roi_name=None, edgecolor=None, zorder=2) -> None:
    x_axis = data.axes_manager["x"]
    y_axis = data.axes_manager["y"]
    scale_x = x_axis.scale
    scale_y = y_axis.scale
    offset_x = x_axis.offset
    offset_y = y_axis.offset
    for i, (left, right, top, bottom) in enumerate(roi_coords):
        # ROI coordinates are defined in notebook as calibrated coordinates (offset=0 frame).
        # hs.plot.plot_images uses calibrated axes with offsets, so shift by axis offsets here.
        x = left + offset_x
        y = top + offset_y
        width = right - left
        height = bottom - top

        # 绘制矩形
        rect = patches.Rectangle(
            (x, y),
            width,
            height,
            linewidth=1,
            edgecolor=edgecolor[i] if edgecolor else "y",
            facecolor="none",
            transform=ax.transData,
            zorder=zorder,
        )
        ax.add_patch(rect)

        # 添加文本标签
        ax.text(
            x + text_shift[i][0] * scale_x,
            y - text_shift[i][1] * scale_y,
            f"{roi_name[i]}" if roi_name else f"ROI {i + 1}",
            fontsize=9,
            color=textcolor[i] if textcolor else "y",
            ha="center",
            va="center",
            transform=ax.transData,
            zorder=zorder + 1,
        )


def rois_plot_pixels(ax, roi_coords, textcolor=None, text_shift=([10, 5], [10, 5]), roi_name=None, edgecolor=None, zorder=2) -> None:
    for i, (left, right, top, bottom) in enumerate(roi_coords):
        x = int(left)
        y = int(top)
        width = int(right - left)
        height = int(bottom - top)

        rect = patches.Rectangle(
            (x, y),
            width,
            height,
            linewidth=1,
            edgecolor=edgecolor[i] if edgecolor else "y",
            facecolor="none",
            transform=ax.transData,
            zorder=zorder,
        )
        ax.add_patch(rect)

        ax.text(
            x + text_shift[i][0],
            y - text_shift[i][1],
            f"{roi_name[i]}" if roi_name else f"ROI {i + 1}",
            fontsize=9,
            color=textcolor[i] if textcolor else "y",
            ha="center",
            va="center",
            transform=ax.transData,
            zorder=zorder + 1,
        )


def rois_plot_pixels_calibrated(
    ax,
    roi_coords,
    data,
    textcolor=None,
    text_shift=([10, 5], [10, 5]),
    roi_name=None,
    edgecolor=None,
    zorder=2,
) -> None:
    # hs.plot.plot_images uses calibrated axes (with offsets), so convert pixel ROI to calibrated coordinates.
    x_axis = data.axes_manager["x"]
    y_axis = data.axes_manager["y"]
    sx, ox = x_axis.scale, x_axis.offset
    sy, oy = y_axis.scale, y_axis.offset

    for i, (left, right, top, bottom) in enumerate(roi_coords):
        x = left * sx + ox
        y = top * sy + oy
        width = (right - left) * sx
        height = (bottom - top) * sy

        rect = patches.Rectangle(
            (x, y),
            width,
            height,
            linewidth=1,
            edgecolor=edgecolor[i] if edgecolor else "y",
            facecolor="none",
            transform=ax.transData,
            zorder=zorder,
        )
        ax.add_patch(rect)

        ax.text(
            x + text_shift[i][0] * sx,
            y - text_shift[i][1] * sy,
            f"{roi_name[i]}" if roi_name else f"ROI {i + 1}",
            fontsize=9,
            color=textcolor[i] if textcolor else "y",
            ha="center",
            va="center",
            transform=ax.transData,
            zorder=zorder + 1,
        )


In [ ]:
# 读取 EDS 的数据

path_FD1 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\EDS")
FD1 = hs.load(path_FD1.joinpath(r"Data", r"0003 - B8_HAADF_67000_x", r"0003 - B8_HAADF_67000_x.emd"))  # type: ignore

path_HC1 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EDS")
HC1_bulk = hs.load(path_HC1.joinpath(r"Data", r"0004-20241211_HAADF_29000_x_EDS", r"0004-20241211_HAADF_29000_x_EDS.emd"))  # type: ignore

HC1_surface = hs.load(path_HC1.joinpath(r"Data", r"0007-20241211_HAADF_115_kx_EDS", r"0007-20241211_HAADF_115_kx_EDS.emd"))  # type: ignore

path_HC2 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.63V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EDS")
HC2 = hs.load(path_HC2.joinpath(r"Data", r"0012-20241211_HAADF_58000_x_EDS", r"0012-20241211_HAADF_58000_x_EDS.emd"))  # type: ignore

path_FC1 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\EDS")
FC1 = hs.load(path_FC1.joinpath(r"Data", r"0020 - 1406 Pristine MnO2 HAADF 10000 x", r"0020 - 1406 Pristine MnO2 HAADF 10000 x.emd"))

path_HD1 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\EDS")
HD1_bulk = hs.load(path_HD1.joinpath(r"Data", r"0015-HAADF_23500_x_EDS_SI", r"0015-HAADF_23500_x_EDS_SI.emd"))  # type: ignore

HD1_surface = hs.load(path_HD1.joinpath(r"Data", r"0013-HAADF_33000_x_EDS_SI", r"0013-HAADF_33000_x_EDS_SI.emd"))  # type: ignore

path_FD2 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B2\EDS")
FD2 = hs.load(path_FD2.joinpath(r"Data", r"0028-HAADF_67000_x_EDS_SI", r"0028-HAADF_67000_x_EDS_SI.emd"))  # type: ignore


In [ ]:
FD1 = convert_units(FD1)
HC1_bulk = convert_units(HC1_bulk)
HC1_surface = convert_units(HC1_surface)
HC2 = convert_units(HC2)
FC1 = convert_units(FC1)
HD1_bulk = convert_units(HD1_bulk)
HD1_surface = convert_units(HD1_surface)
FD2 = convert_units(FD2)

FD1 = element_maps(FD1, elements=["K", "Mn", "Zn"])
HC1_bulk = element_maps(HC1_bulk, elements=["K", "Mn", "Zn"])
HC1_surface = element_maps(HC1_surface, elements=["K", "Mn", "Zn"])
HC2 = element_maps(HC2, elements=["K", "Mn", "Zn"])
FC1 = element_maps(FC1, elements=["K", "Mn", "Zn"])
HD1_bulk = element_maps(HD1_bulk, elements=["K", "Mn", "Zn"])
HD1_surface = element_maps(HD1_surface, elements=["K", "Mn", "Zn"])
FD2 = element_maps(FD2, elements=["K", "Mn", "Zn"])


In [ ]:
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = fig.add_gridspec(
    2,
    3,
    width_ratios=[1, 1, 1],
    height_ratios=[1, 1],
    wspace=0.0,
    hspace=0.0,
    
)
pos_single = (0.04, 0.07, 0.92, 0.86)
pos_pair_left = (0.03, 0.07, 0.45, 0.86)
pos_pair_right = (0.52, 0.07, 0.45, 0.86)
# Keep top aligned with other panels (top = 0.93), but shrink C/F footprint.
pos_tall = (0.05, 0.17, 0.62, 0.76)
pos_single_bottom = (0.04, 0.34, 0.92, 0.86)
pos_pair_left_bottom = (0.03, 0.34, 0.45, 0.86)
pos_pair_right_bottom = (0.52, 0.34, 0.45, 0.86)
pos_tall_bottom = (0.05, 0.44, 0.62, 0.76)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position(pos_single)
# ax.set_box_aspect(0.5)
ax.set_axis_off()
ax.set_anchor("N")

hs.plot.plot_images(
    [FD1["K"], FD1["Mn"], FD1["Zn"]],
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    ax,
    size=200,
    bardata=FD1["K"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=ax.transAxes)

ax.text(
    0.022,
    1.22,
    r"K",
    c="w",
    bbox={"ec": None, "fc": eds_colors["K"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)
ax.text(
    0.122,
    1.22,
    r"Mn",
    c="k",
    bbox={"ec": None, "fc": eds_colors["Mn"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)
ax.text(
    0.27,
    1.22,
    r"Zn",
    c="w",
    bbox={"ec": None, "fc": eds_colors["Zn"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=ax.transAxes,
    fontsize=10,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)

# 标记 Bulk 和 Surface
rois_plot_pixels_calibrated(
    ax,
    roi_coords=[(95, 142, 37, 47), (85, 143, 61, 67)],
    data=FD1["Mn"],
    textcolor=["w", "w"],
    text_shift=([15, 8], [25, -18]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)
ax.text(-0.04, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B Bulk
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position(pos_pair_left)
# ax.set_box_aspect(0.5)
ax.set_axis_off()
ax.set_anchor("N")

hs.plot.plot_images(
    [HC1_bulk["K"], HC1_bulk["Mn"], HC1_bulk["Zn"]],  # type: ignore
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=5000,
    bardata=HC1_bulk["K"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[
        (870, 1200, 870, 1150),
    ],
    data=HC1_bulk["Mn"],
    textcolor=[
        "w",
    ],
    text_shift=([15, 10],),
    roi_name=[
        r"Bulk",
    ],
    edgecolor=[
        "w",
    ],
    zorder=2,
)
ax.text(-0.04, 1.0, r"B", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B Surface
ax = subfig.add_subplot()
ax.set_position(pos_pair_right)
# ax.set_box_aspect(0.5)
ax.set_axis_off()
ax.set_anchor("N")

hs.plot.plot_images(
    [HC1_surface["K"], HC1_surface["Mn"], HC1_surface["Zn"]],  # type: ignore
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=200,
    bardata=HC1_surface["K"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[
        (80, 450, 50, 100),
    ],
    data=HC1_surface["Mn"],
    textcolor=[
        "w",
    ],
    text_shift=([25, 12],),
    roi_name=[
        r"Surface",
    ],
    edgecolor=[
        "w",
    ],
    zorder=2,
)

# 图 C
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position(pos_tall)
# ax.set_box_aspect(0.5)
ax.set_axis_off()
ax.set_anchor("N")

hs.plot.plot_images(
    [HC2["K"], HC2["Mn"], HC2["Zn"]],
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    ax,
    size=500,
    bardata=HC2["K"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[(350, 420, 600, 840), (170, 290, 50, 300)],
    data=HC2["Mn"],
    textcolor=["w", "w"],
    text_shift=([12, 8], [12, -85]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)
ax.text(-0.04, 1.0, r"C", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position(pos_single_bottom)
# ax.set_box_aspect(0.5)
ax.set_axis_off()
ax.set_anchor("N")

hs.plot.plot_images(
    [FC1["K"], FC1["Mn"], FC1["Zn"]],
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=5000,
    bardata=FC1["K"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[(1000, 1700, 2500, 2900), (2300, 2700, 1350, 1650)],
    data=FC1["Mn"],
    textcolor=["w", "w"],
    text_shift=([25, 20], [-10, 15]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)
ax.text(-0.04, 1.0, r"D", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 E Bulk
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position(pos_pair_left_bottom)
# ax.set_box_aspect(0.5)
ax.set_axis_off()
ax.set_anchor("N")

hs.plot.plot_images(
    [HD1_bulk["K"], HD1_bulk["Mn"], HD1_bulk["Zn"]],  # type: ignore
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=2000,
    bardata=HD1_bulk["K"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[
        (730, 1000, 2200, 2700),
    ],
    data=HD1_bulk["Mn"],
    textcolor=[
        "w",
    ],
    text_shift=([14, 10],),
    roi_name=[
        r"Bulk",
    ],
    edgecolor=[
        "w",
    ],
    zorder=2,
)
ax.text(-0.04, 1.0, r"E", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 E Surface
ax = subfig.add_subplot()
ax.set_position(pos_pair_right_bottom)
# ax.set_box_aspect(0.5)
ax.set_axis_off()
ax.set_anchor("N")

hs.plot.plot_images(
    [HD1_surface["K"], HD1_surface["Mn"], HD1_surface["Zn"]],  # type: ignore
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=2000,
    bardata=HD1_surface["K"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[
        (850, 1000, 290, 460),
    ],
    data=HD1_surface["Mn"],
    textcolor=[
        "w",
    ],
    text_shift=([25, 12],),
    roi_name=[
        r"Surface",
    ],
    edgecolor=[
        "w",
    ],
    zorder=2,
)

# 图 F
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position(pos_tall_bottom)
# ax.set_box_aspect(0.5)
ax.set_axis_off()
ax.set_anchor("N")

hs.plot.plot_images(
    [FD2["K"], FD2["Mn"], FD2["Zn"]],
    ax=ax,
    overlay=True,
    colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    ax,
    size=500,
    bardata=FD2["K"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)

# 标记 Bulk 和 Surface
rois_plot(
    ax,
    roi_coords=[(100, 300, 220, 320), (100, 300, 430, 500)],
    data=FD2["Mn"],
    textcolor=["w", "w"],
    text_shift=([4, 4], [7, 4]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)
ax.text(-0.04, 1.0, r"F", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_29_TEM_EDS_02_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_29_TEM_EDS_02_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_29_TEM_EDS_02_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_29_TEM_EDS_02_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 30 -- TEM-EELS - 常规电化学 + TEM-EELS 的各个状态 + 新相

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[0].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[0].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    for item in data:
        if isinstance(item, (list, tuple)):
            results.append([_process(s) for s in item])
        else:
            results.append(_process(item))
    return results


def element_maps(signal_list, elements=(r"K", r"Mn", r"Zn", r"O"), crop_box=None):
    element_maps = {}
    for signal in signal_list:
        if isinstance(signal, list):
            element_maps.setdefault("others", []).extend(signal)
            continue

        try:
            title = signal.metadata["General"]["title"]
        except (KeyError, AttributeError, TypeError):
            title = ""

        if not isinstance(title, str):
            title = str(title)

        match = re.search(r"\b([A-Z][a-z]?)\b\s+[^()]+", title) if title else None
        if not match:
            element_maps.setdefault("others", []).append(signal)
            continue

        element = match.group(1)
        if element in elements:
            if crop_box:
                if signal.axes_manager.navigation_shape:
                    signal_crop = signal.inav[crop_box].copy()
                elif signal.axes_manager.signal_shape:
                    signal_crop = signal.isig[crop_box].copy()
            else:
                signal_crop = signal.copy()

            element_maps[element] = signal_crop
    return element_maps


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def rois_plot(ax, roi_coords, data, textcolor=None, text_shift=None, roi_name=None, edgecolor=None, zorder=2) -> None:
    x_axis = data.axes_manager["x"]
    y_axis = data.axes_manager["y"]
    scale_x = x_axis.scale
    scale_y = y_axis.scale
    if text_shift is None:
        text_shift = [(10, 5)] * len(roi_coords)
    for i, (left, right, top, bottom) in enumerate(roi_coords):
        roi_label = roi_name[i] if roi_name is not None else f"ROI {i + 1}"
        edge_c = edgecolor[i] if edgecolor is not None else "y"
        text_c = textcolor[i] if textcolor is not None else "y"

        # ROI values come from RectangularROI left/right/top/bottom in TEM-EELS notebooks.
        # They are already in calibrated axis units used by hs.plot.plot_images.
        x = left
        y = top
        width = right - left
        height = bottom - top

        # 绘制矩形；若显式传入 None 则跳过该 ROI 边框。
        if edge_c is not None:
            rect = patches.Rectangle(
                (x, y),
                width,
                height,
                linewidth=1,
                edgecolor=edge_c,
                facecolor="none",
                transform=ax.transData,
                zorder=zorder,
            )
            ax.add_patch(rect)

        # 添加文本标签；若 roi_name/textcolor 对应项为 None 则跳过。
        if roi_label is not None and text_c is not None:
            ax.text(
                x + text_shift[i][0] * scale_x,
                y - text_shift[i][1] * scale_y,
                f"{roi_label}",
                fontsize=9,
                color=text_c,
                ha="center",
                va="center",
                transform=ax.transData,
                zorder=zorder + 1,
            )



In [ ]:
# 读取一个常规的 αMnO2 的数据
echem = []
path_filelist = list(Path.joinpath(path_data_root, r"Echem\αMnO2\GCD\1M ZnSO4 + 0.2M MnSO4").glob(r"LC*.xlsx"))
# 读取电化学数据
for path_file in path_filelist[0:1]:
    df = pd.read_excel(path_file, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echem.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echem)):
    echem[i] = echem[i][echem[i]["Cycle"].isin([0, 1, 2])]
    echem[i] = echem[i][echem[i]["Voltage/V"] >= 0]

# 读取 STEM-EELS 的数据
path_pristine = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Pristine\αMnO2\Powders\2023-EMCA\EELS\Results\SI1_37mm_5mm_09eV_2nm_50ms")
path_pristine_maps = list(path_pristine.joinpath(r"EELS Map").glob(r"*.dm4"))
pristine_maps = hs.load(path_pristine_maps)

path_FD1_maps = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\EELS\Results\SI2_80pA_10ms")
FD1_maps = hs.load(path_FD1_maps.joinpath(r"EELS Map").glob(r"*.dm4"))

path_HC1 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EELS\Results\SI 2_630mm_5mm_0.9eVCh_0.9nm_15ms_150pA")
HC1_maps = hs.load(path_HC1.joinpath(r"EELS Map").glob(r"*.dm4"))

path_HC1_new = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EELS\Results\SI 1_630mm_5mm_0.9eVCh_4nm_15ms_150pA")
HC1_maps_new = hs.load(path_HC1_new.joinpath(r"EELS Map").glob(r"*.dm4"))

path_HC2 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.63V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EELS\Results\SI 1_630mm_5mm_0.9eVCh_3nm_15ms_150pA")
HC2_maps = hs.load(path_HC2.joinpath(r"EELS Map").glob(r"*.dm4"))

path_HC2_new = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.63V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EELS\Results\SI 2_630mm_5mm_0.9eVCh_4nm_15ms_150pA")
HC2_maps_new = hs.load(path_HC2_new.joinpath(r"EELS Map").glob(r"*.dm4"))

path_FC1 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\EELS\Results\SI2_EFTEM400m_5mm_50ms_200pA_5nm")
FC1_maps = hs.load(path_FC1.joinpath(r"EELS Map").glob(r"*.dm4"))

path_FC1_new = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\EELS\Results\SI5_EFTEM400m_2nm_50ms_200pA_5nm")
FC1_maps_new = hs.load(path_FC1_new.joinpath(r"EELS Map").glob(r"*.dm4"))

path_HD1 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\EELS\Results\SI3_CL320mm_0.9eV_25ms_1.5nm step_120pA")
HD1_maps = hs.load(path_HD1.joinpath(r"EELS Map").glob(r"*.dm4"))

path_HD1_new = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\EELS\Results\SI1_CL320mm_0.9eV_25ms_2.5nm step_120pA")
HD1_maps_new = hs.load(path_HD1_new.joinpath(r"EELS Map").glob(r"*.dm4"))

path_FD2 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B2\EELS\Results\SI1_100pA_10ms_1.5nm")
FD2_maps = hs.load(path_FD2.joinpath(r"EELS Map").glob(r"*.dm4"))

# TEM-EELS 的比例
path_fc_ratio = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\TEM-EDS-EELS-all").joinpath(r"TEM-EELS_Bulk_Surface_NewPhase.xlsx")
fc_ratio = pd.read_excel(path_fc_ratio, sheet_name=1, header=0, index_col=None).dropna(axis=1, how="all")
# # 清洗数据
# bulk = fc_ratio[fc_ratio["Position"] == "Bulk"].copy(deep=True)
# surface = fc_ratio[fc_ratio["Position"] == "Surface"].copy(deep=True)


In [ ]:
pristine_maps = convert_units(pristine_maps)
FD1_maps = convert_units(FD1_maps)
HC1_maps = convert_units(HC1_maps)
HC1_maps_new = convert_units(HC1_maps_new)
HC2_maps = convert_units(HC2_maps)
HC2_maps_new = convert_units(HC2_maps_new)
FC1_maps = convert_units(FC1_maps)
FC1_maps_new = convert_units(FC1_maps_new)
HD1_maps = convert_units(HD1_maps)
HD1_maps_new = convert_units(HD1_maps_new)
FD2_maps = convert_units(FD2_maps)

pristine_maps = element_maps(pristine_maps, elements=["Mn", "O"], crop_box=(slice(0, 300), slice(None)))
FD1_maps = element_maps(FD1_maps, elements=["O", "Mn", "Zn"], crop_box=(slice(0, 180), slice(None)))
HC1_maps = element_maps(HC1_maps, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
HC1_maps_new = element_maps(HC1_maps_new, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
HC2_maps = element_maps(HC2_maps, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
HC2_maps_new = element_maps(HC2_maps_new, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
FC1_maps = element_maps(FC1_maps, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(0, 70)))
FC1_maps_new = element_maps(FC1_maps_new, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
HD1_maps = element_maps(HD1_maps, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
HD1_maps_new = element_maps(HD1_maps_new, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))
FD2_maps = element_maps(FD2_maps, elements=["O", "Mn", "Zn"], crop_box=(slice(None), slice(None)))


In [ ]:
pristine_rois = [(159, 278, 43, 60), (354, 552, 56, 126)]
FD1_rois = [(11, 53, 43, 60), (55, 93, 73, 81), (89, 124, 10, 26)]
HC1_rois = [(53, 72, 25, 64), (13, 21, 25, 60)]
HC1_rois_new = [(184, 360, 144, 324), (20, 120, 224, 324)]
HC2_rois = [(120, 159, 228, 330), (33, 72, 18, 129)]
HC2_rois_new = [(300, 470, 100, 160), (75, 251, 331, 391)]
FC1_rois = [(34, 67, 24, 31), (16, 56, 45, 59)]
FC1_rois_new = [(62, 98, 74, 106), (14, 30, 92, 126)]
HD1_rois = [(49, 65, 50, 80), (13, 35, 30, 59)]
HD1_rois_new = [(39, 84, 7, 47), (136, 164, 12, 57)]
FD2_rois = [(13, 60, 40, 59), (47, 110, 80, 94)]


In [ ]:
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0, hspace=0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.3)

ax.set_axis_off()
labels = [
    [None, None],
]
for i, data in enumerate(echem):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        if j == 0:
            ax.plot(
                temp.loc[idx_min:, "SpeCap/mAh/g"] + temp.loc[idx_min - 1, "SpeCap/mAh/g"],
                temp.loc[idx_min:, "Voltage/V"],
                ls="-",
                lw=1.0,
                c=colors[j],
                zorder=0,
            )  # 充电
        if j == 1:
            ax.plot(
                temp.loc[:idx_min, "SpeCap/mAh/g"] + 690,
                temp.loc[:idx_min, "Voltage/V"],
                ls="-",
                lw=1.0,
                c=colors[j],
                label=labels[i][j],
                zorder=0,
            )  # 放电

# 插入图 A
axins = ax.inset_axes((-0.03, 0.0, 0.4, 0.4), zorder=1)
hs.plot.plot_images(
    [FD1_maps["Mn"], FD1_maps["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    axins,
    size=20,
    bardata=FD1_maps["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.05, 0, 0), transform=axins.transAxes)
axins.annotate(
    r"FD#A",
    (-0.08, 0.05),
    xytext=(0.5, -0.14),
    fontsize=10,
    c=colors[0],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
axins.text(
    0.03,
    1.20,
    r"Mn",
    c="k",
    bbox={"ec": None, "fc": eels_colors["Mn"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=axins.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)
axins.text(
    0.21,
    1.20,
    r"Zn",
    c="w",
    bbox={"ec": None, "fc": eels_colors["Zn"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=axins.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=FD1_rois,
    data=FD1_maps["Mn"],
    textcolor=["w", "k", "w"],
    text_shift=([10, 10], [45, 8], [-20, -2]),
    roi_name=[r"Bulk", r"Surface", r"ZSH"],
    edgecolor=["w", "w", "w"],
    zorder=2,
)

# 插入图 B
axins = ax.inset_axes((0.18, 0.05, 0.4, 0.4), zorder=1)
hs.plot.plot_images(
    [HC1_maps["Mn"], HC1_maps["Zn"]],  # type: ignore
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    axins,
    size=20,
    bardata=HC1_maps["Mn"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"HC#B",
    (0.1, 1.55),
    xytext=(0.6, 1.1),
    fontsize=10,
    c=colors[0],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=90,angleB=0,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=HC1_rois,
    data=HC1_maps["Mn"],
    textcolor=["k", "w"],
    text_shift=([20, -55], [20, 10]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)

# 插入图 B1
axins = ax.inset_axes((0.16, -0.5, 0.4, 0.4), zorder=1)
hs.plot.plot_images(
    [HC1_maps_new["Mn"], HC1_maps_new["Zn"]],  # type: ignore
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    axins,
    size=500,
    bardata=HC1_maps_new["Mn"],  # type: ignore
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"New Phase",
    (0.1, 1.55),
    xytext=(1.0, 1.1),
    fontsize=10,
    c=colors[0],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops=None,
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=HC1_rois_new,
    data=HC1_maps_new["Mn"],
    textcolor=["k", "k"],
    text_shift=([20, 10], [15, 10]),
    roi_name=[r"#1", r"#2"],
    edgecolor=["w", "w"],
    zorder=2,
)


# 插入图 C
axins = ax.inset_axes((0.33, 0.05, 0.4, 0.5), zorder=1)
hs.plot.plot_images(
    [HC2_maps["Mn"], HC2_maps["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    axins,
    size=500,
    bardata=HC2_maps["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"HC#C",
    (0.0, 1.52),
    xytext=(0.7, 1.07),
    fontsize=10,
    c=colors[0],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=90,angleB=0,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=HC2_rois,
    data=HC2_maps["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, 10], [50, -10]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)

# 插入图 C1
axins = ax.inset_axes((0.33, -0.5, 0.4, 0.4), zorder=1)
hs.plot.plot_images(
    [HC2_maps_new["Mn"], HC2_maps_new["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    axins,
    size=500,
    bardata=HC2_maps_new["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.5, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"New Phase",
    (0.0, 1.52),
    xytext=(1.0, 1.1),
    fontsize=10,
    c=colors[0],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops=None,
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=HC2_rois_new,
    data=HC2_maps_new["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, 10], [30, 10]),
    roi_name=[r"#1", r"#2"],
    edgecolor=["w", "w"],
    zorder=2,
)


# 插入图 D
axins = ax.inset_axes((0.48, 0.6, 0.5, 0.5), zorder=1)
hs.plot.plot_images(
    [FC1_maps["Mn"], FC1_maps["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    axins,
    size=20,
    bardata=FC1_maps["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"FC#D",
    (-0.35, 0.7),
    xytext=(0.0, 0.8),
    fontsize=10,
    c=colors[0],
    rotation=90,
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=90,angleB=0,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=FC1_rois,
    data=FC1_maps["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, 6], [10, 6]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)

# 插入图 D1
axins = ax.inset_axes((0.46, 1.15, 0.5, 0.5), zorder=1)
hs.plot.plot_images(
    [FC1_maps_new["Mn"], FC1_maps_new["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    axins,
    size=50,
    bardata=FC1_maps_new["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.6, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"New Phase",
    (-0.35, 0.7),
    xytext=(0.0, 0.45),
    fontsize=10,
    c=colors[0],
    rotation=90,
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops=None,
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=FC1_rois_new,
    data=FC1_maps_new["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, 6], [10, 6]),
    roi_name=[r"#1", r"#2"],
    edgecolor=["w", "w"],
    zorder=2,
)


# 插入图 E
axins = ax.inset_axes((0.71, 0.7, 0.4, 0.4), zorder=1)

hs.plot.plot_images(
    [HD1_maps["Mn"], HD1_maps["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    axins,
    size=20,
    bardata=HD1_maps["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"HD#E",
    (-1.0, -0.65),
    xytext=(1.0, -0.15),
    fontsize=10,
    c=colors[1],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=90,angleB=0,rad=10",
        "ls": "--",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=HD1_rois,
    data=HD1_maps["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, 6], [10, 6]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)

# 插入图 E1
axins = ax.inset_axes((0.685, 1.25, 0.4, 0.4), zorder=1)

hs.plot.plot_images(
    [HD1_maps_new["Mn"], HD1_maps_new["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    axins,
    size=50,
    bardata=HD1_maps_new["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"New Phase",
    (-1.0, -0.65),
    xytext=(1.0, -0.15),
    fontsize=10,
    c=colors[1],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops=None,
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=HD1_rois_new,
    data=HD1_maps_new["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, -26], [10, -26]),
    roi_name=[r"#1", r"#2"],
    edgecolor=["w", "w"],
    zorder=2,
)


# 插入图 F
axins = ax.inset_axes((0.54, 0.0, 0.4, 0.4), zorder=1)
hs.plot.plot_images(
    [FD2_maps["Mn"], FD2_maps["Zn"]],
    ax=axins,
    overlay=True,
    colors=[eels_colors["Mn"], eels_colors["Zn"]],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100th",
)
scalebar = add_sizebar(
    axins,
    size=50,
    bardata=FD2_maps["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.75, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"FD#F",
    (1.255, 0.10),
    xytext=(1.0, -0.15),
    fontsize=10,
    c=colors[1],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops={
        "arrowstyle": "->",
        "connectionstyle": "angle,angleA=0,angleB=90,rad=10",
        "ls": "-",
        "lw": 1.0,
        "color": "grey",
    },
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=FD2_rois,
    data=FD2_maps["Mn"],
    textcolor=["w", "w"],
    text_shift=([10, 6], [15, 6]),
    roi_name=[r"Bulk", r"Surface"],
    edgecolor=["w", "w"],
    zorder=2,
)


# 插入图 G
axins = ax.inset_axes((0.05, 0.75, 0.35, 0.35), zorder=1)
hs.plot.plot_images(
    [
        pristine_maps["Mn"],
    ],
    ax=axins,
    overlay=True,
    colors=[
        eels_colors["Mn"],
    ],
    axes_decor="off",
    label=None,
    alphas=[1.0, 1.0, 1.0],
    vmax="100thh",
)
scalebar = add_sizebar(
    axins,
    size=100,
    bardata=pristine_maps["Mn"],
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.annotate(
    r"Pristine",
    (0.3, 1.42),
    xytext=(0.25, -0.15),
    fontsize=10,
    c=colors[2],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="normal",
    bbox=None,
    transform=ax.transAxes,
    ha="right",
    va="center",
    arrowprops=None,
)  # bbox=dict(boxstyle='round, pad=0.3', fc="w", ec=colors[0])
# 标记 Bulk 和 Surface
rois_plot(
    axins,
    roi_coords=pristine_rois,
    data=pristine_maps["Mn"],
    textcolor=[None, "w"],
    text_shift=([10, 10], [10, 10]),
    roi_name=[None, r"Bulk"],
    edgecolor=[None, "w"],
    zorder=2,
)

# ax.text(0.04, 1.5, r"A", transform=ax.transAxes, fontsize=8, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# # 图 B
# subfig = fig.add_subfigure(gs[1, 0:2], zorder=0)
# ax = subfig.add_subplot()
# ax.set_position((0.07, -0.3, 0.86, 0.86))
# ax.set_box_aspect(0.4)

# fc_ratio["Samples"] = fc_ratio["Samples"].astype(str).str.strip()
# fc_ratio["Position"] = fc_ratio["Position"].astype(str).str.strip()

# # 类别顺序
# order = ["Bulk", "Surface", "NewPhase"]
# fc_ratio["Position"] = pd.Categorical(fc_ratio["Position"], categories=order, ordered=True)

# # 数值列
# for col in ["Mn/O", "Zn/O"]:
#     fc_ratio[col] = pd.to_numeric(fc_ratio[col], errors="coerce")

# markers = {"Bulk": "o", "Surface": "s", "NewPhase": "*"}
# for pos in ["Bulk", "Surface"]:
#     sub = fc_ratio[fc_ratio["Position"] == pos]
#     if not sub.empty:
#         ax.plot(sub["Samples"], sub["Mn/O"], ls="-", marker=markers[pos], ms=10, c=colors[0], mec=colors[0], mfc=colors[0], zorder=2)

# # NewPhase 散点
# sub_np = fc_ratio[fc_ratio["Position"] == "NewPhase"]
# if not sub_np.empty:
#     ax.scatter(sub_np["Samples"], sub_np["Mn/O"], marker=markers["NewPhase"], s=100, c=colors[0], linewidths=0.5, edgecolors="w", zorder=10)

# ax.set_ylabel(r"Mn/O, Relative Ratio (arb.u.)", fontsize=6)
# ax.set_ylim(-0.05, 0.6)
# ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
# ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))

# # ax.set_xlabel("Sample")
# ax.tick_params(axis="x", rotation=45)
# ax.text(-0.1, 1.0, r"B", transform=ax.transAxes, fontsize=8, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# ax2 = ax.twinx()
# ax2.set_position((0.07, -0.3, 0.86, 0.86))
# ax2.set_box_aspect(0.4)

# ax.set_zorder(3)  # 左轴放在更上层
# ax2.set_zorder(2)  # 右轴放在下层
# ax.patch.set_visible(False)

# markers = {"Bulk": "o", "Surface": "s", "NewPhase": "*"}
# for pos in ["Bulk", "Surface"]:
#     sub = fc_ratio[fc_ratio["Position"] == pos]
#     if not sub.empty:
#         ax2.plot(sub["Samples"], sub["Zn/O"], ls="-", marker=markers[pos], ms=10, c=colors[1], mec=colors[1], mfc=colors[1], zorder=2)

# # NewPhase 散点
# if not sub_np.empty:
#     ax2.scatter(sub_np["Samples"], sub_np["Zn/O"], marker=markers["NewPhase"], s=100, linewidths=0.5, c=colors[1], edgecolors="w", zorder=10)

# ax2.set_ylabel("Zn/O, Relative Ratio (arb.u.)", fontsize=6)
# ax2.set_ylim(-0.02, 0.32)
# ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.1, offset=0))
# ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.05, offset=0))

# # 图例
# custom_handles = [
#     Line2D([0], [0], linestyle="-", marker=None, color=colors[0]),
#     Line2D([0], [0], linestyle="-", marker=None, color=colors[1]),
#     Line2D([0], [0], linestyle="None", marker=markers["Bulk"], color="grey"),
#     Line2D([0], [0], linestyle="None", marker=markers["Surface"], color="grey"),
#     Line2D([0], [0], linestyle="None", marker=markers["NewPhase"], color="grey"),
# ]
# custom_labels = ["Mn/O", "Zn/O", "Bulk", "Surface", "NewPhase"]

# legend1 = ax.legend(custom_handles[0:2], custom_labels[0:2], loc=(0.68, 1.2), ncol=1, frameon=False, fontsize=6, borderpad=0.7, handletextpad=0.4)
# legend2 = ax.legend(custom_handles[2:], custom_labels[2:], loc=(0.8, 1.12), ncol=1, frameon=False, fontsize=6, borderpad=0.7, handletextpad=0.4)

# ax.add_artist(legend1)
# ax.add_artist(legend2)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_30_TEM_EELS_00_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_30_TEM_EELS_00_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_30_TEM_EELS_00_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_30_TEM_EELS_00_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 31 -- XPS - Ratio

In [ ]:
# 读取数据
xps_dir = Path.joinpath(path_data_root, r"XPS+UPS")
xps_ratio_file = xps_dir.joinpath(r"XPS_Ratio.csv")

xps = pd.read_csv(xps_ratio_file, sep=",", header=0, index_col=0, comment="#")
xps = xps.apply(pd.to_numeric, errors="coerce")

weight_counts = {
    r"Mn (IV)": xps.iloc[0, :].values,
    r"Mn (III)": xps.iloc[1, :].values,
    r"Mn (II)": xps.iloc[2, :].values,
}


In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(3.0, 2.5))
gs = fig.add_gridspec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0)

# 图
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

species_map = {
    "Pristine": r"Pristine",
    "1st0.9V": r"FD#A",
    "1st1.53V": r"HC#B",
    "1st1.63V": r"HC#C",
    "1st1.80V": r"FC#D",
    "2nd1.30V": r"HD#E",
    "2nd0.9V": r"FD#F",
}
species = [species_map.get(col, str(col)) for col in xps.columns]

width = 0.5
bottom = np.zeros(len(species))
i = 0
for boolean, weight_count in weight_counts.items():
    p = ax.bar(species, weight_count, width, label=boolean, bottom=bottom, color=colors[i])
    bottom += weight_count
    i += 1

ax.set_ylabel(r"Relative Percentage (%)", fontsize=9)
ax.set_ylim(0, 120)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))
ax.tick_params(axis="both", which="both", labelsize=8)
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0.0, 1.0),
    ncols=3,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.8,
    handletextpad=0.5,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_31_XPS_07_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_31_XPS_07_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_31_XPS_07_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_31_XPS_07_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 32 -- UPS - wf + vb

In [ ]:
path_ups = Path(r"D:\PaperDos.20260304\Data\XPS+UPS\UPS_all-Dino.csv")
df_ups = pd.read_csv(path_ups)

sample_keys = [
    "FD#A",
    "HC#B",
    "HC#C",
    "FC#D",
    "HD#E",
    "FD#F",
]

sample_labels = [
    "FD#A",
    "HC#B",
    "HC#C",
    "FC#D",
    "HD#E",
    "FD#F",
]

wf_values = {
    "FD#A": 4.5681734,
    "HC#B": 4.4875584,
    "HC#C": 4.1186604,
    "FC#D": 4.3366985,
    "HD#E": 4.1662540,
    "FD#F": 4.8094215,
}

df_wf = {}
df_vb = {}
for i, key in enumerate(sample_keys):
    vb_x, vb_y, wf_x, wf_y = df_ups.columns[4 * i : 4 * i + 4]
    df_vb[key] = df_ups[[vb_x, vb_y]].dropna().rename(columns={vb_x: "B.E.", vb_y: "Intensity"})
    df_wf[key] = df_ups[[wf_x, wf_y]].dropna().rename(columns={wf_x: "B.E.", wf_y: "Intensity"})



In [ ]:
plt.close("all")
fig = plt.figure(figsize=(7.0, 5.0))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0, hspace=0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(1.5)

line_colors = [colors[(i + 1) % len(colors)] for i in range(len(sample_keys))]

for i, key in enumerate(sample_keys):
    d = df_wf[key]
    x = d["B.E."]
    y = d["Intensity"].to_numpy(dtype=float) + 50000*i
    ax.plot(x, y, color=line_colors[i], lw=1)
    ax.text(10.9, i * 50000 + 25000, sample_labels[i], fontsize=8, ha="left", va="center")

ax.set_xlabel(r"Binding Energy (eV)", fontsize=9)
ax.set_xlim(11, 5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.set_ylabel(r"Intensity (counts)", fontsize=9)
ax.set_ylim(-5000, 500000)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=100000, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=50000, offset=0))
formatter = ticker.ScalarFormatter()
formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
formatter.set_scientific(True)
ax.yaxis.set_major_formatter(formatter)
ax.tick_params(axis="both", which="both", bottom=True, labelbottom=True, left=True, labelleft=True, labelsize=8)
ax.text(-0.23, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((-0.05, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

wf_plot_values = [wf_values[k] for k in sample_keys]
bar_x = np.arange(len(sample_keys))
ax.bar(bar_x, wf_plot_values, color="#ff6b6b", edgecolor="#e55353", linewidth=1.2)
ax.set_xticks(bar_x)
ax.set_xticklabels(sample_keys, rotation=60, fontsize=8)

ax.set_ylabel("Work function (eV)", fontsize=9)
ax.set_ylim(4.1, 4.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(0.1))

ax.grid(axis="y", linestyle="-", linewidth=0.6, color="#d6deed")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", which="both", bottom=True, labelbottom=True, left=True, labelleft=True, labelsize=8)
ax.text(-0.15, 1.0, r"B", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.12, -0.35, 1.0, 1.0))
ax.set_box_aspect(1.1)

offset = [0, 0, 100000, 100000, 150000, 150000]

for i, key in enumerate(sample_keys):
    d = df_vb[key]
    m = (d["B.E."] >= -0.5) & (d["B.E."] <= 12.5)
    x = d.loc[m, "B.E."].to_numpy(dtype=float)
    y = d.loc[m, "Intensity"].to_numpy(dtype=float)
    ax.plot(x, y+offset[i], color=line_colors[i], lw=1.0)
    # ax.text(4.6, offset[i] + 4.0, sample_labels[i], fontsize=6, ha="left", va="center")

ax.set_xlabel(r"Binding Energy (eV)", fontsize=9)
ax.set_xlim(12.0, 0.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2.0, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1.0, offset=0))

ax.set_ylabel(r"Intensity (counts)", fontsize=9)
ax.set_ylim(-5000, 200000)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=50000, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=25000, offset=0))
formatter = ticker.ScalarFormatter()
formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
formatter.set_scientific(True)
ax.yaxis.set_major_formatter(formatter)
ax.tick_params(axis="both", which="both", bottom=True, labelbottom=True, left=True, labelleft=True, labelsize=8)
ax.text(-0.23, 1.0, r"C", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
subfig = fig.add_subfigure(gs[1, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.05, -0.35, 1.0, 1.0))
ax.set_box_aspect(1.1)

offset = [0, 0, 10000, 10000, 15000, 15000]
for i, key in enumerate(sample_keys):
    d = df_vb[key]
    m = (d["B.E."] >= -1.0) & (d["B.E."] <= 4.0)
    x = d.loc[m, "B.E."].to_numpy(dtype=float)
    y = d.loc[m, "Intensity"].to_numpy(dtype=float)
    ax.plot(x, y+offset[i], color=line_colors[i], lw=1.0)
    ax.hlines(offset[i], xmin=-1.0, xmax=2.2, color="gray", ls='--', lw=0.6, alpha=0.6)
    # ax.text(0.2, offset[i] + 0.08, sample_labels[i], fontsize=7, ha="left", va="bottom")

ax.set_xlabel(r"Binding Energy (eV)", fontsize=9)
ax.set_xlim(4.0, -1.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.set_ylabel(r"Intensity (counts)", fontsize=9)
ax.set_ylim(-2000, 25000)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500, offset=0))
formatter = ticker.ScalarFormatter()
formatter.set_powerlimits((-2, 2))  # 控制何时使用科学计数法的阈值
formatter.set_scientific(True)
ax.yaxis.set_major_formatter(formatter)
ax.tick_params(axis="both", which="both", bottom=True, labelbottom=True, left=True, labelleft=True, labelsize=8)
ax.text(-0.23, 1.0, r"D", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_32_UPS_00_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_32_UPS_00_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_32_UPS_00_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_32_UPS_00_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 33 -- XPS, 2nd0.9V

In [ ]:
# 读取数据
path_xps = Path.joinpath(path_data_root, r"XPS+UPS\ExSitu\αMnO2\Charge\Electrode\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\Results")
Mn_xps = pd.read_csv(
    Path.joinpath(path_xps, r"LC_Mn.txt"),
    sep="\t",
    skiprows=6,
    index_col=None,
    header=1,
    comment="#",
).dropna(axis=1, how="all")
Mn_xps = Mn_xps.iloc[:, 28:]

O_xps = pd.read_csv(Path.joinpath(path_xps, "LC_O.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
O_xps = O_xps.iloc[:, 7:]

Zn_xps = pd.read_csv(Path.joinpath(path_xps, "LC_Zn.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
Zn_xps = Zn_xps.iloc[:, 2:]


In [ ]:
# 数据处理, Mn
envelope_Mn_IV = Mn_xps.iloc[:, 2:10].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_III = Mn_xps.iloc[:, 10:18].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_II = Mn_xps.iloc[:, 18:26].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816


In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0)

# 图 Mn
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 27], label=r"Fit", ls="--", color=colors[1])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 26], label=r"Background", color=colors[2], zorder=2)  # background

ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_IV, color=colors[3], linestyle="--", label=r"Mn(IV)")  # envelope_Mn_IV
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_III, color=colors[5], linestyle="--", label=r"Mn(III)")  # envelope_Mn_III
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_II, color=colors[4], linestyle="--", label=r"Mn(II)", zorder=3)  # envelope_Mn_II

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 2:10], color=colors[3], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 2], y2=Mn_xps.iloc[:, 26], facecolor=colors[3], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 10:18], color=colors[5], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 10], y2=Mn_xps.iloc[:, 26], facecolor=colors[5], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 18:26], color=colors[4], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 18], y2=Mn_xps.iloc[:, 26], facecolor=colors[4], alpha=0.3)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0, 1.0),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.8,
    handletextpad=0.5,
)

ax.text(
    0.55,
    0.95,
    r"Mn(III): 31.2(9) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[5],
)
ax.text(
    0.55,
    0.87,
    r"Mn(IV): 65.2(7) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)
ax.text(
    0.55,
    0.79,
    r"Mn(II): 3.4(3) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)

ax.text(
    1.0,
    1.11,
    r"Mn $\mathit{2p_{3/2}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
)

ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=9,
)
ax.set_xlim(648.5, 638.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0.5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0.5))

ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=9,
)
ax.set_ylim(3000, 25000)
ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500, offset=0))
ax.tick_params(axis="both", which="both", labelsize=8)

# 图 O
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.4, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 1], ls="-", lw=1.0, c=colors[0], label="Data")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 6], ls="--", lw=1.0, c=colors[1], label="Fit")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 5], ls="-", lw=1.0, c=colors[2], label="Background", zorder=6)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 2], ls="-", lw=1.0, c=colors[3], label=r"Lattice Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 2], y2=O_xps.iloc[:, 5], facecolor=colors[3], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 3], ls="-", lw=1.0, c=colors[4], label="Hydroxide, Hydrated\n or Defective Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 3], y2=O_xps.iloc[:, 5], facecolor=colors[4], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 4], ls="-", lw=1.0, c=colors[5], label=r"Water")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 4], y2=O_xps.iloc[:, 5], facecolor=colors[5], alpha=0.5)

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(538.0, 527.0)
ax.set_ylim(1000, 25000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=9,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=9,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=1))
ax.tick_params(axis="both", which="both", labelsize=8)

ax.annotate(
    r"530.09 eV",
    (0.70, 0.82),
    xytext=(0.4, 0.90),
    fontsize=8,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[3],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[3], "ls": "--"},
)

ax.annotate(
    r"531.51 eV",
    (0.6, 0.20),
    xytext=(0.35, 0.3),
    fontsize=8,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[4],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[4], "ls": "--"},
)
ax.annotate(
    r"533.81 eV",
    (0.38, 0.12),
    xytext=(0.05, 0.20),
    fontsize=8,
    c=colors[5],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="bold",
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[5], "ls": "--"},
)
ax.text(
    1.0,
    1.10,
    r"O $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=10,
)

# 图 Zn
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.8, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Zn_xps.iloc[:, 0], Zn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(1018, 1050)
ax.set_ylim(11000, 15000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=9,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=9,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=1000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=500))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.tick_params(axis="both", which="both", labelsize=8)

ax.text(
    1.0,
    1.10,
    r"Zn $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=10,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_33_XPS_06_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_33_XPS_06_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_33_XPS_06_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_33_XPS_06_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 34 -- XPS, 2nd1.30V

In [ ]:
# 读取数据
path_xps = Path.joinpath(path_data_root, r"XPS+UPS\ExSitu\αMnO2\Charge\Electrode\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\Results")
Mn_xps = pd.read_csv(
    Path.joinpath(path_xps, r"LC_Mn.txt"),
    sep="\t",
    skiprows=6,
    index_col=None,
    header=1,
    comment="#",
).dropna(axis=1, how="all")
Mn_xps = Mn_xps.iloc[:, 28:]

O_xps = pd.read_csv(Path.joinpath(path_xps, "LC_O.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
O_xps = O_xps.iloc[:, 7:]

Zn_xps = pd.read_csv(Path.joinpath(path_xps, "LC_Zn.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
Zn_xps = Zn_xps.iloc[:, 2:]


In [ ]:
# 数据处理, Mn
envelope_Mn_IV = Mn_xps.iloc[:, 2:10].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_III = Mn_xps.iloc[:, 10:18].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_II = Mn_xps.iloc[:, 18:26].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816


In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0)

# 图 Mn
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 27], label=r"Fit", ls="--", color=colors[1])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 26], label=r"Background", color=colors[2], zorder=2)  # background

ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_IV, color=colors[3], linestyle="--", label=r"Mn(IV)")  # envelope_Mn_IV
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_III, color=colors[5], linestyle="--", label=r"Mn(III)")  # envelope_Mn_III
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_II, color=colors[4], linestyle="--", label=r"Mn(II)", zorder=3)  # envelope_Mn_II

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 2:10], color=colors[3], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 2], y2=Mn_xps.iloc[:, 26], facecolor=colors[3], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 10:18], color=colors[5], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 10], y2=Mn_xps.iloc[:, 26], facecolor=colors[5], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 18:26], color=colors[4], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 18], y2=Mn_xps.iloc[:, 26], facecolor=colors[4], alpha=0.3)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0, 1.0),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.8,
    handletextpad=0.5,
)

ax.text(
    0.55,
    0.95,
    r"Mn(III): 27.2(2) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[5],
)
ax.text(
    0.55,
    0.85,
    r"Mn(IV): 72.7(0) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)
ax.text(
    0.55,
    0.75,
    r"Mn(II): 0.7(1) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)

ax.text(
    1.0,
    1.11,
    r"Mn $\mathit{2p_{3/2}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=10,
)

ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=9,
)
ax.set_xlim(648.5, 638.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0.5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0.5))

ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=9,
)
ax.set_ylim(2000, 15000)
ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=3000, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1500, offset=0))
ax.tick_params(axis="both", which="both", labelsize=8)

# 图 O
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.4, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 1], ls="-", lw=1.0, c=colors[0], label="Data")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 6], ls="--", lw=1.0, c=colors[1], label="Fit")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 5], ls="-", lw=1.0, c=colors[2], label="Background", zorder=6)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 2], ls="-", lw=1.0, c=colors[3], label=r"Lattice Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 2], y2=O_xps.iloc[:, 5], facecolor=colors[3], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 3], ls="-", lw=1.0, c=colors[4], label="Hydroxide, Hydrated\n or Defective Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 3], y2=O_xps.iloc[:, 5], facecolor=colors[4], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 4], ls="-", lw=1.0, c=colors[5], label=r"Water")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 4], y2=O_xps.iloc[:, 5], facecolor=colors[5], alpha=0.5)

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(535.0, 526.0)
ax.set_ylim(1000, 30000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=9,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=9,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=1))
ax.tick_params(axis="both", which="both", labelsize=8)

# ax.annotate(
#     r"530.08 eV",
#     (0.65, 0.78),
#     xytext=(0.7, 0.90),
#     fontsize=6,
#     fontweight="bold",
#     xycoords="axes fraction",
#     textcoords="axes fraction",
#     c=colors[3],
#     arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[3], "ls": "--"},
# )

# ax.annotate(
#     r"531.04 eV",
#     (0.55, 0.20),
#     xytext=(0.35, 0.35),
#     fontsize=6,
#     fontweight="bold",
#     xycoords="axes fraction",
#     textcoords="axes fraction",
#     c=colors[4],
#     arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[4], "ls": "--"},
# )
# ax.annotate(
#     r"532.77 eV",
#     (0.28, 0.12),
#     xytext=(0.05, 0.25),
#     fontsize=6,
#     c=colors[5],
#     xycoords="axes fraction",
#     textcoords="axes fraction",
#     fontweight="bold",
#     arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[5], "ls": "--"},
# )
ax.text(
    1.0,
    1.10,
    r"O $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=10,
)

# 图 Zn
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.8, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Zn_xps.iloc[:, 0], Zn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(1018, 1050)
ax.set_ylim(7000, 9000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=9,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=9,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=500))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=250))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.tick_params(axis="both", which="both", labelsize=8)

ax.text(
    1.0,
    1.10,
    r"Zn $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=10,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_34_XPS_05_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_34_XPS_05_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_34_XPS_05_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_34_XPS_05_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 35 -- XPS, 1st1.80V

In [ ]:
# 读取数据
path_xps = Path.joinpath(path_data_root, r"XPS+UPS\ExSitu\αMnO2\Charge\Electrode\1st1.80V\1M ZnSO4 + 0.2M MnSO4\Results")
Mn_xps = pd.read_csv(
    Path.joinpath(path_xps, r"LC_Mn.txt"),
    sep="\t",
    skiprows=6,
    index_col=None,
    header=1,
    comment="#",
).dropna(axis=1, how="all")
Mn_xps = Mn_xps.iloc[:, 28:]

O_xps = pd.read_csv(Path.joinpath(path_xps, "LC_O.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
O_xps = O_xps.iloc[:, 7:]

Zn_xps = pd.read_csv(Path.joinpath(path_xps, "LC_Zn.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
Zn_xps = Zn_xps.iloc[:, 2:]


In [ ]:
# 数据处理, Mn
envelope_Mn_IV = Mn_xps.iloc[:, 2:10].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_III = Mn_xps.iloc[:, 10:18].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_II = Mn_xps.iloc[:, 18:26].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816


In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0)

# 图 Mn
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 27], label=r"Fit", ls="--", color=colors[1])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 26], label=r"Background", color=colors[2], zorder=2)  # background

ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_IV, color=colors[3], linestyle="--", label=r"Mn(IV)")  # envelope_Mn_IV
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_III, color=colors[5], linestyle="--", label=r"Mn(III)")  # envelope_Mn_III
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_II, color=colors[4], linestyle="--", label=r"Mn(II)", zorder=3)  # envelope_Mn_II

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 2:10], color=colors[3], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 2], y2=Mn_xps.iloc[:, 26], facecolor=colors[3], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 10:18], color=colors[5], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 10], y2=Mn_xps.iloc[:, 26], facecolor=colors[5], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 18:26], color=colors[4], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 18], y2=Mn_xps.iloc[:, 26], facecolor=colors[4], alpha=0.3)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0, 1.0),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.8,
    handletextpad=0.5,
)

ax.text(
    0.55,
    0.95,
    r"Mn(III): 33.0(4) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[5],
)
ax.text(
    0.55,
    0.85,
    r"Mn(IV): 58.4(5) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)
ax.text(
    0.55,
    0.75,
    r"Mn(II): 8.4(9) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)

ax.text(
    1.0,
    1.11,
    r"Mn $\mathit{2p_{3/2}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=10,
)

ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=9,
)
ax.set_xlim(648.5, 638.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0.5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0.5))

ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=9,
)
ax.set_ylim(3000, 23000)
ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000, offset=3000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500, offset=3000))
ax.tick_params(axis="both", which="both", labelsize=8)

# 图 O
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.4, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 1], ls="-", lw=1.0, c=colors[0], label="Data")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 6], ls="--", lw=1.0, c=colors[1], label="Fit")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 5], ls="-", lw=1.0, c=colors[2], label="Background", zorder=6)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 2], ls="-", lw=1.0, c=colors[3], label=r"Lattice Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 2], y2=O_xps.iloc[:, 5], facecolor=colors[3], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 3], ls="-", lw=1.0, c=colors[4], label="Hydroxide, Hydrated\n or Defective Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 3], y2=O_xps.iloc[:, 5], facecolor=colors[4], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 4], ls="-", lw=1.0, c=colors[5], label=r"Water")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 4], y2=O_xps.iloc[:, 5], facecolor=colors[5], alpha=0.5)

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(538.0, 526.0)
ax.set_ylim(1000, 25000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=9,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=9,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))
ax.tick_params(axis="both", which="both", labelsize=8)

ax.annotate(
    r"530.00 eV",
    (0.65, 0.78),
    xytext=(0.7, 0.90),
    fontsize=8,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[3],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[3], "ls": "--"},
)

ax.annotate(
    r"531.28 eV",
    (0.55, 0.20),
    xytext=(0.35, 0.35),
    fontsize=8,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[4],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[4], "ls": "--"},
)
ax.annotate(
    r"534.29 eV",
    (0.28, 0.12),
    xytext=(0.05, 0.25),
    fontsize=8,
    c=colors[5],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="bold",
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[5], "ls": "--"},
)
ax.text(
    1.0,
    1.10,
    r"O $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=10,
)

# 图 Zn
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.8, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Zn_xps.iloc[:, 0], Zn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(1018, 1050)
ax.set_ylim(10000, 13000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=9,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=9,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=1000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=500))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.tick_params(axis="both", which="both", labelsize=8)

ax.text(
    1.0,
    1.10,
    r"Zn $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=10,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_35_XPS_04_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_35_XPS_04_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_35_XPS_04_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_35_XPS_04_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 36 -- XPS, 1st1.63V

In [ ]:
# 读取数据
path_xps = Path.joinpath(path_data_root, r"XPS+UPS\ExSitu\αMnO2\Charge\Electrode\1st1.63V\1M ZnSO4 + 0.2M MnSO4\Results")
Mn_xps = pd.read_csv(
    Path.joinpath(path_xps, r"LC_Mn.txt"),
    sep="\t",
    skiprows=6,
    index_col=None,
    header=1,
    comment="#",
).dropna(axis=1, how="all")
Mn_xps = Mn_xps.iloc[:, 28:]

O_xps = pd.read_csv(Path.joinpath(path_xps, "LC_O.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
O_xps = O_xps.iloc[:, 7:]

Zn_xps = pd.read_csv(Path.joinpath(path_xps, "LC_Zn.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
Zn_xps = Zn_xps.iloc[:, 2:]


In [ ]:
# 数据处理, Mn
envelope_Mn_IV = Mn_xps.iloc[:, 2:10].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_III = Mn_xps.iloc[:, 10:18].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_II = Mn_xps.iloc[:, 18:26].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816


In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0)

# 图 Mn
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 27], label=r"Fit", ls="--", color=colors[1])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 26], label=r"Background", color=colors[2], zorder=2)  # background

ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_IV, color=colors[3], linestyle="--", label=r"Mn(IV)")  # envelope_Mn_IV
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_III, color=colors[5], linestyle="--", label=r"Mn(III)")  # envelope_Mn_III
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_II, color=colors[4], linestyle="--", label=r"Mn(II)", zorder=3)  # envelope_Mn_II

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 2:10], color=colors[3], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 2], y2=Mn_xps.iloc[:, 26], facecolor=colors[3], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 10:18], color=colors[5], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 10], y2=Mn_xps.iloc[:, 26], facecolor=colors[5], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 18:26], color=colors[4], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 18], y2=Mn_xps.iloc[:, 26], facecolor=colors[4], alpha=0.3)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0, 1.0),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.8,
    handletextpad=0.5,
)

ax.text(
    0.55,
    0.95,
    r"Mn(III): 12.1(9) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[5],
)
ax.text(
    0.55,
    0.85,
    r"Mn(IV): 76.8(4) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)
ax.text(
    0.55,
    0.75,
    r"Mn(II): 10.9(5) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)

ax.text(
    1.0,
    1.11,
    r"Mn $\mathit{2p_{3/2}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=10,
)

ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=9,
)
ax.set_xlim(648.5, 638.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0.5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0.5))

ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=9,
)
ax.set_ylim(3000, 24000)
ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000, offset=3000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500, offset=3000))
ax.tick_params(axis="both", which="both", labelsize=8)

# 图 O
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.4, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 1], ls="-", lw=1.0, c=colors[0], label="Data")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 6], ls="--", lw=1.0, c=colors[1], label="Fit")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 5], ls="-", lw=1.0, c=colors[2], label="Background", zorder=6)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 2], ls="-", lw=1.0, c=colors[3], label=r"Lattice Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 2], y2=O_xps.iloc[:, 5], facecolor=colors[3], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 3], ls="-", lw=1.0, c=colors[4], label="Hydroxide, Hydrated\n or Defective Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 3], y2=O_xps.iloc[:, 5], facecolor=colors[4], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 4], ls="-", lw=1.0, c=colors[5], label=r"Water")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 4], y2=O_xps.iloc[:, 5], facecolor=colors[5], alpha=0.5)

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(536.0, 528.0)
ax.set_ylim(1000, 25000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=9,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=9,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))
ax.tick_params(axis="both", which="both", labelsize=8)

ax.annotate(
    r"530.00 eV",
    (0.74, 0.78),
    xytext=(0.7, 0.90),
    fontsize=8,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[3],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[3], "ls": "--"},
)

ax.annotate(
    r"531.91 eV",
    (0.63, 0.20),
    xytext=(0.38, 0.35),
    fontsize=8,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[4],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[4], "ls": "--"},
)
ax.annotate(
    r"533.13 eV",
    (0.28, 0.14),
    xytext=(0.05, 0.30),
    fontsize=8,
    c=colors[5],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="bold",
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[5], "ls": "--"},
)
ax.text(
    1.0,
    1.10,
    r"O $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=10,
)

# 图 Zn
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.8, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Zn_xps.iloc[:, 0], Zn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(1018, 1050)
ax.set_ylim(11000, 14000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=9,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=9,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=1000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=500))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.tick_params(axis="both", which="both", labelsize=8)

ax.text(
    1.0,
    1.10,
    r"Zn $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=10,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_36_XPS_03_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_36_XPS_03_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_36_XPS_03_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_36_XPS_03_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 37 -- XPS, 1st1.53V

In [ ]:
# 读取数据
path_xps = Path.joinpath(path_data_root, r"XPS+UPS\ExSitu\αMnO2\Charge\Electrode\1st1.53V\1M ZnSO4 + 0.2M MnSO4\Results")
Mn_xps = pd.read_csv(
    Path.joinpath(path_xps, r"LC_Mn.txt"),
    sep="\t",
    skiprows=6,
    index_col=None,
    header=1,
    comment="#",
).dropna(axis=1, how="all")
Mn_xps = Mn_xps.iloc[:, 28:]

O_xps = pd.read_csv(Path.joinpath(path_xps, "LC_O.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
O_xps = O_xps.iloc[:, 7:]

Zn_xps = pd.read_csv(Path.joinpath(path_xps, "LC_Zn.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
Zn_xps = Zn_xps.iloc[:, 2:]


In [ ]:
# 数据处理, Mn
envelope_Mn_IV = Mn_xps.iloc[:, 2:10].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_III = Mn_xps.iloc[:, 10:18].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_II = Mn_xps.iloc[:, 18:26].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816


In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0)

# 图 Mn
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 27], label=r"Fit", ls="--", color=colors[1])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 26], label=r"Background", color=colors[2], zorder=2)  # background

ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_IV, color=colors[3], linestyle="--", label=r"Mn(IV)")  # envelope_Mn_IV
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_III, color=colors[5], linestyle="--", label=r"Mn(III)")  # envelope_Mn_III
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_II, color=colors[4], linestyle="--", label=r"Mn(II)", zorder=3)  # envelope_Mn_II

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 2:10], color=colors[3], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 2], y2=Mn_xps.iloc[:, 26], facecolor=colors[3], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 10:18], color=colors[5], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 10], y2=Mn_xps.iloc[:, 26], facecolor=colors[5], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 18:26], color=colors[4], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 18], y2=Mn_xps.iloc[:, 26], facecolor=colors[4], alpha=0.3)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0, 1.0),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.8,
    handletextpad=0.5,
)

ax.text(
    0.55,
    0.95,
    r"Mn(III): 36.8(1) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[5],
)
ax.text(
    0.55,
    0.85,
    r"Mn(IV): 59.2(6) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)
ax.text(
    0.55,
    0.75,
    r"Mn(II): 3.9(0) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)

ax.text(
    1.0,
    1.11,
    r"Mn $\mathit{2p_{3/2}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=10,
)

ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=9,
)
ax.set_xlim(648.5, 638.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0.5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0.5))

ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=9,
)
ax.set_ylim(3000, 23000)
ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000, offset=3000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500, offset=3000))
ax.tick_params(axis="both", which="both", labelsize=8)

# 图 O
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.4, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 1], ls="-", lw=1.0, c=colors[0], label="Data")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 6], ls="--", lw=1.0, c=colors[1], label="Fit")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 5], ls="-", lw=1.0, c=colors[2], label="Background", zorder=6)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 2], ls="-", lw=1.0, c=colors[3], label=r"Lattice Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 2], y2=O_xps.iloc[:, 5], facecolor=colors[3], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 3], ls="-", lw=1.0, c=colors[4], label="Hydroxide, Hydrated\n or Defective Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 3], y2=O_xps.iloc[:, 5], facecolor=colors[4], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 4], ls="-", lw=1.0, c=colors[5], label=r"Water")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 4], y2=O_xps.iloc[:, 5], facecolor=colors[5], alpha=0.5)

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(536.0, 528.0)
ax.set_ylim(1000, 25000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=9,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=9,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=5000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=2500))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0))
ax.tick_params(axis="both", which="both", labelsize=8)

ax.annotate(
    r"530.10 eV",
    (0.74, 0.78),
    xytext=(0.7, 0.90),
    fontsize=8,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[3],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[3], "ls": "--"},
)

ax.annotate(
    r"531.08 eV",
    (0.56, 0.20),
    xytext=(0.38, 0.35),
    fontsize=8,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[4],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[4], "ls": "--"},
)
ax.annotate(
    r"533.11 eV",
    (0.28, 0.14),
    xytext=(0.05, 0.30),
    fontsize=8,
    c=colors[5],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="bold",
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[5], "ls": "--"},
)
ax.text(
    1.0,
    1.10,
    r"O $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=10,
)

# 图 Zn
subfig = fig.add_subfigure(gs[0, 2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.8, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Zn_xps.iloc[:, 0], Zn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(1018, 1050)
ax.set_ylim(11000, 13000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=9,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=9,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=500))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=250))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.tick_params(axis="both", which="both", labelsize=8)

ax.text(
    1.0,
    1.10,
    r"Zn $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=10,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_37_XPS_02_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_37_XPS_02_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_37_XPS_02_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_37_XPS_02_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 38 -- XPS, Pristine

In [ ]:
# 读取数据
path_xps = Path.joinpath(path_data_root, r"XPS+UPS\ExSitu\αMnO2\Pristine\Powder\Results\2025-ICN2\V3")
Mn_xps = pd.read_csv(
    Path.joinpath(path_xps, r"2025_090_sample4_LC_Mn.txt"),
    sep="\t",
    skiprows=6,
    index_col=None,
    header=1,
    comment="#",
).dropna(axis=1, how="all")
Mn_xps = Mn_xps.iloc[:, 28:]

O_xps = pd.read_csv(Path.joinpath(path_xps, "2025_090_sample4_LC_O.txt"), sep="\t", skiprows=6, index_col=None, header=1, comment="#").dropna(axis=1, how="all")
O_xps = O_xps.iloc[:, 7:]


In [ ]:
# 数据处理, Mn
envelope_Mn_IV = Mn_xps.iloc[:, 2:10].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_III = Mn_xps.iloc[:, 10:18].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816
envelope_Mn_II = Mn_xps.iloc[:, 18:26].sum(axis=1) - (Mn_xps.iloc[:, 26] * 7)  # noqa: N816


In [ ]:
# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1], height_ratios=None, wspace=0, hspace=0)

# 图 Mn
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 1], label=r"Data", ls="-", color=colors[0])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 27], label=r"Fit", ls="--", color=colors[1])
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 26], label=r"Background", color=colors[2], zorder=2)  # background

ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_IV, color=colors[3], linestyle="--", label=r"Mn(IV)")  # envelope_Mn_IV
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_III, color=colors[5], linestyle="--", label=r"Mn(III)")  # envelope_Mn_III
ax.plot(Mn_xps.iloc[:, 0], envelope_Mn_II, color=colors[4], linestyle="--", label=r"Mn(II)", zorder=3)  # envelope_Mn_II

ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 2:10], color=colors[3], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 2], y2=Mn_xps.iloc[:, 26], facecolor=colors[3], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 10:18], color=colors[5], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 10], y2=Mn_xps.iloc[:, 26], facecolor=colors[5], alpha=0.3)
ax.plot(Mn_xps.iloc[:, 0], Mn_xps.iloc[:, 18:26], color=colors[4], alpha=0.5)
for i in range(8):
    ax.fill_between(x=Mn_xps.iloc[:, 0], y1=Mn_xps.iloc[:, i + 18], y2=Mn_xps.iloc[:, 26], facecolor=colors[4], alpha=0.3)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0, 1.0),
    ncols=1,
    frameon=False,
    labelcolor="linecolor",
    fontsize=8,
    columnspacing=0.8,
    handletextpad=0.5,
)

ax.text(
    0.55,
    0.95,
    r"Mn(III): 41.9(3) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[5],
)
ax.text(
    0.55,
    0.85,
    r"Mn(IV): 57.9(6) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)
ax.text(
    0.55,
    0.75,
    r"Mn(II): 0.1(3) at.%",
    horizontalalignment="left",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=8,
    color=colors[3],
)

ax.text(
    1.0,
    1.11,
    r"Mn $\mathit{2p_{3/2}}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=10,
)

ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=9,
)
ax.set_xlim(648.5, 638.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0.5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0.5))

ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=9,
)
ax.set_ylim(1000, 5000)
ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=1000, offset=1000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=500, offset=1000))
ax.tick_params(axis="both", which="both", labelsize=8)

# 图 O
subfig = fig.add_subfigure(gs[0, 1], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.4, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 1], ls="-", lw=1.0, c=colors[0], label="Data")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 6], ls="--", lw=1.0, c=colors[1], label="Fit")
ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 5], ls="-", lw=1.0, c=colors[2], label="Background", zorder=6)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 2], ls="-", lw=1.0, c=colors[3], label=r"Lattice Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 2], y2=O_xps.iloc[:, 5], facecolor=colors[3], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 3], ls="-", lw=1.0, c=colors[4], label="Hydroxide, Hydrated\n or Defective Oxide")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 3], y2=O_xps.iloc[:, 5], facecolor=colors[4], alpha=0.5)

ax.plot(O_xps.iloc[:, 0], O_xps.iloc[:, 4], ls="-", lw=1.0, c=colors[5], label=r"Water")
ax.fill_between(x=O_xps.iloc[:, 0], y1=O_xps.iloc[:, 4], y2=O_xps.iloc[:, 5], facecolor=colors[5], alpha=0.5)

ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
ax.set_xlim(536.5, 527.5)
ax.set_ylim(500, 6000)
ax.set_xlabel(
    r"Bind Energy (eV)",
    fontsize=9,
)
ax.set_ylabel(
    r"Intensity (counts)",
    fontsize=9,
)

ax.yaxis.set_major_formatter(ticker.EngFormatter(sep=" "))
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=1000))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=500))
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2, offset=0.5))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1, offset=0.5))
ax.tick_params(axis="both", which="both", labelsize=8)

ax.annotate(
    r"530.03 eV",
    (0.70, 0.75),
    xytext=(0.65, 0.90),
    fontsize=8,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[3],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[3], "ls": "--"},
)

ax.annotate(
    r"531.39 eV",
    (0.56, 0.20),
    xytext=(0.38, 0.35),
    fontsize=8,
    fontweight="bold",
    xycoords="axes fraction",
    textcoords="axes fraction",
    c=colors[4],
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[4], "ls": "--"},
)
ax.annotate(
    r"533.96 eV",
    (0.25, 0.14),
    xytext=(0.05, 0.30),
    fontsize=8,
    c=colors[5],
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontweight="bold",
    arrowprops={"arrowstyle": patches.ArrowStyle("->"), "color": colors[5], "ls": "--"},
)
ax.text(
    1.0,
    1.11,
    r"O $\mathit{1s}$",
    horizontalalignment="right",
    verticalalignment="top",
    transform=ax.transAxes,
    fontsize=10,
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_38_XPS_01_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_38_XPS_01_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_38_XPS_01_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_38_XPS_01_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()


## FigureSI 39 -- XRD - Operando XRD, EMD, being from pH Buffer, 1 M + 0.2 M, 30uA - V1

In [ ]:
# 读取数据

# Operando XRD, EMD, from pH Buffer, 1 M + 0.2 M, 30 uA - V1

# 1) 读取电化学数据
path_filelist = list(Path.joinpath(path_data_root, r"XRD\Operando\EMD\2025-MSPD\Results\IS16_C\Echem\A").glob(r"**\*.txt"))
echem_all = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break

    df = pd.read_csv(path_file, sep="	", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = pd.to_datetime(df["time/s"], format="mixed", errors="coerce")
    df["cycle number"] = pd.to_numeric(df["cycle number"], errors="coerce").astype("Int64")
    df["Voltage/V"] = df["Ewe/V"] - df["Ece/V"]
    echem_all.append(df)

echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 2) 读取谱线时间戳（按当前固定格式：Wave / Date / filename）
path_xrd = Path.joinpath(path_data_root, r"XRD\Operando\EMD\2025-MSPD\Results\IS16_C\Data")
path_time_spectrum = path_xrd.joinpath(r"Time_index_spectrum.csv.xlsx")

spectrum_time_all = pd.read_excel(path_time_spectrum)
required_cols = {"Wave", "Date", "filename"}

spectrum_time_all["spec_time"] = pd.to_datetime(
    spectrum_time_all["Date"],
    format=r"%Y-%m-%d_%H:%M:%S",
    errors="coerce",
)

spectrum_time_all["spectrum_col"] = spectrum_time_all["filename"].astype(str).str.strip().str.replace(r"\.xye$", "", regex=True)
spectrum_time_all = spectrum_time_all.dropna(subset=["spec_time", "spectrum_col"]).sort_values("spec_time").reset_index(drop=True)

# 3) 读取 XRD 全谱数据
path_dspacing = path_xrd.joinpath(r"spectrum_all_d_spacing.csv")

spectrum_all = pd.read_csv(path_dspacing, index_col=0, header=0)
spectrum_all.index = pd.to_numeric(spectrum_all.index, errors="coerce")
spectrum_all = spectrum_all.loc[~spectrum_all.index.isna(), :]
spectrum_all.columns = [str(c).strip().replace(".xye", "") for c in spectrum_all.columns]


In [ ]:
# 过滤 16_Ca_* 谱线并与时间索引对齐（后续单元直接使用）

# 1) 时间表仅保留 16_Ca_fxxxxx
spectrum_time_all["spectrum_col"] = spectrum_time_all["spectrum_col"].astype(str).str.strip().str.replace(r"\.xye$", "", regex=True)
mask_ca = spectrum_time_all["spectrum_col"].str.match(r"^16_Ca_f\d+$", na=False)
spectrum_time_all = spectrum_time_all.loc[mask_ca].copy().sort_values("spec_time").reset_index(drop=True)

# 2) 谱图矩阵仅保留 16_Ca_fxxxxx 列
spectrum_all.columns = [str(c).strip().replace(".xye", "") for c in spectrum_all.columns]
spec_cols_ca = [c for c in spectrum_all.columns if c.startswith("16_Ca_f")]
spectrum_all = spectrum_all.loc[:, spec_cols_ca].copy()

# 3) 取交集并按时间顺序重排，确保 time/spectrum 完全对齐
common_cols = [c for c in spectrum_time_all["spectrum_col"].tolist() if c in spectrum_all.columns]

spectrum_time_all = spectrum_time_all[spectrum_time_all["spectrum_col"].isin(common_cols)].copy()
spectrum_time_all = spectrum_time_all.drop_duplicates(subset=["spectrum_col"], keep="first").sort_values("spec_time").reset_index(drop=True)
spectrum_all = spectrum_all.reindex(columns=spectrum_time_all["spectrum_col"].tolist())

In [ ]:
# 清洗数据 + 建立谱线时间映射

# 1) 清理 echem_all 起始异常段,估计了一下 index.
echem_all.drop(echem_all.index[:7050], inplace=True)

# 2) 只保留第一圈充放电与第二圈充电
selected_echem = echem_all[echem_all["cycle number"].isin([1, 2])]
selected_echem = selected_echem[selected_echem["Voltage/V"] >= 0.8].copy()
selected_echem = selected_echem.dropna(subset=["time/s", "Voltage/V", "<I>/mA"]).sort_values(by="time/s").reset_index(drop=True)
selected_echem["charge_time"] = (selected_echem["time/s"] - selected_echem["time/s"].iloc[0]).dt.total_seconds() / 3600

# 3) 最近邻时间匹配：谱线 -> 电化学
selected_spectrum_time = (
    pd.merge_asof(
        spectrum_time_all[["spectrum_col", "spec_time"]].sort_values(by="spec_time"),
        selected_echem[["time/s", "Voltage/V", "<I>/mA", "charge_time"]].sort_values(by="time/s"),
        left_on="spec_time",
        right_on="time/s",
        direction="nearest",
        tolerance=pd.Timedelta("5s"),
    )
    .dropna(subset=["spectrum_col", "spec_time", "time/s", "Voltage/V", "charge_time"], inplace=False)
    .sort_values(by="spec_time")
    .reset_index(drop=True)
)

# 4) 统一时间零点
time_zero = selected_echem["time/s"].iloc[0]
selected_spectrum_time["spec_charge_time"] = (
    selected_spectrum_time["spec_time"] - time_zero
).dt.total_seconds() / 3600
selected_spectrum_time["spec_time_h"] = selected_spectrum_time["spec_charge_time"]
selected_spectrum_time["map_dt_s"] = (
    selected_spectrum_time["time/s"] - selected_spectrum_time["spec_time"]
).dt.total_seconds().abs()
selected_spectrum_time["zero_consistency_dt_s"] = (
    selected_spectrum_time["charge_time"] - selected_spectrum_time["spec_charge_time"]
).abs() * 3600

# 5) 选择 d-spacing 区间并建立谱线-时间映射
d_spacing_range = (0.5, 14.01)
spectrum_all = spectrum_all.loc[(spectrum_all.index >= d_spacing_range[0]) & (spectrum_all.index <= d_spacing_range[1])].copy()

# 按当前数据格式：以 filename 作为谱线列键进行映射
selected_spectrum_time["spectrum_col"] = selected_spectrum_time["spectrum_col"].astype(str).str.strip().str.replace(r"\.xye$", "", regex=True)
selected_spectrum_time = selected_spectrum_time.dropna(subset=["spectrum_col"]).copy()

selected_cols = [col for col in spectrum_all.columns if col in set(selected_spectrum_time["spectrum_col"]) ]
selected_spectrum = spectrum_all.loc[:, selected_cols]
selected_spectrum_time = (
    selected_spectrum_time[selected_spectrum_time["spectrum_col"].isin(selected_spectrum.columns)]
    .drop_duplicates(subset=["spectrum_col"], keep="first")
    .sort_values(by="spec_charge_time")
    .reset_index(drop=True)
)

selected_spectrum_plot = selected_spectrum.reindex(columns=selected_spectrum_time["spectrum_col"].to_list())
selected_spectrum_time["range_idx"] = pd.Series(np.arange(1, len(selected_spectrum_time) + 1), dtype="Int64")
spec_time_arr = selected_spectrum_time["spec_time_h"].to_numpy(dtype=float)

# 统一时间显示窗口：覆盖电化学与谱线时间，并上下留 15 min
pad_h = 0.25
time_min = min(float(selected_echem["charge_time"].min()), float(spec_time_arr.min()))
time_max = max(float(selected_echem["charge_time"].max()), float(spec_time_arr.max()))
y_min_plot = time_min - pad_h
y_max_plot = time_max + pad_h

# 时间边界（给后续扩展绘图使用）
spectrum_time_edges = np.empty(spec_time_arr.size + 1, dtype=float)
if spec_time_arr.size == 1:
    spectrum_time_edges[0] = spec_time_arr[0] - 0.5
    spectrum_time_edges[1] = spec_time_arr[0] + 0.5
else:
    spectrum_time_edges[1:-1] = 0.5 * (spec_time_arr[:-1] + spec_time_arr[1:])
    spectrum_time_edges[0] = spec_time_arr[0] - 0.5 * (spec_time_arr[1] - spec_time_arr[0])
    spectrum_time_edges[-1] = spec_time_arr[-1] + 0.5 * (spec_time_arr[-1] - spec_time_arr[-2])


In [ ]:
%matplotlib inline
plt.close("all")

# 画图
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 4, 4], height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Voltage/V"], selected_echem["charge_time"], ls="-", lw=1.0, c=colors[0], label=r"Voltage")

# 添加索引文本标注
# index_labels = [1, 30, 74, 110, 148, 185, 220, 255, 278]
index_labels = [1, 16, 36, 53, 71, 88, 108, 124, 135]

range_idx_int = pd.to_numeric(selected_spectrum_time["range_idx"], errors="coerce").astype("Int64")
special_mask = range_idx_int.isin(index_labels)

plot_points = selected_spectrum_time.copy()
plot_points["is_special"] = special_mask.to_numpy()
special_points = plot_points[plot_points["is_special"]].copy()

for _, row in plot_points.iterrows():
    if bool(row["is_special"]):
        ax.scatter(
            row["Voltage/V"],
            row["spec_time_h"],
            c=colors[0],
            edgecolors="face",
            marker="*",
            s=50,
            zorder=5,
        )
    else:
        ax.scatter(
            row["Voltage/V"],
            row["spec_time_h"],
            c="lightgrey",
            edgecolors="face",
            s=30,
            zorder=1,
        )
ax.set_xlabel(
    r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$",
    fontsize=9,
)
ax.set_xlim(0.95, 1.65)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.35, offset=-0.1))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.175, offset=-0.1))

ax.set_ylabel(r"Duration Time (hour)", fontsize=9)
ax.set_ylim(y_min_plot, y_max_plot)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))
ax.tick_params(
    axis="both",
    which="both",
    labelcolor="k",
    bottom=True,
    left=True,
    labelbottom=True,
    labelleft=True,
    labelsize=8,
)

ax2 = ax.twiny()
ax2.set_position((0, 0, 1, 1))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"] * 1000, selected_echem["charge_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(
    r"$\mathrm{Current  \ (\mu A)}$",
    fontsize=11,
)
ax2.set_xlim(-60, 60)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=30, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=15, offset=0))
ax2.tick_params(axis="both", which="both", labelsize=9, right=True, labelright=True)
ax2.set_ylim(y_min_plot, y_max_plot)

ax2.tick_params(
    axis="both",
    which="both",
    labelcolor="k",
    top=True,
    labeltop=True,
    labelsize=8,
)

ax.text(-0.6, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.05, 0.035, 0.94, 0.94))
ax.set_box_aspect(1.0)

scan_cols_plot = selected_spectrum_time["spectrum_col"].astype(str).to_list()
selected_spectrum_plot = selected_spectrum.reindex(columns=scan_cols_plot)
spec_time_arr = selected_spectrum_time["spec_time_h"].to_numpy(dtype=float)
d_axis = selected_spectrum_plot.index.to_numpy(dtype=float)
special_y_time = special_points["spec_time_h"].dropna().astype(float).tolist()

def _axis_edges(arr):
    arr = np.asarray(arr, dtype=float)
    edges = np.empty(arr.size + 1, dtype=float)
    if arr.size == 1:
        edges[0] = arr[0] - 0.5
        edges[1] = arr[0] + 0.5
    else:
        edges[1:-1] = 0.5 * (arr[:-1] + arr[1:])
        edges[0] = arr[0] - 0.5 * (arr[1] - arr[0])
        edges[-1] = arr[-1] + 0.5 * (arr[-1] - arr[-2])
    return edges

z_main = selected_spectrum_plot.T.to_numpy(dtype=float)
t_edges = _axis_edges(spec_time_arr)
d_edges = _axis_edges(d_axis)
vmin_main = float(np.nanpercentile(z_main, 2.0))
vmax_main = float(np.nanpercentile(z_main, 99.5))
if vmax_main <= vmin_main:
    vmax_main = vmin_main + 1.0
main_cmap = "coolwarm"
main_norm = mpl.colors.Normalize(vmin=vmin_main, vmax=vmax_main)

pcm_main = ax.pcolormesh(
    d_axis,
    spec_time_arr,
    z_main,
    cmap=main_cmap,
    norm=main_norm,
    shading="gouraud",
)

ax.set_yticks([])
ax.set_ylim(float(spec_time_arr.min()), float(spec_time_arr.max()))
ax.set_xlim(float(d_axis.min()), float(d_axis.max()))
ax.set_xlabel(
    r"$\mathrm{d \ Spacing \ (\AA)}$",
    fontsize=9,
)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=2.0, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=1.0, offset=0))
ax.tick_params(axis="x", which="both", direction="out", labelsize=8, top=False)

# Add colorbar and adjust ticks afterwards
colorbar = fig.colorbar(
    pcm_main,
    cax=ax.inset_axes((1.03, 0.1, 0.08, 0.8)),
    orientation="vertical",
)
colorbar.set_ticks([])
ax.text(
    1.08,
    0.97,
    "High",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="center",
    fontfamily="Arial",
)
ax.text(
    1.08,
    0.08,
    "Low",
    transform=ax.transAxes,
    fontsize=8,
    va="top",
    ha="center",
    fontfamily="Arial",
)

ax.text(-0.05, 1.0, r"B", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C

subfig = fig.add_subfigure(gs[0, 2], zorder=0)

opxrd_ref_d_spacing = [[8.01], [5.98], [4.62], [2.48], [1.43], [1.15]]
opxrd_colors = ["blue", "blue", "blue", "blue", "blue", "blue"]
vmin_vmax = [(None, None), (None, None), (None, None), (None, None), (None, None), (None, None)]
for i in range(len(opxrd_ref_d_spacing)):
    ax = subfig.add_axes((0.1 + i * 0.14, 0.035, 0.12, 0.94), zorder=0)
    temp = selected_spectrum_plot.loc[(selected_spectrum_plot.index >= opxrd_ref_d_spacing[i][0] - 0.4) & (selected_spectrum_plot.index <= opxrd_ref_d_spacing[i][0] + 0.4)]
    if temp.empty:
        continue
    temp_d = temp.index.to_numpy(dtype=float)
    temp_z = temp.T.to_numpy(dtype=float)
    temp_d_edges = _axis_edges(temp_d)
    local_vmin = vmin_vmax[i][0] if vmin_vmax[i][0] is not None else float(np.nanpercentile(temp_z, 2.0))
    local_vmax = vmin_vmax[i][1] if vmin_vmax[i][1] is not None else float(np.nanpercentile(temp_z, 99.5))
    if local_vmax <= local_vmin:
        local_vmax = local_vmin + 1.0
    local_norm = mpl.colors.Normalize(vmin=local_vmin, vmax=local_vmax)
    ax.pcolormesh(temp_d, spec_time_arr, temp_z, cmap=main_cmap, norm=local_norm, shading="gouraud")
    # ax.axvline(opxrd_ref_d_spacing[i][0], ls="--", lw=1.0, color="grey", alpha=0.3)
    # ax.text(0.9 + 0.01 * i, -0.15, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(float(spec_time_arr.min()), float(spec_time_arr.max()))
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_d_spacing[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(
        axis="both",
        which="both",
        labelcolor="k",
        bottom=True,
        left=True,
        labelbottom=True,
        labelleft=True,
        labelsize=8,
    )
    for y_sp in special_y_time:
        ax.axhline(y=y_sp, lw=0.6, ls="--", color=colors[0], alpha=0.5)
ax.text(-1.0, -0.15, r"$\mathrm{d \ Spacing\ (\AA)}$", transform=ax.transAxes, fontsize=9, va="center", ha="right", fontfamily="Arial", fontweight="bold", c="blue")

ax.text(-6.0, 1.0, r"C", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_39_XRD_03_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_39_XRD_03_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_39_XRD_03_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
# plt.savefig(
#     Path.joinpath(path_out, r"PaperDos_FigureSI_39_XRD_03_V1_900.svg"),
#     pad_inches=0.05,
#     bbox_inches="tight",
#     dpi=900,
#     transparent=False,
# )

plt.gcf().set_facecolor("white")
plt.show()


In [ ]:
%matplotlib inline
plt.close("all")
# gridspec inside gridspec
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 0.3, 0.3))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Voltage/V"], selected_echem["charge_time"], ls="-", lw=1.0, c=colors[0], label=r"Voltage")

# 添加索引文本标注
# index_labels = [1, 30, 74, 110, 148, 185, 220, 255, 278]
index_labels = [1, 16, 36, 53, 71, 88, 108, 124, 135]
range_idx_int = pd.to_numeric(selected_spectrum_time["range_idx"], errors="coerce").astype("Int64")
special_mask = range_idx_int.isin(index_labels)

plot_points = selected_spectrum_time.copy()
plot_points["is_special"] = special_mask.to_numpy()
special_points = plot_points[plot_points["is_special"]].copy()

for _, row in plot_points.iterrows():
    if bool(row["is_special"]):
        ax.scatter(
            row["Voltage/V"],
            row["spec_time_h"],
            c=colors[0],
            edgecolors="face",
            marker="*",
            s=50,
            zorder=5,
        )
    else:
        ax.scatter(
            row["Voltage/V"],
            row["spec_time_h"],
            c="lightgrey",
            edgecolors="face",
            s=30,
            zorder=1,
        )
ax.set_xlabel(
    r"Voltage (V)",
    fontsize=9,
)
ax.set_xlim(0.95, 1.65)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.7, offset=0.25))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.35, offset=0.25))

ax.set_ylabel(r"Duration Time (hour)", fontsize=9)
ax.set_ylim(y_min_plot, y_max_plot)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))
ax.tick_params(
    axis="both",
    which="both",
    labelcolor="k",
    bottom=True,
    left=True,
    labelbottom=True,
    labelleft=True,
    labelsize=8,
)

ax2 = ax.twiny()
ax2.set_position((0, 0, 0.3, 0.3))
ax2.set_box_aspect(3.0)

ax2.plot(selected_echem["<I>/mA"] * 1000, selected_echem["charge_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")

ax2.set_xlabel(
    r"$\mathrm{Current  \ (\mu A)}$",
    fontsize=9,
)
ax2.set_xlim(-60, 60)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=60, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=30, offset=0))
ax2.set_ylim(y_min_plot, y_max_plot)
ax2.tick_params(
    axis="both",
    which="both",
    labelcolor="k",
    top=True,
    labeltop=True,
    labelsize=8,
)

ax.text(-1.0, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 0], zorder=0)

scan_cols_plot = selected_spectrum_time["spectrum_col"].astype(str).to_list()
selected_spectrum_plot = selected_spectrum.reindex(columns=scan_cols_plot)
spec_time_arr = selected_spectrum_time["spec_time_h"].to_numpy(dtype=float)
special_y_time = special_points["spec_time_h"].dropna().astype(float).tolist()

opxrd_ref_d_spacing = [
    [10.93],
    [5.47],
    [4.17],
    [3.51],
    [3.20],
    [2.85],
    [2.74],
    [2.70],
    [2.59],
    [2.47],
    [2.33],
    [2.21],
    [2.06],
    [1.96],
    [1.86],
    [1.81],
    [1.78],
    [1.70],
    [1.65],
    [1.60],
    [1.56],
    [1.47],
    [1.30],
    [1.29],
    [1.15],
]
vmin_vmax = [
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
    (None, None),
]
n_ref = len(opxrd_ref_d_spacing)
for i in range(0, n_ref // 3-1):
    # print(i)
    ax = subfig.add_axes((0.23 + i * 0.09, 0.0, 0.08, 0.3), zorder=0)
    temp = selected_spectrum_plot.loc[(selected_spectrum_plot.index >= opxrd_ref_d_spacing[i][0] - 0.4) & (selected_spectrum_plot.index <= opxrd_ref_d_spacing[i][0] + 0.4)]
    if temp.empty:
        continue
    temp_d_grid, temp_t_grid = np.meshgrid(temp.index.to_numpy(dtype=float), spec_time_arr)
    ax.pcolormesh(temp_d_grid, temp_t_grid, temp.T.to_numpy(dtype=float), cmap="coolwarm", shading="gouraud", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1])
    # ax.axvline(opxrd_ref_d_spacing[i][0], ls="--", lw=1.0, color="grey", alpha=0.3)
    # ax.text(0.75 + 0.001 * i, -0.32, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(float(spec_time_arr.min()), float(spec_time_arr.max()))
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_d_spacing[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, labelbottom=True)
    for y_sp in special_y_time:
        ax.axhline(y=y_sp, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(n_ref // 3-1, 2 * n_ref // 3):
    # print(i)
    ax = subfig.add_axes((0.05 + (i - n_ref // 3 + 1) * 0.09, -0.43, 0.08, 0.3), zorder=0)
    temp = selected_spectrum_plot.loc[(selected_spectrum_plot.index >= opxrd_ref_d_spacing[i][0] - 0.4) & (selected_spectrum_plot.index <= opxrd_ref_d_spacing[i][0] + 0.4)]
    if temp.empty:
        continue
    temp_d_grid, temp_t_grid = np.meshgrid(temp.index.to_numpy(dtype=float), spec_time_arr)
    ax.pcolormesh(temp_d_grid, temp_t_grid, temp.T.to_numpy(dtype=float), cmap="coolwarm", shading="gouraud", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1])
    # ax.axvline(opxrd_ref_d_spacing[i][0], ls="--", lw=1.0, color="grey", alpha=0.3)
    # ax.text(0.75 + 0.001 * i, -0.23, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(float(spec_time_arr.min()), float(spec_time_arr.max()))
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_d_spacing[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, labelbottom=True)
    for y_sp in special_y_time:
        ax.axhline(y=y_sp, lw=0.6, ls="--", color=colors[0], alpha=0.5)

for i in range(2 * n_ref // 3, n_ref):
    # print(i)
    ax = subfig.add_axes((0.05 + (i - 2 * n_ref // 3) * 0.09, -0.82, 0.08, 0.3), zorder=0)
    temp = selected_spectrum_plot.loc[(selected_spectrum_plot.index >= opxrd_ref_d_spacing[i][0] - 0.4) & (selected_spectrum_plot.index <= opxrd_ref_d_spacing[i][0] + 0.4)]
    if temp.empty:
        continue
    temp_d_grid, temp_t_grid = np.meshgrid(temp.index.to_numpy(dtype=float), spec_time_arr)
    ax.pcolormesh(temp_d_grid, temp_t_grid, temp.T.to_numpy(dtype=float), cmap="coolwarm", shading="gouraud", vmin=vmin_vmax[i][0], vmax=vmin_vmax[i][1])
    # ax.axvline(opxrd_ref_d_spacing[i][0], ls="--", lw=1.0, color="grey", alpha=0.3)
    # ax.text(0.75 + 0.001 * i, -0.23, f"{opxrd_ref_d_spacing[i][0]}", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", c="blue")
    ax.yaxis.set_ticks([])
    ax.set_ylim(float(spec_time_arr.min()), float(spec_time_arr.max()))
    ax.set_xlim(temp.index.min(), temp.index.max())
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_d_spacing[i][0]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, labelbottom=True)
    for y_sp in special_y_time:
        ax.axhline(y=y_sp, lw=0.6, ls="--", color=colors[0], alpha=0.5)

ax.text(-1.1, 2.55, r"$\mathrm{d \ Spacing\ (\AA)}$", transform=ax.transAxes, fontsize=9, va="center", ha="right", fontfamily="Arial", fontweight="bold", c="blue")

ax.text(-6.8, 3.75, r"B", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_39_XRD_04_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_39_XRD_04_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_39_XRD_04_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
# plt.savefig(
#     Path.joinpath(path_out, r"PaperDos_FigureSI_39_XRD_04_V1_900.svg"),
#     pad_inches=0.05,
#     bbox_inches="tight",
#     dpi=900,
#     transparent=False,
# )

plt.gcf().set_facecolor("white")
plt.show()



## FigureSI 40 -- Operando XRD of Zn, EMD 0.9 -1.8 V

In [ ]:
# 读取数据

# Operando XRD, EMD, no pH buffer, 1 M + 0.2 M, 30 uA - V2

# 1) 读取电化学数据
path_filelist = list(Path.joinpath(path_data_root, r"XRD\Operando\EMD\2025-MSPD\Results\IS17_D\Echem\B").glob(r"**\*.txt"))
echem_all = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break

    df = pd.read_csv(path_file, sep="	", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = pd.to_datetime(df["time/s"], format="mixed", errors="coerce")
    df["cycle number"] = pd.to_numeric(df["cycle number"], errors="coerce").astype("Int64")
    df["Voltage/V"] = df["Ewe/V"] - df["Ece/V"]
    echem_all.append(df)

echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 2) 读取谱线时间戳（当前格式：Wave / Date / filename）
path_xrd = Path.joinpath(path_data_root, r"XRD\Operando\EMD\2025-MSPD\Results\IS17_D\Data")
path_time_spectrum = path_xrd.joinpath(r"Time_index_spectrum.xlsx")

spectrum_time_all = pd.read_excel(path_time_spectrum)
required_cols = {"Wave", "Date", "filename"}

spectrum_time_all["spec_time"] = pd.to_datetime(
    spectrum_time_all["Date"],
    format=r"%Y-%m-%d_%H:%M:%S",
    errors="coerce",
)
spectrum_time_all["spectrum_col"] = spectrum_time_all["filename"].astype(str).str.strip().str.replace(r"\.xye$", "", regex=True)
spectrum_time_all = spectrum_time_all.dropna(subset=["spec_time", "spectrum_col"]).sort_values("spec_time").reset_index(drop=True)

# 3) 读取 XRD 全谱数据
path_dspacing = path_xrd.joinpath(r"spectrum_all_d_spacing.csv")
if not path_dspacing.exists():
    raise FileNotFoundError(f"FigureMS 02: missing {path_dspacing}")

spectrum_all = pd.read_csv(path_dspacing, index_col=0, header=0)
spectrum_all.index = pd.to_numeric(spectrum_all.index, errors="coerce")
spectrum_all = spectrum_all.loc[~spectrum_all.index.isna(), :]
spectrum_all.columns = [str(c).strip().replace(".xye", "") for c in spectrum_all.columns]

In [ ]:
# 过滤 17_Da_* 谱线并与时间索引对齐（后续单元直接使用）

# 1) 时间表仅保留 17_Da_fxxxxx
spectrum_time_all["spectrum_col"] = spectrum_time_all["spectrum_col"].astype(str).str.strip().str.replace(r"\.xye$", "", regex=True)
mask_ca = spectrum_time_all["spectrum_col"].str.match(r"^17_Da_f\d+$", na=False)
spectrum_time_all = spectrum_time_all.loc[mask_ca].copy().sort_values("spec_time").reset_index(drop=True)

# 2) 谱图矩阵仅保留 17_Da_fxxxxx 列
spectrum_all.columns = [str(c).strip().replace(".xye", "") for c in spectrum_all.columns]
spec_cols_ca = [c for c in spectrum_all.columns if c.startswith("17_Da_f")]
spectrum_all = spectrum_all.loc[:, spec_cols_ca].copy()

# 3) 取交集并按时间顺序重排，确保 time/spectrum 完全对齐
common_cols = [c for c in spectrum_time_all["spectrum_col"].tolist() if c in spectrum_all.columns]

spectrum_time_all = spectrum_time_all[spectrum_time_all["spectrum_col"].isin(common_cols)].copy()
spectrum_time_all = spectrum_time_all.drop_duplicates(subset=["spectrum_col"], keep="first").sort_values("spec_time").reset_index(drop=True)
spectrum_all = spectrum_all.reindex(columns=spectrum_time_all["spectrum_col"].tolist())

In [ ]:
# 清洗数据 + 校验谱线-时间映射

# 1) 清理 echem_all 起始异常段,估计了一下 index.
echem_all.drop(echem_all.index[:6500], inplace=True)

# 2) 只保留第一圈充放电与第二圈充电
selected_echem = echem_all[echem_all["cycle number"].isin([1, 2])]
selected_echem = selected_echem[selected_echem["Voltage/V"] >= 0.8].copy()
selected_echem = selected_echem.dropna(subset=["time/s", "Voltage/V", "<I>/mA"]).sort_values(by="time/s").reset_index(drop=True)
selected_echem["charge_time"] = (selected_echem["time/s"] - selected_echem["time/s"].iloc[0]).dt.total_seconds() / 3600

# 3) 最近邻时间匹配：谱线 -> 电化学（每条谱线只匹配一个最近电化学点）
selected_spectrum_time = (
    pd.merge_asof(
        spectrum_time_all[["spectrum_col", "spec_time"]].sort_values(by="spec_time"),
        selected_echem[["time/s", "Voltage/V", "<I>/mA", "charge_time"]].sort_values(by="time/s"),
        left_on="spec_time",
        right_on="time/s",
        direction="nearest",
        tolerance=pd.Timedelta("5s"),
    )
    .dropna(subset=["spectrum_col", "spec_time", "time/s", "Voltage/V", "charge_time"], inplace=False)
    .sort_values(by="spec_time")
    .reset_index(drop=True)
)

if selected_spectrum_time.empty:
    raise ValueError("FigureMS 02: no matched spectra after time mapping")

# 时间零点一致性检查：电化学与谱线都相对 selected_echem 首时刻
time_zero = selected_echem["time/s"].iloc[0]
selected_spectrum_time["spec_charge_time"] = (
    selected_spectrum_time["spec_time"] - time_zero
).dt.total_seconds() / 3600
selected_spectrum_time["map_dt_s"] = (
    selected_spectrum_time["time/s"] - selected_spectrum_time["spec_time"]
).dt.total_seconds().abs()
selected_spectrum_time["zero_consistency_dt_s"] = (
    selected_spectrum_time["charge_time"] - selected_spectrum_time["spec_charge_time"]
).abs() * 3600

# 4) 选择 d-spacing 区间并建立谱线-时间映射
d_spacing_range = (0.5, 15.0)
spectrum_all = spectrum_all.loc[(spectrum_all.index >= d_spacing_range[0]) & (spectrum_all.index <= d_spacing_range[1])]

selected_cols = [col for col in spectrum_all.columns if col in set(selected_spectrum_time["spectrum_col"]) ]

selected_spectrum = spectrum_all.loc[:, selected_cols]
selected_spectrum_time = (
    selected_spectrum_time[selected_spectrum_time["spectrum_col"].isin(selected_spectrum.columns)]
    .drop_duplicates(subset=["spectrum_col"], keep="first")
    .sort_values(by="spec_charge_time")
    .reset_index(drop=True)
)
selected_spectrum = selected_spectrum.reindex(columns=selected_spectrum_time["spectrum_col"].to_list())
selected_spectrum_time["range_idx"] = pd.Series(np.arange(1, len(selected_spectrum_time) + 1), dtype="Int64")

spectrum_time_axis = selected_spectrum_time["spec_charge_time"].to_numpy(dtype=float)

# 若时间非单调，按时间重排，避免后续绘图错位
if np.any(np.diff(spectrum_time_axis) < 0):
    order = np.argsort(spectrum_time_axis)
    selected_spectrum = selected_spectrum.iloc[:, order]
    selected_spectrum_time = selected_spectrum_time.iloc[order].reset_index(drop=True)
    spectrum_time_axis = spectrum_time_axis[order]

# 时间边界（给 pcolormesh 使用）
spectrum_time_edges = np.empty(spectrum_time_axis.size + 1, dtype=float)
if spectrum_time_axis.size == 1:
    spectrum_time_edges[0] = spectrum_time_axis[0] - 0.5
    spectrum_time_edges[1] = spectrum_time_axis[0] + 0.5
else:
    spectrum_time_edges[1:-1] = 0.5 * (spectrum_time_axis[:-1] + spectrum_time_axis[1:])
    spectrum_time_edges[0] = spectrum_time_axis[0] - 0.5 * (spectrum_time_axis[1] - spectrum_time_axis[0])
    spectrum_time_edges[-1] = spectrum_time_axis[-1] + 0.5 * (spectrum_time_axis[-1] - spectrum_time_axis[-2])



In [ ]:
# 画图
%matplotlib inline
plt.close("all")

fig = plt.figure(figsize=(14.0, 15.0))
gs = gridspec.GridSpec(3, 2, width_ratios=[1, 1], height_ratios=[1, 1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 0.3, 0.3))
ax.set_box_aspect(3.0)

ax.plot(selected_echem["Voltage/V"], selected_echem["charge_time"], ls="-", lw=1.0, c=colors[0], label=r"Voltage")

index_labels = [1, 13, 23, 31, 39, 48, 61, 75, 85, 95, 102, 112, 123]

# 统一时间显示窗口：覆盖电化学与谱线时间，并在上下各增加 15 min 余量
pad_h = 0.25
time_min = min(float(selected_echem["charge_time"].min()), float(spectrum_time_edges[0]))
time_max = max(float(selected_echem["charge_time"].max()), float(spectrum_time_edges[-1]))
y_min_plot = time_min - pad_h
y_max_plot = time_max + pad_h + 1
special_y_time = selected_spectrum_time[selected_spectrum_time["range_idx"].isin(index_labels)]["charge_time"].dropna().astype(float).tolist()
for i, idx in enumerate(selected_spectrum_time["range_idx"].tolist()):
    if idx in index_labels:
        value = selected_spectrum_time.loc[selected_spectrum_time["range_idx"] == idx, ["Voltage/V", "charge_time"]]
        ax.scatter(value["Voltage/V"], value["charge_time"], c=colors[2], edgecolors='face', marker="*", s=30, zorder=5)
    else:
        value = selected_spectrum_time.loc[selected_spectrum_time.index == i, ["Voltage/V", "charge_time"]]
        ax.scatter(value["Voltage/V"], value["charge_time"], c="lightgrey", edgecolors='face', s=10, zorder=1)

ax.set_xlabel(r"Voltage (V)", fontsize=9)
ax.set_xlim(0.8, 2.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.2, offset=-0.4))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.6, offset=-0.4))

ax.set_ylabel(r"Duration Time (h)", fontsize=9)
ax.set_ylim(y_min_plot, y_max_plot)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))

ax2 = ax.twiny()
ax2.set_position((0, 0, 0.3, 0.3))
ax2.set_box_aspect(3.0)
ax2.plot(selected_echem["<I>/mA"] * 1000, selected_echem["charge_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")
ax2.set_xlabel(r"$\mathrm{Current  \ (\mu A)}$", fontsize=9)
ax2.set_xlim(-60, 60)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=60, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=30, offset=0))
ax2.tick_params(axis="both", which="both", labelsize=9, right=True, labelright=True)
ax2.set_ylim(y_min_plot, y_max_plot)

ax.text(-0.9, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
opxrd_ref_d_spacing = [1.578523, 1.821636, 2.32571, 2.473, 2.536224, 2.671445, 2.714108, 3.128303, 3.230361, 3.408803, 3.51733, 3.644164, 4.172575, 5.465435, 10.9373]
vmin_vmax = [
    (None, None), (None, None), (None, None), (None, None), (None, None), (None, None),
    (None, None), (None, None), (None, None), (None, None), (None, None), (None, None),
    (27800, 34000), (None, None), (None, None)
]
peak_width = [
    (0.1, 0.1), (0.1, 0.1), (0.1, 0.1), (0.1, 0.1), (0.1, 0.1), (0.1, 0.1),
    (0.1, 0.1), (0.1, 0.1), (0.1, 0.1), (0.1, 0.1), (0.1, 0.1), (0.1, 0.1),
    (0.1, 0.1), (0.1, 0.1), (0.3, 0.3)
]


# 上排峰窗
for i in range(0, len(opxrd_ref_d_spacing) // 2 - 1):
    ax = subfig.add_axes((0.225 + i * 0.055, 0.0, 0.05, 0.3), zorder=0)
    temp = selected_spectrum.loc[
        (selected_spectrum.index >= opxrd_ref_d_spacing[i] - peak_width[i][0])
        & (selected_spectrum.index <= opxrd_ref_d_spacing[i] + peak_width[i][1])
    ]
    if temp.empty:
        continue

    x_axis = temp.index.to_numpy(dtype=float)
    z_map = temp.to_numpy(dtype=float).T
    ax.pcolormesh(
        x_axis,
        spectrum_time_axis,
        z_map,
        cmap="coolwarm",
        shading="gouraud",
        vmin=vmin_vmax[i][0],
        vmax=vmin_vmax[i][1],
    )
    # ax.axvline(opxrd_ref_d_spacing[i], ls="--", lw=1.0, color="w", alpha=0.3)
    ax.yaxis.set_ticks([])
    ax.set_ylim(y_min_plot, y_max_plot)
    ax.set_xlim(float(x_axis.min()), float(x_axis.max()))
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_d_spacing[i]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, labelbottom=True)
    for y_sp in special_y_time:
        ax.axhline(y=y_sp, lw=0.6, ls="--", color=colors[1], alpha=0.3)

# 下排峰窗
for i in range(len(opxrd_ref_d_spacing) // 2 - 1, len(opxrd_ref_d_spacing)):
    ax = subfig.add_axes((0.06 + (i - len(opxrd_ref_d_spacing) // 2 + 1) * 0.055, -0.40, 0.05, 0.3), zorder=0)
    temp = selected_spectrum.loc[
        (selected_spectrum.index >= opxrd_ref_d_spacing[i] - peak_width[i][0])
        & (selected_spectrum.index <= opxrd_ref_d_spacing[i] + peak_width[i][1])
    ]
    if temp.empty:
        continue

    x_axis = temp.index.to_numpy(dtype=float)
    z_map = temp.to_numpy(dtype=float).T
    ax.pcolormesh(
        x_axis,
        spectrum_time_axis,
        z_map,
        cmap="coolwarm",
        shading="gouraud",
        vmin=vmin_vmax[i][0],
        vmax=vmin_vmax[i][1],
    )
    # ax.axvline(opxrd_ref_d_spacing[i], ls="--", lw=1.0, color="w", alpha=0.3)
    ax.yaxis.set_ticks([])
    ax.set_ylim(y_min_plot, y_max_plot)
    ax.set_xlim(float(x_axis.min()), float(x_axis.max()))
    ax.xaxis.set_major_locator(ticker.FixedLocator([opxrd_ref_d_spacing[i]]))
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, labelbottom=True)
    for y_sp in special_y_time:
        ax.axhline(y=y_sp, lw=0.6, ls="--", color=colors[1], alpha=0.3)

ax.text(-0.5, 1.15, r"$\mathrm{d \ Spacing \ (\AA)}$", transform=ax.transAxes, fontsize=9, va="center", ha="right", fontfamily="Arial", fontweight="bold")
ax.text(-5.6, 2.35, r"B", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_40_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_40_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_40_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
# plt.savefig(
#     Path.joinpath(path_out, r"PaperDos_FigureSI_40_V0_900.svg"),
#     pad_inches=0.05,
#     bbox_inches="tight",
#     dpi=900,
#     transparent=False,
# )

plt.gcf().set_facecolor("white")
plt.show()

## FigureSI 41 -- Echem - 0.2 M MnSO4 + 1stDischarge -V1 + 1stCharge -V1

In [ ]:
# 1st0.9V - 1M ZnSO4 + 0.2M MnSO4 - ZHS

# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r"Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st0.9V\1M ZnSO4 + 0.2M MnSO4\Results\ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echema = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echema.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echema)):
    echema[i] = echema[i][echema[i].iloc[:, 0].isin([0, 1, 2, 3])]

# 1st0.9V - 1M ZnSO4 - ZHS

# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r"Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st0.9V\1M ZnSO4\Results\ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echemb = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemb.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemb)):
    echemb[i] = echemb[i][echemb[i].iloc[:, 0].isin([0, 1, 2, 3])]

# # 1st0.9V - 1M ZnSO4 + 0.2M MnSO4 - No ZHS

# # 电化学上的时间
# path_filelist = list(Path.joinpath(path_data_root, r"Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st0.9V\1M ZnSO4 + 0.2M MnSO4\Results\No ZHS\17#-03-2023").glob(r"**\*.xlsx"))
# echemc = []
# # 读取电化学数据
# for path in path_filelist:
#     df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
#     # 转换数据格式
#     df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
#     df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
#     df["Cycle"] = df["Cycle"].astype(np.int16)
#     echemc.append(df)

# # 选取每个数据的第一到第二圈
# for i in range(len(echemc)):
#     echemc[i] = echemc[i][echemc[i].iloc[:, 0].isin([0, 1, 2, 3])]

# 1st0.9V - 1M ZnSO4 - No ZHS

# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r"Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st0.9V\1M ZnSO4\Results\No ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echemd = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemd.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemd)):
    echemd[i] = echemd[i][echemd[i].iloc[:, 0].isin([0, 1, 2, 3])]

In [ ]:
# 1st1.80V - 1M ZnSO4 + 0.2M MnSO4 - ZHS

# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r"Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st1.80V\1M ZnSO4 + 0.2M MnSO4\Results\ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echema1 = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echema1.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echema1)):
    echema1[i] = echema1[i][echema1[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echema1[i] = echema1[i][echema1[i].iloc[:, 2] >= 0]

# 1st1.80V - 1M ZnSO4 - ZHS

# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r"Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st1.80V\1M ZnSO4\Results\ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echemb1 = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemb1.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemb1)):
    echemb1[i] = echemb1[i][echemb1[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echemb1[i] = echemb1[i][echemb1[i].iloc[:, 2] >= 0]

# # 1st1.80V - 1M ZnSO4 + 0.2M MnSO4 - No ZHS

# # 电化学上的时间
# path_filelist = list(
#     Path.joinpath(path_data_root, r"Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st1.80V\1M ZnSO4 + 0.2M MnSO4\Results\No ZHS\17#-03-2023").glob(r"**\*.xlsx")
# )
# echemc1 = []
# # 读取电化学数据
# for path in path_filelist:
#     df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
#     # 转换数据格式
#     df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
#     df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
#     df["Cycle"] = df["Cycle"].astype(np.int16)
#     echemc1.append(df)

# # 选取每个数据的第一到第二圈
# for i in range(len(echemc1)):
#     echemc1[i] = echemc1[i][echemc1[i].iloc[:, 0].isin([0, 1, 2, 3])]
#     echemc1[i] = echemc1[i][echemc1[i].iloc[:, 2] >= 0]

# 1st1.80V - 1M ZnSO4 - No ZHS

# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r"Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st1.80V\1M ZnSO4\Results\No ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echemd1 = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemd1.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemd1)):
    echemd1[i] = echemd1[i][echemd1[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echemd1[i] = echemd1[i][echemd1[i].iloc[:, 2] >= 0]

In [ ]:
# 1st1.80V - 1M ZnSO4 + 0.2M MnSO4 - No ZHS - 0.2 M MnSO4 测试

# 电化学上的时间
path_filelist = list(
    Path.joinpath(path_data_root, r"Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st1.80V\1M ZnSO4 + 0.2M MnSO4\Results\No ZHS\17#-03-2023").glob(r"**\*.xlsx")
)
echeme = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echeme.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echeme)):
    echeme[i] = echeme[i][echeme[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echeme[i] = echeme[i][echeme[i].iloc[:, 2] >= 0]

# 1st0.9V - 1M ZnSO4 + 0.2M MnSO4 - No ZHS - 0.2 M MnSO4 测试

# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r"Echem\αMnO2\GCD\0.2M MnSO4\ExtendedCapacity\1st0.9V\1M ZnSO4 + 0.2M MnSO4\Results\No ZHS\17#-03-2023").glob(r"**\*.xlsx"))
echemf = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemf.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemf)):
    echemf[i] = echemf[i][echemf[i].iloc[:, 0].isin([0, 1, 2, 3])]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 8.0))
gs = gridspec.GridSpec(4, 2, width_ratios=[1, 1], height_ratios=[1, 1, 1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$" + "\n" + r"$\mathrm{1^{st} \ Discharge}$", None, None]]
for i, data in enumerate(echema):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[:idx_min, "SpeCap/mAh/g"], temp.loc[:idx_min, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 2 :, "SpeCap/mAh/g"], temp.loc[idx_min + 2 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(305, 0.9, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=9, labelpad=1.0)
ax.set_xlim(-0.1, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=8)
ax.legend()
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=7)
ax.text(0.05, 0.07, r"No Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=8, c="k")
ax.text(-0.21, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
subfig = fig.add_subfigure(gs[0, 1])
ax = subfig.add_axes((0.0, 0, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4, 1^{st} \ Discharge}$", None, None]]
for i, data in enumerate(echemb):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[:idx_min, "SpeCap/mAh/g"], temp.loc[:idx_min, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 2 :, "SpeCap/mAh/g"], temp.loc[idx_min + 2 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(360, 0.9, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=9, labelpad=1.0)
ax.set_xlim(-0.1, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=8)
ax.legend()
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=7)
ax.text(0.05, 0.07, r"No Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=8, c="k")
ax.text(-0.21, 1.0, r"B", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
subfig = fig.add_subfigure(gs[1, 0])
ax = subfig.add_axes((0, -0.3, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$" + "\n" + r"$\mathrm{1^{st} \ Charge}$", None, None]]
for i, data in enumerate(echema1):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[:idx_min, "SpeCap/mAh/g"], temp.loc[:idx_min, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 2 :, "SpeCap/mAh/g"], temp.loc[idx_min + 2 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(285, 1.80, marker="*", markersize=10, color=colors[1], alpha=0.5)


# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=9, labelpad=1.0)
ax.set_xlim(-0.1, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=8)
ax.legend()
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=7)
ax.text(0.03, 0.07, r"No Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=8, c="k")

ax.text(-0.21, 1.0, r"C", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
subfig = fig.add_subfigure(gs[1, 1])
ax = subfig.add_axes((0.0, -0.3, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4, 1^{st} \ Charge}$", None, None]]
for i, data in enumerate(echemb1):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(138, 1.80, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=9, labelpad=1.0)
ax.set_xlim(-0.1, 300)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=50, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=25, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=8)
ax.legend()
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=7)
ax.text(0.03, 0.07, r"No Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=8, c="k")

ax.text(-0.21, 1.0, r"D", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 E
subfig = fig.add_subfigure(gs[2, 0])
ax = subfig.add_axes((0.0, -0.6, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$", None, None]]
for i, data in enumerate(echemf):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[idx_min + 2 :, "SpeCap/mAh/g"], temp.loc[idx_min + 2 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
        ax.plot(temp.loc[:idx_min, "SpeCap/mAh/g"], temp.loc[:idx_min, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)

ax.plot(348, 0.9, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=9, labelpad=1.0)
ax.set_xlim(-0.1, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=8)
ax.legend()
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=7)
ax.text(0.03, 0.07, r"Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=8, c="k")
ax.text(-0.21, 1.0, r"E", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 F
subfig = fig.add_subfigure(gs[2, 1])
ax = subfig.add_axes((0.0, -0.6, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4, 1^{st} \ Discharge}$", None, None]]
for i, data in enumerate(echemd):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[:idx_min, "SpeCap/mAh/g"], temp.loc[:idx_min, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 2 :, "SpeCap/mAh/g"], temp.loc[idx_min + 2 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(290, 0.93, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=9, labelpad=1.0)
ax.set_xlim(-0.1, 300)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=50, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=25, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=8)
ax.legend()
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=7)
ax.text(0.05, 0.07, r"Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=8, c="k")
ax.text(-0.21, 1.0, r"F", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 G
subfig = fig.add_subfigure(gs[3, 0])
ax = subfig.add_axes((0.0, -0.9, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$" + "\n" + r"$\mathrm{1^{st} \ Charge}$", None, None]]
for i, data in enumerate(echeme):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)

ax.plot(320, 1.80, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=9, labelpad=1.0)
ax.set_xlim(-0.1, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=8)
ax.legend()
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=7)
ax.text(0.03, 0.07, r"Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=8, c="k")
ax.text(-0.21, 1.0, r"G", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 H
subfig = fig.add_subfigure(gs[3, 1])
ax = subfig.add_axes((0.0, -0.9, 1, 1))
ax.set_box_aspect(0.8)

labels = [[r"$\mathrm{0.2 \ M \ MnSO_4, Cycles}$", None, None], [r"$\mathrm{1 \ M \ ZnSO_4, 1^{st} \ Charge}$", None, None]]
for i, data in enumerate(echemd1):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 2 :, "SpeCap/mAh/g"], temp.loc[idx_min + 2 :, "Voltage/V"], ls="-", lw=1.0, c=colors[i], label=None, zorder=0)
ax.plot(200, 1.80, marker="*", markersize=10, color=colors[1], alpha=0.5)

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=9, labelpad=1.0)
ax.set_xlim(-0.1, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9, labelpad=1.0)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=-0.15))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=-0.15))

ax.tick_params(axis="both", direction="out", labelsize=8)
ax.legend()
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], loc="upper right", bbox_to_anchor=(1.0, 0.68), ncols=1, frameon=False, labelcolor="linecolor", fontsize=7)
ax.text(0.03, 0.07, r"Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=8, c="k")
ax.text(-0.21, 1.0, r"H", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_41_Echem_06_V1_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_41_Echem_06_V1_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_41_Echem_06_V1_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_41_Echem_06_V1_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

## FigureSI 22 -- HRTEM 1st1.53V TEM

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [ ]:
path_tem = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\TEM\Results\01\TEM1.53V")
HAADF = hs.load(path_tem.joinpath(r"0010-20241211_Ceta_340_kx_Ceta.dm4"))

TEM_1 = hs.load([path_tem.joinpath(r"FFT of 0010-20241211_Ceta_340_kx_Ceta.dm4"), path_tem.joinpath(r"IFFT of Untitled.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled.dm4")])
TEM_2 = hs.load([path_tem.joinpath(r"FFT of 0010-20241211_Ceta_340_kx_Ceta_1.dm4"), path_tem.joinpath(r"IFFT of Untitled.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled.dm4")])
TEM_3 = hs.load([path_tem.joinpath(r"FFT of 0010-20241211_Ceta_340_kx_Ceta_2.dm4"), path_tem.joinpath(r"IFFT of Untitled.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled.dm4")])

HAADF = convert_units(HAADF)[0]
TEM_1 = convert_units([TEM_1])[0]
TEM_2 = convert_units([TEM_2])[0]
TEM_3 = convert_units([TEM_3])[0]

In [ ]:
# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(4, 4, width_ratios=[1, 1, 1, 1], height_ratios=[1, 1, 1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0:2, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(1.0)

vmin, vmax = np.percentile(HAADF.data, [1.0, 99.0])
ax.imshow(HAADF.data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")

scalebar = add_sizebar(
    ax,
    size=10,
    bardata=HAADF,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()

ax.text(
    0.23,
    0.95,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=11,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.1,
    1.0,
    r"A",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, 0.0, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_1[2].axes_manager[-1].axis,
    y1=TEM_1[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[1],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 5.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-30, 50)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.12,
    1.0,
    r"B",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B1
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_axes((-0.15, 0.0, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_1[1].data, [1.0, 99.0])
ax.imshow(TEM_1[1].data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")
add_sizebar(ax, 1, TEM_1[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 B2
subfig = fig.add_subfigure(gs[1, 3], zorder=0)
ax = subfig.add_axes((-0.05, 0.0, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_1[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog", interpolation="bilinear")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 C
subfig = fig.add_subfigure(gs[2, 0:2], zorder=0)
ax = subfig.add_axes((0.00, -0.15, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_2[2].axes_manager[-1].axis,
    y1=TEM_2[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[0],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 5.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-30, 45)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.1,
    1.0,
    r"C",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C1
subfig = fig.add_subfigure(gs[3, 0], zorder=0)
ax = subfig.add_axes((0.07, -0.16, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_2[1].data, [1.0, 99.0])
ax.imshow(TEM_2[1].data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")
add_sizebar(ax, 1, TEM_2[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 C2
subfig = fig.add_subfigure(gs[3, 1], zorder=0)
ax = subfig.add_axes((0.15, -0.16, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_2[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog", interpolation="bilinear")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 D
subfig = fig.add_subfigure(gs[2, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, -0.15, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_3[2].axes_manager[-1].axis,
    y1=TEM_3[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[4],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 5.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-30, 45)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.1,
    1.0,
    r"D",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 D1
subfig = fig.add_subfigure(gs[3, 2], zorder=0)
ax = subfig.add_axes((-0.15, -0.15, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_3[1].data, [1.0, 99.0])
ax.imshow(TEM_3[1].data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")
add_sizebar(ax, 1, TEM_3[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 D2
subfig = fig.add_subfigure(gs[3, 3], zorder=0)
ax = subfig.add_axes((-0.07, -0.15, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_3[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog", interpolation="bilinear")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_22_TEM_TEM_00_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_22_TEM_TEM_00_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_22_TEM_TEM_00_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_22_TEM_TEM_00_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

## FigureSI 23 -- HRTEM 1st1.63V TEM

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [ ]:
path_tem = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.63V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\TEM\Results\01\TEM163V")
HAADF = hs.load(path_tem.joinpath(r"0004-20241211_Ceta_210_kx_Ceta.dm4"))

TEM_1 = hs.load([path_tem.joinpath(r"FFT of 0004-20241211_Ceta_210_kx_Ceta.dm4"), path_tem.joinpath(r"IFFT of Untitled.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled.dm4")])

HAADF = convert_units(HAADF)[0]
TEM_1 = convert_units([TEM_1])[0]

In [ ]:
# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(4, 4, width_ratios=[1, 1, 1, 1], height_ratios=[1, 1, 1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0:2, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

vmin, vmax = np.percentile(HAADF.data, [1.0, 99.0])
ax.imshow(HAADF.data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")

scalebar = add_sizebar(
    ax,
    size=20,
    bardata=HAADF,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()

ax.text(
    0.23,
    0.95,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=11,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.05,
    1.0,
    r"A",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, 0.0, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_1[2].axes_manager[-1].axis,
    y1=TEM_1[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[1],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 5.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-20, 30)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.12,
    1.0,
    r"B",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B1
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_axes((-0.15, 0.0, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_1[1].data, [1.0, 99.0])
ax.imshow(TEM_1[1].data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")
add_sizebar(ax, 1, TEM_1[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 B2
subfig = fig.add_subfigure(gs[1, 3], zorder=0)
ax = subfig.add_axes((-0.05, 0.0, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_1[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog", interpolation="bilinear")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_23_TEM_TEM_01_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_23_TEM_TEM_01_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_23_TEM_TEM_01_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_23_TEM_TEM_01_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

## FigureSI 24 -- HRTEM 1st1.80V TEM

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [ ]:
path_tem = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\TEM\Results\01\Workspace 1")
HAADF = hs.load(path_tem.joinpath(r"0024 - 0947 Pristine MnO2 HAADF 2.60 Mx.emd.conv_HAADF.dm4"))

TEM_1 = hs.load([
    path_tem.joinpath(r"FFT of 0024 - 0947 Pristine MnO2 HAADF 2.60 Mx.emd.conv_HAADF.dm4"),
    path_tem.joinpath(r"IFFT of Untitled.dm4"),
    path_tem.joinpath(r"Live Profile Of IFFT of Untitled.dm4"),
])
TEM_2 = hs.load([
    path_tem.joinpath(r"FFT of 0024 - 0947 Pristine MnO2 HAADF 2.60 Mx.emd.conv_HAADF_1.dm4"),
    path_tem.joinpath(r"IFFT of Untitled_1.dm4"),
    path_tem.joinpath(r"Live Profile Of IFFT of Untitled_1.dm4"),
])
TEM_3 = hs.load([
    path_tem.joinpath(r"FFT of 0024 - 0947 Pristine MnO2 HAADF 2.60 Mx.emd.conv_HAADF_2.dm4"),
    path_tem.joinpath(r"IFFT of Untitled_2.dm4"),
    path_tem.joinpath(r"Live Profile Of IFFT of Untitled_2.dm4"),
])

HAADF = convert_units(HAADF)[0]
TEM_1 = convert_units([TEM_1])[0]
TEM_2 = convert_units([TEM_2])[0]
TEM_3 = convert_units([TEM_3])[0]

In [ ]:
# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = gridspec.GridSpec(4, 4, width_ratios=[1, 1, 1, 1], height_ratios=[1, 1, 1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0:2, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
ax.set_box_aspect(1.0)

vmin, vmax = np.percentile(HAADF.data, [1.0, 99.0])
ax.imshow(HAADF.data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")

scalebar = add_sizebar(
    ax,
    size=5,
    bardata=HAADF,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()

ax.text(
    0.23,
    0.95,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=11,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.1,
    1.0,
    r"A",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, 0.0, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_1[2].axes_manager[-1].axis,
    y1=TEM_1[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[1],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 3.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-2800, 3500)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.12,
    1.0,
    r"B",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B1
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_axes((-0.15, 0.0, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_1[1].data, [1.0, 99.0])
ax.imshow(TEM_1[1].data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")
add_sizebar(ax, 1, TEM_1[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 B2
subfig = fig.add_subfigure(gs[1, 3], zorder=0)
ax = subfig.add_axes((-0.05, 0.0, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_1[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog", interpolation="bilinear")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 C
subfig = fig.add_subfigure(gs[2, 0:2], zorder=0)
ax = subfig.add_axes((0.00, -0.15, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_2[2].axes_manager[-1].axis,
    y1=TEM_2[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[0],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 2.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-1500, 2000)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.1,
    1.0,
    r"C",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C1
subfig = fig.add_subfigure(gs[3, 0], zorder=0)
ax = subfig.add_axes((0.07, -0.16, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_2[1].data, [1.0, 99.0])
ax.imshow(TEM_2[1].data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")
add_sizebar(ax, 1, TEM_2[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 C2
subfig = fig.add_subfigure(gs[3, 1], zorder=0)
ax = subfig.add_axes((0.15, -0.16, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_2[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog", interpolation="bilinear")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 D
subfig = fig.add_subfigure(gs[2, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, -0.15, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_3[2].axes_manager[-1].axis,
    y1=TEM_3[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[4],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 3.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-4000, 5000)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.1,
    1.0,
    r"D",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 D1
subfig = fig.add_subfigure(gs[3, 2], zorder=0)
ax = subfig.add_axes((-0.15, -0.15, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_3[1].data, [1.0, 99.0])
ax.imshow(TEM_3[1].data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")
add_sizebar(ax, 1, TEM_3[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 D2
subfig = fig.add_subfigure(gs[3, 3], zorder=0)
ax = subfig.add_axes((-0.07, -0.15, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_3[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog", interpolation="bilinear")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_24_TEM_TEM_02_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_24_TEM_TEM_02_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_24_TEM_TEM_02_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_24_TEM_TEM_02_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

## FigureSI 25 -- HRTEM 2nd13.0V TEM

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [ ]:
path_tem = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\TEM\Results\01\TEM130V")
HAADF = hs.load(path_tem.joinpath(r"0023-Ceta_420_kx_Camera_Ceta.dm4"))

TEM_1 = hs.load([path_tem.joinpath(r"FFT of 0023-Ceta_420_kx_Camera_Ceta.dm4"), path_tem.joinpath(r"IFFT of Untitled.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled.dm4")])
TEM_2 = hs.load([path_tem.joinpath(r"FFT of 0023-Ceta_420_kx_Camera_Ceta_1.dm4"), path_tem.joinpath(r"IFFT of Untitled_1.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled_1.dm4")])
TEM_3 = hs.load([path_tem.joinpath(r"FFT of 0023-Ceta_420_kx_Camera_Ceta_2.dm4"), path_tem.joinpath(r"IFFT of Untitled_2.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled_2.dm4")])
TEM_4 = hs.load([path_tem.joinpath(r"FFT of 0023-Ceta_420_kx_Camera_Ceta_3.dm4"), path_tem.joinpath(r"IFFT of Untitled_3.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled_3.dm4")])

HAADF = convert_units(HAADF)[0]
TEM_1 = convert_units([TEM_1])[0]
TEM_2 = convert_units([TEM_2])[0]
TEM_3 = convert_units([TEM_3])[0]
TEM_4 = convert_units([TEM_4])[0]

path_tem2 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\TEM\Results\02\TEM130V-2")
HAADF2 = hs.load(path_tem2.joinpath(r"0026-Ceta_800_kx_Camera_Ceta.dm4"))

TEM2_1 = hs.load([path_tem2.joinpath(r"FFT of 0026-Ceta_800_kx_Camera_Ceta.dm4"), path_tem2.joinpath(r"IFFT of Untitled.dm4"), path_tem2.joinpath(r"Live Profile Of IFFT of Untitled.dm4")])

HAADF2 = convert_units(HAADF2)[0]
TEM2_1 = convert_units([TEM2_1])[0]

In [ ]:
# 画图
fig = plt.figure(figsize=(7.0, 7.5))
gs = gridspec.GridSpec(6, 4, width_ratios=[1, 1, 1, 1], height_ratios=[1, 1, 1, 1, 1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0:2, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

vmin, vmax = np.percentile(HAADF.data, [1.0, 99.0])
ax.imshow(HAADF.data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")

scalebar = add_sizebar(
    ax,
    size=10,
    bardata=HAADF,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()

ax.text(
    0.23,
    0.95,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=11,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.1,
    1.0,
    r"A",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, 0.0, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_1[2].axes_manager[-1].axis,
    y1=TEM_1[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[1],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 3.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-50, 70)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.12,
    1.0,
    r"B",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B1
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_axes((-0.15, 0.0, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_1[1].data, [1.0, 99.0])
ax.imshow(TEM_1[1].data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")
add_sizebar(ax, 1, TEM_1[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 B2
subfig = fig.add_subfigure(gs[1, 3], zorder=0)
ax = subfig.add_axes((-0.05, 0.0, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_1[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog", interpolation="bilinear")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 C
subfig = fig.add_subfigure(gs[2, 0:2], zorder=0)
ax = subfig.add_axes((0.00, -0.15, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_2[2].axes_manager[-1].axis,
    y1=TEM_2[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[0],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 2.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-150, 200)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.1,
    1.0,
    r"C",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C1
subfig = fig.add_subfigure(gs[3, 0], zorder=0)
ax = subfig.add_axes((0.07, -0.16, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_2[1].data, [1.0, 99.0])
ax.imshow(TEM_2[1].data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")
add_sizebar(ax, 1, TEM_2[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 C2
subfig = fig.add_subfigure(gs[3, 1], zorder=0)
ax = subfig.add_axes((0.15, -0.16, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_2[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog", interpolation="bilinear")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 D
subfig = fig.add_subfigure(gs[2, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, -0.15, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_4[2].axes_manager[-1].axis,
    y1=TEM_4[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[4],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 3.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-150, 200)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.1,
    1.0,
    r"D",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 D1
subfig = fig.add_subfigure(gs[3, 2], zorder=0)
ax = subfig.add_axes((-0.15, -0.15, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_4[1].data, [1.0, 99.0])
ax.imshow(TEM_4[1].data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")
add_sizebar(ax, 1, TEM_4[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 D2
subfig = fig.add_subfigure(gs[3, 3], zorder=0)
ax = subfig.add_axes((-0.07, -0.15, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_4[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog", interpolation="bilinear")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 E
subfig = fig.add_subfigure(gs[4:6, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, -0.2, 1.0, 1.0))
# ax.set_box_aspect(1.0)

vmin, vmax = np.percentile(HAADF2.data, [1.0, 99.0])
ax.imshow(HAADF2.data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")

scalebar = add_sizebar(
    ax,
    size=5,
    bardata=HAADF2,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()

ax.text(
    0.23,
    0.95,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=11,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.1,
    1.0,
    r"E",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 F
subfig = fig.add_subfigure(gs[4, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, -0.4, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM2_1[2].axes_manager[-1].axis,
    y1=TEM2_1[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[3],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 2.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-250, 300)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.12,
    1.0,
    r"F",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 F1
subfig = fig.add_subfigure(gs[5, 2], zorder=0)
ax = subfig.add_axes((-0.15, -0.4, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM2_1[1].data, [1.0, 99.0])
ax.imshow(TEM2_1[1].data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")
add_sizebar(ax, 1, TEM2_1[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 F2
subfig = fig.add_subfigure(gs[5, 3], zorder=0)
ax = subfig.add_axes((-0.05, -0.4, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM2_1[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog", interpolation="bilinear")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_25_TEM_TEM_03_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_25_TEM_TEM_03_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_25_TEM_TEM_03_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_25_TEM_TEM_03_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

## FigureSI 26 -- HRTEM 2nd0.9V TEM

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[1].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[1].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 9},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    def _process(s):
        am = s.axes_manager
        if len(am.shape) >= 2:
            if len(am.navigation_shape) == 2 and getattr(am.navigation_axes[0], "units", None) == r"µm":
                am.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(am.signal_shape) == 2 and getattr(am.signal_axes[0], "units", None) == r"µm":
                am.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        return s

    results = []
    if not isinstance(data, (list, tuple)):
        results.append(_process(data))
    else:
        for item in data:
            if isinstance(item, (list, tuple)):
                results.append([_process(s) for s in item])
            else:
                results.append(_process(item))

    return results


eels_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])

In [ ]:
path_tem = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B2\TEM\Results\01\TEM0.9V")
HAADF = hs.load(path_tem.joinpath(r"0025-Ceta_265_kx_Camera_Ceta.dm4"))

TEM_1 = hs.load([path_tem.joinpath(r"FFT of 0025-Ceta_265_kx_Camera_Ceta.dm4"), path_tem.joinpath(r"IFFT of Untitled.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled.dm4")])
TEM_2 = hs.load([path_tem.joinpath(r"FFT of 0025-Ceta_265_kx_Camera_Ceta_1.dm4"), path_tem.joinpath(r"IFFT of Untitled_1.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled_1.dm4")])
TEM_3 = hs.load([path_tem.joinpath(r"FFT of 0025-Ceta_265_kx_Camera_Ceta_2.dm4"), path_tem.joinpath(r"IFFT of Untitled_2.dm4"), path_tem.joinpath(r"Live Profile Of IFFT of Untitled_2.dm4")])

HAADF = convert_units(HAADF)[0]
TEM_1 = convert_units([TEM_1])[0]
TEM_2 = convert_units([TEM_2])[0]
TEM_3 = convert_units([TEM_3])[0]

path_tem2 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B2\TEM\Results\02\TEM0.9V-2")
HAADF2 = hs.load(path_tem2.joinpath(r"0016-Ceta_420_kx_Camera_Ceta.dm4"))

TEM2_1 = hs.load([path_tem2.joinpath(r"FFT of 0016-Ceta_420_kx_Camera_Ceta.dm4"), path_tem2.joinpath(r"IFFT of Untitled.dm4"), path_tem2.joinpath(r"Live Profile Of IFFT of Untitled.dm4")])

HAADF2 = convert_units(HAADF2)[0]
TEM2_1 = convert_units([TEM2_1])[0]

In [ ]:
# 画图
fig = plt.figure(figsize=(7.0, 7.5))
gs = gridspec.GridSpec(6, 4, width_ratios=[1, 1, 1, 1], height_ratios=[1, 1, 1, 1, 1, 1], wspace=0, hspace=0, figure=fig)

# 图 A
subfig = fig.add_subfigure(gs[0:2, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, 0, 1.0, 1.0))
# ax.set_box_aspect(1.0)

vmin, vmax = np.percentile(HAADF.data, [1.0, 99.0])
ax.imshow(HAADF.data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")

scalebar = add_sizebar(
    ax,
    size=10,
    bardata=HAADF,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()

ax.text(
    0.23,
    0.95,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=11,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.1,
    1.0,
    r"A",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B
subfig = fig.add_subfigure(gs[0, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, 0.0, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_1[2].axes_manager[-1].axis,
    y1=TEM_1[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[1],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 4.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-150, 200)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.12,
    1.0,
    r"B",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 B1
subfig = fig.add_subfigure(gs[1, 2], zorder=0)
ax = subfig.add_axes((-0.15, 0.0, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_1[1].data, [1.0, 99.0])
ax.imshow(TEM_1[1].data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")
add_sizebar(ax, 1, TEM_1[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 B2
subfig = fig.add_subfigure(gs[1, 3], zorder=0)
ax = subfig.add_axes((-0.05, 0.0, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_1[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog", interpolation="bilinear")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 C
subfig = fig.add_subfigure(gs[2, 0:2], zorder=0)
ax = subfig.add_axes((0.00, -0.15, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_2[2].axes_manager[-1].axis,
    y1=TEM_2[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[0],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 6.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=1.0, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-40, 40)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.1,
    1.0,
    r"C",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 C1
subfig = fig.add_subfigure(gs[3, 0], zorder=0)
ax = subfig.add_axes((0.07, -0.16, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_2[1].data, [1.0, 99.0])
ax.imshow(TEM_2[1].data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")
add_sizebar(ax, 1, TEM_2[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 C2
subfig = fig.add_subfigure(gs[3, 1], zorder=0)
ax = subfig.add_axes((0.15, -0.16, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_2[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog", interpolation="bilinear")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 D
subfig = fig.add_subfigure(gs[2, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, -0.15, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM_3[2].axes_manager[-1].axis,
    y1=TEM_3[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[4],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 3.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-15, 20)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.1,
    1.0,
    r"D",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 D1
subfig = fig.add_subfigure(gs[3, 2], zorder=0)
ax = subfig.add_axes((-0.15, -0.15, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM_3[1].data, [1.0, 99.0])
ax.imshow(TEM_3[1].data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")
add_sizebar(ax, 1, TEM_3[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 D2
subfig = fig.add_subfigure(gs[3, 3], zorder=0)
ax = subfig.add_axes((-0.07, -0.15, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM_3[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog", interpolation="bilinear")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 E
subfig = fig.add_subfigure(gs[4:6, 0:2], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0, -0.2, 1.0, 1.0))
# ax.set_box_aspect(1.0)

vmin, vmax = np.percentile(HAADF2.data, [1.0, 99.0])
ax.imshow(HAADF2.data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")

scalebar = add_sizebar(
    ax,
    size=5,
    bardata=HAADF2,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()

ax.text(
    0.23,
    0.95,
    r"HAADF",
    transform=ax.transAxes,
    fontsize=11,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)
ax.text(
    -0.1,
    1.0,
    r"E",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 F
subfig = fig.add_subfigure(gs[4, 2:4], zorder=0)
ax = subfig.add_axes((-0.1, -0.4, 1.0, 1.0), zorder=0)
ax.set_box_aspect(0.5)

ax.fill_between(
    x=TEM2_1[2].axes_manager[-1].axis,
    y1=TEM2_1[2].data,
    y2=0,
    ls="-",
    lw=1.0,
    color=colors[3],
    label=r"Distance",
)

ax.set_xlabel(r"Distance (nm)", fontsize=9)
ax.set_xlim(0, 3.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.25, offset=0))

ax.set_ylabel(
    r"Intensity (arb.u.)",
    fontsize=9,
)
ax.set_ylim(-90, 110)
ax.set_yticks([])

ax.tick_params(
    axis="both",
    labelsize=9,
    which="both",
    bottom=True,
    labelbottom=True,
)
ax.text(
    -0.12,
    1.0,
    r"F",
    transform=ax.transAxes,
    fontsize=13,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
)

# 图 F1
subfig = fig.add_subfigure(gs[5, 2], zorder=0)
ax = subfig.add_axes((-0.15, -0.4, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

vmin, vmax = np.percentile(TEM2_1[1].data, [1.0, 99.0])
ax.imshow(TEM2_1[1].data, cmap="gray", vmin=vmin, vmax=vmax, interpolation="bilinear")
add_sizebar(ax, 1, TEM2_1[1], "w")
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"iFFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)

# 图 F2
subfig = fig.add_subfigure(gs[5, 3], zorder=0)
ax = subfig.add_axes((-0.05, -0.4, 0.8, 0.8), zorder=0)
ax.set_aspect(1.0)

FFT = np.log10(np.abs(TEM2_1[0]) + 1e-10)
vmin, vmax = np.percentile(FFT.data, [1.0, 99.9])
ax.imshow(FFT.data, cmap="gray", vmin=vmin, vmax=vmax, norm="symlog", interpolation="bilinear")
scalebar = add_sizebar(
    ax,
    size=10,
    bardata=FFT,
    color="w",
)
scalebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.0, 0, 0), transform=ax.transAxes)
ax.set_axis_off()
ax.text(
    0.95,
    0.9,
    r"FFT",
    transform=ax.transAxes,
    fontsize=9,
    va="center",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    color="w",
)


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_26_TEM_TEM_04_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_26_TEM_TEM_04_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_26_TEM_TEM_04_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigureSI_26_TEM_TEM_04_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()

## FigureSI 47 -- XRD 和 EXAFS 相关性分析

In [ ]:
def minmax_norm(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    vmin, vmax = s.min(skipna=True), s.max(skipna=True)
    return (s - vmin) / (vmax - vmin)

In [ ]:
# 读取数据
csv_path = path_data_root.joinpath(r"XRD_XAS_相关性数据_2.csv")
df = pd.read_csv(csv_path)

# 直接使用原始 SOC 列，不再做拼接
xas_soc = pd.to_numeric(df["XAS_SOC"], errors="coerce")
xrd_soc = pd.to_numeric(df["XRD_SOC"], errors="coerce")

# 归一化
df = pd.DataFrame({
    "xas_x": xas_soc,
    "xrd_x": xrd_soc,
    "xas_p1": minmax_norm(df["XAS_R_peak1_A"]),
    "xas_p2": minmax_norm(df["XAS_R_peak2_A"]),
    "xas_i1": minmax_norm(df["XAS_R_peak1_intensity"]),
    "xas_i2": minmax_norm(df["XAS_R_peak2_intensity"]),
    "xrd_p1": minmax_norm(df["XRD_d_peak1_A"]),
    "xrd_p2": minmax_norm(df["XRD_d_peak2_A"]),
    "xrd_i1": minmax_norm(df["XRD_d_peak1_intensity"]),
    "xrd_i2": minmax_norm(df["XRD_d_peak2_intensity"]),
})

In [ ]:
# 画图
%matplotlib inline
plt.close("all")

fig, axL = plt.subplots(figsize=(3.2, 2.0), dpi=300)

axL.set_box_aspect(0.8)
axR = axL.twinx()
axR.set_box_aspect(0.8)

# Phase spans: charge/discharge/charge
phase_spans = [
    (0.0, 1.0, "charge", "#f8f5d7"),
    (1.0, 2.0, "discharge", "#e8f1fb"),
    (2.0, 3.0, "charge", "#f8f5d7"),
]
for x0, x1, lbl, col in phase_spans:
    axL.axvspan(x0, x1, color=col, alpha=0.35, zorder=0)

# Left axis: normalized peak
xrd_offset = 1.3
l1, = axL.plot(df["xas_x"], df["xas_p1"], color=colors[0], marker="o", ms=3.0, lw=1.0, ls="--", label=r"EXAFS Mn-O (d)")
l2, = axL.plot(df["xas_x"], df["xas_p2"], color=colors[1], marker="s", ms=3.0, lw=1.0, ls="--", label=r"EXAFS Mn-Mn (d)")
l3, = axL.plot(df["xrd_x"], df["xrd_p1"], color=colors[2], marker="^", ms=3.0, lw=1.0, ls="--", label=r"XRD ZSH (d)")
l4, = axL.plot(df["xrd_x"], df["xrd_p2"], color=colors[3], marker="D", ms=3.0, lw=1.0, ls="--", label=r"XRD MnO2 (d)")

# Right axis: normalized intensity
r1, = axR.plot(df["xas_x"], df["xas_i1"] + xrd_offset, color=colors[0], alpha=0.7, marker="o", ms=3.0, lw=1.0, ls="-", label=r"EXAFS Mn-O (i)")
r2, = axR.plot(df["xas_x"], df["xas_i2"] + xrd_offset, color=colors[1], alpha=0.7, marker="s", ms=3.0, lw=1.0, ls="-", label=r"EXAFS Mn-Mn (i)")
r3, = axR.plot(df["xrd_x"], df["xrd_i1"] + xrd_offset, color=colors[2], alpha=0.7, marker="^", ms=3.0, lw=1.0, ls="-", label=r"XRD ZSH (i)")
r4, = axR.plot(df["xrd_x"], df["xrd_i2"] + xrd_offset, color=colors[3], alpha=0.7, marker="D", ms=3.0, lw=1.0, ls="-", label=r"XRD MnO2 (i)")

axL.set_xlabel(r"Charge and Discharge", fontsize=9)
axL.set_xticks([0, 1, 2, 3])
axL.set_ylabel("Normalized Peak Position", fontsize=9)
axR.set_ylabel("Normalized Peak Intensity", fontsize=9)

axL.set_ylim(-0.1, 2.5)
axR.set_ylim(-0.1, 2.5)
axL.set_yticks([])
axR.set_yticks([])

axL.tick_params(axis="both", labelsize=8)
axR.tick_params(axis="both", labelsize=8)

# Diagonal arrows with rounded labels to indicate y-axis mapping
axL.annotate(
    "",
    xy=(0.005, 0.25),
    xycoords="axes fraction",
    xytext=(0.15, 0.35),
    textcoords="axes fraction",
    fontsize=7,
    ha="left",
    va="center",
    bbox=dict(boxstyle="round,pad=0.22,rounding_size=0.15", fc="white", ec="black", lw=0.8,  alpha=0.95),
    arrowprops=dict(
        arrowstyle="-|>",
        lw=0.8,
        color=colors[0],
        ls="--",
        shrinkA=2,
        shrinkB=0,
        connectionstyle="angle3,angleA=0,angleB=90",
    ),
)

axR.annotate(
    "",
    xy=(0.98, 0.5),
    xycoords="axes fraction",
    xytext=(0.85, 0.65),
    textcoords="axes fraction",
    fontsize=7,
    ha="left",
    va="center",
    bbox=dict(boxstyle="round,pad=0.22,rounding_size=0.15", fc="white", ec="black", lw=0.8, alpha=0.95),
    arrowprops=dict(
        arrowstyle="-|>",
        lw=0.8,
        color=colors[1],
        shrinkA=2,
        shrinkB=0,
        connectionstyle="angle3,angleA=180,angleB=90",
    ),
)
axL.text(0.5, 2.43, "Charge", ha="center", va="top", fontsize=8)
axL.text(1.5, 2.43, "Discharge", ha="center", va="top", fontsize=8)
axL.text(2.5, 2.43, "Charge", ha="center", va="top", fontsize=8)
axL.grid(alpha=0.2)

all_lines = [l1,l2,l3,l4,r1,r2,r3,r4]
axL.legend(all_lines, [ln.get_label() for ln in all_lines], ncol=1, labelcolor="linecolor", frameon=False, fontsize=7, loc="upper left", bbox_to_anchor=(1.1, 1.07))

# 保存图片
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureSI_47_V0_300.tif"), dpi=300, bbox_inches="tight", pad_inches=0.05, pil_kwargs={"compression": "tiff_lzw"})
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureSI_47_V0_600.tif"), dpi=600, bbox_inches="tight", pad_inches=0.05, pil_kwargs={"compression": "tiff_lzw"})
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureSI_47_V0_600.png"), dpi=600, bbox_inches="tight", pad_inches=0.05)
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureSI_47_V0_900.svg"), dpi=900, bbox_inches="tight", pad_inches=0.05)

plt.gcf().set_facecolor("white")
plt.show()

## TOC

In [ ]:
# 读取一个常规的 αMnO2 的数据
echem = []
path_filelist = list(Path.joinpath(path_data_root, r"Echem\αMnO2\GCD\1M ZnSO4 + 0.2M MnSO4").glob(r"LC*.xlsx"))
# 读取电化学数据
for path_file in path_filelist[0:1]:
    df = pd.read_excel(path_file, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echem.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echem)):
    echem[i] = echem[i][echem[i]["Cycle"].isin([0, 1, 2])]
    echem[i] = echem[i][echem[i]["Voltage/V"] >= 0]


In [ ]:
plt.close("all")

# 画图
fig = plt.figure(figsize=(7.0, 5.0))
gs = fig.add_gridspec(2, 1, width_ratios=None, height_ratios=[1, 1], wspace=0, hspace=0)

# 图 A
subfig = fig.add_subfigure(gs[0, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.3)

ax.set_axis_off()
labels = [
    [None, None],
]
for _, data in enumerate(echem):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        if j == 0:
            ax.plot(
                temp.loc[idx_min:, "SpeCap/mAh/g"],
                temp.loc[idx_min:, "Voltage/V"],
                ls="-",
                lw=1.0,
                c=colors[j],
                zorder=0,
            )  # 充电

# 图 B
subfig = fig.add_subfigure(gs[1, 0], zorder=0)
ax = subfig.add_subplot()
ax.set_position((0.0, 0.0, 1.0, 1.0))
ax.set_box_aspect(0.3)

ax.set_axis_off()
labels = [
    [None, None],
]
for i, data in enumerate(echem):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        if j == 1:
            ax.plot(
                temp.loc[:idx_min, "SpeCap/mAh/g"],
                temp.loc[:idx_min, "Voltage/V"],
                ls="-",
                lw=1.0,
                c=colors[j],
                label=labels[i][j],
                zorder=0,
            )  # 放电


# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_Figure_TOC_00_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_Figure_TOC_00_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_Figure_TOC_00_V0_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_Figure_TOC_00_V0_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)

plt.gcf().set_facecolor("white")
plt.show()
